# HGSOC malignant subtypes and fibroblast-associated meta-interactions

See [README.md](README.md) for the execution order, upstream inputs, commands and paper mapping.
Run the unified entry point to save executed copies and local outputs. Scientific variants and their parameters are retained below.


## 1. Imports and device setup

In [ ]:
import json
from SpiderNet.utils import *
from SpiderNet.config import *
from dataclasses import fields
import scanpy as sc
import gseapy as gp

cuda_available = torch.cuda.is_available()
if cuda_available:
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")


## 2. Configure input and output paths

The training setup JSON overrides `PROCESSED_DATA_DIR` and `OUTPUT_ROOT` when present. `DATA_ROOT` remains as configured below; some auxiliary blocks also specify their own input paths.


In [ ]:
from workflow_paths import (PROCESSED_DATA_DIR, RESULTS_ROOT, DATA_ROOT, OUTPUT_ROOT, input_path, output_path, saved, ensure_output)
ensure_output()


## 3. Load the trained run configuration


In [ ]:
import re
from workflow_paths import run_dirs as _configured_run_dirs
run_dirs = dict(_configured_run_dirs)
run_dir = Path(run_dirs["run_dir"])
match = re.search(r"Result_dim(\d+)", str(run_dir))
if match is None:
    raise ValueError(f"Cannot parse dim_envir from run directory: {run_dir}")
dim_envir = int(match.group(1))
print(run_dirs)


## 4. Load processed HGSOC data

In [ ]:
from SpiderNet.io import load_processed_data
processed = load_processed_data(PROCESSED_DATA_DIR)

[f.name for f in fields(processed)]


In [ ]:
processed.spidernet_data = [data.to(device) for data in processed.spidernet_data]


In [ ]:
# Check whether the six checkpoint/exhaustion markers are measured.

np.isin(('PDCD1', 'LAG3', 'TIGIT', 'HAVCR2', 'CTLA4', 'TOX'),np.array(processed.adata_all.var.index))


## 5. Identify and characterize MI-guided malignant subtypes


### 5.1. Select the malignant-cell population


In [ ]:
cellclass_choose = 'Malignant'


### 5.2. Construct the MI-induced cell embedding

The imported helper aggregates per-cell sending/receiving MI strengths, selects MI features using `MIlevel_agg_threshold`, and computes PCA and UMAP coordinates.


In [ ]:
from SpiderNet.analysis import MIinduced_cellembedding
MIinduced_cellembedding(
    cellclass_choose=cellclass_choose,
    file_savepath_main=run_dirs["run_dir"] + "/",
    Factor_envir_list_path=input_path(run_dirs["run_dir"] + "/Factor_envir_list.pkl"),
    adata_copy_path=input_path(str(PROCESSED_DATA_DIR / "adata_all.h5ad")),
    device=device,
    SpiderNet_data_pyg_list_path=input_path(str(PROCESSED_DATA_DIR / "SpiderNet_data_pyg_list.pkl")),
    LR_list_merge_path=input_path(run_dirs["run_dir"] + "/LR_list_merge.pkl"),
    Avg_MI_cellclass_pair_merge_use_path=input_path(run_dirs["run_dir"] + "/Avg_MI_cellclass_pair_merge_use.pkl"),
    dim_envir=dim_envir,
    MIlevel_agg_threshold=0.6,
    metadata_sample_path=input_path(str(PROCESSED_DATA_DIR / "metadata_sample.csv")),
    embedding_method="PCA",
    show=True,
)


### 5.3. Louvain clustering and UMAP visualization


In [ ]:
# Cluster the MI_PCA representation with 25 neighbors and resolution 0.17.
# Labels are shifted to one-based strings before export and C5 annotation.

adata_choose_path = run_dirs['run_dir'] + "/" + ("adata_choose_" + str(cellclass_choose) + ".h5ad")
adata_choose = sc.read_h5ad(input_path(adata_choose_path))
print(adata_choose)
##
# sc.pp.neighbors(adata_choose, use_rep='MI_PCA', n_neighbors=20)
# # sc.tl.louvain(adata_choose, resolution=0.17, key_added='MI_louvain')
# sc.tl.louvain(adata_choose, resolution=0.18, key_added='MI_louvain')

sc.pp.neighbors(adata_choose, use_rep='MI_PCA', n_neighbors=25,)
sc.tl.louvain(adata_choose, resolution=0.17, key_added='MI_louvain')
# sc.tl.louvain(adata_choose, resolution=0.20, key_added='MI_louvain')
# sc.tl.louvain(adata_choose, resolution=0.30, key_added='MI_louvain')

##
# -----------------------------
# Step 1: Normalize cluster labels (start from 1, as string)
# -----------------------------
adata_choose.obs["MI_louvain"] = (
    adata_choose.obs["MI_louvain"].astype(int) + 1
).astype(str)

cluster_assign = adata_choose.obs["MI_louvain"]

# -----------------------------
# Step 2: Save cluster assignment (cell-level)
# -----------------------------
cluster_assign.to_csv(
    f"{run_dirs['run_dir']}/Cluster_assignment_{cellclass_choose}.csv"
)

# -----------------------------
# Step 3: Save barcode-indexed cluster assignment
# -----------------------------
cluster_assign_df = pd.DataFrame(
    {
        "MI_louvain": cluster_assign.values,
        "MI_louvain_C": "C" + cluster_assign.values
    },
    index=adata_choose.obs["barcode"].tolist()
)

cluster_assign_df.to_csv(
    f"{run_dirs['run_dir']}/Cluster_assignment_{cellclass_choose}_barcode.csv"
)

# -----------------------------
# Step 4: Print summary
# -----------------------------
print(
    f"Number of clusters identified in {cellclass_choose}: "
    f"{cluster_assign.nunique()}"
)

adata_choose.obs["Malignant_C5"] = adata_choose.obs["MI_louvain"].apply(lambda x: "C5" if x == "5" else "Other")
# -----------------------------
# Step 5: Save updated AnnData
# -----------------------------
adata_choose.write_h5ad(adata_choose_path)


In [ ]:
import matplotlib.pyplot as plt

plt.close()
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Prepare cluster info
subtype_cat = adata_choose.obs["MI_louvain"].astype("category")
subtype_codes = subtype_cat.cat.codes
subtype_names = subtype_cat.cat.categories
n_subtypes = len(subtype_names)

# Custom color palette
# custom_colors = [
# '#e6f29f', '#fee089', '#645599','#90c6b6', '#ac352f', '#2e87b6'
# ]
# custom_colors = [
# '#82CCE2', '#D1CABE', '#2E7FB9','#FED881', '#9F3B38', '#519384',
#     '#636491'
# ]
custom_colors = [
'#D4ECF1', '#EFE8E5', '#B7CCE5','#FFF2D2', '#9F3B38', '#B9CEC7',
    '#A6A2B9'
]

# Assign palette entries in cluster order, cycling if needed.
base_colors = custom_colors
colors = [base_colors[i % len(base_colors)] for i in range(n_subtypes)]
print(colors)

# Plot UMAP clusters
for i, name in enumerate(subtype_names):
    idx = (subtype_codes == i)
    ax.scatter(
        adata_choose.obsm["MI_UMAP"][idx, 0],
        adata_choose.obsm["MI_UMAP"][idx, 1],
        c=[colors[i]], s=2, edgecolor="none",
        rasterized=True,
        label=f"{cellclass_choose} C{i + 1}"   # Display one-based cluster numbers.
    )

# Axis labels
ax.set_xlabel("UMAP 1", fontsize=18)
ax.set_ylabel("UMAP 2", fontsize=18)

# Legend
ax.legend(
    fontsize=17, scatterpoints=1, markerscale=8,
    handlelength=2.0, loc="center left",
    bbox_to_anchor=(1.02, 0.5), frameon=False
)

# Style cleanup
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# Save as white-background figure
filename = f"{run_dirs['run_dir']}/UMAP_{cellclass_choose}_clusters.png"
plt.savefig(filename, format="png", bbox_inches="tight", dpi=300, facecolor="white")
filename_pdf = f"{run_dirs['run_dir']}/UMAP_{cellclass_choose}_clusters.pdf"
plt.savefig(filename_pdf, format="pdf", bbox_inches="tight", dpi=300, facecolor="white")
plt.show()
plt.close()


### 5.4. Sample annotations, KEGG scores and spatial neighborhoods

The following blocks evaluate sample-label silhouette width, four KEGG module scores and embedding compactness, followed by CD8 T-cell neighborhoods and gene-program comparisons. The early KEGG scoring block reads a reference CSV exported later in this notebook; retain that file as an upstream input for a sequential run.


In [ ]:
from SpiderNet.analysis import Feature_show_umap
Feature_show_umap(
    cellclass_choose=cellclass_choose,
    adata_choose_path=input_path(run_dirs["run_dir"] + f"/adata_choose_{cellclass_choose}.h5ad"),
    file_savepath_main=run_dirs["run_dir"],
    obsm_show="MI_UMAP",
    adata_copy_path=input_path(str(PROCESSED_DATA_DIR / "adata_all.h5ad")),
    show=True,
)


In [ ]:
# Evaluate sample-label separation in MI_UMAP, using the seeded 30% subsample.
# The labels come from samples; this is not a malignant-subtype silhouette score.

##Compute silhouette score
import numpy as np
from sklearn.metrics import silhouette_samples

# embedding and labels
X = adata_choose.obsm['MI_UMAP']
labels = adata_choose.obs['samples'].values

# subsample
np.random.seed(0)
n_cells = X.shape[0]
n_sub = max(int(n_cells * 0.30), 1000)
idx = np.random.choice(n_cells, size=n_sub, replace=False)

X_sub = X[idx]
labels_sub = labels[idx]
print(f"Computing silhouette score on {n_sub} subsampled cells (out of {n_cells})...")

# compute silhouette on subsampled cells
sil_scores_sub = silhouette_samples(X_sub, labels_sub, metric='euclidean')
asw_sub = sil_scores_sub.mean()

print("Subsampled mean silhouette score:", asw_sub)


In [ ]:
# Four KEGG pathway module scores on the malignant MI UMAP
# Requires C5_four_KEGG_pathway_reference_genes_long.csv, produced by the later
# KEGG reference-gene export block. Each score averages malignant-cell gene
# z-scores (ddof=1), replacing undefined z-scores with zero. Figures and the
# cell-level score table are written under OUT_PREFIX in the selected run.

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

KEGG_PATHWAYS_TO_SCORE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

OUT_PREFIX = "Malignant_KEGG_module_score_MI_UMAP_continuous"

out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _safe_name(x):
    x = str(x)
    x = x.replace("/", "_").replace("-", "_")
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _first_indexer(values, query_values):
    mapping = {}

    for i, v in enumerate(values):
        v = str(v)
        if v not in mapping:
            mapping[v] = i

    return np.array(
        [mapping.get(str(q), -1) for q in query_values],
        dtype=int
    )


def _make_composite(sample_values, id_values):
    return np.array(
        [f"{str(s)}||{str(i)}" for s, i in zip(sample_values, id_values)],
        dtype=object
    )


def _add_expr_candidate(candidates, name, obj):
    if obj is not None and hasattr(obj, "var_names") and hasattr(obj, "obs") and hasattr(obj, "X"):
        candidates.append((name, obj))


def _build_expr_candidates():
    candidates = []

    if "adata_choose" in globals():
        _add_expr_candidate(candidates, "adata_choose", adata_choose)

    if "processed" in globals():
        _add_expr_candidate(candidates, "processed.adata_all", getattr(processed, "adata_all", None))
        _add_expr_candidate(candidates, "processed.adata", getattr(processed, "adata", None))

        if hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
            try:
                import anndata as ad

                adata_list_concat = ad.concat(
                    processed.adata_list,
                    join="outer",
                    index_unique=None,
                    merge="same",
                )

                _add_expr_candidate(candidates, "concat(processed.adata_list)", adata_list_concat)

            except Exception as e:
                print(f"[Info] Could not concatenate processed.adata_list: {e}")

    for var_name in [
        "adata_all_full",
        "adata_raw",
        "adata_full",
        "adata_ori",
        "adata_original",
        "adata_copy",
        "adata",
        "adata_all",
    ]:
        if var_name in globals():
            _add_expr_candidate(candidates, var_name, globals()[var_name])

    seen = set()
    unique_candidates = []

    for name, obj in candidates:
        if id(obj) not in seen:
            unique_candidates.append((name, obj))
            seen.add(id(obj))

    return unique_candidates


def _match_malignant_rows_to_expr_adata(expr_adata):
    if expr_adata is adata_choose:
        return np.arange(adata_choose.n_obs), "direct adata_choose row order"

    malignant_obs_names = adata_choose.obs_names.astype(str).to_numpy()

    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = malignant_obs_names.copy()

    expr_obs_names = expr_adata.obs_names.astype(str).to_numpy()

    if "barcode" in expr_adata.obs.columns:
        expr_barcodes = expr_adata.obs["barcode"].astype(str).to_numpy()
    else:
        expr_barcodes = expr_obs_names.copy()

    sample_col_choose = _sample_col_from_obs(adata_choose)
    sample_col_expr = _sample_col_from_obs(expr_adata)

    if sample_col_choose is not None and sample_col_expr is not None:
        malignant_sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
        expr_sample_values = expr_adata.obs[sample_col_expr].astype(str).to_numpy()

        query_comp_barcode = _make_composite(malignant_sample_values, malignant_barcodes)
        expr_comp_barcode = _make_composite(expr_sample_values, expr_barcodes)

        idx = _first_indexer(expr_comp_barcode, query_comp_barcode)
        if np.all(idx >= 0):
            return idx, f"sample + barcode using adata_choose['{sample_col_choose}'] and expr['{sample_col_expr}']"

        query_comp_obs = _make_composite(malignant_sample_values, malignant_obs_names)
        expr_comp_obs = _make_composite(expr_sample_values, expr_obs_names)

        idx = _first_indexer(expr_comp_obs, query_comp_obs)
        if np.all(idx >= 0):
            return idx, f"sample + obs_names using adata_choose['{sample_col_choose}'] and expr['{sample_col_expr}']"

    idx = _first_indexer(expr_barcodes, malignant_barcodes)
    if np.all(idx >= 0):
        return idx, "barcode -> expression barcode"

    idx = _first_indexer(expr_obs_names, malignant_obs_names)
    if np.all(idx >= 0):
        return idx, "adata_choose obs_names -> expression obs_names"

    idx = _first_indexer(expr_obs_names, malignant_barcodes)
    if np.all(idx >= 0):
        return idx, "adata_choose barcode -> expression obs_names"

    return None, None


def _get_X_array(adata, rows, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub


def _get_mi_umap(adata):
    obsm_candidates = [
        "MI_UMAP", "X_MI_UMAP", "MI_umap", "X_mi_umap",
        "X_umap", "umap"
    ]

    for key in obsm_candidates:
        if key in adata.obsm:
            coords = np.asarray(adata.obsm[key])
            if coords.ndim == 2 and coords.shape[1] >= 2:
                print(f"Using MI_UMAP coordinates from adata_choose.obsm['{key}']")
                return coords[:, :2], f"obsm:{key}"

    obs_col_pairs = [
        ("MI_UMAP1", "MI_UMAP2"),
        ("MI_UMAP_1", "MI_UMAP_2"),
        ("MI_umap1", "MI_umap2"),
        ("MI_umap_1", "MI_umap_2"),
        ("UMAP1", "UMAP2"),
        ("UMAP_1", "UMAP_2"),
        ("umap1", "umap2"),
        ("umap_1", "umap_2"),
    ]

    for c1, c2 in obs_col_pairs:
        if c1 in adata.obs.columns and c2 in adata.obs.columns:
            print(f"Using MI_UMAP coordinates from adata_choose.obs[['{c1}', '{c2}']]")
            return adata.obs[[c1, c2]].to_numpy(dtype=float), f"obs:{c1},{c2}"

    raise KeyError(
        "Cannot find MI_UMAP coordinates. Expected adata_choose.obsm['MI_UMAP'], "
        "['X_MI_UMAP'], ['MI_umap'], ['X_umap'] or obs columns such as MI_UMAP1/MI_UMAP2."
    )


# =============================
# Check required variables
# =============================
if "adata_choose" not in globals():
    raise NameError("adata_choose is required.")

if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Load four KEGG pathway gene sets
# =============================
if "c5_kegg_reference_gene_df" in globals():
    kegg_gene_df = c5_kegg_reference_gene_df.copy()
else:
    candidate_paths = [
        Path(out_dir) / "TCGA_OV_KEGG_gene_signature" / "C5_four_KEGG_pathway_reference_genes_long.csv",
        (DATA_ROOT / 'TCGA_OV_KEGG_gene_signature/C5_four_KEGG_pathway_reference_genes_long.csv'),
        (DATA_ROOT / 'C5_four_KEGG_pathway_reference_genes_long.csv'),
    ]

    kegg_gene_df = None

    for candidate_path in candidate_paths:
        if input_path(candidate_path).exists():
            print(f"Loading KEGG pathway genes from:\n{candidate_path}")
            kegg_gene_df = pd.read_csv(input_path(candidate_path))
            break

    if kegg_gene_df is None:
        raise FileNotFoundError(
            "Cannot find c5_kegg_reference_gene_df in memory or saved KEGG pathway-gene CSV.\n"
            "Please run the KEGG reference gene export cell first, or provide:\n"
            "C5_four_KEGG_pathway_reference_genes_long.csv"
        )

required_cols = {"Requested_Pathway", "Gene"}
missing_cols = required_cols - set(kegg_gene_df.columns)

if len(missing_cols) > 0:
    raise KeyError(
        f"KEGG pathway-gene table is missing required columns: {missing_cols}. "
        f"Available columns: {list(kegg_gene_df.columns)}"
    )

kegg_gene_df = (
    kegg_gene_df
    .loc[
        kegg_gene_df["Requested_Pathway"].isin(KEGG_PATHWAYS_TO_SCORE),
        ["Requested_Pathway", "Gene"]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)

if kegg_gene_df.empty:
    raise ValueError("No KEGG pathway genes found for selected four pathways.")

# =============================
# Build KEGG gene-set dictionary
# =============================
geneset_dict = {}

for pathway in KEGG_PATHWAYS_TO_SCORE:
    genes_cur = (
        kegg_gene_df
        .loc[kegg_gene_df["Requested_Pathway"] == pathway, "Gene"]
        .astype(str)
        .tolist()
    )

    genes_cur = list(dict.fromkeys([g.strip() for g in genes_cur if g.strip()]))

    score_name = f"KEGG_{_safe_name(pathway)}_score"
    geneset_dict[score_name] = genes_cur

score_title_map = {
    f"KEGG_{_safe_name('HIF-1 signaling pathway')}_score": "HIF-1 signaling",
    f"KEGG_{_safe_name('PD-L1 expression and PD-1 checkpoint pathway in cancer')}_score": "PD-1/PD-L1 checkpoint",
    f"KEGG_{_safe_name('PI3K-Akt signaling pathway')}_score": "PI3K-Akt signaling",
    f"KEGG_{_safe_name('ECM-receptor interaction')}_score": "ECM-receptor interaction",
}

all_genes = []

for genes in geneset_dict.values():
    all_genes.extend(genes)

all_genes = list(dict.fromkeys([str(g).strip() for g in all_genes if str(g).strip()]))

geneset_input_long = []

for score_name, genes in geneset_dict.items():
    for gene in genes:
        geneset_input_long.append({
            "module": score_name,
            "module_label": score_title_map.get(score_name, score_name),
            "Gene": gene,
        })

geneset_input_long = pd.DataFrame(geneset_input_long)

geneset_input_long.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_input_genesets_long.csv"),
    index=False,
)

print("Input KEGG gene-set sizes:")
_display_df(
    geneset_input_long
    .groupby(["module", "module_label"], as_index=False)
    .agg(n_input_genes=("Gene", "nunique"))
)

# =============================
# Find expression source
# =============================
expr_candidates = _build_expr_candidates()

candidate_summary = []
best = None

for name, cand in expr_candidates:
    resolved_cur, missing_cur = _resolve_gene_names(cand, all_genes)
    n_genes_cur = len(resolved_cur)

    row_idx_cur, match_mode_cur = _match_malignant_rows_to_expr_adata(cand)
    rows_ok = row_idx_cur is not None

    per_module_found = {}

    for score_name, genes in geneset_dict.items():
        per_module_found[score_name] = len([g for g in genes if g in resolved_cur])

    candidate_summary.append({
        "candidate": name,
        "n_obs": cand.n_obs,
        "n_vars": cand.n_vars,
        "n_total_module_genes_found": n_genes_cur,
        "malignant_rows_matched": rows_ok,
        "match_mode": match_mode_cur if match_mode_cur is not None else "NA",
        **{f"n_found__{k}": v for k, v in per_module_found.items()},
        "first_10_found_genes": ", ".join(list(resolved_cur.keys())[:10]),
    })

    all_modules_have_gene = all(v > 0 for v in per_module_found.values())

    if n_genes_cur > 0 and rows_ok and all_modules_have_gene:
        if best is None or n_genes_cur > best["n_genes"]:
            best = {
                "name": name,
                "adata": cand,
                "resolved": resolved_cur,
                "missing": missing_cur,
                "row_idx": row_idx_cur,
                "match_mode": match_mode_cur,
                "n_genes": n_genes_cur,
                "per_module_found": per_module_found,
            }

candidate_summary_df = pd.DataFrame(candidate_summary)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False,
)

print("Expression object search summary:")
_display_df(candidate_summary_df)

if best is None:
    raise ValueError(
        "Cannot find an expression AnnData that contains at least one gene for every KEGG module "
        "and matches malignant cells. Please check expression_object_search_summary."
    )

expr_source_name = best["name"]
expr_adata = best["adata"]
expr_rows = best["row_idx"]
resolved_genes = best["resolved"]
missing_genes = best["missing"]

available_genes = [g for g in all_genes if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

print(f"Selected expression source: {expr_source_name}")
print(f"Malignant row matching mode: {best['match_mode']}")
print(f"Total KEGG module genes found: {len(available_genes)} / {len(all_genes)}")

if len(missing_genes) > 0:
    print(f"[Warning] Missing KEGG module genes skipped: {missing_genes}")

# =============================
# Extract expression and compute z-scores
# =============================
X_sub = _get_X_array(
    adata=expr_adata,
    rows=expr_rows,
    genes_actual=actual_gene_names,
)

expr_gene_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=adata_choose.obs_names,
)

gene_mean = expr_gene_df.mean(axis=0)
gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)

expr_z = (expr_gene_df - gene_mean) / gene_std
expr_z = expr_z.fillna(0)

# =============================
# Compute KEGG module scores
# =============================
if "barcode" in adata_choose.obs.columns:
    barcode_values = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    barcode_values = adata_choose.obs_names.astype(str).to_numpy()

cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

module_score_df = pd.DataFrame({
    "barcode": barcode_values,
    "MI_louvain": cluster_clean,
    "MalignantCluster": cluster_display,
}, index=adata_choose.obs_names)

module_gene_count_records = []

for score_name, genes in geneset_dict.items():
    genes_cur = [g for g in genes if g in expr_z.columns]

    if len(genes_cur) == 0:
        print(f"[Warning] No available genes for {score_name}. Skipping.")
        continue

    module_score_df[score_name] = expr_z[genes_cur].mean(axis=1).to_numpy()
    adata_choose.obs[score_name] = module_score_df[score_name].to_numpy()

    module_gene_count_records.append({
        "module": score_name,
        "module_label": score_title_map.get(score_name, score_name),
        "n_genes_used": len(genes_cur),
        "genes_used": ", ".join(genes_cur),
    })

module_gene_count_df = pd.DataFrame(module_gene_count_records)

score_cols = [r["module"] for r in module_gene_count_records]

if len(score_cols) == 0:
    raise ValueError("No KEGG module scores were computed.")

module_gene_count_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_module_gene_count.csv"),
    index=False,
)

print("KEGG module genes used:")
_display_df(module_gene_count_df[["module_label", "n_genes_used", "genes_used"]])

# =============================
# Get MI_UMAP coordinates
# =============================
mi_umap, mi_umap_source = _get_mi_umap(adata_choose)

module_score_df["MI_UMAP1"] = mi_umap[:, 0]
module_score_df["MI_UMAP2"] = mi_umap[:, 1]

module_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_module_scores.csv"),
    index=False,
)

# =============================
# Plot continuous KEGG module scores on MI_UMAP
# =============================
n_panels = len(score_cols)
n_cols = 2
n_rows = int(np.ceil(n_panels / n_cols))

plot_summary_records = []

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.2 * n_cols, 2.9 * n_rows),
    squeeze=False,
)

for ax_i, score_name in enumerate(score_cols):
    ax = axes.flat[ax_i]

    score_values = module_score_df[score_name].to_numpy(dtype=float)
    finite_score = score_values[np.isfinite(score_values)]

    if len(finite_score) == 0:
        vmin, vmax = 0, 1
        score_mean = np.nan
        score_median = np.nan
    else:
        vmin, vmax = np.nanpercentile(finite_score, [1, 99])

        if vmin == vmax:
            vmin = np.nanmin(finite_score)
            vmax = np.nanmax(finite_score)

        if vmin == vmax:
            vmin = vmin - 1e-6
            vmax = vmax + 1e-6

        score_mean = float(np.nanmean(finite_score))
        score_median = float(np.nanmedian(finite_score))

    plot_summary_records.append({
        "module": score_name,
        "module_label": score_title_map.get(score_name, score_name),
        "n_finite_cells": int(len(finite_score)),
        "score_mean": score_mean,
        "score_median": score_median,
        "vmin_p1": vmin,
        "vmax_p99": vmax,
        "umap_source": mi_umap_source,
    })

    sc_plot = ax.scatter(
        module_score_df["MI_UMAP1"],
        module_score_df["MI_UMAP2"],
        c=score_values,
        cmap="viridis",
        s=0.8,
        linewidths=0,
        alpha=0.9,
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )

    ax.set_title(
        score_title_map.get(score_name, score_name),
        fontsize=8,
    )

    ax.set_xlabel("MI_UMAP1")
    ax.set_ylabel("MI_UMAP2")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out", length=2)

    ax.set_xticks([])
    ax.set_yticks([])

    cbar = plt.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.03)
    cbar.set_label("Module score", fontsize=6)
    cbar.ax.tick_params(labelsize=5)

# hide unused panels
for j in range(n_panels, n_rows * n_cols):
    axes.flat[j].axis("off")

fig.suptitle(
    "KEGG module scores on MI_UMAP",
    y=1.02,
    fontsize=10,
)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_MI_UMAP_continuous_KEGG_module_scores.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_MI_UMAP_continuous_KEGG_module_scores.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

# =============================
# Save plot summary and score table
# =============================
plot_summary_df = pd.DataFrame(plot_summary_records)

plot_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_MI_UMAP_continuous_score_summary.csv"),
    index=False,
)

module_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_module_scores_with_coordinates.csv"),
    index=False,
)

print("Continuous KEGG module-score UMAP summary:")
_display_df(plot_summary_df)

# =============================
# Optional: save updated adata_choose
# =============================
if "adata_choose_path" in globals():
    try:
        adata_choose.write_h5ad(adata_choose_path)
    except Exception as e:
        print(f"[Warning] Could not save adata_choose to adata_choose_path: {e}")


In [ ]:
# ==============================================================
# Top-cell compactness ratio for four KEGG pathway module scores
# --------------------------------------------------------------
# For each KEGG pathway score:
#   1. Select top 20% malignant cells by feature/module score.
#   2. Compute average pairwise distance among top cells in embedding space:
#        D_top = mean_{i,j in T} ||z_i - z_j||
#   3. Compute average pairwise distance among all cells:
#        D_all = mean_{i,j} ||z_i - z_j||
#   4. Compactness = log2(D_all / D_top)
#
# Interpretation:
#   Compactness > 0  : top-score cells are more spatially compact in embedding space
#                      than the full malignant-cell population.
#   Compactness = 0  : top-score cells have similar spread as all cells.
#   Compactness < 0  : top-score cells are more dispersed than all cells.
#
# Note:
#   Exact pairwise distances can be memory-heavy for many cells.
#   This cell uses exact pairwise distances for small sets and random pair sampling
#   for large sets.
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

# =============================
# Settings
# =============================
TOP_FRAC = 0.20

# Exact pairwise calculation is used only when n <= this threshold.
# For larger n, random pair sampling is used to avoid O(n^2) memory.
EXACT_PAIRWISE_MAX_N = 5000

# Number of random pairs used when approximate mode is needed.
N_RANDOM_PAIRS = 300000

RANDOM_SEED = 0

COMPACTNESS_OUT_PREFIX = "Malignant_KEGG_top20_compactness_ratio"

rng = np.random.default_rng(RANDOM_SEED)

# =============================
# Required variables from previous cell
# =============================
if "module_score_df" not in globals():
    raise NameError(
        "module_score_df is required. Please run the KEGG module-score UMAP cell first."
    )

if "score_cols" not in globals() or len(score_cols) == 0:
    score_cols = [
        col for col in module_score_df.columns
        if col.startswith("KEGG_") and col.endswith("_score")
    ]

if len(score_cols) == 0:
    raise ValueError(
        "No KEGG score columns found. Expected columns like 'KEGG_..._score'."
    )

if "score_title_map" not in globals():
    score_title_map = {col: col for col in score_cols}

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)

# =============================
# Get embedding coordinates
# =============================
if {"MI_UMAP1", "MI_UMAP2"}.issubset(module_score_df.columns):
    Z = module_score_df[["MI_UMAP1", "MI_UMAP2"]].to_numpy(dtype=float)
    embedding_source = "module_score_df[['MI_UMAP1', 'MI_UMAP2']]"

elif "adata_choose" in globals():
    mi_umap, mi_umap_source = _get_mi_umap(adata_choose)
    Z = np.asarray(mi_umap[:, :2], dtype=float)
    embedding_source = mi_umap_source

else:
    raise KeyError(
        "Cannot find embedding coordinates. Expected module_score_df with "
        "'MI_UMAP1' and 'MI_UMAP2', or adata_choose with MI_UMAP coordinates."
    )

if Z.shape[0] != module_score_df.shape[0]:
    raise ValueError(
        f"Embedding row number ({Z.shape[0]}) does not match module_score_df "
        f"row number ({module_score_df.shape[0]})."
    )

finite_embed_mask = np.isfinite(Z).all(axis=1)

if finite_embed_mask.sum() < 3:
    raise ValueError("Fewer than 3 cells have finite embedding coordinates.")

# =============================
# Helper functions
# =============================
def _mean_pairwise_distance(coords, exact_max_n=5000, n_random_pairs=300000, rng=None):
    """
    Compute mean pairwise Euclidean distance.

    Uses exact scipy.spatial.distance.pdist when n is small.
    Uses random pair sampling when n is large.
    """
    coords = np.asarray(coords, dtype=float)
    coords = coords[np.isfinite(coords).all(axis=1)]

    n = coords.shape[0]

    if n < 2:
        return np.nan, n, "too_few_cells", 0

    if n <= exact_max_n:
        d = pdist(coords, metric="euclidean")
        return float(np.mean(d)), n, "exact_pdist", int(len(d))

    if rng is None:
        rng = np.random.default_rng(0)

    # Sample ordered pairs with replacement, excluding self-pairs.
    i = rng.integers(0, n, size=n_random_pairs)
    j = rng.integers(0, n, size=n_random_pairs)

    same = i == j
    while np.any(same):
        j[same] = rng.integers(0, n, size=int(np.sum(same)))
        same = i == j

    dist = np.linalg.norm(coords[i] - coords[j], axis=1)

    return float(np.mean(dist)), n, "sampled_random_pairs", int(n_random_pairs)


# =============================
# Compute D_all once using all valid cells
# =============================
Z_all = Z[finite_embed_mask]

D_all, n_all_embed, D_all_mode, n_all_pairs_used = _mean_pairwise_distance(
    Z_all,
    exact_max_n=EXACT_PAIRWISE_MAX_N,
    n_random_pairs=N_RANDOM_PAIRS,
    rng=rng,
)

if not np.isfinite(D_all) or D_all <= 0:
    raise ValueError(f"Invalid D_all: {D_all}")

print(f"Embedding source: {embedding_source}")
print(
    f"D_all = {D_all:.6f} "
    f"(n_cells={n_all_embed}, mode={D_all_mode}, n_pairs_used={n_all_pairs_used})"
)

# =============================
# Compute compactness per KEGG pathway score
# =============================
compactness_records = []

for score_col in score_cols:
    score_values = module_score_df[score_col].to_numpy(dtype=float)

    valid_mask = finite_embed_mask & np.isfinite(score_values)

    n_valid = int(valid_mask.sum())

    if n_valid < 3:
        compactness_records.append({
            "module": score_col,
            "module_label": score_title_map.get(score_col, score_col),
            "top_fraction": TOP_FRAC,
            "n_valid_cells": n_valid,
            "n_top_cells": 0,
            "score_threshold_top20": np.nan,
            "D_all": D_all,
            "D_top": np.nan,
            "compactness_log2_Dall_over_Dtop": np.nan,
            "D_all_mode": D_all_mode,
            "D_top_mode": "too_few_valid_cells",
            "n_all_pairs_used": n_all_pairs_used,
            "n_top_pairs_used": 0,
            "embedding_source": embedding_source,
        })
        continue

    score_valid = score_values[valid_mask]
    Z_valid = Z[valid_mask]

    # Select top 20% cells.
    # Select ceil(TOP_FRAC * n_valid) cells, with a minimum of two,
    # resolving score ties according to the order returned by numpy.argsort.
    n_top = max(2, int(np.ceil(TOP_FRAC * n_valid)))

    order_desc = np.argsort(-score_valid)
    top_idx = order_desc[:n_top]

    Z_top = Z_valid[top_idx]
    top_scores = score_valid[top_idx]

    score_threshold = float(np.nanmin(top_scores))

    D_top, n_top_embed, D_top_mode, n_top_pairs_used = _mean_pairwise_distance(
        Z_top,
        exact_max_n=EXACT_PAIRWISE_MAX_N,
        n_random_pairs=N_RANDOM_PAIRS,
        rng=rng,
    )

    if np.isfinite(D_top) and D_top > 0:
        compactness = float(np.log2(D_all / D_top))
    else:
        compactness = np.nan

    compactness_records.append({
        "module": score_col,
        "module_label": score_title_map.get(score_col, score_col),
        "top_fraction": TOP_FRAC,
        "n_valid_cells": n_valid,
        "n_top_cells": int(n_top_embed),
        "score_threshold_top20": score_threshold,
        "top_score_mean": float(np.nanmean(top_scores)),
        "top_score_median": float(np.nanmedian(top_scores)),
        "all_score_mean": float(np.nanmean(score_valid)),
        "all_score_median": float(np.nanmedian(score_valid)),
        "D_all": D_all,
        "D_top": D_top,
        "compactness_log2_Dall_over_Dtop": compactness,
        "D_all_mode": D_all_mode,
        "D_top_mode": D_top_mode,
        "n_all_pairs_used": n_all_pairs_used,
        "n_top_pairs_used": n_top_pairs_used,
        "embedding_source": embedding_source,
    })

compactness_df = pd.DataFrame(compactness_records)

compactness_df = compactness_df.sort_values(
    "compactness_log2_Dall_over_Dtop",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

compactness_csv = os.path.join(
    out_dir,
    f"{COMPACTNESS_OUT_PREFIX}_summary.csv",
)

compactness_df.to_csv(compactness_csv, index=False)

print("Top-cell compactness ratio summary:")
_display_df(compactness_df)

print(f"Saved compactness summary to:\n{compactness_csv}")

# =============================
# Plot compactness barplot
# =============================
plot_df = compactness_df.dropna(
    subset=["compactness_log2_Dall_over_Dtop"]
).copy()

if not plot_df.empty:
    fig_width = max(3.4, 1.05 * plot_df.shape[0])
    fig, ax = plt.subplots(figsize=(fig_width, 2.8))

    x = np.arange(plot_df.shape[0])
    y = plot_df["compactness_log2_Dall_over_Dtop"].to_numpy(dtype=float)

    ax.bar(
        x,
        y,
        width=0.65,
        edgecolor="black",
        linewidth=0.6,
    )

    ax.axhline(0, color="black", linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df["module_label"].astype(str).tolist(),
        rotation=35,
        ha="right",
    )

    ax.set_ylabel(r"Compactness = log$_2$(D$_{all}$/D$_{top}$)")
    ax.set_title("Top 20% KEGG-score cell compactness in MI_UMAP")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out", length=2)

    for xi, yi in zip(x, y):
        ax.text(
            xi,
            yi + 0.03 * np.sign(yi if yi != 0 else 1),
            f"{yi:.2f}",
            ha="center",
            va="bottom" if yi >= 0 else "top",
            fontsize=6,
        )

    plt.tight_layout()

    compactness_png = os.path.join(
        out_dir,
        f"{COMPACTNESS_OUT_PREFIX}_barplot.png",
    )

    compactness_pdf = os.path.join(
        out_dir,
        f"{COMPACTNESS_OUT_PREFIX}_barplot.pdf",
    )

    plt.savefig(compactness_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(compactness_pdf, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close()

    print(f"Saved compactness barplot to:\n{compactness_png}\n{compactness_pdf}")

else:
    print("No finite compactness values available; skipped barplot.")

In [ ]:
# Export the four KEGG scores created by the preceding UMAP scoring block.

# ==============================================================
# Export malignant-cell KEGG module scores for downstream baseline comparison
# --------------------------------------------------------------
# This cell should be run immediately after the KEGG module-score MI_UMAP cell.
# It saves one row per malignant cell with:
#   barcode, sample, MI_louvain cluster, malignant-cluster label,
#   and the four KEGG pathway module scores.
# ==============================================================

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# -----------------------------
# Settings
# -----------------------------
KEGG_PATHWAYS_TO_EXPORT = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

SPIDERNET_KEGG_SCORE_EXPORT_PATH = (
    Path(run_dirs["run_dir"])
    / "HGSOC_malignant_cell_KEGG_module_scores_SpiderNet_MI_louvain.csv"
)

SPIDERNET_KEGG_SCORE_METADATA_PATH = (
    Path(run_dirs["run_dir"])
    / "HGSOC_malignant_cell_KEGG_module_scores_SpiderNet_MI_louvain_metadata.csv"
)

# -----------------------------
# Helper functions
# -----------------------------
def _safe_name_export(x):
    x = str(x)
    x = x.replace("/", "_").replace("-", "_")
    x = re.sub(r"[^\w]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _clean_cluster_label_export(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _sample_col_from_obs_export(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _display_df_export(df):
    if "display" in globals():
        display(df)
    else:
        print(df)

# -----------------------------
# Locate score columns
# -----------------------------
expected_score_cols = [
    f"KEGG_{_safe_name_export(pathway)}_score"
    for pathway in KEGG_PATHWAYS_TO_EXPORT
]

score_label_map = {
    f"KEGG_{_safe_name_export('HIF-1 signaling pathway')}_score": "HIF-1 signaling",
    f"KEGG_{_safe_name_export('PD-L1 expression and PD-1 checkpoint pathway in cancer')}_score": "PD-1/PD-L1 checkpoint",
    f"KEGG_{_safe_name_export('PI3K-Akt signaling pathway')}_score": "PI3K-Akt signaling",
    f"KEGG_{_safe_name_export('ECM-receptor interaction')}_score": "ECM-receptor interaction",
}

missing_score_cols = []
for col in expected_score_cols:
    if "module_score_df" in globals() and col in module_score_df.columns:
        continue
    if col in adata_choose.obs.columns:
        continue
    missing_score_cols.append(col)

if len(missing_score_cols) > 0:
    raise KeyError(
        "The following KEGG module-score columns are missing. "
        "Please run the KEGG module-score MI_UMAP cell first:\n"
        + "\n".join(missing_score_cols)
    )

# -----------------------------
# Build one-row-per-malignant-cell export table
# -----------------------------
if "barcode" in adata_choose.obs.columns:
    barcode_values = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    barcode_values = adata_choose.obs_names.astype(str).to_numpy()

sample_col = _sample_col_from_obs_export(adata_choose)
if sample_col is not None:
    sample_values = adata_choose.obs[sample_col].astype(str).to_numpy()
else:
    print("[Warning] No sample column found in adata_choose.obs. Using '__unknown_sample__'.")
    sample_col = "__unknown_sample__"
    sample_values = np.array(["__unknown_sample__"] * adata_choose.n_obs, dtype=object)

if "MI_louvain" in adata_choose.obs.columns:
    mi_louvain_values = adata_choose.obs["MI_louvain"].astype(str).to_numpy()
else:
    raise KeyError("Cannot find 'MI_louvain' in adata_choose.obs.")

mi_louvain_clean = np.array([
    _clean_cluster_label_export(x)
    for x in mi_louvain_values
], dtype=object)

export_df = pd.DataFrame({
    "cell_id": adata_choose.obs_names.astype(str),
    "barcode": barcode_values,
    "sample": sample_values,
    "sample_source_col": sample_col,
    "MI_louvain": mi_louvain_values,
    "MI_louvain_clean": mi_louvain_clean,
    "MalignantCluster": [f"Malignant C{x}" for x in mi_louvain_clean],
})

# Pull module scores from module_score_df when available; otherwise from adata_choose.obs
for col in expected_score_cols:
    if "module_score_df" in globals() and col in module_score_df.columns:
        values = pd.Series(
            module_score_df[col].to_numpy(),
            index=module_score_df.index.astype(str),
        ).reindex(adata_choose.obs_names.astype(str)).to_numpy()
    else:
        values = adata_choose.obs[col].to_numpy()

    export_df[col] = values

# Optional coordinates if available from the previous cell
if "module_score_df" in globals():
    for coord_col in ["MI_UMAP1", "MI_UMAP2"]:
        if coord_col in module_score_df.columns:
            export_df[coord_col] = pd.Series(
                module_score_df[coord_col].to_numpy(),
                index=module_score_df.index.astype(str),
            ).reindex(adata_choose.obs_names.astype(str)).to_numpy()

# Save
SPIDERNET_KEGG_SCORE_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(SPIDERNET_KEGG_SCORE_EXPORT_PATH, index=False)

metadata_df = pd.DataFrame({
    "score_col": expected_score_cols,
    "score_label": [score_label_map[c] for c in expected_score_cols],
    "pathway": KEGG_PATHWAYS_TO_EXPORT,
    "export_csv": str(SPIDERNET_KEGG_SCORE_EXPORT_PATH),
})
metadata_df.to_csv(SPIDERNET_KEGG_SCORE_METADATA_PATH, index=False)

print(f"Saved malignant-cell KEGG module-score table to:\n{SPIDERNET_KEGG_SCORE_EXPORT_PATH}")
print(f"Saved metadata to:\n{SPIDERNET_KEGG_SCORE_METADATA_PATH}")
print(f"Exported cells: {export_df.shape[0]:,}")
print("Cluster counts:")
_display_df_export(
    export_df["MalignantCluster"].value_counts().rename_axis("MalignantCluster").reset_index(name="n_cells")
)

print("Preview:")
_display_df_export(export_df.head())


In [ ]:
# CD8 T-cell neighborhoods of malignant cells
# Count CD8.T.cell neighbors among up to ten non-self spatial neighbors within
# each sample/slice. The next blocks export cell/cluster summaries, the fraction
# with a CD8 T-cell neighbor, and the 1/2/>=3-neighbor count distribution.

import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.neighbors import NearestNeighbors

# =============================
# Settings
# =============================
K_NEIGH = 10
CD8_LABEL = "CD8.T.cell"
CELL_SUBTYPE_COL = "cell.subtypes"
CLUSTER_COL = "MI_louvain"
OUT_COL = "neighbor_CD8T_prop_spatial10NN"

# =============================
# Publication-style global settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Get full AnnData
# =============================
adata_all = processed.adata_all

if CELL_SUBTYPE_COL not in adata_all.obs.columns:
    raise KeyError(
        f"Cannot find '{CELL_SUBTYPE_COL}' in processed.adata_all.obs. "
        f"Available columns are:\n{list(adata_all.obs.columns)}"
    )

if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Please run the malignant Louvain clustering cell first."
    )

# =============================
# Get spatial coordinates
# =============================
if "spatial" in adata_all.obsm:
    spatial_all = np.asarray(adata_all.obsm["spatial"], dtype=float)
else:
    spatial_col_candidates = [
        ("x", "y"),
        ("X", "Y"),
        ("center_x", "center_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("x_centroid", "y_centroid"),
        ("X_centroid", "Y_centroid"),
    ]
    spatial_cols = next(
        (
            cols for cols in spatial_col_candidates
            if cols[0] in adata_all.obs.columns and cols[1] in adata_all.obs.columns
        ),
        None
    )
    if spatial_cols is None:
        raise KeyError(
            "Cannot find spatial coordinates. Expected adata_all.obsm['spatial'] "
            "or x/y-like columns in adata_all.obs."
        )
    spatial_all = adata_all.obs.loc[:, list(spatial_cols)].to_numpy(dtype=float)

# =============================
# Match malignant cells to full AnnData
# =============================
if "barcode" in adata_choose.obs.columns:
    malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

adata_all_index = pd.Index(adata_all.obs_names.astype(str))
malignant_full_idx = adata_all_index.get_indexer(malignant_barcodes)

# Fallback: match through adata_all.obs["barcode"] if needed
if np.any(malignant_full_idx < 0) and "barcode" in adata_all.obs.columns:
    all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
    barcode_to_idx = {}
    for i, b in enumerate(all_barcodes):
        if b not in barcode_to_idx:
            barcode_to_idx[b] = i
    malignant_full_idx = np.array(
        [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
        dtype=int
    )

missing_mask = malignant_full_idx < 0
if np.any(missing_mask):
    raise ValueError(
        f"{missing_mask.sum()} malignant cells could not be matched to processed.adata_all. "
        "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names."
    )

# =============================
# Sample/slice column
# Compute spatial neighbors separately within each sample/slice when available.
# =============================
sample_col_candidates = [
    "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
    "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
]

sample_col = next(
    (col for col in sample_col_candidates if col in adata_all.obs.columns),
    None
)

if sample_col is None:
    print(
        "[Warning] No sample/slice column found in adata_all.obs. "
        "Spatial 10-NN will be computed across all cells together."
    )
    sample_values = np.array(["__all_cells__"] * adata_all.n_obs)
else:
    print(f"Using sample/slice column for within-sample spatial KNN: {sample_col}")
    sample_values = adata_all.obs[sample_col].astype(str).to_numpy()

# =============================
# Compute CD8.T.cell proportion among spatial 10-NN
# =============================
cell_subtypes = adata_all.obs[CELL_SUBTYPE_COL].astype(str).to_numpy()
is_cd8 = cell_subtypes == CD8_LABEL

cd8_prop = np.full(adata_choose.n_obs, np.nan, dtype=float)
cd8_count = np.full(adata_choose.n_obs, np.nan, dtype=float)
neighbor_n = np.full(adata_choose.n_obs, np.nan, dtype=float)

# map full-cell index -> malignant-row position in adata_choose
full_idx_to_mal_pos = pd.Series(
    np.arange(len(malignant_full_idx)),
    index=malignant_full_idx
)

for sample in pd.unique(sample_values[malignant_full_idx]):
    sample_idx = np.where(sample_values == sample)[0]
    mal_idx_sample = malignant_full_idx[sample_values[malignant_full_idx] == sample]

    if len(sample_idx) <= 1 or len(mal_idx_sample) == 0:
        continue

    k_fit = min(K_NEIGH + 1, len(sample_idx))

    nn = NearestNeighbors(n_neighbors=k_fit, algorithm="auto")
    nn.fit(spatial_all[sample_idx, :])

    _, neigh_local = nn.kneighbors(spatial_all[mal_idx_sample, :])

    for row_i, cell_full_idx in enumerate(mal_idx_sample):
        neigh_global = sample_idx[neigh_local[row_i]]

        # remove self if present, then keep exactly 10 nearest available neighbors
        neigh_global = neigh_global[neigh_global != cell_full_idx][:K_NEIGH]

        mal_pos = int(full_idx_to_mal_pos.loc[cell_full_idx])

        if len(neigh_global) == 0:
            continue

        cd8_count[mal_pos] = np.sum(is_cd8[neigh_global])
        neighbor_n[mal_pos] = len(neigh_global)
        cd8_prop[mal_pos] = cd8_count[mal_pos] / neighbor_n[mal_pos]

# add to adata_choose
adata_choose.obs[OUT_COL] = cd8_prop
adata_choose.obs["neighbor_CD8T_count_spatial10NN"] = cd8_count
adata_choose.obs["neighbor_n_spatial10NN"] = neighbor_n
adata_choose.obs["has_neighbor_CD8T_spatial10NN"] = cd8_count > 0


In [ ]:
# =============================
# Prepare output table
# =============================
plot_df = pd.DataFrame({
    "barcode": malignant_barcodes,
    "MI_louvain": adata_choose.obs[CLUSTER_COL].astype(str).to_numpy(),
    "neighbor_CD8T_prop": cd8_prop,
    "neighbor_CD8T_count": cd8_count,
    "neighbor_n": neighbor_n,
})

def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x

plot_df["MI_louvain_clean"] = plot_df["MI_louvain"].map(_clean_cluster_label)
plot_df["MalignantCluster"] = "Malignant C" + plot_df["MI_louvain_clean"]
plot_df = plot_df.dropna(subset=["neighbor_CD8T_prop"]).copy()

plot_df["has_neighbor_CD8T"] = plot_df["neighbor_CD8T_count"] > 0

# cluster order
cluster_order = sorted(
    plot_df["MI_louvain_clean"].unique(),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)
display_order = [f"Malignant C{x}" for x in cluster_order]

plot_df["MalignantCluster"] = pd.Categorical(
    plot_df["MalignantCluster"],
    categories=display_order,
    ordered=True
)

# =============================
# Summary tables
# =============================
bar_df = (
    plot_df
    .groupby("MalignantCluster", observed=True)
    .agg(
        n_malignant_cells=("barcode", "count"),
        n_with_CD8T_neighbor=("has_neighbor_CD8T", "sum"),
        mean_CD8T_prop_all=("neighbor_CD8T_prop", "mean"),
        median_CD8T_prop_all=("neighbor_CD8T_prop", "median"),
    )
    .reset_index()
)

bar_df["fraction_with_CD8T_neighbor"] = (
    bar_df["n_with_CD8T_neighbor"] / bar_df["n_malignant_cells"]
)

plot_df_pos = plot_df.loc[plot_df["has_neighbor_CD8T"]].copy()

box_summary_df = (
    plot_df_pos
    .groupby("MalignantCluster", observed=True)
    .agg(
        n_CD8T_neighbor_positive_cells=("barcode", "count"),
        mean_CD8T_prop_positive=("neighbor_CD8T_prop", "mean"),
        median_CD8T_prop_positive=("neighbor_CD8T_prop", "median"),
        std_CD8T_prop_positive=("neighbor_CD8T_prop", "std"),
    )
    .reset_index()
)

summary_df = bar_df.merge(box_summary_df, on="MalignantCluster", how="left")

# save cell-level and summary tables
plot_df.to_csv(
    f"{run_dirs['run_dir']}/Malignant_neighbor_CD8T_spatial10NN_celllevel.csv",
    index=False
)

summary_df.to_csv(
    f"{run_dirs['run_dir']}/Malignant_neighbor_CD8T_spatial10NN_cluster_summary.csv",
    index=False
)

# save updated malignant AnnData
adata_choose.write_h5ad(adata_choose_path)

print(summary_df)

# =============================
# Plot colors
# =============================
edge_colors = ["#82CCE2", "#D1CABE", "#2E7FB9", "#FED881", "#9F3B38", "#519384", "#636491"]
fill_colors = ["#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2", "#E1B6A7", "#B9CEC7", "#A6A2B9"]

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}

bar_color = "#6FA8BF"

# ==============================================================
# Plot 1: Barplot
# Fraction of malignant cells with >=1 neighboring CD8T cell
# ==============================================================
fig, ax = plt.subplots(figsize=(3.4, 2.6))

sns.barplot(
    data=bar_df,
    x="MalignantCluster",
    y="fraction_with_CD8T_neighbor",
    order=display_order,
    color=bar_color,
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)

# label each bar with count: n_with_CD8T / total
for i, row in bar_df.set_index("MalignantCluster").loc[display_order].reset_index().iterrows():
    y = row["fraction_with_CD8T_neighbor"]
    label = f"{int(row['n_with_CD8T_neighbor'])}/{int(row['n_malignant_cells'])}"
    ax.text(
        i,
        y + 0.015,
        label,
        ha="center",
        va="bottom",
        fontsize=5.5,
        rotation=90
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_xlabel("Malignant cluster")
ax.set_ylabel("Fraction with ≥1 neighboring CD8 T cell\n(spatial 10-NN)")
ax.set_ylim(0, min(1.05, max(0.12, bar_df["fraction_with_CD8T_neighbor"].max() * 1.25)))

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_fraction_malignant_cells_with_neighbor_CD8T_spatial10NN_by_MI_louvain.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_fraction_malignant_cells_with_neighbor_CD8T_spatial10NN_by_MI_louvain.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


In [ ]:
# ==============================================================
# Plot 2: Stacked barplot
# Among malignant cells with non-zero CD8T neighbors,
# show the distribution of CD8T-neighbor count: 1, 2, >=3
# ==============================================================

plot_df_nonzero = plot_df.loc[
    np.isfinite(plot_df["neighbor_CD8T_count"]) &
    (plot_df["neighbor_CD8T_count"] > 0)
].copy()

if plot_df_nonzero.shape[0] == 0:
    print("No malignant cells have non-zero neighboring CD8.T.cell count. Plot is skipped.")
else:
    # Convert count into interpretable categories
    plot_df_nonzero["CD8T_neighbor_count_group"] = plot_df_nonzero["neighbor_CD8T_count"].astype(int).astype(str)
    plot_df_nonzero.loc[
        plot_df_nonzero["neighbor_CD8T_count"] >= 3,
        "CD8T_neighbor_count_group"
    ] = "≥3"

    count_order = ["1", "2", "≥3"]

    # Compute within-cluster fraction
    stacked_df = (
        plot_df_nonzero
        .groupby(["MalignantCluster", "CD8T_neighbor_count_group"], observed=True)
        .size()
        .reset_index(name="n_cells")
    )

    total_nonzero_df = (
        plot_df_nonzero
        .groupby("MalignantCluster", observed=True)
        .size()
        .reset_index(name="n_nonzero_cells")
    )

    stacked_df = stacked_df.merge(total_nonzero_df, on="MalignantCluster", how="left")
    stacked_df["fraction"] = stacked_df["n_cells"] / stacked_df["n_nonzero_cells"]

    # Make sure all cluster x count combinations exist
    display_order_nonzero = [
        c for c in display_order
        if c in plot_df_nonzero["MalignantCluster"].astype(str).unique()
    ]

    full_index = pd.MultiIndex.from_product(
        [display_order_nonzero, count_order],
        names=["MalignantCluster", "CD8T_neighbor_count_group"]
    )

    stacked_df = (
        stacked_df
        .set_index(["MalignantCluster", "CD8T_neighbor_count_group"])
        .reindex(full_index)
        .reset_index()
    )

    stacked_df["n_cells"] = stacked_df["n_cells"].fillna(0)
    stacked_df["n_nonzero_cells"] = stacked_df["n_nonzero_cells"].fillna(
        stacked_df["MalignantCluster"].map(
            total_nonzero_df.set_index("MalignantCluster")["n_nonzero_cells"]
        )
    )
    stacked_df["fraction"] = stacked_df["fraction"].fillna(0)

    stacked_wide = (
        stacked_df
        .pivot(
            index="MalignantCluster",
            columns="CD8T_neighbor_count_group",
            values="fraction"
        )
        .loc[display_order_nonzero, count_order]
        .fillna(0)
    )

    # Save table
    stacked_df.to_csv(
        f"{run_dirs['run_dir']}/Malignant_nonzero_CD8T_neighbor_count_distribution_spatial10NN.csv",
        index=False
    )

    # Colors for 1, 2, >=3 CD8T neighbors
    count_colors = {
        "1": "#D4ECF1",
        "2": "#82CCE2",
        "≥3": "#2E7FB9",
    }

    fig, ax = plt.subplots(figsize=(3.4, 2.8))

    bottom = np.zeros(len(stacked_wide))

    for count_group in count_order:
        values = stacked_wide[count_group].values

        ax.bar(
            np.arange(len(stacked_wide)),
            values,
            bottom=bottom,
            color=count_colors[count_group],
            edgecolor="black",
            linewidth=0.5,
            width=0.65,
            label=f"{count_group} CD8T neighbors"
        )

        bottom += values

    # Add n labels above bars
    n_map = total_nonzero_df.set_index("MalignantCluster")["n_nonzero_cells"].to_dict()

    for i, cluster in enumerate(stacked_wide.index):
        n_cur = int(n_map.get(cluster, 0))
        ax.text(
            i,
            1.03,
            f"n={n_cur}",
            ha="center",
            va="bottom",
            fontsize=5.5,
            rotation=90
        )

    ax.set_xticks(np.arange(len(stacked_wide)))
    ax.set_xticklabels(stacked_wide.index, rotation=30, ha="right")

    ax.set_xlabel("Malignant cluster")
    ax.set_ylabel("Fraction among non-zero cells")
    ax.set_ylim(0, 1.12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

    ax.legend(
        frameon=False,
        fontsize=6,
        title="CD8T count in 10-NN",
        title_fontsize=6,
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig(
        f"{run_dirs['run_dir']}/StackedBar_nonzero_CD8T_neighbor_count_distribution_spatial10NN_by_MI_louvain.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.savefig(
        f"{run_dirs['run_dir']}/StackedBar_nonzero_CD8T_neighbor_count_distribution_spatial10NN_by_MI_louvain.pdf",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()


In [ ]:
# Shared scoring helper for the following TIL-UP, KEGG, CAF and CD8 blocks.
# This implementation uses reference-cell means/stds without control-gene subtraction.

# ==============================================================
# Helper: zscore_mean_global module scoring across slices
# --------------------------------------------------------------
# MODULE_SCORE_METHOD = "zscore_mean_global"
#
# For each gene set:
#   1. Collect the relevant reference cells across all slices/samples
#      in the selected expression AnnData.
#   2. Compute one global mean and one global standard deviation per gene
#      using the concatenated reference-cell expression matrix.
#   3. For the requested rows/cells, transform each gene by the same
#      global mean/std.
#   4. Average the global z-scored genes to obtain one score per cell.
#
# This avoids per-slice mean/std re-estimation, so all slices share the
# same z-score reference scale.
# ============================================================== 

import numpy as np
import pandas as pd
from scipy import sparse

MODULE_SCORE_METHOD = "zscore_mean_global"

_ZSCORE_MEAN_GLOBAL_CACHE = {}


def _spidernet_find_sample_col(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _spidernet_resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = []
    missing = []
    seen = set()

    for gene in genes:
        gene = str(gene).strip()
        if not gene:
            continue

        if gene in var_names:
            actual = gene
        elif gene.upper() in upper_to_actual:
            actual = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)
            continue

        if actual not in seen:
            resolved.append(actual)
            seen.add(actual)

    return resolved, missing


def _spidernet_get_X_dense(adata, rows, genes_actual):
    rows = np.asarray(rows, dtype=int)
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub.astype(float, copy=False)


def _spidernet_global_gene_zscore_stats(expr_adata, genes_used, reference_row_idx=None, ddof=1):
    """Compute global mean/std per gene using reference rows pooled across slices."""
    if reference_row_idx is None:
        reference_row_idx = np.arange(expr_adata.n_obs, dtype=int)
    else:
        reference_row_idx = np.asarray(reference_row_idx, dtype=int)

    valid_ref = (reference_row_idx >= 0) & (reference_row_idx < expr_adata.n_obs)
    reference_row_idx = reference_row_idx[valid_ref]

    cache_key = (
        id(expr_adata),
        tuple(map(str, expr_adata.var_names.astype(str))),
        tuple(genes_used),
        tuple(reference_row_idx.tolist()),
        int(ddof),
        int(expr_adata.n_obs),
        int(expr_adata.n_vars),
    )

    if cache_key in _ZSCORE_MEAN_GLOBAL_CACHE:
        return _ZSCORE_MEAN_GLOBAL_CACHE[cache_key]

    if reference_row_idx.size == 0:
        gene_mean = np.full(len(genes_used), np.nan, dtype=float)
        gene_std = np.full(len(genes_used), np.nan, dtype=float)
    else:
        X_ref = _spidernet_get_X_dense(expr_adata, reference_row_idx, genes_used)
        gene_mean = np.nanmean(X_ref, axis=0)
        gene_std = np.nanstd(X_ref, axis=0, ddof=int(ddof))

    gene_std = np.asarray(gene_std, dtype=float)
    gene_std[~np.isfinite(gene_std) | (gene_std <= 0)] = np.nan

    stats = (gene_mean, gene_std, reference_row_idx)
    _ZSCORE_MEAN_GLOBAL_CACHE[cache_key] = stats
    return stats


def _spidernet_zscore_mean_global_scores_for_rows(
    expr_adata,
    row_idx,
    genes,
    reference_row_idx=None,
    sample_values=None,
    sample_col=None,
    score_name="__spidernet_zscore_mean_global__",
    ddof=1,
    fill_all_missing_with_nan=True,
):
    """Compute mean global z-scored expression over genes for requested rows.

    Parameters
    ----------
    expr_adata:
        Expression AnnData containing all slices/samples to be used for scoring.
    row_idx:
        Rows/cells for which scores should be returned.
    genes:
        Gene set to score.
    reference_row_idx:
        Rows pooled across all slices/samples to estimate global gene mean/std.
        If None, row_idx defines the reference population. For cell-type-specific
        scores, pass all cells of the relevant cell type across slices.
    sample_values, sample_col, score_name:
        Kept for backward compatibility with earlier AddModule-style calls.
    ddof:
        Standard deviation degrees of freedom. ddof=1 matches pandas sample std.
    """
    row_idx = np.asarray(row_idx, dtype=int)
    out = np.full(row_idx.size, np.nan, dtype=float)

    genes_used, genes_missing = _spidernet_resolve_gene_names(expr_adata, genes)
    if len(genes_used) == 0 or row_idx.size == 0:
        return out, genes_used, genes_missing

    if reference_row_idx is None:
        # Default to the requested target population, pooled across all slices.
        # The requested rows define the reference population when none is supplied.
        reference_row_idx = row_idx

    gene_mean, gene_std, reference_row_idx_used = _spidernet_global_gene_zscore_stats(
        expr_adata=expr_adata,
        genes_used=genes_used,
        reference_row_idx=reference_row_idx,
        ddof=ddof,
    )

    valid = (row_idx >= 0) & (row_idx < expr_adata.n_obs)
    if not np.any(valid):
        return out, genes_used, genes_missing

    X_target = _spidernet_get_X_dense(expr_adata, row_idx[valid], genes_used)

    with np.errstate(invalid="ignore", divide="ignore"):
        Z_target = (X_target - gene_mean.reshape(1, -1)) / gene_std.reshape(1, -1)

    # Average across genes after global z-scoring. Genes with zero/undefined
    # global std are ignored by nanmean. If all genes are invalid for a row,
    # the score remains NaN.
    score_valid = np.nanmean(Z_target, axis=1)
    out[valid] = score_valid
    return out, genes_used, genes_missing


In [ ]:
# ==============================================================
# Malignant TIL-UP gene-set module score across malignant clusters
# --------------------------------------------------------------
# Input:
#   D:\SpiderNet\Data\HGSOC\Malignant_TIL_upgenes.csv
#
# The file can be:
#   - one column without header, or
#   - one column with header such as UP/Gene/Genes
#
# Score:
#   TIL_UP_score
#
# For the TIL-UP gene set:
#   - intersect genes with current HGSOC expression matrix
#   - average gene-wise z-scores using malignant reference cells pooled across slices
#   - use the zscore_mean_global score as module score
#   - aggregate at sample_id × malignant cluster level
#   - plot sample-level boxplot across malignant clusters
# ==============================================================

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import PathPatch
from scipy import sparse
from scipy.stats import ranksums

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

TIL_UP_GENESET_PATH = (DATA_ROOT / 'Malignant_TIL_upgenes.csv')

OUT_PREFIX = "Malignant_TIL_UP_module_score_by_cluster"

MIN_CELLS_PER_SAMPLE_CLUSTER = 10

# Reference cluster for pairwise annotation
REF_CLUSTER = "Malignant C5"

# One-sided Wilcoxon rank-sum test direction:
#   "greater": test whether C5 > other cluster
#   "less":    test whether C5 < other cluster
TEST_ALTERNATIVE = "greater"

out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _first_indexer(values, query_values):
    mapping = {}

    for i, v in enumerate(values):
        v = str(v)
        if v not in mapping:
            mapping[v] = i

    return np.array(
        [mapping.get(str(q), -1) for q in query_values],
        dtype=int
    )


def _make_composite(sample_values, id_values):
    return np.array(
        [f"{str(s)}||{str(i)}" for s, i in zip(sample_values, id_values)],
        dtype=object
    )


def _add_expr_candidate(candidates, name, obj):
    if obj is not None and hasattr(obj, "var_names") and hasattr(obj, "obs") and hasattr(obj, "X"):
        candidates.append((name, obj))


def _build_expr_candidates():
    """
    Collect possible AnnData expression objects from current notebook.
    Prefer the object with the largest number of TIL-UP genes and successful malignant-cell row matching.
    """
    candidates = []

    if "adata_choose" in globals():
        _add_expr_candidate(candidates, "adata_choose", adata_choose)

    if "processed" in globals():
        _add_expr_candidate(candidates, "processed.adata_all", getattr(processed, "adata_all", None))
        _add_expr_candidate(candidates, "processed.adata", getattr(processed, "adata", None))

        if hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
            try:
                import anndata as ad

                adata_list_concat = ad.concat(
                    processed.adata_list,
                    join="outer",
                    index_unique=None,
                    merge="same",
                )

                _add_expr_candidate(candidates, "concat(processed.adata_list)", adata_list_concat)

            except Exception as e:
                print(f"[Info] Could not concatenate processed.adata_list: {e}")

    for var_name in [
        "adata_all_full",
        "adata_raw",
        "adata_full",
        "adata_ori",
        "adata_original",
        "adata_copy",
        "adata",
        "adata_all",
    ]:
        if var_name in globals():
            _add_expr_candidate(candidates, var_name, globals()[var_name])

    # deduplicate by object id
    seen = set()
    unique_candidates = []

    for name, obj in candidates:
        if id(obj) not in seen:
            unique_candidates.append((name, obj))
            seen.add(id(obj))

    return unique_candidates


def _match_malignant_rows_to_expr_adata(expr_adata):
    """
    Match malignant cells in adata_choose to expression AnnData.
    If expr_adata is adata_choose itself, use direct row order.
    Otherwise try:
      1. sample + barcode
      2. sample + obs_names
      3. barcode
      4. obs_names
    """
    if expr_adata is adata_choose:
        return np.arange(adata_choose.n_obs), "direct adata_choose row order"

    malignant_obs_names = adata_choose.obs_names.astype(str).to_numpy()

    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = malignant_obs_names.copy()

    expr_obs_names = expr_adata.obs_names.astype(str).to_numpy()

    if "barcode" in expr_adata.obs.columns:
        expr_barcodes = expr_adata.obs["barcode"].astype(str).to_numpy()
    else:
        expr_barcodes = expr_obs_names.copy()

    sample_col_choose = _sample_col_from_obs(adata_choose)
    sample_col_expr = _sample_col_from_obs(expr_adata)

    if sample_col_choose is not None and sample_col_expr is not None:
        malignant_sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
        expr_sample_values = expr_adata.obs[sample_col_expr].astype(str).to_numpy()

        query_comp_barcode = _make_composite(malignant_sample_values, malignant_barcodes)
        expr_comp_barcode = _make_composite(expr_sample_values, expr_barcodes)

        idx = _first_indexer(expr_comp_barcode, query_comp_barcode)
        if np.all(idx >= 0):
            return idx, f"sample + barcode using adata_choose['{sample_col_choose}'] and expr['{sample_col_expr}']"

        query_comp_obs = _make_composite(malignant_sample_values, malignant_obs_names)
        expr_comp_obs = _make_composite(expr_sample_values, expr_obs_names)

        idx = _first_indexer(expr_comp_obs, query_comp_obs)
        if np.all(idx >= 0):
            return idx, f"sample + obs_names using adata_choose['{sample_col_choose}'] and expr['{sample_col_expr}']"

    # barcode matching
    idx = _first_indexer(expr_barcodes, malignant_barcodes)
    if np.all(idx >= 0):
        return idx, "barcode -> expression barcode"

    # obs_names matching
    idx = _first_indexer(expr_obs_names, malignant_obs_names)
    if np.all(idx >= 0):
        return idx, "adata_choose obs_names -> expression obs_names"

    # malignant barcode to expression obs_names
    idx = _first_indexer(expr_obs_names, malignant_barcodes)
    if np.all(idx >= 0):
        return idx, "adata_choose barcode -> expression obs_names"

    return None, None


def _get_X_array(adata, rows, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


# =============================
# Check malignant cluster column
# =============================
if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Load TIL-UP gene set
# =============================
if not input_path(TIL_UP_GENESET_PATH).exists():
    raise FileNotFoundError(
        f"Cannot find TIL-UP gene-set file:\n{TIL_UP_GENESET_PATH}"
    )

# Read as no-header file first; this is robust for one-column gene lists.
til_gene_raw = pd.read_csv(input_path(TIL_UP_GENESET_PATH), header=None)

# Flatten all values, because the file may be one column or accidentally have extra blank columns.
til_up_genes = (
    pd.Series(til_gene_raw.values.ravel())
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)

# Remove empty values and possible header-like values
til_up_genes = [
    g for g in til_up_genes
    if len(g) > 0
    and g.lower() != "nan"
    and g.upper() not in {"UP", "GENE", "GENES", "TIL_UP", "TIL_UP_GENES"}
]

til_up_genes = list(dict.fromkeys(til_up_genes))

if len(til_up_genes) == 0:
    raise ValueError(
        f"No genes were loaded from:\n{TIL_UP_GENESET_PATH}"
    )

til_geneset_dict = {
    "TIL_UP_score": til_up_genes,
}

all_til_genes = til_up_genes.copy()

print(f"TIL-UP genes loaded: {len(til_up_genes)}")

til_gene_long = pd.DataFrame({
    "TIL_gene_set": "TIL_UP_score",
    "Gene": til_up_genes,
})

til_gene_long.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_input_gene_set_long.csv"),
    index=False,
)

# =============================
# Malignant cluster labels
# =============================
cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()

cluster_order = sorted(
    np.unique(cluster_clean),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

display_order = [f"Malignant C{x}" for x in cluster_order]
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

if "barcode" in adata_choose.obs.columns:
    malignant_barcodes_out = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    malignant_barcodes_out = adata_choose.obs_names.astype(str).to_numpy()

# =============================
# Find best expression AnnData
# =============================
expr_candidates = _build_expr_candidates()

candidate_summary = []
best = None

for name, cand in expr_candidates:
    resolved_cur, missing_cur = _resolve_gene_names(cand, all_til_genes)
    n_genes_cur = len(resolved_cur)

    row_idx_cur, match_mode_cur = _match_malignant_rows_to_expr_adata(cand)
    rows_ok = row_idx_cur is not None

    n_up_found = len([g for g in til_up_genes if g in resolved_cur])

    candidate_summary.append({
        "candidate": name,
        "n_obs": cand.n_obs,
        "n_vars": cand.n_vars,
        "n_TIL_UP_genes_found": n_genes_cur,
        "n_UP_genes_found": n_up_found,
        "malignant_rows_matched": rows_ok,
        "match_mode": match_mode_cur if match_mode_cur is not None else "NA",
        "first_10_found_genes": ", ".join(list(resolved_cur.keys())[:10]),
    })

    if n_genes_cur > 0 and rows_ok and n_up_found > 0:
        if best is None or n_genes_cur > best["n_genes"]:
            best = {
                "name": name,
                "adata": cand,
                "resolved": resolved_cur,
                "missing": missing_cur,
                "row_idx": row_idx_cur,
                "match_mode": match_mode_cur,
                "n_genes": n_genes_cur,
                "n_up_found": n_up_found,
            }

candidate_summary_df = pd.DataFrame(candidate_summary)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False,
)

print("Expression object search summary:")
_display_df(candidate_summary_df)

if best is None:
    raise ValueError(
        "Cannot find an expression AnnData that contains TIL-UP genes "
        "and matches malignant cells. Please check the expression_object_search_summary table."
    )

expr_source_name = best["name"]
expr_adata = best["adata"]
expr_rows = best["row_idx"]
resolved_genes = best["resolved"]
missing_genes = best["missing"]

available_genes = [g for g in all_til_genes if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

print(f"Selected expression source: {expr_source_name}")
print(f"Malignant row matching mode: {best['match_mode']}")
print(f"TIL-UP genes found: {len(available_genes)} / {len(all_til_genes)}")

if len(missing_genes) > 0:
    print(f"[Warning] Missing TIL-UP genes skipped: {missing_genes}")

til_gene_available_df = til_gene_long.loc[
    til_gene_long["Gene"].isin(available_genes)
].copy()

til_gene_available_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_genes_used.csv"),
    index=False,
)

print("Genes used for TIL-UP score:")
_display_df(
    til_gene_available_df
    .groupby("TIL_gene_set", as_index=False)
    .agg(
        n_genes=("Gene", "nunique"),
        genes=("Gene", lambda x: ", ".join(sorted(x.unique())))
    )
)

# =============================
# Compute TIL-UP module score with zscore_mean_global
# =============================
# Estimate gene means and sample standard deviations (ddof=1) from the
# specified reference rows pooled across slices, then average valid gene z-scores.
# Genes with zero or undefined standard deviation are omitted by nanmean.
score_wide = pd.DataFrame({
    "barcode": malignant_barcodes_out,
    "MI_louvain": cluster_clean,
    "MalignantCluster": cluster_display,
}, index=adata_choose.obs_names)

score_gene_count_records = []

for score_name, genes in til_geneset_dict.items():
    score_cur, genes_used_cur, genes_missing_cur = _spidernet_zscore_mean_global_scores_for_rows(
        expr_adata=expr_adata,
        row_idx=expr_rows,
        reference_row_idx=expr_rows,
        genes=genes,
        sample_col=_spidernet_find_sample_col(expr_adata),
        score_name=f"__{score_name}_zscore_mean_global__",
        ddof=1,
    )

    if len(genes_used_cur) == 0:
        print(f"[Warning] No available genes for {score_name}. Skipping.")
        continue

    score_wide[score_name] = score_cur

    score_gene_count_records.append({
        "TIL_score": score_name,
        "score_method": MODULE_SCORE_METHOD,
        "n_genes_used": len(genes_used_cur),
        "genes_used": ", ".join(genes_used_cur),
        "n_missing_genes": len(genes_missing_cur),
        "missing_genes": ", ".join(genes_missing_cur),
    })

score_gene_count_df = pd.DataFrame(score_gene_count_records)

score_cols = ["TIL_UP_score"] if "TIL_UP_score" in score_wide.columns else []

if len(score_cols) == 0:
    raise ValueError("TIL_UP_score was not computed.")

# save score to adata_choose.obs
adata_choose.obs["TIL_UP_score"] = score_wide["TIL_UP_score"].to_numpy()

score_long = score_wide.melt(
    id_vars=["barcode", "MI_louvain", "MalignantCluster"],
    value_vars=score_cols,
    var_name="TIL_score",
    value_name="module_score",
)

score_long["MalignantCluster"] = pd.Categorical(
    score_long["MalignantCluster"],
    categories=display_order,
    ordered=True,
)

score_long["TIL_score"] = pd.Categorical(
    score_long["TIL_score"],
    categories=score_cols,
    ordered=True,
)

# =============================
# Add sample information
# =============================
sample_col_choose = _sample_col_from_obs(adata_choose)

if sample_col_choose is not None:
    sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
    print(f"Using sample column from adata_choose: {sample_col_choose}")
else:
    sample_col_expr = _sample_col_from_obs(expr_adata)

    if sample_col_expr is not None:
        sample_values = expr_adata.obs.iloc[expr_rows][sample_col_expr].astype(str).to_numpy()
        print(f"Using sample column from expression AnnData: {sample_col_expr}")
    else:
        print(
            "[Warning] No sample column found. "
            "All malignant cells will be treated as one pseudo-sample."
        )
        sample_values = np.array(["__all_cells__"] * adata_choose.n_obs)

barcode_to_sample = pd.Series(
    sample_values,
    index=malignant_barcodes_out,
)

score_long["sample_id"] = score_long["barcode"].map(barcode_to_sample).astype(str)
score_wide["sample_id"] = sample_values

# =============================
# Sample-level aggregation
# =============================
sample_score_df = (
    score_long
    .groupby(["sample_id", "MalignantCluster", "TIL_score"], observed=True)
    .agg(
        mean_module_score=("module_score", "mean"),
        median_module_score=("module_score", "median"),
        n_cells=("barcode", "count"),
    )
    .reset_index()
)

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_CLUSTER
].copy()

sample_score_df_plot["MalignantCluster"] = pd.Categorical(
    sample_score_df_plot["MalignantCluster"],
    categories=display_order,
    ordered=True,
)

sample_score_df_plot["TIL_score"] = pd.Categorical(
    sample_score_df_plot["TIL_score"],
    categories=score_cols,
    ordered=True,
)

# =============================
# Summary + cluster vs others p-values
# =============================
cluster_summary_df = (
    sample_score_df_plot
    .groupby(["TIL_score", "MalignantCluster"], observed=True)
    .agg(
        n_samples=("sample_id", "nunique"),
        mean_sample_level_score=("mean_module_score", "mean"),
        median_sample_level_score=("mean_module_score", "median"),
        std_sample_level_score=("mean_module_score", "std"),
        total_cells=("n_cells", "sum"),
    )
    .reset_index()
)

pval_records = []

for score_name in score_cols:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["TIL_score"].astype(str) == score_name
    ].copy()

    for cluster in display_order:
        x = sub.loc[
            sub["MalignantCluster"].astype(str) == cluster,
            "mean_module_score"
        ].to_numpy(dtype=float)

        y = sub.loc[
            sub["MalignantCluster"].astype(str) != cluster,
            "mean_module_score"
        ].to_numpy(dtype=float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) > 0 and len(y) > 0:
            _, p_greater = ranksums(x, y, alternative="greater")
            _, p_less = ranksums(x, y, alternative="less")
            _, p_two = ranksums(x, y, alternative="two-sided")
        else:
            p_greater = np.nan
            p_less = np.nan
            p_two = np.nan

        pval_records.append({
            "TIL_score": score_name,
            "MalignantCluster": cluster,
            "n_sample_cluster_values": len(x),
            "n_other_sample_cluster_values": len(y),
            "p_greater_than_other_clusters": p_greater,
            "p_less_than_other_clusters": p_less,
            "p_two_sided": p_two,
        })

pval_df = pd.DataFrame(pval_records)

# =============================
# Pairwise Wilcoxon rank-sum test:
# Malignant C5 vs each other malignant cluster
# one-sided
# =============================
pairwise_c5_records = []

for score_name in score_cols:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["TIL_score"].astype(str) == score_name
    ].copy()

    if REF_CLUSTER not in sub["MalignantCluster"].astype(str).unique():
        print(f"[Warning] {REF_CLUSTER} is not found. Skip pairwise C5 tests.")
        continue

    ref_vals = sub.loc[
        sub["MalignantCluster"].astype(str) == REF_CLUSTER,
        "mean_module_score"
    ].to_numpy(dtype=float)
    ref_vals = ref_vals[np.isfinite(ref_vals)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            p_one = np.nan
            star = "-"
            n_other = len(ref_vals)
        else:
            other_vals = sub.loc[
                sub["MalignantCluster"].astype(str) == cluster,
                "mean_module_score"
            ].to_numpy(dtype=float)
            other_vals = other_vals[np.isfinite(other_vals)]

            if len(ref_vals) > 0 and len(other_vals) > 0:
                stat, p_one = ranksums(
                    ref_vals,
                    other_vals,
                    alternative=TEST_ALTERNATIVE,
                )
            else:
                stat = np.nan
                p_one = np.nan

            star = p_to_star_with_ns(p_one)
            n_other = len(other_vals)

        pairwise_c5_records.append({
            "TIL_score": score_name,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "alternative": TEST_ALTERNATIVE,
            "n_reference": len(ref_vals),
            "n_other": n_other,
            "wilcoxon_rank_sum_statistic": stat,
            "p_one_sided": p_one,
            "star": star,
        })

pairwise_c5_pval_df = pd.DataFrame(pairwise_c5_records)

# =============================
# Save outputs
# =============================
score_wide.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score_wide.csv"),
    index=False,
)

score_long.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score_long.csv"),
    index=False,
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_all.csv"),
    index=False,
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_min{MIN_CELLS_PER_SAMPLE_CLUSTER}cells.csv"),
    index=False,
)

cluster_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_summary.csv"),
    index=False,
)

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_vs_others_ranksum_pvalues.csv"),
    index=False,
)

pairwise_c5_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_C5_vs_each_cluster_ranksum_pvalues_one_sided.csv"),
    index=False,
)

score_gene_count_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_gene_count_used_for_scoring.csv"),
    index=False,
)

# Save zscore_mean_global scores and metadata used for downstream checking.
score_wide.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_zscore_mean_global_celllevel_score_wide.csv"),
    index=False,
)

if "adata_choose_path" in globals():
    try:
        adata_choose.write_h5ad(adata_choose_path)
    except Exception as e:
        print(f"[Warning] Could not save adata_choose to adata_choose_path: {e}")

print("Gene counts used for TIL-UP module scoring:")
_display_df(score_gene_count_df)

print("Sample-level TIL-UP score table:")
_display_df(sample_score_df_plot.head())

print("Cluster summary:")
_display_df(cluster_summary_df)

print("Cluster vs others rank-sum p-values:")
_display_df(pval_df)

print("Pairwise C5 vs each cluster one-sided rank-sum p-values:")
_display_df(pairwise_c5_pval_df)

# =============================
# Plot: sample-level boxplot for TIL-UP score
# Cluster-colored boxes with one-sided C5 comparison annotations.
# Sample points, sample-size annotations and whisker caps are omitted.
# ==============================================================

# Reference-style edge and fill palettes
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


fig, ax = plt.subplots(figsize=(3.8, 3))

sns.boxplot(
    data=sample_score_df_plot,
    x="MalignantCluster",
    y="mean_module_score",
    order=display_order,
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    linewidth=0.8,
    width=0.65,
    saturation=1,
    ax=ax,
)

# =============================
# Apply robust fill and edge colors
# Also recolor median / whisker lines by cluster
# =============================
box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

for patch in box_patches:
    x_center = get_patch_x_center(patch)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
    cluster = display_order[nearest_idx]

    patch.set_facecolor(fill_palette[cluster])
    patch.set_edgecolor(edge_palette[cluster])
    patch.set_linewidth(0.8)

for line in ax.lines:
    xdata = np.asarray(line.get_xdata(), dtype=float)

    if xdata.size == 0 or np.any(~np.isfinite(xdata)):
        continue

    x_mid = np.mean(xdata)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
    cluster = display_order[nearest_idx]

    line.set_color(edge_palette[cluster])
    line.set_linewidth(0.8)

# =============================
# One-line annotation: vs C5
# =============================
if sample_score_df_plot.shape[0] > 0:
    y_min = sample_score_df_plot["mean_module_score"].min()
    y_max = sample_score_df_plot["mean_module_score"].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    y_annot = y_max + 0.08 * y_range

    pairwise_plot_df = pairwise_c5_pval_df.copy()
    pairwise_plot_df["other_cluster"] = pd.Categorical(
        pairwise_plot_df["other_cluster"],
        categories=display_order,
        ordered=True,
    )
    pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

    star_map = dict(zip(
        pairwise_plot_df["other_cluster"].astype(str),
        pairwise_plot_df["star"].astype(str),
    ))

    # label on the left of the annotation row
    ax.text(
        -0.72,
        y_annot,
        "vs C5:",
        ha="right",
        va="center",
        fontsize=6,
        color="black",
        clip_on=False,
    )

    # one star/ns/- per cluster, aligned with each box
    for i, cluster in enumerate(display_order):
        annot_text = star_map.get(cluster, "NA")
        ax.text(
            i,
            y_annot,
            annot_text,
            ha="center",
            va="center",
            fontsize=6,
            color="black",
            clip_on=False,
        )

    ax.set_ylim(
        y_min - 0.08 * y_range,
        y_max + 0.18 * y_range
    )

ax.set_title("TIL-up gene-set score", fontsize=7)
ax.set_xlabel("Malignant cluster")
ax.set_ylabel("TIL-UP module score\nsample-level mean z-scored genes")

ax.set_xticklabels(display_order, rotation=35, ha="right")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

In [ ]:
# ==============================================================
# Single-cell-level comparison: TIL-UP zscore_mean_global score by malignant cluster
# --------------------------------------------------------------
# This complements the previous sample-level cluster comparison.
# No points are drawn to keep the plot readable for many cells.
#
# Plot style:
#   - Reference-style cluster-specific fill + edge colors
#   - Box/median/whisker lines colored by cluster
#   - One-line C5 comparison annotation:
#       "vs C5: ** ns * **** - ** *"
#
# Statistics:
#   - Cell-level Wilcoxon rank-sum test
#   - REF_CLUSTER vs each other malignant cluster
# ==============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from scipy.stats import ranksums

OUT_PREFIX_SINGLECELL = "Malignant_TIL_UP_module_score_by_cluster_singlecell"

REF_CLUSTER = "Malignant C5"

# One-sided test direction:
#   "greater": test whether C5 > other cluster
#   "less":    test whether C5 < other cluster
#   "two-sided": two-sided comparison
TEST_ALTERNATIVE_SINGLECELL = "greater"

if "score_long" not in globals():
    raise NameError(
        "Run the TIL-UP zscore_mean_global score cell above before this single-cell comparison cell."
    )

if "display_order" not in globals():
    raise NameError(
        "display_order is not found. Please run the TIL-UP zscore_mean_global score cell above first."
    )

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)


# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _p_to_label(p):
    if not np.isfinite(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


# =============================
# Prepare single-cell dataframe
# =============================
single_cell_df = score_long.copy()

single_cell_df["module_score"] = pd.to_numeric(
    single_cell_df["module_score"],
    errors="coerce",
)

single_cell_df = single_cell_df.loc[
    np.isfinite(single_cell_df["module_score"].to_numpy(dtype=float))
].copy()

single_cell_df["MalignantCluster"] = pd.Categorical(
    single_cell_df["MalignantCluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_cell_df = single_cell_df.loc[
    single_cell_df["MalignantCluster"].notna()
].copy()

single_cell_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_celllevel_scores.csv"),
    index=False,
)


# =============================
# Single-cell summary
# =============================
single_summary_df = (
    single_cell_df
    .groupby(["TIL_score", "MalignantCluster"], observed=True)
    .agg(
        n_cells=("barcode", "count"),
        n_samples=("sample_id", "nunique"),
        mean_module_score=("module_score", "mean"),
        median_module_score=("module_score", "median"),
        std_module_score=("module_score", "std"),
    )
    .reset_index()
)


# =============================
# Cell-level C5 vs each cluster test
# =============================
single_pval_records = []

for score_name in single_cell_df["TIL_score"].astype(str).unique():
    sub_score = single_cell_df.loc[
        single_cell_df["TIL_score"].astype(str) == score_name
    ].copy()

    ref_values = sub_score.loc[
        sub_score["MalignantCluster"].astype(str) == REF_CLUSTER,
        "module_score",
    ].to_numpy(dtype=float)

    ref_values = ref_values[np.isfinite(ref_values)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            pval = np.nan
            star = "-"
            p_label = "-"
            n_other = len(ref_values)
        else:
            other_values = sub_score.loc[
                sub_score["MalignantCluster"].astype(str) == cluster,
                "module_score",
            ].to_numpy(dtype=float)

            other_values = other_values[np.isfinite(other_values)]

            if len(ref_values) > 0 and len(other_values) > 0:
                stat, pval = ranksums(
                    ref_values,
                    other_values,
                    alternative=TEST_ALTERNATIVE_SINGLECELL,
                )
            else:
                stat, pval = np.nan, np.nan

            star = p_to_star_with_ns(pval)
            p_label = _p_to_label(pval)
            n_other = len(other_values)

        single_pval_records.append({
            "TIL_score": score_name,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "test": "cell-level Wilcoxon rank-sum",
            "alternative": TEST_ALTERNATIVE_SINGLECELL,
            "n_cells_ref": len(ref_values),
            "n_cells_other": n_other,
            "statistic": stat,
            "pvalue": pval,
            "pvalue_label": p_label,
            "star": star,
        })

single_pval_df = pd.DataFrame(single_pval_records)

single_pval_df["other_cluster"] = pd.Categorical(
    single_pval_df["other_cluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_pval_df = single_pval_df.sort_values(
    ["TIL_score", "other_cluster"]
).reset_index(drop=True)


# =============================
# Save statistics
# =============================
single_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_summary.csv"),
    index=False,
)

single_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_C5_vs_each_cluster_pvalues.csv"),
    index=False,
)


# =============================
# Reference-style color palettes
# Same style as sample-level TIL-UP plot
# =============================
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}


# =============================
# Plot: single-cell boxplot without points
# =============================
plt.close("all")

fig, ax = plt.subplots(
    figsize=(max(4.0, 0.48 * len(display_order)), 3.0)
)

sns.boxplot(
    data=single_cell_df,
    x="MalignantCluster",
    y="module_score",
    order=display_order,
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    width=0.65,
    linewidth=0.8,
    saturation=1,
    ax=ax,
)

# =============================
# Apply fill and edge colors robustly
# Also recolor median / whisker lines by cluster
# =============================
box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

for patch in box_patches:
    x_center = get_patch_x_center(patch)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
    cluster = display_order[nearest_idx]

    patch.set_facecolor(fill_palette[cluster])
    patch.set_edgecolor(edge_palette[cluster])
    patch.set_linewidth(0.8)

for line in ax.lines:
    xdata = np.asarray(line.get_xdata(), dtype=float)

    if xdata.size == 0 or np.any(~np.isfinite(xdata)):
        continue

    x_mid = np.mean(xdata)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
    cluster = display_order[nearest_idx]

    line.set_color(edge_palette[cluster])
    line.set_linewidth(0.8)


# =============================
# One-line annotation: vs C5
# =============================
if single_cell_df.shape[0] > 0:
    y_min = single_cell_df["module_score"].min()
    y_max = single_cell_df["module_score"].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    y_annot = y_max + 0.08 * y_range

    # If there is only one TIL_score, this is straightforward.
    # If multiple scores exist, use the first one shown in this plot.
    plot_score_name = single_cell_df["TIL_score"].astype(str).unique()[0]

    pairwise_plot_df = single_pval_df.loc[
        single_pval_df["TIL_score"].astype(str) == plot_score_name
    ].copy()

    pairwise_plot_df["other_cluster"] = pd.Categorical(
        pairwise_plot_df["other_cluster"].astype(str),
        categories=display_order,
        ordered=True,
    )

    pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

    star_map = dict(zip(
        pairwise_plot_df["other_cluster"].astype(str),
        pairwise_plot_df["star"].astype(str),
    ))

    p_label_map = dict(zip(
        pairwise_plot_df["other_cluster"].astype(str),
        pairwise_plot_df["pvalue_label"].astype(str),
    ))

    # label on the left of the annotation row
    ax.text(
        -0.72,
        y_annot,
        "vs C5:",
        ha="right",
        va="center",
        fontsize=6,
        color="black",
        clip_on=False,
    )

    # one star/ns/- per cluster, aligned with each box
    for i, cluster in enumerate(display_order):
        annot_text = star_map.get(cluster, "NA")

        ax.text(
            i,
            y_annot,
            annot_text,
            ha="center",
            va="center",
            fontsize=6,
            color="black",
            clip_on=False,
        )

    ax.set_ylim(
        y_min - 0.08 * y_range,
        y_max + 0.18 * y_range,
    )


# =============================
# Axis and style
# =============================
ax.set_title("Single-cell TIL-up gene-set score", fontsize=7)
ax.set_xlabel("Malignant cluster")
ax.set_ylabel("TIL-UP zscore_mean_global score")

ax.set_xticklabels(display_order, rotation=35, ha="right")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

for ext in ["png", "pdf"]:
    fig.savefig(
        os.path.join(
            out_dir,
            f"{OUT_PREFIX_SINGLECELL}_boxplot_no_points_with_C5_test.{ext}"
        ),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

plt.show()
plt.close()


# =============================
# Display outputs
# =============================
print("Single-cell TIL-UP comparison summary:")
_display_df(single_summary_df)

print("Cell-level C5 vs each malignant cluster rank-sum test:")
_display_df(single_pval_df)

In [ ]:
# ==============================================================
# C5-enriched KEGG pathway module scores across malignant clusters
# --------------------------------------------------------------
# Pathways:
#   1. HIF-1 signaling pathway
#   2. PD-L1 expression and PD-1 checkpoint pathway in cancer
#   3. PI3K-Akt signaling pathway
#   4. ECM-receptor interaction
#
# For each pathway:
#   - Use genes overlapped between KEGG reference and current HGSOC dataset
#   - average gene-wise z-scores using malignant reference cells pooled across slices
#   - aggregate at sample_id × malignant cluster level
#   - plot sample-level boxplots across malignant clusters
# ==============================================================

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import PathPatch
from scipy import sparse
from scipy.stats import ranksums

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

KEGG_PATHWAYS_TO_SCORE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

OUT_PREFIX = "Malignant_C5_four_KEGG_pathway_module_score_by_cluster"

MIN_CELLS_PER_SAMPLE_CLUSTER = 10

# Reference cluster for pairwise annotation
REF_CLUSTER = "Malignant C5"

# One-sided Wilcoxon rank-sum test direction:
#   "greater": test whether C5 > other cluster
#   "less":    test whether C5 < other cluster
TEST_ALTERNATIVE = "greater"

out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene)
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


# =============================
# Check malignant cluster column
# =============================
if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Load pathway-gene table
# Use the in-memory reference table or its saved CSV from the KEGG export block.
# =============================
if "c5_kegg_reference_gene_df" in globals():
    pathway_gene_df = c5_kegg_reference_gene_df.copy()
else:
    candidate_path = (
        Path(out_dir)
        / "TCGA_OV_KEGG_gene_signature"
        / "C5_four_KEGG_pathway_reference_genes_long.csv"
    )
    if not input_path(candidate_path).exists():
        raise FileNotFoundError(
            "Cannot find c5_kegg_reference_gene_df in memory or saved pathway-gene CSV:\n"
            f"{candidate_path}\n"
            "Please run the KEGG reference gene export cell first."
        )
    pathway_gene_df = pd.read_csv(input_path(candidate_path))

required_cols = {"Requested_Pathway", "Gene"}
missing_cols = required_cols - set(pathway_gene_df.columns)
if len(missing_cols) > 0:
    raise KeyError(
        f"pathway_gene_df is missing required columns: {missing_cols}. "
        f"Available columns: {list(pathway_gene_df.columns)}"
    )

pathway_gene_df = pathway_gene_df.loc[
    pathway_gene_df["Requested_Pathway"].isin(KEGG_PATHWAYS_TO_SCORE),
    ["Requested_Pathway", "Gene"]
].drop_duplicates().copy()

if pathway_gene_df.empty:
    raise ValueError("No pathway genes found for the selected four KEGG pathways.")

print("Pathway gene counts before expression extraction:")
display(
    pathway_gene_df
    .groupby("Requested_Pathway", as_index=False)
    .agg(n_genes=("Gene", "nunique"))
)

# =============================
# Malignant cluster labels
# =============================
cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()

cluster_order = sorted(
    np.unique(cluster_clean),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

display_order = [f"Malignant C{x}" for x in cluster_order]
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

# =============================
# Resolve expression source
# Prefer adata_choose; fallback to processed.adata_all if needed
# =============================
all_pathway_genes = sorted(pathway_gene_df["Gene"].astype(str).unique())

resolved_choose, missing_choose = _resolve_gene_names(adata_choose, all_pathway_genes)

adata_all = None
malignant_full_idx = None

# Use adata_choose if it contains all pathway genes
if len(missing_choose) == 0:
    expr_source = "adata_choose"
    expr_adata = adata_choose
    expr_rows = np.arange(adata_choose.n_obs)
    resolved_genes = resolved_choose

# Otherwise use processed.adata_all
else:
    print(
        f"[Info] {len(missing_choose)} pathway genes are missing from adata_choose. "
        "Trying processed.adata_all..."
    )

    if "processed" not in globals() or not hasattr(processed, "adata_all"):
        raise ValueError(
            "Some pathway genes are missing from adata_choose, and processed.adata_all is not available."
        )

    adata_all = processed.adata_all
    resolved_all, missing_all = _resolve_gene_names(adata_all, all_pathway_genes)

    if len(resolved_all) == 0:
        raise ValueError("None of the selected KEGG pathway genes are found in processed.adata_all.")

    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

    adata_all_index = pd.Index(adata_all.obs_names.astype(str))
    malignant_full_idx = adata_all_index.get_indexer(malignant_barcodes)

    if np.any(malignant_full_idx < 0) and "barcode" in adata_all.obs.columns:
        all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
        barcode_to_idx = {}
        for i, b in enumerate(all_barcodes):
            if b not in barcode_to_idx:
                barcode_to_idx[b] = i

        malignant_full_idx = np.array(
            [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
            dtype=int
        )

    missing_match = malignant_full_idx < 0
    if np.any(missing_match):
        raise ValueError(
            f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
            "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names."
        )

    expr_source = "processed.adata_all"
    expr_adata = adata_all
    expr_rows = malignant_full_idx
    resolved_genes = resolved_all

print(f"Expression source: {expr_source}")
print(f"Number of pathway genes available for scoring: {len(resolved_genes)}")

# Keep only genes actually available in expression matrix
available_genes = sorted(resolved_genes.keys())

pathway_gene_df_available = pathway_gene_df.loc[
    pathway_gene_df["Gene"].astype(str).isin(available_genes)
].drop_duplicates().copy()

if pathway_gene_df_available.empty:
    raise ValueError("No selected pathway genes are available in the expression matrix.")

pathway_gene_count_df = (
    pathway_gene_df_available
    .groupby("Requested_Pathway", as_index=False)
    .agg(n_genes_used=("Gene", "nunique"))
)

print("Pathway gene counts used for module scoring:")
display(pathway_gene_count_df)

# =============================
# Compute pathway module scores with zscore_mean_global
# =============================
# Estimate gene means and sample standard deviations (ddof=1) from the
# specified reference rows pooled across slices, then average valid gene z-scores.
# Genes with zero or undefined standard deviation are omitted by nanmean.
pathway_score_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    genes_cur = (
        pathway_gene_df_available
        .loc[pathway_gene_df_available["Requested_Pathway"] == pathway, "Gene"]
        .astype(str)
        .unique()
        .tolist()
    )

    score_cur, genes_used_cur, genes_missing_cur = _spidernet_zscore_mean_global_scores_for_rows(
        expr_adata=expr_adata,
        row_idx=expr_rows,
        reference_row_idx=expr_rows,
        genes=genes_cur,
        sample_col=_spidernet_find_sample_col(expr_adata),
        score_name=f"__KEGG_{re.sub(r'[^A-Za-z0-9]+', '_', pathway)}_zscore_mean_global__",
        ddof=1,
    )

    if len(genes_used_cur) == 0:
        print(f"[Warning] No available genes for pathway: {pathway}. Skipping.")
        continue

    tmp = pd.DataFrame({
        "barcode": (
            adata_choose.obs["barcode"].astype(str).to_numpy()
            if "barcode" in adata_choose.obs.columns
            else adata_choose.obs_names.astype(str).to_numpy()
        ),
        "MI_louvain": cluster_clean,
        "MalignantCluster": cluster_display,
        "Pathway": pathway,
        "score_method": MODULE_SCORE_METHOD,
        "n_genes_used": len(genes_used_cur),
        "genes_used": ", ".join(genes_used_cur),
        "n_missing_genes": len(genes_missing_cur),
        "missing_genes": ", ".join(genes_missing_cur),
        "pathway_module_score": score_cur,
    })

    pathway_score_records.append(tmp)

    # also save each pathway score into adata_choose.obs
    safe_name = (
        pathway
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("__", "_")
    )
    adata_choose.obs[f"KEGG_{safe_name}_module_score"] = score_cur

score_df = pd.concat(pathway_score_records, axis=0, ignore_index=True)

score_df["MalignantCluster"] = pd.Categorical(
    score_df["MalignantCluster"],
    categories=display_order,
    ordered=True
)

score_df["Pathway"] = pd.Categorical(
    score_df["Pathway"],
    categories=KEGG_PATHWAYS_TO_SCORE,
    ordered=True
)

# =============================
# Add sample information
# =============================
sample_col_candidates = [
    "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
    "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
]

sample_col_choose = next(
    (col for col in sample_col_candidates if col in adata_choose.obs.columns),
    None
)

if sample_col_choose is not None:
    sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
    print(f"Using sample column from adata_choose: {sample_col_choose}")
else:
    # fallback to processed.adata_all
    if adata_all is None and "processed" in globals() and hasattr(processed, "adata_all"):
        adata_all = processed.adata_all

        if "barcode" in adata_choose.obs.columns:
            malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
        else:
            malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

        adata_all_index = pd.Index(adata_all.obs_names.astype(str))
        malignant_full_idx_tmp = adata_all_index.get_indexer(malignant_barcodes)

        if np.any(malignant_full_idx_tmp < 0) and "barcode" in adata_all.obs.columns:
            all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
            barcode_to_idx = {}
            for i, b in enumerate(all_barcodes):
                if b not in barcode_to_idx:
                    barcode_to_idx[b] = i

            malignant_full_idx_tmp = np.array(
                [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
                dtype=int
            )

        if not np.any(malignant_full_idx_tmp < 0):
            malignant_full_idx = malignant_full_idx_tmp

    sample_values = None

    if adata_all is not None and malignant_full_idx is not None:
        sample_col_all = next(
            (col for col in sample_col_candidates if col in adata_all.obs.columns),
            None
        )

        if sample_col_all is not None:
            sample_values = adata_all.obs.iloc[malignant_full_idx][sample_col_all].astype(str).to_numpy()
            print(f"Using sample column from processed.adata_all: {sample_col_all}")

    if sample_values is None:
        print(
            "[Warning] No sample column found. "
            "All malignant cells will be treated as one pseudo-sample."
        )
        sample_values = np.array(["__all_cells__"] * adata_choose.n_obs)

# map sample values to long score_df
barcode_to_sample = pd.Series(
    sample_values,
    index=(
        adata_choose.obs["barcode"].astype(str).to_numpy()
        if "barcode" in adata_choose.obs.columns
        else adata_choose.obs_names.astype(str).to_numpy()
    )
)

score_df["sample_id"] = score_df["barcode"].map(barcode_to_sample).astype(str)

# =============================
# Sample-level aggregation
# =============================
sample_score_df = (
    score_df
    .groupby(["sample_id", "MalignantCluster", "Pathway"], observed=True)
    .agg(
        mean_pathway_module_score=("pathway_module_score", "mean"),
        median_pathway_module_score=("pathway_module_score", "median"),
        n_cells=("barcode", "count"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_CLUSTER
].copy()

sample_score_df_plot["MalignantCluster"] = pd.Categorical(
    sample_score_df_plot["MalignantCluster"],
    categories=display_order,
    ordered=True
)

sample_score_df_plot["Pathway"] = pd.Categorical(
    sample_score_df_plot["Pathway"],
    categories=KEGG_PATHWAYS_TO_SCORE,
    ordered=True
)

# =============================
# Summary + cluster vs others p-values
# =============================
cluster_summary_df = (
    sample_score_df_plot
    .groupby(["Pathway", "MalignantCluster"], observed=True)
    .agg(
        n_samples=("sample_id", "nunique"),
        mean_sample_level_score=("mean_pathway_module_score", "mean"),
        median_sample_level_score=("mean_pathway_module_score", "median"),
        std_sample_level_score=("mean_pathway_module_score", "std"),
        total_cells=("n_cells", "sum"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)

pval_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["Pathway"].astype(str) == pathway
    ].copy()

    for cluster in display_order:
        x = sub.loc[
            sub["MalignantCluster"].astype(str) == cluster,
            "mean_pathway_module_score"
        ].to_numpy(dtype=float)

        y = sub.loc[
            sub["MalignantCluster"].astype(str) != cluster,
            "mean_pathway_module_score"
        ].to_numpy(dtype=float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) > 0 and len(y) > 0:
            _, p_greater = ranksums(x, y, alternative="greater")
            _, p_less = ranksums(x, y, alternative="less")
            _, p_two = ranksums(x, y, alternative="two-sided")
        else:
            p_greater = np.nan
            p_less = np.nan
            p_two = np.nan

        pval_records.append({
            "Pathway": pathway,
            "MalignantCluster": cluster,
            "n_sample_cluster_values": len(x),
            "n_other_sample_cluster_values": len(y),
            "p_greater_than_other_clusters": p_greater,
            "p_less_than_other_clusters": p_less,
            "p_two_sided": p_two,
        })

pval_df = pd.DataFrame(pval_records)

# =============================
# Pairwise Wilcoxon rank-sum test:
# Malignant C5 vs each other malignant cluster
# one-sided, separately for each pathway
# =============================
pairwise_c5_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["Pathway"].astype(str) == pathway
    ].copy()

    if REF_CLUSTER not in sub["MalignantCluster"].astype(str).unique():
        print(f"[Warning] {REF_CLUSTER} is not found for {pathway}. Skip pairwise C5 tests.")
        continue

    ref_vals = sub.loc[
        sub["MalignantCluster"].astype(str) == REF_CLUSTER,
        "mean_pathway_module_score"
    ].to_numpy(dtype=float)
    ref_vals = ref_vals[np.isfinite(ref_vals)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            p_one = np.nan
            star = "-"
            n_other = len(ref_vals)
        else:
            other_vals = sub.loc[
                sub["MalignantCluster"].astype(str) == cluster,
                "mean_pathway_module_score"
            ].to_numpy(dtype=float)
            other_vals = other_vals[np.isfinite(other_vals)]

            if len(ref_vals) > 0 and len(other_vals) > 0:
                stat, p_one = ranksums(
                    ref_vals,
                    other_vals,
                    alternative=TEST_ALTERNATIVE,
                )
            else:
                stat = np.nan
                p_one = np.nan

            star = p_to_star_with_ns(p_one)
            n_other = len(other_vals)

        pairwise_c5_records.append({
            "Pathway": pathway,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "alternative": TEST_ALTERNATIVE,
            "n_reference": len(ref_vals),
            "n_other": n_other,
            "wilcoxon_rank_sum_statistic": stat,
            "p_one_sided": p_one,
            "star": star,
        })

pairwise_c5_pval_df = pd.DataFrame(pairwise_c5_records)

# =============================
# Save outputs
# =============================
score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score.csv"),
    index=False
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_all.csv"),
    index=False
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_min{MIN_CELLS_PER_SAMPLE_CLUSTER}cells.csv"),
    index=False
)

cluster_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_summary.csv"),
    index=False
)

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_vs_others_ranksum_pvalues.csv"),
    index=False
)

pairwise_c5_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_C5_vs_each_cluster_ranksum_pvalues_one_sided.csv"),
    index=False
)

# save zscore_mean_global cell-level scores used for scoring
score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_zscore_mean_global_celllevel_scores_long.csv"),
    index=False
)

if "adata_choose_path" in globals():
    adata_choose.write_h5ad(adata_choose_path)

print("Sample-level pathway score table:")
display(sample_score_df_plot.head())

print("Cluster summary:")
display(cluster_summary_df)

print("Cluster vs others rank-sum p-values:")
display(pval_df)

print("Pairwise C5 vs each cluster one-sided rank-sum p-values:")
display(pairwise_c5_pval_df)

# =============================
# Plot: sample-level boxplot faceted by pathway
# Compact pathway facets with cluster-colored boxes and one-sided C5
# comparison annotations. Sample points and whisker caps are omitted.
# ==============================================================

# Reference-style edge and fill palettes
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}

g = sns.catplot(
    data=sample_score_df_plot,
    x="MalignantCluster",
    y="mean_pathway_module_score",
    col="Pathway",
    col_wrap=2,
    order=display_order,
    kind="box",
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    linewidth=0.8,
    width=0.65,
    height=1.3,
    aspect=1.35,
    sharey=False,
    saturation=1,
)

# =============================
# Apply robust fill and edge colors
# Also recolor median / whisker lines by cluster
# =============================
for ax in g.axes.flat:
    box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

    for patch in box_patches:
        x_center = get_patch_x_center(patch)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
        cluster = display_order[nearest_idx]

        patch.set_facecolor(fill_palette[cluster])
        patch.set_edgecolor(edge_palette[cluster])
        patch.set_linewidth(0.8)

    for line in ax.lines:
        xdata = np.asarray(line.get_xdata(), dtype=float)

        if xdata.size == 0 or np.any(~np.isfinite(xdata)):
            continue

        x_mid = np.mean(xdata)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
        cluster = display_order[nearest_idx]

        line.set_color(edge_palette[cluster])
        line.set_linewidth(0.8)


# =============================
# Final formatting + one-line annotation for each pathway
# =============================
short_title_map = {
    "HIF-1 signaling pathway": "HIF-1 signaling",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer": "PD-1/PD-L1 checkpoint",
    "PI3K-Akt signaling pathway": "PI3K-Akt signaling",
    "ECM-receptor interaction": "ECM-receptor interaction",
}

for ax, pathway in zip(g.axes.flat, KEGG_PATHWAYS_TO_SCORE):
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["Pathway"].astype(str) == pathway
    ].copy()

    if sub.shape[0] > 0:
        y_min = sub["mean_pathway_module_score"].min()
        y_max = sub["mean_pathway_module_score"].max()
        y_range = y_max - y_min if y_max > y_min else 1.0

        y_annot = y_max + 0.08 * y_range

        pairwise_plot_df = pairwise_c5_pval_df.loc[
            pairwise_c5_pval_df["Pathway"].astype(str) == pathway
        ].copy()

        pairwise_plot_df["other_cluster"] = pd.Categorical(
            pairwise_plot_df["other_cluster"],
            categories=display_order,
            ordered=True,
        )
        pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

        star_map = dict(zip(
            pairwise_plot_df["other_cluster"].astype(str),
            pairwise_plot_df["star"].astype(str),
        ))

        # label on the left of the annotation row
        ax.text(
            -0.72,
            y_annot,
            "vs C5:",
            ha="right",
            va="center",
            fontsize=5.5,
            color="black",
            clip_on=False,
        )

        # one star/ns/- per cluster, aligned with each box
        for i, cluster in enumerate(display_order):
            annot_text = star_map.get(cluster, "NA")
            ax.text(
                i,
                y_annot,
                annot_text,
                ha="center",
                va="center",
                fontsize=5.5,
                color="black",
                clip_on=False,
            )

        ax.set_ylim(
            y_min - 0.08 * y_range,
            y_max + 0.18 * y_range
        )

    ax.set_title(short_title_map.get(pathway, pathway), fontsize=7)
    ax.set_xlabel("")
    ax.set_ylabel("Pathway module score\nsample-level mean z-scored genes")

    ax.set_xticklabels(display_order, rotation=35, ha="right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    "C5-enriched KEGG pathway module scores across malignant clusters",
    y=1.05,
    fontsize=8
)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()

In [ ]:
# ==============================================================
# Single-cell-level comparison: KEGG zscore_mean_global pathway scores by malignant cluster
# --------------------------------------------------------------
# This complements the previous sample-level cluster comparison.
# No points are drawn to keep the plot readable for many cells.
#
# Plot style:
#   - Reference-style cluster-specific fill + edge colors
#   - Box/median/whisker lines colored by cluster
#   - One-line C5 comparison annotation for each pathway:
#       "vs C5: ** ns * **** - ** *"
#
# Statistics:
#   - Cell-level Wilcoxon rank-sum test
#   - REF_CLUSTER vs each other malignant cluster
#   - One-sided by default: C5 > other cluster
# ==============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from scipy.stats import ranksums

OUT_PREFIX_SINGLECELL = "Malignant_C5_four_KEGG_pathway_module_score_by_cluster_singlecell"

REF_CLUSTER = "Malignant C5"

# One-sided test direction:
#   "greater": test whether C5 > other cluster
#   "less":    test whether C5 < other cluster
#   "two-sided": two-sided comparison
TEST_ALTERNATIVE_SINGLECELL = "greater"

if "score_df" not in globals() or "pathway_module_score" not in score_df.columns:
    raise NameError(
        "Run the KEGG zscore_mean_global score cell above before this single-cell comparison cell."
    )

if "display_order" not in globals():
    raise NameError(
        "display_order is not found. Please run the KEGG zscore_mean_global score cell above first."
    )

if "KEGG_PATHWAYS_TO_SCORE" not in globals():
    raise NameError(
        "KEGG_PATHWAYS_TO_SCORE is not found. Please run the KEGG score cell above first."
    )

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)


# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _p_to_label(p):
    if not np.isfinite(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


# =============================
# Prepare single-cell dataframe
# =============================
single_cell_df = score_df.copy()

single_cell_df["pathway_module_score"] = pd.to_numeric(
    single_cell_df["pathway_module_score"],
    errors="coerce",
)

single_cell_df = single_cell_df.loc[
    np.isfinite(single_cell_df["pathway_module_score"].to_numpy(dtype=float))
].copy()

single_cell_df["MalignantCluster"] = pd.Categorical(
    single_cell_df["MalignantCluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_cell_df["Pathway"] = pd.Categorical(
    single_cell_df["Pathway"].astype(str),
    categories=KEGG_PATHWAYS_TO_SCORE,
    ordered=True,
)

single_cell_df = single_cell_df.loc[
    single_cell_df["MalignantCluster"].notna()
    & single_cell_df["Pathway"].notna()
].copy()

single_cell_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_celllevel_scores.csv"),
    index=False,
)


# =============================
# Single-cell summary
# =============================
single_summary_df = (
    single_cell_df
    .groupby(["Pathway", "MalignantCluster"], observed=True)
    .agg(
        n_cells=("barcode", "count"),
        n_samples=("sample_id", "nunique"),
        mean_pathway_module_score=("pathway_module_score", "mean"),
        median_pathway_module_score=("pathway_module_score", "median"),
        std_pathway_module_score=("pathway_module_score", "std"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)


# =============================
# Cell-level C5 vs each cluster test
# =============================
single_pval_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    sub_pathway = single_cell_df.loc[
        single_cell_df["Pathway"].astype(str) == pathway
    ].copy()

    ref_values = sub_pathway.loc[
        sub_pathway["MalignantCluster"].astype(str) == REF_CLUSTER,
        "pathway_module_score",
    ].to_numpy(dtype=float)

    ref_values = ref_values[np.isfinite(ref_values)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            pval = np.nan
            star = "-"
            p_label = "-"
            n_other = len(ref_values)
        else:
            other_values = sub_pathway.loc[
                sub_pathway["MalignantCluster"].astype(str) == cluster,
                "pathway_module_score",
            ].to_numpy(dtype=float)

            other_values = other_values[np.isfinite(other_values)]

            if len(ref_values) > 0 and len(other_values) > 0:
                stat, pval = ranksums(
                    ref_values,
                    other_values,
                    alternative=TEST_ALTERNATIVE_SINGLECELL,
                )
            else:
                stat, pval = np.nan, np.nan

            star = p_to_star_with_ns(pval)
            p_label = _p_to_label(pval)
            n_other = len(other_values)

        single_pval_records.append({
            "Pathway": pathway,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "test": "cell-level Wilcoxon rank-sum",
            "alternative": TEST_ALTERNATIVE_SINGLECELL,
            "n_cells_ref": len(ref_values),
            "n_cells_other": n_other,
            "statistic": stat,
            "pvalue": pval,
            "pvalue_label": p_label,
            "star": star,
        })

single_pval_df = pd.DataFrame(single_pval_records)

single_pval_df["Pathway"] = pd.Categorical(
    single_pval_df["Pathway"].astype(str),
    categories=KEGG_PATHWAYS_TO_SCORE,
    ordered=True,
)

single_pval_df["other_cluster"] = pd.Categorical(
    single_pval_df["other_cluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_pval_df = single_pval_df.sort_values(
    ["Pathway", "other_cluster"]
).reset_index(drop=True)


# =============================
# Save statistics
# =============================
single_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_summary.csv"),
    index=False,
)

single_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_C5_vs_each_cluster_pvalues.csv"),
    index=False,
)


# =============================
# Reference-style color palettes
# Same style as sample-level KEGG plot
# =============================
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}


# =============================
# Plot: single-cell boxplot faceted by pathway
# =============================
plt.close("all")

g = sns.catplot(
    data=single_cell_df,
    x="MalignantCluster",
    y="pathway_module_score",
    col="Pathway",
    col_wrap=2,
    col_order=KEGG_PATHWAYS_TO_SCORE,
    order=display_order,
    kind="box",
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    linewidth=0.8,
    width=0.65,
    height=2.2,
    aspect=1.35,
    sharey=False,
    saturation=1,
)


# =============================
# Apply robust fill and edge colors
# Also recolor median / whisker lines by cluster
# =============================
for ax in g.axes.flat:
    box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

    for patch in box_patches:
        x_center = get_patch_x_center(patch)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
        cluster = display_order[nearest_idx]

        patch.set_facecolor(fill_palette[cluster])
        patch.set_edgecolor(edge_palette[cluster])
        patch.set_linewidth(0.8)

    for line in ax.lines:
        xdata = np.asarray(line.get_xdata(), dtype=float)

        if xdata.size == 0 or np.any(~np.isfinite(xdata)):
            continue

        x_mid = np.mean(xdata)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
        cluster = display_order[nearest_idx]

        line.set_color(edge_palette[cluster])
        line.set_linewidth(0.8)


# =============================
# Final formatting + one-line annotation for each pathway
# =============================
short_title_map = {
    "HIF-1 signaling pathway": "HIF-1 signaling",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer": "PD-1/PD-L1 checkpoint",
    "PI3K-Akt signaling pathway": "PI3K-Akt signaling",
    "ECM-receptor interaction": "ECM-receptor interaction",
}

for ax, pathway in zip(g.axes.flat, KEGG_PATHWAYS_TO_SCORE):
    sub = single_cell_df.loc[
        single_cell_df["Pathway"].astype(str) == pathway
    ].copy()

    if sub.shape[0] > 0:
        y_min = sub["pathway_module_score"].min()
        y_max = sub["pathway_module_score"].max()
        y_range = y_max - y_min if y_max > y_min else 1.0

        y_annot = y_max + 0.08 * y_range

        pairwise_plot_df = single_pval_df.loc[
            single_pval_df["Pathway"].astype(str) == pathway
        ].copy()

        pairwise_plot_df["other_cluster"] = pd.Categorical(
            pairwise_plot_df["other_cluster"].astype(str),
            categories=display_order,
            ordered=True,
        )

        pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

        star_map = dict(zip(
            pairwise_plot_df["other_cluster"].astype(str),
            pairwise_plot_df["star"].astype(str),
        ))

        # label on the left of the annotation row
        ax.text(
            -0.72,
            y_annot,
            "vs C5:",
            ha="right",
            va="center",
            fontsize=5.5,
            color="black",
            clip_on=False,
        )

        # one star/ns/- per cluster, aligned with each box
        for i, cluster in enumerate(display_order):
            annot_text = star_map.get(cluster, "NA")

            ax.text(
                i,
                y_annot,
                annot_text,
                ha="center",
                va="center",
                fontsize=5.5,
                color="black",
                clip_on=False,
            )

        ax.set_ylim(
            y_min - 0.08 * y_range,
            y_max + 0.18 * y_range,
        )

    ax.set_title(short_title_map.get(pathway, pathway), fontsize=7)
    ax.set_xlabel("")
    ax.set_ylabel("Pathway module score\nsingle-cell zscore_mean_global")

    ax.set_xticklabels(display_order, rotation=35, ha="right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    "Single-cell C5-enriched KEGG pathway scores across malignant clusters",
    y=1.03,
    fontsize=8,
)

plt.tight_layout()

for ext in ["png", "pdf"]:
    g.fig.savefig(
        os.path.join(
            out_dir,
            f"{OUT_PREFIX_SINGLECELL}_boxplot_no_points_with_C5_test.{ext}"
        ),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

plt.show()
plt.close()


# =============================
# Display outputs
# =============================
print("Single-cell KEGG comparison summary:")
_display_df(single_summary_df)

print("Cell-level C5 vs each malignant cluster rank-sum test:")
_display_df(single_pval_df)

In [ ]:
# Uses CAF markers from the current namespace when available, otherwise the
# explicit fallback marker sets below; the selected genes are exported.

# ==============================================================
# Overall CAF score in Fibroblasts stratified by neighboring malignant C5
# --------------------------------------------------------------
# For each Fibroblast cell:
#   Group 1: spatial 10-NN contains >=1 malignant C5 cell
#   Group 2: spatial 10-NN contains no malignant C5 cell
#
# CAF score:
#   Mean gene-wise z-score over CAF markers, using fibroblasts pooled across slices.
#
# Visualization:
#   sample-level paired CAF score comparison
# ==============================================================

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from scipy.stats import wilcoxon, ranksums

# =============================
# Settings
# =============================
K_NEIGH = 10

CELL_SUBTYPE_COL = "cell.subtypes"
FIBRO_LABEL = "Fibroblast"

CLUSTER_COL = "MI_louvain"
C5_LABEL = "5"

GROUP_WITH_C5 = "Fibroblast with neighboring malignant C5"
GROUP_WITHOUT_C5 = "Fibroblast without neighboring malignant C5"
GROUP_ORDER = [GROUP_WITHOUT_C5, GROUP_WITH_C5]

MIN_CELLS_PER_SAMPLE_GROUP = 10

CAF_SCORE_COL = "CAF_score_all_markers"
OUT_PREFIX = "Fibroblast_overall_CAF_score_by_neighbor_malignantC5_spatial10NN"

if "run_dirs" in globals() and isinstance(run_dirs, dict) and "run_dir" in run_dirs:
    out_dir = run_dirs["run_dir"]
else:
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _get_spatial_matrix(adata):
    if "spatial" in adata.obsm:
        return np.asarray(adata.obsm["spatial"], dtype=float)

    spatial_col_candidates = [
        ("x", "y"),
        ("X", "Y"),
        ("center_x", "center_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("x_centroid", "y_centroid"),
        ("X_centroid", "Y_centroid"),
    ]

    spatial_cols = next(
        (
            cols for cols in spatial_col_candidates
            if cols[0] in adata.obs.columns and cols[1] in adata.obs.columns
        ),
        None
    )

    if spatial_cols is None:
        raise KeyError(
            "Cannot find spatial coordinates. Expected adata.obsm['spatial'] "
            "or x/y-like columns in adata.obs."
        )

    return adata.obs.loc[:, list(spatial_cols)].to_numpy(dtype=float)


def _flatten_gene_list(x):
    if x is None:
        return []

    if isinstance(x, str):
        x = x.replace(";", ",").replace("|", ",")
        return [g.strip() for g in x.split(",") if g.strip()]

    if isinstance(x, dict):
        for key in ["genes", "Genes", "gene", "Gene", "markers", "Markers", "marker", "Marker"]:
            if key in x:
                return _flatten_gene_list(x[key])

        out = []
        for v in x.values():
            out.extend(_flatten_gene_list(v))
        return out

    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series, pd.Index)):
        out = []
        for v in list(x):
            out.extend(_flatten_gene_list(v))
        return out

    x = str(x).strip()
    return [x] if x else []


def _extract_caf_modules_from_notebook():
    candidate_var_names = [
        "caf_modules",
        "CAF_modules",
        "caf_marker_dict",
        "CAF_marker_dict",
        "caf_markers",
        "CAF_markers",
        "caf_marker_genes",
        "CAF_marker_genes",
        "CAF_gene_sets",
        "caf_gene_sets",
        "caf_signature_dict",
        "CAF_signature_dict",
    ]

    for var_name in candidate_var_names:
        if var_name not in globals():
            continue

        obj = globals()[var_name]

        if isinstance(obj, dict):
            parsed = {}

            for module, genes in obj.items():
                genes_use = _flatten_gene_list(genes)
                genes_use = [
                    str(g).strip()
                    for g in genes_use
                    if str(g).strip()
                ]
                genes_use = list(dict.fromkeys(genes_use))

                if len(genes_use) > 0:
                    parsed[str(module)] = genes_use

            if len(parsed) > 0:
                print(f"Using CAF markers from notebook variable: {var_name}")
                return parsed

        if isinstance(obj, pd.DataFrame):
            module_col_candidates = [
                "CAF_module", "caf_module", "module", "Module",
                "state", "State", "signature", "Signature",
                "cell_state", "Cell_state", "CAF_state", "caf_state"
            ]

            gene_col_candidates = [
                "Gene", "gene", "genes", "Genes",
                "marker", "Marker", "markers", "Markers"
            ]

            module_col = next((c for c in module_col_candidates if c in obj.columns), None)
            gene_col = next((c for c in gene_col_candidates if c in obj.columns), None)

            if module_col is not None and gene_col is not None:
                parsed = {}

                for _, row in obj[[module_col, gene_col]].dropna().iterrows():
                    module = str(row[module_col])
                    genes_use = _flatten_gene_list(row[gene_col])
                    parsed.setdefault(module, [])
                    parsed[module].extend(genes_use)

                parsed = {
                    m: list(dict.fromkeys([
                        str(g).strip()
                        for g in genes
                        if str(g).strip()
                    ]))
                    for m, genes in parsed.items()
                }

                parsed = {
                    m: genes
                    for m, genes in parsed.items()
                    if len(genes) > 0
                }

                if len(parsed) > 0:
                    print(f"Using CAF markers from notebook DataFrame: {var_name}")
                    return parsed

    print("[Warning] No notebook CAF marker variable found. Using fallback CAF marker sets.")

    return {
        "myCAF": [
            "ACTA2", "TAGLN", "MYL9", "TPM2", "CNN1", "CALD1",
            "COL1A1", "COL1A2"
        ],
        "iCAF": [
            "IL6", "CXCL12", "CXCL14", "LIF", "CCL2", "PTGS2"
        ],
        "apCAF": [
            "HLA-DRA", "HLA-DRB1", "CD74", "CIITA"
        ],
        "meCAF": [
            "COL11A1", "THBS2", "MMP11", "ITGA11", "FN1",
            "VCAN", "SPARC", "SULF1", "LOX", "PLOD2"
        ],
        "periCAF": [
            "RGS5", "PDGFRB", "MCAM", "NOTCH3", "TAGLN"
        ],
        "prolCAF": [
            "MKI67", "TOP2A", "PCNA"
        ],
    }


def _resolve_gene_names_in_adata(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if not gene:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _first_indexer(values, query_values):
    mapping = {}
    for i, v in enumerate(values):
        v = str(v)
        if v not in mapping:
            mapping[v] = i

    return np.array(
        [mapping.get(str(q), -1) for q in query_values],
        dtype=int
    )


def _make_composite(sample_values, id_values):
    return np.array(
        [f"{str(s)}||{str(i)}" for s, i in zip(sample_values, id_values)],
        dtype=object
    )


def _match_fibro_rows_to_expr_adata(expr_adata, spatial_adata_all, fibro_full_idx, spatial_sample_values):
    spatial_obs_names = spatial_adata_all.obs_names[fibro_full_idx].astype(str).to_numpy()

    if "barcode" in spatial_adata_all.obs.columns:
        spatial_barcodes = (
            spatial_adata_all.obs.iloc[fibro_full_idx]["barcode"]
            .astype(str)
            .to_numpy()
        )
    else:
        spatial_barcodes = spatial_obs_names.copy()

    expr_obs_names = expr_adata.obs_names.astype(str).to_numpy()

    if "barcode" in expr_adata.obs.columns:
        expr_barcodes = expr_adata.obs["barcode"].astype(str).to_numpy()
    else:
        expr_barcodes = expr_obs_names.copy()

    expr_sample_col = _sample_col_from_obs(expr_adata)

    if expr_sample_col is not None:
        expr_sample_values = expr_adata.obs[expr_sample_col].astype(str).to_numpy()
        spatial_sample_fibro = spatial_sample_values[fibro_full_idx].astype(str)

        query_comp_barcode = _make_composite(spatial_sample_fibro, spatial_barcodes)
        expr_comp_barcode = _make_composite(expr_sample_values, expr_barcodes)

        idx = _first_indexer(expr_comp_barcode, query_comp_barcode)
        if np.all(idx >= 0):
            return idx, f"sample + barcode using expression sample column '{expr_sample_col}'"

        query_comp_obs = _make_composite(spatial_sample_fibro, spatial_obs_names)
        expr_comp_obs = _make_composite(expr_sample_values, expr_obs_names)

        idx = _first_indexer(expr_comp_obs, query_comp_obs)
        if np.all(idx >= 0):
            return idx, f"sample + obs_names using expression sample column '{expr_sample_col}'"

    idx = _first_indexer(expr_barcodes, spatial_barcodes)
    if np.all(idx >= 0):
        return idx, "barcode -> expression barcode"

    idx = _first_indexer(expr_obs_names, spatial_obs_names)
    if np.all(idx >= 0):
        return idx, "spatial obs_names -> expression obs_names"

    idx = _first_indexer(expr_obs_names, spatial_barcodes)
    if np.all(idx >= 0):
        return idx, "spatial barcode -> expression obs_names"

    return None, None


def _add_expr_candidate(candidates, name, obj):
    if obj is not None and hasattr(obj, "var_names") and hasattr(obj, "obs") and hasattr(obj, "X"):
        candidates.append((name, obj))


def _build_expr_candidates():
    candidates = []

    if "processed" in globals():
        _add_expr_candidate(candidates, "processed.adata_all", getattr(processed, "adata_all", None))
        _add_expr_candidate(candidates, "processed.adata", getattr(processed, "adata", None))

        if hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
            try:
                import anndata as ad
                adata_list_concat = ad.concat(
                    processed.adata_list,
                    join="outer",
                    index_unique=None,
                    merge="same",
                )
                _add_expr_candidate(candidates, "concat(processed.adata_list)", adata_list_concat)
            except Exception as e:
                print(f"[Info] Could not concatenate processed.adata_list: {e}")

    for var_name in [
        "adata_all_full",
        "adata_raw",
        "adata_full",
        "adata_ori",
        "adata_original",
        "adata_copy",
        "adata",
        "adata_all",
        "adata_choose",
    ]:
        if var_name in globals():
            _add_expr_candidate(candidates, var_name, globals()[var_name])

    seen = set()
    unique_candidates = []

    for name, obj in candidates:
        if id(obj) not in seen:
            unique_candidates.append((name, obj))
            seen.add(id(obj))

    return unique_candidates


def _get_X_array(adata, rows, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub


def _p_to_label(p):
    if pd.isna(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


# ==============================================================
# Step 1. Spatial object and annotations
# ==============================================================

if "processed" not in globals() or not hasattr(processed, "adata_all"):
    raise NameError("processed.adata_all is required for spatial 10-NN grouping.")

adata_spatial = processed.adata_all

if CELL_SUBTYPE_COL not in adata_spatial.obs.columns:
    raise KeyError(
        f"Cannot find '{CELL_SUBTYPE_COL}' in processed.adata_all.obs. "
        f"Available columns are:\n{list(adata_spatial.obs.columns)}"
    )

if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Please run malignant clustering first."
    )

spatial_all = _get_spatial_matrix(adata_spatial)

sample_col = _sample_col_from_obs(adata_spatial)

if sample_col is None:
    print(
        "[Warning] No sample/slice column found in processed.adata_all.obs. "
        "Spatial 10-NN will be computed across all cells together."
    )
    sample_values = np.array(["__all_cells__"] * adata_spatial.n_obs)
else:
    print(f"Using sample/slice column for within-sample spatial KNN: {sample_col}")
    sample_values = adata_spatial.obs[sample_col].astype(str).to_numpy()


# ==============================================================
# Step 2. Match malignant cells and define malignant C5
# ==============================================================

if "barcode" in adata_choose.obs.columns:
    malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

spatial_index = pd.Index(adata_spatial.obs_names.astype(str))
malignant_full_idx = spatial_index.get_indexer(malignant_barcodes)

if np.any(malignant_full_idx < 0) and "barcode" in adata_spatial.obs.columns:
    spatial_barcodes_all = adata_spatial.obs["barcode"].astype(str).to_numpy()

    barcode_to_idx = {}
    for i, b in enumerate(spatial_barcodes_all):
        if b not in barcode_to_idx:
            barcode_to_idx[b] = i

    malignant_full_idx = np.array(
        [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
        dtype=int
    )

missing_match = malignant_full_idx < 0

if np.any(missing_match):
    raise ValueError(
        f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
        "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names "
        "or processed.adata_all.obs['barcode']."
    )

malignant_cluster_clean = (
    adata_choose.obs[CLUSTER_COL]
    .astype(str)
    .map(_clean_cluster_label)
    .to_numpy()
)

malignant_c5_full_idx = malignant_full_idx[malignant_cluster_clean == C5_LABEL]

if len(malignant_c5_full_idx) == 0:
    raise ValueError(
        f"No malignant C5 cells found using adata_choose.obs['{CLUSTER_COL}'] == {C5_LABEL}."
    )

is_malignant_c5 = np.zeros(adata_spatial.n_obs, dtype=bool)
is_malignant_c5[malignant_c5_full_idx] = True

print(f"Matched malignant cells: {len(malignant_full_idx)}")
print(f"Matched malignant C5 cells: {len(malignant_c5_full_idx)}")


# ==============================================================
# Step 3. Identify Fibroblast cells
# ==============================================================

cell_subtypes = adata_spatial.obs[CELL_SUBTYPE_COL].astype(str).to_numpy()
is_fibro = cell_subtypes == FIBRO_LABEL

if is_fibro.sum() == 0:
    is_fibro = pd.Series(cell_subtypes).str.contains(
        "fibro",
        case=False,
        regex=False
    ).to_numpy()
    print("[Info] Exact 'Fibroblast' label not found. Using contains('fibro').")

fibro_full_idx = np.where(is_fibro)[0]

if len(fibro_full_idx) == 0:
    raise ValueError(
        f"No Fibroblast cells found in processed.adata_all.obs['{CELL_SUBTYPE_COL}']."
    )

print(f"Total Fibroblast cells: {len(fibro_full_idx)}")


# ==============================================================
# Step 4. Spatial 10-NN grouping for Fibroblast cells
# ==============================================================

has_neighbor_malignant_c5 = np.full(adata_spatial.n_obs, False, dtype=bool)
neighbor_malignant_c5_count = np.full(adata_spatial.n_obs, np.nan, dtype=float)
neighbor_n = np.full(adata_spatial.n_obs, np.nan, dtype=float)

for sample in pd.unique(sample_values[fibro_full_idx]):
    sample_idx = np.where(sample_values == sample)[0]
    fibro_idx_sample = fibro_full_idx[sample_values[fibro_full_idx] == sample]

    if len(sample_idx) <= 1 or len(fibro_idx_sample) == 0:
        continue

    k_fit = min(K_NEIGH + 1, len(sample_idx))

    nn = NearestNeighbors(n_neighbors=k_fit, algorithm="auto")
    nn.fit(spatial_all[sample_idx, :])

    _, neigh_local = nn.kneighbors(spatial_all[fibro_idx_sample, :])

    for row_i, fibro_cell_full_idx in enumerate(fibro_idx_sample):
        neigh_global = sample_idx[neigh_local[row_i]]
        neigh_global = neigh_global[neigh_global != fibro_cell_full_idx][:K_NEIGH]

        if len(neigh_global) == 0:
            continue

        c5_count_cur = np.sum(is_malignant_c5[neigh_global])

        neighbor_malignant_c5_count[fibro_cell_full_idx] = c5_count_cur
        neighbor_n[fibro_cell_full_idx] = len(neigh_global)
        has_neighbor_malignant_c5[fibro_cell_full_idx] = c5_count_cur > 0

fibro_group_values = np.where(
    has_neighbor_malignant_c5[fibro_full_idx],
    GROUP_WITH_C5,
    GROUP_WITHOUT_C5
)

fibro_sample_values = sample_values[fibro_full_idx]
fibro_barcodes = adata_spatial.obs_names[fibro_full_idx].astype(str).to_numpy()

adata_spatial.obs["Fibroblast_neighbor_malignantC5_spatial10NN"] = "Non-Fibroblast"

adata_spatial.obs.loc[
    adata_spatial.obs_names[fibro_full_idx[~has_neighbor_malignant_c5[fibro_full_idx]]],
    "Fibroblast_neighbor_malignantC5_spatial10NN"
] = GROUP_WITHOUT_C5

adata_spatial.obs.loc[
    adata_spatial.obs_names[fibro_full_idx[has_neighbor_malignant_c5[fibro_full_idx]]],
    "Fibroblast_neighbor_malignantC5_spatial10NN"
] = GROUP_WITH_C5

adata_spatial.obs["neighbor_malignantC5_count_spatial10NN_for_Fibroblast"] = neighbor_malignant_c5_count
adata_spatial.obs["neighbor_n_spatial10NN_for_Fibroblast"] = neighbor_n

print("Fibroblast group sizes:")
print(pd.Series(fibro_group_values).value_counts())


# ==============================================================
# Step 5. Get CAF marker genes
# ==============================================================

caf_modules = _extract_caf_modules_from_notebook()

caf_gene_rows = []

for module, genes in caf_modules.items():
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        caf_gene_rows.append({
            "CAF_module": str(module),
            "Gene": gene,
        })

caf_gene_df = pd.DataFrame(caf_gene_rows).drop_duplicates()

if caf_gene_df.empty:
    raise ValueError("CAF marker table is empty. Please check CAF marker definitions in the notebook.")

caf_gene_union = list(dict.fromkeys(caf_gene_df["Gene"].astype(str).tolist()))

print("CAF marker genes used for overall CAF score:")
_display_df(
    caf_gene_df
    .groupby("CAF_module", as_index=False)
    .agg(
        n_genes=("Gene", "nunique"),
        genes=("Gene", lambda x: ", ".join(sorted(x.unique())))
    )
)


# ==============================================================
# Step 6. Find expression AnnData containing CAF markers
# ==============================================================

expr_candidates = _build_expr_candidates()

candidate_summary = []
best = None

for name, cand in expr_candidates:
    resolved_cur, missing_cur = _resolve_gene_names_in_adata(cand, caf_gene_union)
    n_genes_cur = len(resolved_cur)

    row_idx_cur, match_mode_cur = _match_fibro_rows_to_expr_adata(
        expr_adata=cand,
        spatial_adata_all=adata_spatial,
        fibro_full_idx=fibro_full_idx,
        spatial_sample_values=sample_values,
    )

    rows_ok = row_idx_cur is not None

    candidate_summary.append({
        "candidate": name,
        "n_obs": cand.n_obs,
        "n_vars": cand.n_vars,
        "n_CAF_markers_found": n_genes_cur,
        "fibro_rows_matched": rows_ok,
        "match_mode": match_mode_cur if match_mode_cur is not None else "NA",
        "first_10_found_genes": ", ".join(list(resolved_cur.keys())[:10]),
    })

    if n_genes_cur > 0 and rows_ok:
        if best is None or n_genes_cur > best["n_genes"]:
            best = {
                "name": name,
                "adata": cand,
                "resolved": resolved_cur,
                "missing": missing_cur,
                "row_idx": row_idx_cur,
                "match_mode": match_mode_cur,
                "n_genes": n_genes_cur,
            }

candidate_summary_df = pd.DataFrame(candidate_summary)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False
)

print("Expression object search summary:")
_display_df(candidate_summary_df)

if best is None:
    raise ValueError(
        "Cannot find an expression AnnData that both contains CAF markers and matches Fibroblast cells.\n"
        "Please check the table 'expression_object_search_summary'. "
        "The most common issue is using a SpiderNet/HVG-filtered object instead of the full expression object."
    )

expr_source_name = best["name"]
expr_adata = best["adata"]
expr_rows_fibro = best["row_idx"]
resolved_genes = best["resolved"]
missing_genes = best["missing"]

available_genes = [g for g in caf_gene_union if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

if len(available_genes) == 0:
    raise ValueError("None of the CAF marker genes are available in the selected expression object.")

caf_gene_df_available = caf_gene_df.loc[
    caf_gene_df["Gene"].isin(available_genes)
].copy()

print(f"Selected expression source: {expr_source_name}")
print(f"Fibroblast row matching mode: {best['match_mode']}")
print(f"Number of CAF marker genes found: {len(available_genes)} / {len(caf_gene_union)}")

if len(missing_genes) > 0:
    print(f"[Warning] Missing CAF marker genes skipped: {missing_genes}")

print("Available CAF marker genes:")
_display_df(
    caf_gene_df_available
    .groupby("CAF_module", as_index=False)
    .agg(
        n_genes=("Gene", "nunique"),
        genes=("Gene", lambda x: ", ".join(sorted(x.unique())))
    )
)


# ==============================================================
# Step 7. Extract CAF marker expression in Fibroblast cells
# ==============================================================

X_fibro = _get_X_array(
    adata=expr_adata,
    rows=expr_rows_fibro,
    genes_actual=actual_gene_names,
)

expr_df = pd.DataFrame(
    X_fibro,
    columns=available_genes,
    index=fibro_barcodes,
)

expr_df["barcode"] = fibro_barcodes
expr_df["sample_id"] = fibro_sample_values
expr_df["Fibroblast_group"] = fibro_group_values
expr_df["neighbor_malignantC5_count"] = neighbor_malignant_c5_count[fibro_full_idx]
expr_df["neighbor_n"] = neighbor_n[fibro_full_idx]

expr_df["Fibroblast_group"] = pd.Categorical(
    expr_df["Fibroblast_group"],
    categories=GROUP_ORDER,
    ordered=True,
)


# ==============================================================
# Step 8. Compute overall CAF score with zscore_mean_global
# ==============================================================
# Estimate gene means and sample standard deviations (ddof=1) from the
# specified reference rows pooled across slices, then average valid gene z-scores.
# Genes with zero or undefined standard deviation are omitted by nanmean.

score_cur, genes_used_cur, genes_missing_cur = _spidernet_zscore_mean_global_scores_for_rows(
    expr_adata=expr_adata,
    row_idx=expr_rows_fibro,
    reference_row_idx=expr_rows_fibro,
    genes=available_genes,
    sample_col=_spidernet_find_sample_col(expr_adata),
    score_name="__CAF_score_all_markers_zscore_mean_global__",
    ddof=1,
)

if len(genes_used_cur) == 0:
    raise ValueError("No available CAF marker genes for zscore_mean_global.")

expr_df[CAF_SCORE_COL] = score_cur
expr_df["CAF_score_method"] = MODULE_SCORE_METHOD

score_df = expr_df[
    [
        "barcode",
        "sample_id",
        "Fibroblast_group",
        CAF_SCORE_COL,
        "CAF_score_method",
        "neighbor_malignantC5_count",
        "neighbor_n",
    ]
].copy()

score_df["Fibroblast_group"] = pd.Categorical(
    score_df["Fibroblast_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score.csv"),
    index=False,
)

# Save raw marker expression matrix used for zscore_mean_global scoring
expr_marker_out = expr_df[available_genes].copy()
expr_marker_out["barcode"] = fibro_barcodes
expr_marker_out["sample_id"] = fibro_sample_values
expr_marker_out["Fibroblast_group"] = fibro_group_values
expr_marker_out[CAF_SCORE_COL] = expr_df[CAF_SCORE_COL].to_numpy()
expr_marker_out["CAF_score_method"] = MODULE_SCORE_METHOD
expr_marker_out["genes_used_for_CAF_score"] = ", ".join(genes_used_cur)
expr_marker_out["missing_genes_for_CAF_score"] = ", ".join(genes_missing_cur)
expr_marker_out["neighbor_malignantC5_count"] = neighbor_malignant_c5_count[fibro_full_idx]
expr_marker_out["neighbor_n"] = neighbor_n[fibro_full_idx]

expr_marker_out.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_marker_expression_used_for_CAF_zscore_mean_global_score.csv"),
    index=False,
)


# ==============================================================
# Step 9. Sample-level aggregation
# ==============================================================

sample_score_df = (
    score_df
    .groupby(["sample_id", "Fibroblast_group"], observed=True)
    .agg(
        mean_CAF_score=(CAF_SCORE_COL, "mean"),
        median_CAF_score=(CAF_SCORE_COL, "median"),
        n_cells=("barcode", "count"),
    )
    .reset_index()
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_all.csv"),
    index=False,
)

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_GROUP
].copy()

sample_score_df_plot["Fibroblast_group"] = pd.Categorical(
    sample_score_df_plot["Fibroblast_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_min{MIN_CELLS_PER_SAMPLE_GROUP}cells.csv"),
    index=False,
)

sample_score_wide = (
    sample_score_df_plot
    .pivot(
        index="sample_id",
        columns="Fibroblast_group",
        values="mean_CAF_score"
    )
)

sample_score_wide = sample_score_wide.dropna(subset=GROUP_ORDER, how="any")

if sample_score_wide.shape[0] >= 2:
    try:
        stat, pval_score = wilcoxon(
            sample_score_wide[GROUP_WITH_C5],
            sample_score_wide[GROUP_WITHOUT_C5],
            alternative="greater",
        )
        pval_label = f"paired Wilcoxon P={pval_score:.2e}"
        test_name = "paired Wilcoxon"
    except ValueError:
        pval_score = np.nan
        pval_label = "paired Wilcoxon P=NA"
        test_name = "paired Wilcoxon"
elif sample_score_wide.shape[0] == 1:
    pval_score = np.nan
    pval_label = "Only one paired sample"
    test_name = "NA"
else:
    x = sample_score_df_plot.loc[
        sample_score_df_plot["Fibroblast_group"] == GROUP_WITH_C5,
        "mean_CAF_score"
    ].to_numpy(dtype=float)

    y = sample_score_df_plot.loc[
        sample_score_df_plot["Fibroblast_group"] == GROUP_WITHOUT_C5,
        "mean_CAF_score"
    ].to_numpy(dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) > 0 and len(y) > 0:
        stat, pval_score = ranksums(x, y, alternative="greater")
        pval_label = f"sample-level ranksum P={pval_score:.2e}"
        test_name = "sample-level ranksum"
    else:
        pval_score = np.nan
        pval_label = "P=NA"
        test_name = "NA"

pval_df = pd.DataFrame([{
    "score": CAF_SCORE_COL,
    "test": test_name,
    "alternative": "with_neighboring_malignant_C5_greater",
    "n_paired_samples": sample_score_wide.shape[0],
    "pvalue": pval_score,
}])

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_pvalue.csv"),
    index=False,
)

caf_gene_df_available.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_CAF_marker_genes_used.csv"),
    index=False,
)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False,
)

print("Sample-level CAF score:")
_display_df(sample_score_df_plot)

print("Paired sample table:")
_display_df(sample_score_wide)

print(pval_label)

print("P-value table:")
_display_df(pval_df)


# ==============================================================
# Step 10. Plot: sample-level paired CAF score visualization
# ==============================================================

palette_score = {
    GROUP_WITHOUT_C5: "#D4ECF1",
    GROUP_WITH_C5: "#9F3B38",
}

fig, ax = plt.subplots(figsize=(2.4, 2.8))

# light cell-level background
sns.violinplot(
    data=score_df,
    x="Fibroblast_group",
    y=CAF_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    inner=None,
    linewidth=0.6,
    cut=0,
    alpha=0.35,
    ax=ax,
)

sns.boxplot(
    data=score_df,
    x="Fibroblast_group",
    y=CAF_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    showfliers=False,
    width=0.20,
    linewidth=0.8,
    boxprops={"facecolor": "none"},
    ax=ax,
)

# sample-level paired points and lines
if sample_score_wide.shape[0] > 0:
    for sample_id, row in sample_score_wide.iterrows():
        ax.plot(
            [0, 1],
            [row[GROUP_WITHOUT_C5], row[GROUP_WITH_C5]],
            color="black",
            linewidth=0.5,
            alpha=0.20,
            zorder=3,
        )

    ax.scatter(
        np.zeros(sample_score_wide.shape[0]),
        sample_score_wide[GROUP_WITHOUT_C5],
        s=3,
        color="black",
        alpha=0.75,
        zorder=4,
    )

    ax.scatter(
        np.ones(sample_score_wide.shape[0]),
        sample_score_wide[GROUP_WITH_C5],
        s=3,
        color="black",
        alpha=0.75,
        zorder=4,
    )

ax.set_xticklabels(
    ["No neighboring\nmalignant C5", "With neighboring\nmalignant C5"],
    rotation=30,
    ha="right",
)

ax.set_xlabel("")
ax.set_ylabel("CAF score\nmean z-scored CAF markers")
ax.set_title(pval_label, fontsize=6.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_paired_CAF_score.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_paired_CAF_score.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

In [ ]:
# ==============================================================
# Single-cell-level comparison: CAF zscore_mean_global score in Fibroblasts
# --------------------------------------------------------------
# Groups are defined by whether each Fibroblast has a neighboring malignant C5
# cell in spatial 10-NN. No points are drawn to keep the plot readable.
#
# Plot style:
#   - Light violin background for single-cell distribution
#   - Boxplot with no fill
#   - No cell-level points
#   - P-value / stars annotated on the plot
# ==============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ranksums

OUT_PREFIX_SINGLECELL = "Fibroblast_overall_CAF_score_by_neighbor_malignantC5_spatial10NN_singlecell"

# One-sided test direction:
#   "greater": test whether Fibroblast with neighboring malignant C5 has higher CAF score
#   "less":    test whether Fibroblast with neighboring malignant C5 has lower CAF score
#   "two-sided": two-sided comparison
TEST_ALTERNATIVE_SINGLECELL = "greater"

if "score_df" not in globals() or CAF_SCORE_COL not in score_df.columns:
    raise NameError(
        "Run the Fibroblast CAF zscore_mean_global score cell above before this single-cell comparison cell."
    )

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)


# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _p_to_label(p):
    if not np.isfinite(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


# =============================
# Prepare single-cell dataframe
# =============================
single_cell_df = score_df.copy()

single_cell_df[CAF_SCORE_COL] = pd.to_numeric(
    single_cell_df[CAF_SCORE_COL],
    errors="coerce",
)

single_cell_df = single_cell_df.loc[
    np.isfinite(single_cell_df[CAF_SCORE_COL].to_numpy(dtype=float))
].copy()

single_cell_df["Fibroblast_group"] = pd.Categorical(
    single_cell_df["Fibroblast_group"].astype(str),
    categories=GROUP_ORDER,
    ordered=True,
)

single_cell_df = single_cell_df.loc[
    single_cell_df["Fibroblast_group"].notna()
].copy()


# =============================
# Cell-level rank-sum test
# =============================
x = single_cell_df.loc[
    single_cell_df["Fibroblast_group"].astype(str) == GROUP_WITH_C5,
    CAF_SCORE_COL,
].to_numpy(dtype=float)

y = single_cell_df.loc[
    single_cell_df["Fibroblast_group"].astype(str) == GROUP_WITHOUT_C5,
    CAF_SCORE_COL,
].to_numpy(dtype=float)

x = x[np.isfinite(x)]
y = y[np.isfinite(y)]

if len(x) > 0 and len(y) > 0:
    stat, pval = ranksums(
        x,
        y,
        alternative=TEST_ALTERNATIVE_SINGLECELL,
    )
else:
    stat, pval = np.nan, np.nan

pval_label = _p_to_label(pval)
pval_star = p_to_star_with_ns(pval)


# =============================
# Summary tables
# =============================
single_summary_df = (
    single_cell_df
    .groupby("Fibroblast_group", observed=True)
    .agg(
        n_cells=("barcode", "count"),
        n_samples=("sample_id", "nunique"),
        mean_CAF_score=(CAF_SCORE_COL, "mean"),
        median_CAF_score=(CAF_SCORE_COL, "median"),
        std_CAF_score=(CAF_SCORE_COL, "std"),
    )
    .reset_index()
)

single_pval_df = pd.DataFrame([{
    "score": CAF_SCORE_COL,
    "comparison": f"{GROUP_WITH_C5} vs {GROUP_WITHOUT_C5}",
    "test": "cell-level Wilcoxon rank-sum",
    "alternative": TEST_ALTERNATIVE_SINGLECELL,
    "n_cells_with_neighbor_malignantC5": len(x),
    "n_cells_without_neighbor_malignantC5": len(y),
    "statistic": stat,
    "pvalue": pval,
    "pvalue_label": pval_label,
    "star": pval_star,
}])

single_cell_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_celllevel_scores.csv"),
    index=False,
)

single_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_summary.csv"),
    index=False,
)

single_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_pvalue.csv"),
    index=False,
)


# =============================
# Plot style
# =============================
palette_score = {
    GROUP_WITHOUT_C5: "#D4ECF1",
    GROUP_WITH_C5: "#9F3B38",
}

edge_palette = {
    GROUP_WITHOUT_C5: "#82CCE2",
    GROUP_WITH_C5: "#9F3B38",
}

x_tick_labels = [
    "No neighboring\nmalignant C5",
    "With neighboring\nmalignant C5",
]


# =============================
# Plot: single-cell distribution without points
# =============================
plt.close("all")

fig, ax = plt.subplots(figsize=(2.4, 2.8))

# Light violin background: cell-level distribution
sns.violinplot(
    data=single_cell_df,
    x="Fibroblast_group",
    y=CAF_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    inner=None,
    linewidth=0.6,
    cut=0,
    saturation=1,
    ax=ax,
)

# Make violin semi-transparent and set edge colors
for i, collection in enumerate(ax.collections):
    if i >= len(GROUP_ORDER):
        continue
    group = GROUP_ORDER[i]
    collection.set_alpha(0.35)
    collection.set_edgecolor(edge_palette[group])
    collection.set_linewidth(0.6)

# Boxplot overlay: no fill, no points
sns.boxplot(
    data=single_cell_df,
    x="Fibroblast_group",
    y=CAF_SCORE_COL,
    order=GROUP_ORDER,
    showfliers=False,
    showcaps=False,
    width=0.22,
    linewidth=0.8,
    boxprops={"facecolor": "none", "zorder": 3},
    whiskerprops={"linewidth": 0.8},
    medianprops={"linewidth": 0.8},
    ax=ax,
)

# Recolor boxplot lines by group
for patch_i, patch in enumerate(ax.patches):
    if patch_i >= len(GROUP_ORDER):
        continue
    group = GROUP_ORDER[patch_i]
    patch.set_facecolor("none")
    patch.set_edgecolor(edge_palette[group])
    patch.set_linewidth(0.8)

for line in ax.lines:
    xdata = np.asarray(line.get_xdata(), dtype=float)

    if xdata.size == 0 or np.any(~np.isfinite(xdata)):
        continue

    x_mid = np.mean(xdata)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(GROUP_ORDER)) - x_mid)))
    group = GROUP_ORDER[nearest_idx]

    line.set_color(edge_palette[group])
    line.set_linewidth(0.8)


# =============================
# P-value annotation
# =============================
if single_cell_df.shape[0] > 0:
    y_min = single_cell_df[CAF_SCORE_COL].min()
    y_max = single_cell_df[CAF_SCORE_COL].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    y_bracket = y_max + 0.08 * y_range
    y_text = y_max + 0.13 * y_range

    ax.plot(
        [0, 0, 1, 1],
        [
            y_bracket - 0.015 * y_range,
            y_bracket,
            y_bracket,
            y_bracket - 0.015 * y_range,
        ],
        color="black",
        linewidth=0.6,
        clip_on=False,
    )

    ax.text(
        0.5,
        y_text,
        f"{pval_star} ({pval_label})",
        ha="center",
        va="bottom",
        fontsize=6,
        color="black",
        clip_on=False,
    )

    ax.set_ylim(
        y_min - 0.08 * y_range,
        y_max + 0.22 * y_range,
    )


# =============================
# Axis formatting
# =============================
ax.set_xticklabels(
    x_tick_labels,
    rotation=30,
    ha="right",
)

ax.set_xlabel("")
ax.set_ylabel("CAF score\nsingle-cell zscore_mean_global")
ax.set_title("Fibroblast CAF score by neighboring malignant C5", fontsize=7)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

for ext in ["png", "pdf"]:
    fig.savefig(
        os.path.join(
            out_dir,
            f"{OUT_PREFIX_SINGLECELL}_boxplot_no_points_with_test.{ext}"
        ),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

plt.show()
plt.close()


# =============================
# Display outputs
# =============================
print("Single-cell Fibroblast CAF comparison summary:")
_display_df(single_summary_df)

print("Cell-level Fibroblast CAF rank-sum test:")
_display_df(single_pval_df)

In [ ]:
# ==============================================================
# CD8 exhaustion module score stratified by neighboring malignant C5
# --------------------------------------------------------------
# For each CD8.T.cell:
#   Group 1: spatial 10-NN contains >=1 malignant C5 cell
#   Group 2: spatial 10-NN contains no malignant C5 cell
#
# CD8 exhaustion score:
#   Mean gene-wise z-score over exhaustion markers, using CD8 T cells pooled across slices.
#
# Visualization:
#   sample-level paired comparison between
#   near-C5 CD8T cells vs not-near-C5 CD8T cells
# ==============================================================

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from scipy.stats import wilcoxon, ranksums

# =============================
# Settings
# =============================
K_NEIGH = 10

CELL_SUBTYPE_COL = "cell.subtypes"
CD8_LABEL = "CD8.T.cell"

CLUSTER_COL = "MI_louvain"
C5_LABEL = "5"

GROUP_WITH_C5 = "CD8T with neighboring malignant C5"
GROUP_WITHOUT_C5 = "CD8T without neighboring malignant C5"
GROUP_ORDER = [GROUP_WITHOUT_C5, GROUP_WITH_C5]

MIN_CELLS_PER_SAMPLE_GROUP = 10

CD8_EXHAUSTION_SCORE_COL = "CD8_exhaustion_score"

# Default CD8 exhaustion/checkpoint marker genes.
# The analysis uses the six markers listed below.
CD8_EXHAUSTION_GENES = [
    "PDCD1", "LAG3", "TIGIT", "HAVCR2", "CTLA4", "TOX"
]

OUT_PREFIX = "CD8T_exhaustion_score_by_neighbor_malignantC5_spatial10NN"

# sample-level test direction
# "greater": test whether near-C5 CD8T has higher exhaustion score
# "less":    test whether near-C5 CD8T has lower exhaustion score
# "two-sided": two-sided comparison
TEST_ALTERNATIVE = "greater"

if "run_dirs" in globals() and isinstance(run_dirs, dict) and "run_dir" in run_dirs:
    out_dir = run_dirs["run_dir"]
else:
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _get_spatial_matrix(adata):
    if "spatial" in adata.obsm:
        return np.asarray(adata.obsm["spatial"], dtype=float)

    spatial_col_candidates = [
        ("x", "y"),
        ("X", "Y"),
        ("center_x", "center_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("x_centroid", "y_centroid"),
        ("X_centroid", "Y_centroid"),
    ]

    spatial_cols = next(
        (
            cols for cols in spatial_col_candidates
            if cols[0] in adata.obs.columns and cols[1] in adata.obs.columns
        ),
        None
    )

    if spatial_cols is None:
        raise KeyError(
            "Cannot find spatial coordinates. Expected adata.obsm['spatial'] "
            "or x/y-like columns in adata.obs."
        )

    return adata.obs.loc[:, list(spatial_cols)].to_numpy(dtype=float)


def _resolve_gene_names_in_adata(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if not gene:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _first_indexer(values, query_values):
    mapping = {}

    for i, v in enumerate(values):
        v = str(v)
        if v not in mapping:
            mapping[v] = i

    return np.array(
        [mapping.get(str(q), -1) for q in query_values],
        dtype=int
    )


def _make_composite(sample_values, id_values):
    return np.array(
        [f"{str(s)}||{str(i)}" for s, i in zip(sample_values, id_values)],
        dtype=object
    )


def _add_expr_candidate(candidates, name, obj):
    if obj is not None and hasattr(obj, "var_names") and hasattr(obj, "obs") and hasattr(obj, "X"):
        candidates.append((name, obj))


def _build_expr_candidates():
    candidates = []

    if "processed" in globals():
        _add_expr_candidate(candidates, "processed.adata_all", getattr(processed, "adata_all", None))
        _add_expr_candidate(candidates, "processed.adata", getattr(processed, "adata", None))

        if hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
            try:
                import anndata as ad
                adata_list_concat = ad.concat(
                    processed.adata_list,
                    join="outer",
                    index_unique=None,
                    merge="same",
                )
                _add_expr_candidate(candidates, "concat(processed.adata_list)", adata_list_concat)
            except Exception as e:
                print(f"[Info] Could not concatenate processed.adata_list: {e}")

    for var_name in [
        "adata_all_full",
        "adata_raw",
        "adata_full",
        "adata_ori",
        "adata_original",
        "adata_copy",
        "adata",
        "adata_all",
        "adata_choose",
    ]:
        if var_name in globals():
            _add_expr_candidate(candidates, var_name, globals()[var_name])

    seen = set()
    unique_candidates = []

    for name, obj in candidates:
        if id(obj) not in seen:
            unique_candidates.append((name, obj))
            seen.add(id(obj))

    return unique_candidates


def _match_cd8_rows_to_expr_adata(expr_adata, spatial_adata_all, cd8_full_idx, spatial_sample_values):
    if expr_adata is spatial_adata_all:
        return cd8_full_idx, "direct processed.adata_all row order"

    spatial_obs_names = spatial_adata_all.obs_names[cd8_full_idx].astype(str).to_numpy()

    if "barcode" in spatial_adata_all.obs.columns:
        spatial_barcodes = (
            spatial_adata_all.obs.iloc[cd8_full_idx]["barcode"]
            .astype(str)
            .to_numpy()
        )
    else:
        spatial_barcodes = spatial_obs_names.copy()

    expr_obs_names = expr_adata.obs_names.astype(str).to_numpy()

    if "barcode" in expr_adata.obs.columns:
        expr_barcodes = expr_adata.obs["barcode"].astype(str).to_numpy()
    else:
        expr_barcodes = expr_obs_names.copy()

    expr_sample_col = _sample_col_from_obs(expr_adata)

    if expr_sample_col is not None:
        expr_sample_values = expr_adata.obs[expr_sample_col].astype(str).to_numpy()
        spatial_sample_cd8 = spatial_sample_values[cd8_full_idx].astype(str)

        query_comp_barcode = _make_composite(spatial_sample_cd8, spatial_barcodes)
        expr_comp_barcode = _make_composite(expr_sample_values, expr_barcodes)

        idx = _first_indexer(expr_comp_barcode, query_comp_barcode)
        if np.all(idx >= 0):
            return idx, f"sample + barcode using expression sample column '{expr_sample_col}'"

        query_comp_obs = _make_composite(spatial_sample_cd8, spatial_obs_names)
        expr_comp_obs = _make_composite(expr_sample_values, expr_obs_names)

        idx = _first_indexer(expr_comp_obs, query_comp_obs)
        if np.all(idx >= 0):
            return idx, f"sample + obs_names using expression sample column '{expr_sample_col}'"

    idx = _first_indexer(expr_barcodes, spatial_barcodes)
    if np.all(idx >= 0):
        return idx, "barcode -> expression barcode"

    idx = _first_indexer(expr_obs_names, spatial_obs_names)
    if np.all(idx >= 0):
        return idx, "spatial obs_names -> expression obs_names"

    idx = _first_indexer(expr_obs_names, spatial_barcodes)
    if np.all(idx >= 0):
        return idx, "spatial barcode -> expression obs_names"

    return None, None


def _get_X_array(adata, rows, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[rows, :][:, gene_idx]

    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)

    return X_sub


def _p_to_label(p):
    if pd.isna(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


# ==============================================================
# Step 1. Spatial object and annotations
# ==============================================================

if "processed" not in globals() or not hasattr(processed, "adata_all"):
    raise NameError("processed.adata_all is required for spatial 10-NN grouping.")

adata_spatial = processed.adata_all

if CELL_SUBTYPE_COL not in adata_spatial.obs.columns:
    raise KeyError(
        f"Cannot find '{CELL_SUBTYPE_COL}' in processed.adata_all.obs. "
        f"Available columns are:\n{list(adata_spatial.obs.columns)}"
    )

if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Please run malignant clustering first."
    )

spatial_all = _get_spatial_matrix(adata_spatial)

sample_col = _sample_col_from_obs(adata_spatial)

if sample_col is None:
    print(
        "[Warning] No sample/slice column found in processed.adata_all.obs. "
        "Spatial 10-NN will be computed across all cells together."
    )
    sample_values = np.array(["__all_cells__"] * adata_spatial.n_obs)
else:
    print(f"Using sample/slice column for within-sample spatial KNN: {sample_col}")
    sample_values = adata_spatial.obs[sample_col].astype(str).to_numpy()


# ==============================================================
# Step 2. Match malignant cells and define malignant C5
# ==============================================================

if "barcode" in adata_choose.obs.columns:
    malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

spatial_index = pd.Index(adata_spatial.obs_names.astype(str))
malignant_full_idx = spatial_index.get_indexer(malignant_barcodes)

if np.any(malignant_full_idx < 0) and "barcode" in adata_spatial.obs.columns:
    spatial_barcodes_all = adata_spatial.obs["barcode"].astype(str).to_numpy()

    barcode_to_idx = {}
    for i, b in enumerate(spatial_barcodes_all):
        if b not in barcode_to_idx:
            barcode_to_idx[b] = i

    malignant_full_idx = np.array(
        [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
        dtype=int
    )

missing_match = malignant_full_idx < 0

if np.any(missing_match):
    raise ValueError(
        f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
        "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names "
        "or processed.adata_all.obs['barcode']."
    )

malignant_cluster_clean = (
    adata_choose.obs[CLUSTER_COL]
    .astype(str)
    .map(_clean_cluster_label)
    .to_numpy()
)

malignant_c5_full_idx = malignant_full_idx[malignant_cluster_clean == C5_LABEL]

if len(malignant_c5_full_idx) == 0:
    raise ValueError(
        f"No malignant C5 cells found using adata_choose.obs['{CLUSTER_COL}'] == {C5_LABEL}."
    )

is_malignant_c5 = np.zeros(adata_spatial.n_obs, dtype=bool)
is_malignant_c5[malignant_c5_full_idx] = True

print(f"Matched malignant cells: {len(malignant_full_idx)}")
print(f"Matched malignant C5 cells: {len(malignant_c5_full_idx)}")


# ==============================================================
# Step 3. Identify CD8 T cells
# ==============================================================

cell_subtypes = adata_spatial.obs[CELL_SUBTYPE_COL].astype(str).to_numpy()

is_cd8 = cell_subtypes == CD8_LABEL

if is_cd8.sum() == 0:
    is_cd8 = pd.Series(cell_subtypes).str.contains(
        "CD8",
        case=False,
        regex=False
    ).to_numpy()
    print("[Info] Exact 'CD8.T.cell' label not found. Using contains('CD8').")

cd8_full_idx = np.where(is_cd8)[0]

if len(cd8_full_idx) == 0:
    raise ValueError(
        f"No CD8 T cells found in processed.adata_all.obs['{CELL_SUBTYPE_COL}']."
    )

print(f"Total CD8 T cells: {len(cd8_full_idx)}")


# ==============================================================
# Step 4. Spatial 10-NN grouping for CD8 T cells
# ==============================================================

has_neighbor_malignant_c5 = np.full(adata_spatial.n_obs, False, dtype=bool)
neighbor_malignant_c5_count = np.full(adata_spatial.n_obs, np.nan, dtype=float)
neighbor_n = np.full(adata_spatial.n_obs, np.nan, dtype=float)

for sample in pd.unique(sample_values[cd8_full_idx]):
    sample_idx = np.where(sample_values == sample)[0]
    cd8_idx_sample = cd8_full_idx[sample_values[cd8_full_idx] == sample]

    if len(sample_idx) <= 1 or len(cd8_idx_sample) == 0:
        continue

    k_fit = min(K_NEIGH + 1, len(sample_idx))

    nn = NearestNeighbors(n_neighbors=k_fit, algorithm="auto")
    nn.fit(spatial_all[sample_idx, :])

    _, neigh_local = nn.kneighbors(spatial_all[cd8_idx_sample, :])

    for row_i, cd8_cell_full_idx in enumerate(cd8_idx_sample):
        neigh_global = sample_idx[neigh_local[row_i]]
        neigh_global = neigh_global[neigh_global != cd8_cell_full_idx][:K_NEIGH]

        if len(neigh_global) == 0:
            continue

        c5_count_cur = np.sum(is_malignant_c5[neigh_global])

        neighbor_malignant_c5_count[cd8_cell_full_idx] = c5_count_cur
        neighbor_n[cd8_cell_full_idx] = len(neigh_global)
        has_neighbor_malignant_c5[cd8_cell_full_idx] = c5_count_cur > 0

cd8_group_values = np.where(
    has_neighbor_malignant_c5[cd8_full_idx],
    GROUP_WITH_C5,
    GROUP_WITHOUT_C5
)

cd8_sample_values = sample_values[cd8_full_idx]
cd8_barcodes = adata_spatial.obs_names[cd8_full_idx].astype(str).to_numpy()

adata_spatial.obs["CD8T_neighbor_malignantC5_spatial10NN"] = "Non-CD8T"

adata_spatial.obs.loc[
    adata_spatial.obs_names[cd8_full_idx[~has_neighbor_malignant_c5[cd8_full_idx]]],
    "CD8T_neighbor_malignantC5_spatial10NN"
] = GROUP_WITHOUT_C5

adata_spatial.obs.loc[
    adata_spatial.obs_names[cd8_full_idx[has_neighbor_malignant_c5[cd8_full_idx]]],
    "CD8T_neighbor_malignantC5_spatial10NN"
] = GROUP_WITH_C5

adata_spatial.obs["neighbor_malignantC5_count_spatial10NN_for_CD8T"] = neighbor_malignant_c5_count
adata_spatial.obs["neighbor_n_spatial10NN_for_CD8T"] = neighbor_n

print("CD8T group sizes:")
print(pd.Series(cd8_group_values).value_counts())


# ==============================================================
# Step 5. Prepare CD8 exhaustion gene set
# ==============================================================

cd8_exhaustion_gene_df = pd.DataFrame({
    "Gene_set": "CD8_exhaustion",
    "Gene": list(dict.fromkeys([str(g).strip() for g in CD8_EXHAUSTION_GENES if str(g).strip()]))
})

if cd8_exhaustion_gene_df.empty:
    raise ValueError("CD8_EXHAUSTION_GENES is empty.")

cd8_exhaustion_gene_union = cd8_exhaustion_gene_df["Gene"].astype(str).tolist()

print("CD8 exhaustion genes requested:")
_display_df(cd8_exhaustion_gene_df)


# ==============================================================
# Step 6. Find expression AnnData containing exhaustion genes
# ==============================================================

expr_candidates = _build_expr_candidates()

candidate_summary = []
best = None

for name, cand in expr_candidates:
    resolved_cur, missing_cur = _resolve_gene_names_in_adata(cand, cd8_exhaustion_gene_union)
    n_genes_cur = len(resolved_cur)

    row_idx_cur, match_mode_cur = _match_cd8_rows_to_expr_adata(
        expr_adata=cand,
        spatial_adata_all=adata_spatial,
        cd8_full_idx=cd8_full_idx,
        spatial_sample_values=sample_values,
    )

    rows_ok = row_idx_cur is not None

    candidate_summary.append({
        "candidate": name,
        "n_obs": cand.n_obs,
        "n_vars": cand.n_vars,
        "n_CD8_exhaustion_genes_found": n_genes_cur,
        "CD8_rows_matched": rows_ok,
        "match_mode": match_mode_cur if match_mode_cur is not None else "NA",
        "first_10_found_genes": ", ".join(list(resolved_cur.keys())[:10]),
    })

    if n_genes_cur > 0 and rows_ok:
        if best is None or n_genes_cur > best["n_genes"]:
            best = {
                "name": name,
                "adata": cand,
                "resolved": resolved_cur,
                "missing": missing_cur,
                "row_idx": row_idx_cur,
                "match_mode": match_mode_cur,
                "n_genes": n_genes_cur,
            }

candidate_summary_df = pd.DataFrame(candidate_summary)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False
)

print("Expression object search summary:")
_display_df(candidate_summary_df)

if best is None:
    raise ValueError(
        "Cannot find an expression AnnData that both contains CD8 exhaustion genes "
        "and matches CD8T cells. Please check the expression_object_search_summary table."
    )

expr_source_name = best["name"]
expr_adata = best["adata"]
expr_rows_cd8 = best["row_idx"]
resolved_genes = best["resolved"]
missing_genes = best["missing"]

available_genes = [g for g in cd8_exhaustion_gene_union if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

if len(available_genes) == 0:
    raise ValueError("None of the CD8 exhaustion genes are available in the selected expression object.")

cd8_exhaustion_gene_df_available = cd8_exhaustion_gene_df.loc[
    cd8_exhaustion_gene_df["Gene"].isin(available_genes)
].copy()

print(f"Selected expression source: {expr_source_name}")
print(f"CD8T row matching mode: {best['match_mode']}")
print(f"Number of CD8 exhaustion genes found: {len(available_genes)} / {len(cd8_exhaustion_gene_union)}")

if len(missing_genes) > 0:
    print(f"[Warning] Missing CD8 exhaustion genes skipped: {missing_genes}")

print("Available CD8 exhaustion genes:")
_display_df(cd8_exhaustion_gene_df_available)


# ==============================================================
# Step 7. Extract exhaustion marker expression in CD8T cells
# ==============================================================

X_cd8 = _get_X_array(
    adata=expr_adata,
    rows=expr_rows_cd8,
    genes_actual=actual_gene_names,
)

expr_df = pd.DataFrame(
    X_cd8,
    columns=available_genes,
    index=cd8_barcodes,
)

expr_df["barcode"] = cd8_barcodes
expr_df["sample_id"] = cd8_sample_values
expr_df["CD8T_group"] = cd8_group_values
expr_df["neighbor_malignantC5_count"] = neighbor_malignant_c5_count[cd8_full_idx]
expr_df["neighbor_n"] = neighbor_n[cd8_full_idx]

expr_df["CD8T_group"] = pd.Categorical(
    expr_df["CD8T_group"],
    categories=GROUP_ORDER,
    ordered=True,
)


# ==============================================================
# Step 8. Compute CD8 exhaustion module score with zscore_mean_global
# ==============================================================
# Estimate gene means and sample standard deviations (ddof=1) from the
# specified reference rows pooled across slices, then average valid gene z-scores.
# Genes with zero or undefined standard deviation are omitted by nanmean.

score_cur, genes_used_cur, genes_missing_cur = _spidernet_zscore_mean_global_scores_for_rows(
    expr_adata=expr_adata,
    row_idx=expr_rows_cd8,
    reference_row_idx=expr_rows_cd8,
    genes=available_genes,
    sample_col=_spidernet_find_sample_col(expr_adata),
    score_name="__CD8_exhaustion_zscore_mean_global__",
    ddof=1,
)

if len(genes_used_cur) == 0:
    raise ValueError("No available CD8 exhaustion genes for zscore_mean_global.")

expr_df[CD8_EXHAUSTION_SCORE_COL] = score_cur
expr_df["CD8_exhaustion_score_method"] = MODULE_SCORE_METHOD

score_df = expr_df[
    [
        "barcode",
        "sample_id",
        "CD8T_group",
        CD8_EXHAUSTION_SCORE_COL,
        "CD8_exhaustion_score_method",
        "neighbor_malignantC5_count",
        "neighbor_n",
    ]
].copy()

score_df["CD8T_group"] = pd.Categorical(
    score_df["CD8T_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score.csv"),
    index=False,
)

# Save raw marker expression matrix used for zscore_mean_global scoring
expr_marker_out = expr_df[available_genes].copy()
expr_marker_out["barcode"] = cd8_barcodes
expr_marker_out["sample_id"] = cd8_sample_values
expr_marker_out["CD8T_group"] = cd8_group_values
expr_marker_out[CD8_EXHAUSTION_SCORE_COL] = expr_df[CD8_EXHAUSTION_SCORE_COL].to_numpy()
expr_marker_out["CD8_exhaustion_score_method"] = MODULE_SCORE_METHOD
expr_marker_out["genes_used_for_CD8_exhaustion_score"] = ", ".join(genes_used_cur)
expr_marker_out["missing_genes_for_CD8_exhaustion_score"] = ", ".join(genes_missing_cur)
expr_marker_out["neighbor_malignantC5_count"] = neighbor_malignant_c5_count[cd8_full_idx]
expr_marker_out["neighbor_n"] = neighbor_n[cd8_full_idx]

expr_marker_out.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_marker_expression_used_for_CD8_exhaustion_zscore_mean_global_score.csv"),
    index=False,
)


# ==============================================================
# Step 9. Sample-level aggregation
# ==============================================================

sample_score_df = (
    score_df
    .groupby(["sample_id", "CD8T_group"], observed=True)
    .agg(
        mean_CD8_exhaustion_score=(CD8_EXHAUSTION_SCORE_COL, "mean"),
        median_CD8_exhaustion_score=(CD8_EXHAUSTION_SCORE_COL, "median"),
        n_cells=("barcode", "count"),
    )
    .reset_index()
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_all.csv"),
    index=False,
)

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_GROUP
].copy()

sample_score_df_plot["CD8T_group"] = pd.Categorical(
    sample_score_df_plot["CD8T_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_min{MIN_CELLS_PER_SAMPLE_GROUP}cells.csv"),
    index=False,
)

sample_score_wide = (
    sample_score_df_plot
    .pivot(
        index="sample_id",
        columns="CD8T_group",
        values="mean_CD8_exhaustion_score"
    )
)

sample_score_wide = sample_score_wide.dropna(subset=GROUP_ORDER, how="any")

# Prefer paired Wilcoxon when at least two samples have both groups.
if sample_score_wide.shape[0] >= 2:
    try:
        stat, pval_score = wilcoxon(
            sample_score_wide[GROUP_WITH_C5],
            sample_score_wide[GROUP_WITHOUT_C5],
            alternative=TEST_ALTERNATIVE,
        )
        pval_label = f"paired Wilcoxon P={pval_score:.2e}"
        test_name = "paired Wilcoxon"
    except ValueError:
        pval_score = np.nan
        pval_label = "paired Wilcoxon P=NA"
        test_name = "paired Wilcoxon"

else:
    x = sample_score_df_plot.loc[
        sample_score_df_plot["CD8T_group"] == GROUP_WITH_C5,
        "mean_CD8_exhaustion_score"
    ].to_numpy(dtype=float)

    y = sample_score_df_plot.loc[
        sample_score_df_plot["CD8T_group"] == GROUP_WITHOUT_C5,
        "mean_CD8_exhaustion_score"
    ].to_numpy(dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) > 0 and len(y) > 0:
        stat, pval_score = ranksums(
            x,
            y,
            alternative=TEST_ALTERNATIVE,
        )
        pval_label = f"sample-level ranksum P={pval_score:.2e}"
        test_name = "sample-level ranksum"
    else:
        pval_score = np.nan
        pval_label = "P=NA"
        test_name = "NA"

pval_df = pd.DataFrame([{
    "score": CD8_EXHAUSTION_SCORE_COL,
    "test": test_name,
    "alternative": TEST_ALTERNATIVE,
    "n_paired_samples": sample_score_wide.shape[0],
    "pvalue": pval_score,
    "n_genes_used": len(available_genes),
    "genes_used": ", ".join(available_genes),
}])

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_pvalue.csv"),
    index=False,
)

cd8_exhaustion_gene_df_available.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_CD8_exhaustion_genes_used.csv"),
    index=False,
)

candidate_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_expression_object_search_summary.csv"),
    index=False,
)

print("Sample-level CD8 exhaustion score:")
_display_df(sample_score_df_plot)

print("Paired sample table:")
_display_df(sample_score_wide)

print(pval_label)

print("P-value table:")
_display_df(pval_df)


# ==============================================================
# Step 10. Plot: sample-level paired CD8 exhaustion score
# ==============================================================

palette_score = {
    GROUP_WITHOUT_C5: "#D4ECF1",
    GROUP_WITH_C5: "#9F3B38",
}

fig, ax = plt.subplots(figsize=(2.4, 2.8))

# light cell-level background
sns.violinplot(
    data=score_df,
    x="CD8T_group",
    y=CD8_EXHAUSTION_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    inner=None,
    linewidth=0.6,
    cut=0,
    alpha=0.35,
    ax=ax,
)

sns.boxplot(
    data=score_df,
    x="CD8T_group",
    y=CD8_EXHAUSTION_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    showfliers=False,
    width=0.20,
    linewidth=0.8,
    boxprops={"facecolor": "none"},
    ax=ax,
)

# sample-level paired points and lines
if sample_score_wide.shape[0] > 0:
    for sample_id, row in sample_score_wide.iterrows():
        ax.plot(
            [0, 1],
            [row[GROUP_WITHOUT_C5], row[GROUP_WITH_C5]],
            color="black",
            linewidth=0.5,
            alpha=0.20,
            zorder=3,
        )

    ax.scatter(
        np.zeros(sample_score_wide.shape[0]),
        sample_score_wide[GROUP_WITHOUT_C5],
        s=3,
        color="black",
        alpha=0.75,
        zorder=4,
    )

    ax.scatter(
        np.ones(sample_score_wide.shape[0]),
        sample_score_wide[GROUP_WITH_C5],
        s=3,
        color="black",
        alpha=0.75,
        zorder=4,
    )

ax.set_xticklabels(
    ["No neighboring\nmalignant C5", "With neighboring\nmalignant C5"],
    rotation=30,
    ha="right",
)

ax.set_xlabel("")
ax.set_ylabel("CD8 exhaustion score\nmean z-scored exhaustion genes")
ax.set_title(pval_label, fontsize=6.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_paired_CD8_exhaustion_score.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_paired_CD8_exhaustion_score.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

In [ ]:
# ==============================================================
# Single-cell-level comparison: CD8 exhaustion zscore_mean_global score
# --------------------------------------------------------------
# Groups are defined by whether each CD8 T cell has a neighboring malignant C5
# cell in spatial 10-NN. No points are drawn to keep the plot readable.
#
# Plot style:
#   - Light violin background for single-cell distribution
#   - Boxplot with no fill
#   - No cell-level points
#   - P-value / stars annotated on the plot
# ==============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from scipy.stats import ranksums

OUT_PREFIX_SINGLECELL = "CD8T_exhaustion_score_by_neighbor_malignantC5_spatial10NN_singlecell"

# One-sided test direction:
#   "greater": test whether CD8T with neighboring malignant C5 has higher exhaustion score
#   "less":    test whether CD8T with neighboring malignant C5 has lower exhaustion score
#   "two-sided": two-sided comparison
TEST_ALTERNATIVE_SINGLECELL = TEST_ALTERNATIVE if "TEST_ALTERNATIVE" in globals() else "greater"

if "score_df" not in globals() or CD8_EXHAUSTION_SCORE_COL not in score_df.columns:
    raise NameError(
        "Run the CD8 exhaustion zscore_mean_global score cell above before this single-cell comparison cell."
    )

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)


# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _p_to_label(p):
    if not np.isfinite(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


# =============================
# Prepare single-cell dataframe
# =============================
single_cell_df = score_df.copy()

single_cell_df[CD8_EXHAUSTION_SCORE_COL] = pd.to_numeric(
    single_cell_df[CD8_EXHAUSTION_SCORE_COL],
    errors="coerce",
)

single_cell_df = single_cell_df.loc[
    np.isfinite(single_cell_df[CD8_EXHAUSTION_SCORE_COL].to_numpy(dtype=float))
].copy()

single_cell_df["CD8T_group"] = pd.Categorical(
    single_cell_df["CD8T_group"].astype(str),
    categories=GROUP_ORDER,
    ordered=True,
)

single_cell_df = single_cell_df.loc[
    single_cell_df["CD8T_group"].notna()
].copy()


# =============================
# Cell-level rank-sum test
# =============================
x = single_cell_df.loc[
    single_cell_df["CD8T_group"].astype(str) == GROUP_WITH_C5,
    CD8_EXHAUSTION_SCORE_COL,
].to_numpy(dtype=float)

y = single_cell_df.loc[
    single_cell_df["CD8T_group"].astype(str) == GROUP_WITHOUT_C5,
    CD8_EXHAUSTION_SCORE_COL,
].to_numpy(dtype=float)

x = x[np.isfinite(x)]
y = y[np.isfinite(y)]

if len(x) > 0 and len(y) > 0:
    stat, pval = ranksums(
        x,
        y,
        alternative=TEST_ALTERNATIVE_SINGLECELL,
    )
else:
    stat, pval = np.nan, np.nan

pval_label = _p_to_label(pval)
pval_star = p_to_star_with_ns(pval)


# =============================
# Summary tables
# =============================
single_summary_df = (
    single_cell_df
    .groupby("CD8T_group", observed=True)
    .agg(
        n_cells=("barcode", "count"),
        n_samples=("sample_id", "nunique"),
        mean_CD8_exhaustion_score=(CD8_EXHAUSTION_SCORE_COL, "mean"),
        median_CD8_exhaustion_score=(CD8_EXHAUSTION_SCORE_COL, "median"),
        std_CD8_exhaustion_score=(CD8_EXHAUSTION_SCORE_COL, "std"),
    )
    .reset_index()
)

single_pval_df = pd.DataFrame([{
    "score": CD8_EXHAUSTION_SCORE_COL,
    "comparison": f"{GROUP_WITH_C5} vs {GROUP_WITHOUT_C5}",
    "test": "cell-level Wilcoxon rank-sum",
    "alternative": TEST_ALTERNATIVE_SINGLECELL,
    "n_cells_with_neighbor_malignantC5": len(x),
    "n_cells_without_neighbor_malignantC5": len(y),
    "statistic": stat,
    "pvalue": pval,
    "pvalue_label": pval_label,
    "star": pval_star,
}])

single_cell_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_celllevel_scores.csv"),
    index=False,
)

single_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_summary.csv"),
    index=False,
)

single_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_pvalue.csv"),
    index=False,
)


# =============================
# Plot style
# =============================
palette_score = {
    GROUP_WITHOUT_C5: "#D4ECF1",
    GROUP_WITH_C5: "#9F3B38",
}

edge_palette = {
    GROUP_WITHOUT_C5: "#82CCE2",
    GROUP_WITH_C5: "#9F3B38",
}

x_tick_labels = [
    "No neighboring\nmalignant C5",
    "With neighboring\nmalignant C5",
]


# =============================
# Plot: single-cell distribution without points
# =============================
plt.close("all")

fig, ax = plt.subplots(figsize=(2.4, 2.8))

# Light violin background: cell-level distribution
sns.violinplot(
    data=single_cell_df,
    x="CD8T_group",
    y=CD8_EXHAUSTION_SCORE_COL,
    order=GROUP_ORDER,
    palette=palette_score,
    inner=None,
    linewidth=0.6,
    cut=0,
    saturation=1,
    ax=ax,
)

# Make violin semi-transparent and set edge colors
# Violin bodies are stored in ax.collections.
for i, collection in enumerate(ax.collections):
    if i >= len(GROUP_ORDER):
        continue

    group = GROUP_ORDER[i]
    collection.set_alpha(0.35)
    collection.set_edgecolor(edge_palette[group])
    collection.set_linewidth(0.6)


# Boxplot overlay: no fill, no points
sns.boxplot(
    data=single_cell_df,
    x="CD8T_group",
    y=CD8_EXHAUSTION_SCORE_COL,
    order=GROUP_ORDER,
    showfliers=False,
    showcaps=False,
    width=0.22,
    linewidth=0.8,
    boxprops={"facecolor": "none", "zorder": 3},
    whiskerprops={"linewidth": 0.8},
    medianprops={"linewidth": 0.8},
    ax=ax,
)

# Recolor boxplot patches and lines by group
box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

for patch_i, patch in enumerate(box_patches):
    if patch_i >= len(GROUP_ORDER):
        continue

    group = GROUP_ORDER[patch_i]
    patch.set_facecolor("none")
    patch.set_edgecolor(edge_palette[group])
    patch.set_linewidth(0.8)

for line in ax.lines:
    xdata = np.asarray(line.get_xdata(), dtype=float)

    if xdata.size == 0 or np.any(~np.isfinite(xdata)):
        continue

    x_mid = np.mean(xdata)
    nearest_idx = int(np.argmin(np.abs(np.arange(len(GROUP_ORDER)) - x_mid)))
    group = GROUP_ORDER[nearest_idx]

    line.set_color(edge_palette[group])
    line.set_linewidth(0.8)


# =============================
# P-value annotation
# =============================
if single_cell_df.shape[0] > 0:
    y_min = single_cell_df[CD8_EXHAUSTION_SCORE_COL].min()
    y_max = single_cell_df[CD8_EXHAUSTION_SCORE_COL].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    y_bracket = y_max + 0.08 * y_range
    y_text = y_max + 0.13 * y_range

    ax.plot(
        [0, 0, 1, 1],
        [
            y_bracket - 0.015 * y_range,
            y_bracket,
            y_bracket,
            y_bracket - 0.015 * y_range,
        ],
        color="black",
        linewidth=0.6,
        clip_on=False,
    )

    ax.text(
        0.5,
        y_text,
        f"{pval_star} ({pval_label})",
        ha="center",
        va="bottom",
        fontsize=6,
        color="black",
        clip_on=False,
    )

    ax.set_ylim(
        y_min - 0.08 * y_range,
        y_max + 0.22 * y_range,
    )


# =============================
# Axis formatting
# =============================
ax.set_xticklabels(
    x_tick_labels,
    rotation=30,
    ha="right",
)

ax.set_xlabel("")
ax.set_ylabel("CD8 exhaustion score\nsingle-cell zscore_mean_global")
ax.set_title("CD8 exhaustion score by neighboring malignant C5", fontsize=7)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.tight_layout()

for ext in ["png", "pdf"]:
    fig.savefig(
        os.path.join(
            out_dir,
            f"{OUT_PREFIX_SINGLECELL}_boxplot_no_points_with_test.{ext}"
        ),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

plt.show()
plt.close()


# =============================
# Display outputs
# =============================
print("Single-cell CD8 exhaustion comparison summary:")
_display_df(single_summary_df)

print("Cell-level CD8 exhaustion rank-sum test:")
_display_df(single_pval_df)

In [ ]:
# Antigen-presentation gene expression across malignant clusters
# Plot the seven genes in GENES_TO_SHOW: HLA-A, HLA-B, HLA-C, B2M, TAP1, TAP2
# and NLRC5. Export cell-level expression and per-cluster summaries.

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

GENES_TO_SHOW = [
    "HLA-A", "HLA-B", "HLA-C",
    "B2M",
    "TAP1", "TAP2",
    "NLRC5"
]

OUT_PREFIX = "Malignant_antigen_presentation_gene_expression_by_cluster"

# output directory
out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style global settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Check malignant cluster column
# =============================
if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Helper: clean malignant cluster labels
# =============================
def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x

cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()
cluster_order = sorted(
    np.unique(cluster_clean),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

display_order = [f"Malignant C{x}" for x in cluster_order]
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

# =============================
# Resolve genes from adata_choose first
# If some genes are missing, try processed.adata_all by matching malignant barcodes
# =============================
def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {g.upper(): g for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing

resolved_choose, missing_choose = _resolve_gene_names(adata_choose, GENES_TO_SHOW)

use_adata_all = False

if len(missing_choose) == 0:
    expr_source = "adata_choose"
    expr_adata = adata_choose
    expr_rows = np.arange(adata_choose.n_obs)
    resolved_genes = resolved_choose
else:
    print(
        f"[Info] These genes are missing from adata_choose: {missing_choose}. "
        "Trying processed.adata_all..."
    )

    if "processed" not in globals() or not hasattr(processed, "adata_all"):
        raise ValueError(
            "Some genes are missing from adata_choose, and processed.adata_all is not available."
        )

    adata_all = processed.adata_all
    resolved_all, missing_all = _resolve_gene_names(adata_all, GENES_TO_SHOW)

    if len(missing_all) > 0:
        raise ValueError(
            f"These genes are not found in either adata_choose or processed.adata_all: {missing_all}"
        )

    # match malignant cells from adata_choose to processed.adata_all
    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

    adata_all_index = pd.Index(adata_all.obs_names.astype(str))
    malignant_full_idx = adata_all_index.get_indexer(malignant_barcodes)

    # fallback: match through adata_all.obs["barcode"]
    if np.any(malignant_full_idx < 0) and "barcode" in adata_all.obs.columns:
        all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
        barcode_to_idx = {}
        for i, b in enumerate(all_barcodes):
            if b not in barcode_to_idx:
                barcode_to_idx[b] = i

        malignant_full_idx = np.array(
            [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
            dtype=int
        )

    missing_match = malignant_full_idx < 0
    if np.any(missing_match):
        raise ValueError(
            f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
            "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names."
        )

    expr_source = "processed.adata_all"
    expr_adata = adata_all
    expr_rows = malignant_full_idx
    resolved_genes = resolved_all

print(f"Expression source: {expr_source}")

available_genes = [g for g in GENES_TO_SHOW if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

print("Genes used:")
for g, actual_g in zip(available_genes, actual_gene_names):
    if g != actual_g:
        print(f"  {g} -> {actual_g}")
    else:
        print(f"  {g}")

# =============================
# Extract expression
# =============================
gene_idx = [expr_adata.var_names.get_loc(g) for g in actual_gene_names]

X_sub = expr_adata.X[expr_rows, :][:, gene_idx]

if sparse.issparse(X_sub):
    X_sub = X_sub.toarray()
else:
    X_sub = np.asarray(X_sub)

expr_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=adata_choose.obs_names
)

expr_df["MalignantCluster"] = cluster_display
expr_df["MI_louvain"] = cluster_clean

if "barcode" in adata_choose.obs.columns:
    expr_df["barcode"] = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    expr_df["barcode"] = adata_choose.obs_names.astype(str).to_numpy()

expr_long = expr_df.melt(
    id_vars=["barcode", "MI_louvain", "MalignantCluster"],
    value_vars=available_genes,
    var_name="Gene",
    value_name="Expression"
)

expr_long["MalignantCluster"] = pd.Categorical(
    expr_long["MalignantCluster"],
    categories=display_order,
    ordered=True
)

expr_long["Gene"] = pd.Categorical(
    expr_long["Gene"],
    categories=available_genes,
    ordered=True
)

# =============================
# Summary tables
# =============================
summary_df = (
    expr_long
    .groupby(["Gene", "MalignantCluster"], observed=True)
    .agg(
        n_cells=("barcode", "count"),
        mean_expression=("Expression", "mean"),
        median_expression=("Expression", "median"),
        pct_positive=("Expression", lambda x: 100 * np.mean(np.asarray(x) > 0)),
    )
    .reset_index()
)

expr_long.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_long.csv"),
    index=False
)

summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_summary.csv"),
    index=False
)

print(summary_df)

# =============================
# Plot: boxplot by malignant cluster, faceted by gene
# =============================
fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

cluster_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}

g = sns.catplot(
    data=expr_long,
    x="MalignantCluster",
    y="Expression",
    col="Gene",
    col_wrap=3,
    order=display_order,
    kind="box",
    palette=cluster_palette,
    showfliers=False,
    linewidth=0.8,
    width=0.65,
    height=2.15,
    aspect=1.05,
    sharey=False,
)

for ax, gene in zip(g.axes.flat, available_genes):
    ax.set_title(gene, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("Expression")

    ax.set_xticklabels(
        display_order,
        rotation=35,
        ha="right"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    "Antigen-presentation gene expression across malignant clusters",
    y=1.03,
    fontsize=9
)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_boxplot.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_boxplot.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


In [ ]:
# Antigen-presentation module scores across malignant clusters
# Average malignant-cell z-scores for the seven genes in GENES_TO_SCORE, then
# summarize sample-by-cluster means and compare each cluster with the others.
# Undefined gene z-scores are replaced by zero before averaging.

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse
from scipy.stats import ranksums

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

GENES_TO_SCORE = [
    "HLA-A", "HLA-B", "HLA-C",
    "B2M",
    "TAP1", "TAP2",
    "NLRC5"
]

MODULE_SCORE_COL = "antigen_presentation_module_score"
OUT_PREFIX = "Malignant_antigen_presentation_module_score_by_cluster"

out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style global settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Check malignant cluster column
# =============================
if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Helper: clean malignant cluster labels
# =============================
def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x

cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()

cluster_order = sorted(
    np.unique(cluster_clean),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

display_order = [f"Malignant C{x}" for x in cluster_order]
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

# =============================
# Resolve genes
# Prefer adata_choose.
# If some genes are missing, fallback to processed.adata_all by barcode matching.
# =============================
def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {g.upper(): g for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing

resolved_choose, missing_choose = _resolve_gene_names(adata_choose, GENES_TO_SCORE)

# This will be filled only if processed.adata_all is needed
malignant_full_idx = None
adata_all = None

if len(missing_choose) == 0:
    expr_source = "adata_choose"
    expr_adata = adata_choose
    expr_rows = np.arange(adata_choose.n_obs)
    resolved_genes = resolved_choose
else:
    print(
        f"[Info] These genes are missing from adata_choose: {missing_choose}. "
        "Trying processed.adata_all..."
    )

    if "processed" not in globals() or not hasattr(processed, "adata_all"):
        raise ValueError(
            "Some genes are missing from adata_choose, and processed.adata_all is not available."
        )

    adata_all = processed.adata_all
    resolved_all, missing_all = _resolve_gene_names(adata_all, GENES_TO_SCORE)

    if len(missing_all) > 0:
        raise ValueError(
            f"These genes are not found in either adata_choose or processed.adata_all: {missing_all}"
        )

    # match malignant cells from adata_choose to processed.adata_all
    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

    adata_all_index = pd.Index(adata_all.obs_names.astype(str))
    malignant_full_idx = adata_all_index.get_indexer(malignant_barcodes)

    # fallback: match through adata_all.obs["barcode"]
    if np.any(malignant_full_idx < 0) and "barcode" in adata_all.obs.columns:
        all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
        barcode_to_idx = {}
        for i, b in enumerate(all_barcodes):
            if b not in barcode_to_idx:
                barcode_to_idx[b] = i

        malignant_full_idx = np.array(
            [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
            dtype=int
        )

    missing_match = malignant_full_idx < 0
    if np.any(missing_match):
        raise ValueError(
            f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
            "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names."
        )

    expr_source = "processed.adata_all"
    expr_adata = adata_all
    expr_rows = malignant_full_idx
    resolved_genes = resolved_all

print(f"Expression source: {expr_source}")

available_genes = [g for g in GENES_TO_SCORE if g in resolved_genes]
actual_gene_names = [resolved_genes[g] for g in available_genes]

print("Genes used for antigen-presentation module score:")
for g, actual_g in zip(available_genes, actual_gene_names):
    if g != actual_g:
        print(f"  {g} -> {actual_g}")
    else:
        print(f"  {g}")

if len(available_genes) == 0:
    raise ValueError("No antigen-presentation genes are available for scoring.")

# =============================
# Extract expression matrix
# =============================
gene_idx = [expr_adata.var_names.get_loc(g) for g in actual_gene_names]

X_sub = expr_adata.X[expr_rows, :][:, gene_idx]

if sparse.issparse(X_sub):
    X_sub = X_sub.toarray()
else:
    X_sub = np.asarray(X_sub)

expr_gene_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=adata_choose.obs_names
)

# =============================
# Compute module score
# z-score each marker across malignant cells, then average
# =============================
gene_mean = expr_gene_df.mean(axis=0)
gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)

expr_z = (expr_gene_df - gene_mean) / gene_std
expr_z = expr_z.fillna(0)

module_score = expr_z.mean(axis=1).to_numpy()

# Save score to adata_choose
adata_choose.obs[MODULE_SCORE_COL] = module_score

# =============================
# Add barcode, cluster, sample information
# =============================
if "barcode" in adata_choose.obs.columns:
    barcode_values = adata_choose.obs["barcode"].astype(str).to_numpy()
else:
    barcode_values = adata_choose.obs_names.astype(str).to_numpy()

# Find sample column from adata_choose first
sample_col_candidates = [
    "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
    "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
]

sample_col_choose = next(
    (col for col in sample_col_candidates if col in adata_choose.obs.columns),
    None
)

sample_values = None

if sample_col_choose is not None:
    sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
    print(f"Using sample column from adata_choose: {sample_col_choose}")
else:
    # fallback to processed.adata_all if available
    if adata_all is None and "processed" in globals() and hasattr(processed, "adata_all"):
        adata_all = processed.adata_all

        if "barcode" in adata_choose.obs.columns:
            malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
        else:
            malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

        adata_all_index = pd.Index(adata_all.obs_names.astype(str))
        malignant_full_idx_tmp = adata_all_index.get_indexer(malignant_barcodes)

        if np.any(malignant_full_idx_tmp < 0) and "barcode" in adata_all.obs.columns:
            all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
            barcode_to_idx = {}
            for i, b in enumerate(all_barcodes):
                if b not in barcode_to_idx:
                    barcode_to_idx[b] = i

            malignant_full_idx_tmp = np.array(
                [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
                dtype=int
            )

        if not np.any(malignant_full_idx_tmp < 0):
            malignant_full_idx = malignant_full_idx_tmp

    if adata_all is not None and malignant_full_idx is not None:
        sample_col_all = next(
            (col for col in sample_col_candidates if col in adata_all.obs.columns),
            None
        )

        if sample_col_all is not None:
            sample_values = adata_all.obs.iloc[malignant_full_idx][sample_col_all].astype(str).to_numpy()
            print(f"Using sample column from processed.adata_all: {sample_col_all}")

if sample_values is None:
    print(
        "[Warning] No sample column found. "
        "All malignant cells will be treated as one pseudo-sample."
    )
    sample_values = np.array(["__all_cells__"] * adata_choose.n_obs)

# =============================
# Cell-level score table
# =============================
score_df = pd.DataFrame({
    "barcode": barcode_values,
    "sample_id": sample_values,
    "MI_louvain": cluster_clean,
    "MalignantCluster": cluster_display,
    MODULE_SCORE_COL: module_score,
})

score_df["MalignantCluster"] = pd.Categorical(
    score_df["MalignantCluster"],
    categories=display_order,
    ordered=True
)

# Also save individual gene expression + z-scored expression if useful
expr_gene_out = expr_gene_df.copy()
expr_gene_out["barcode"] = barcode_values
expr_gene_out["sample_id"] = sample_values
expr_gene_out["MalignantCluster"] = cluster_display
expr_gene_out[MODULE_SCORE_COL] = module_score

expr_z_out = expr_z.copy()
expr_z_out.columns = [f"{g}_z" for g in available_genes]
expr_z_out["barcode"] = barcode_values
expr_z_out["sample_id"] = sample_values
expr_z_out["MalignantCluster"] = cluster_display
expr_z_out[MODULE_SCORE_COL] = module_score

# =============================
# Sample-level aggregation
# =============================
sample_score_df = (
    score_df
    .groupby(["sample_id", "MalignantCluster"], observed=True)
    .agg(
        mean_antigen_presentation_score=(MODULE_SCORE_COL, "mean"),
        median_antigen_presentation_score=(MODULE_SCORE_COL, "median"),
        n_cells=("barcode", "count"),
    )
    .reset_index()
)

# Optional: filter very small sample-cluster groups
# Set MIN_CELLS_PER_SAMPLE_CLUSTER > 0 if needed
MIN_CELLS_PER_SAMPLE_CLUSTER = 10

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_CLUSTER
].copy()

sample_score_df_plot["MalignantCluster"] = pd.Categorical(
    sample_score_df_plot["MalignantCluster"],
    categories=display_order,
    ordered=True
)

# =============================
# Summary statistics across samples
# =============================
cluster_summary_df = (
    sample_score_df_plot
    .groupby("MalignantCluster", observed=True)
    .agg(
        n_samples=("sample_id", "nunique"),
        mean_sample_level_score=("mean_antigen_presentation_score", "mean"),
        median_sample_level_score=("mean_antigen_presentation_score", "median"),
        std_sample_level_score=("mean_antigen_presentation_score", "std"),
        total_cells=("n_cells", "sum"),
    )
    .reset_index()
)

# Optional: compare each cluster against all other clusters at sample-level
pval_records = []
for cluster in display_order:
    x = sample_score_df_plot.loc[
        sample_score_df_plot["MalignantCluster"].astype(str) == cluster,
        "mean_antigen_presentation_score"
    ].to_numpy(dtype=float)

    y = sample_score_df_plot.loc[
        sample_score_df_plot["MalignantCluster"].astype(str) != cluster,
        "mean_antigen_presentation_score"
    ].to_numpy(dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) > 0 and len(y) > 0:
        _, p_less = ranksums(x, y, alternative="less")
        _, p_greater = ranksums(x, y, alternative="greater")
        _, p_two = ranksums(x, y, alternative="two-sided")
    else:
        p_less = np.nan
        p_greater = np.nan
        p_two = np.nan

    pval_records.append({
        "MalignantCluster": cluster,
        "n_sample_cluster_values": len(x),
        "n_other_sample_cluster_values": len(y),
        "p_less_than_other_clusters": p_less,
        "p_greater_than_other_clusters": p_greater,
        "p_two_sided": p_two,
    })

pval_df = pd.DataFrame(pval_records)

# =============================
# Save outputs
# =============================
score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score.csv"),
    index=False
)

expr_gene_out.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_gene_expression.csv"),
    index=False
)

expr_z_out.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_gene_zscore.csv"),
    index=False
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_all.csv"),
    index=False
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_min{MIN_CELLS_PER_SAMPLE_CLUSTER}cells.csv"),
    index=False
)

cluster_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_summary.csv"),
    index=False
)

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_vs_others_ranksum_pvalues.csv"),
    index=False
)

if "adata_choose_path" in globals():
    adata_choose.write_h5ad(adata_choose_path)

print("Sample-level score table:")
print(sample_score_df_plot.head())

print("Cluster summary:")
print(cluster_summary_df)

print("Cluster vs others rank-sum p-values:")
print(pval_df)

# =============================
# Plot: sample-level boxplot across malignant clusters
# =============================
fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

cluster_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}

fig, ax = plt.subplots(figsize=(3.8, 2.8))

sns.boxplot(
    data=sample_score_df_plot,
    x="MalignantCluster",
    y="mean_antigen_presentation_score",
    order=display_order,
    palette=cluster_palette,
    showfliers=False,
    linewidth=0.8,
    width=0.65,
    ax=ax,
)

sns.stripplot(
    data=sample_score_df_plot,
    x="MalignantCluster",
    y="mean_antigen_presentation_score",
    order=display_order,
    color="black",
    size=2.2,
    alpha=0.65,
    jitter=0.22,
    ax=ax,
)

# add n sample labels
n_sample_map = (
    sample_score_df_plot
    .groupby("MalignantCluster", observed=True)["sample_id"]
    .nunique()
    .to_dict()
)

y_min = sample_score_df_plot["mean_antigen_presentation_score"].min()
y_max = sample_score_df_plot["mean_antigen_presentation_score"].max()
y_range = y_max - y_min if y_max > y_min else 1.0

for i, cluster in enumerate(display_order):
    n_cur = int(n_sample_map.get(cluster, 0))
    ax.text(
        i,
        y_max + 0.06 * y_range,
        f"n={n_cur}",
        ha="center",
        va="bottom",
        fontsize=5.8,
        rotation=90,
    )

ax.set_xlabel("Malignant cluster")
ax.set_ylabel("Antigen-presentation module score\nsample-level mean z-scored genes")

ax.set_ylim(
    y_min - 0.08 * y_range,
    y_max + 0.18 * y_range
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.xticks(rotation=35, ha="right")
plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


### 5.5. CancerSEA functional states across malignant clusters

The ten-state analysis, the separate three-state sample/cell comparisons, and the patient/sample heatmaps use distinct score tables. Their reference populations and standard-deviation conventions are defined within each block.


In [ ]:
# Ten CancerSEA states: pooled malignant-cell z-scores (ddof=0).
# Gene indices follow processed.adata_list[0].var_names and assume the same
# gene order in adata_choose.X. Undefined z-scores are replaced by zero.

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import PathPatch
from scipy import sparse
from scipy.stats import ranksums

# =============================
# Publication-style global settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Prepare gene sets
# =============================
geneset_df = pd.read_csv(input_path(DATA_ROOT / "CancerSEA_OV/functional_geneset_list_df.csv"))
geneset_df = geneset_df[geneset_df["Gene"].isin(processed.adata_list[0].var_names)]

geneset_index_dict = {
    term: np.where(
        np.isin(
            processed.adata_list[0].var_names,
            geneset_df.loc[geneset_df["GeneSet"] == term, "Gene"]
        )
    )[0]
    for term in np.unique(geneset_df["GeneSet"])
}

# =============================
# Compute CancerSEA scores
# Global z-score using all selected malignant cells across slices + gene-set mean
# =============================
X = adata_choose.X
exp = X.toarray() if sparse.issparse(X) else np.asarray(X)
exp = exp.astype(float, copy=False)

gene_mean = np.nanmean(exp, axis=0)
gene_std = np.nanstd(exp, axis=0)
gene_std = np.where(gene_std > 0, gene_std, np.nan)

exp_z = (exp - gene_mean) / gene_std
exp_z = np.nan_to_num(exp_z, nan=0.0, posinf=0.0, neginf=0.0)

for term, idxs in geneset_index_dict.items():
    adata_choose.obs[f"{term}_score"] = (
        np.mean(exp_z[:, idxs], axis=1) if len(idxs) > 0 else np.zeros(adata_choose.n_obs)
    )

# =============================
# Convert to long format
# =============================
functional_states = [
    "Angiogenesis_score", "Apoptosis_score", "Cell Cycle_score",
    "Differentiation_score", "EMT_score", "Hypoxia_score",
    "Inflammation_score", "Metastasis_score",
    "Quiescence_score", "Stemness_score"
]

df_long = adata_choose.obs.melt(
    id_vars=["MI_louvain"],
    value_vars=functional_states,
    var_name="FunctionalState",
    value_name="Score"
).dropna()

df_long["MI_louvain"] = df_long["MI_louvain"].astype(str)

df_long.to_csv(
    f"{run_dirs['run_dir']}/FunctionalState_scores_by_MI_louvain_long.csv",
    index=False
)

# =============================
# Define order and labels
# =============================
order = sorted(df_long["MI_louvain"].unique(), key=lambda x: int(x))
cluster_labels = [f"C{x}" for x in order]

# =============================
# Define edge and fill palettes
# =============================
edge_colors = ["#82CCE2", "#D1CABE", "#2E7FB9", "#FED881", "#9F3B38", "#519384", "#636491"]
fill_colors = ["#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2", "#E1B6A7", "#B9CEC7", "#A6A2B9"]

edge_palette = {order[i]: edge_colors[i % len(edge_colors)] for i in range(len(order))}
fill_palette = {order[i]: fill_colors[i % len(fill_colors)] for i in range(len(order))}

# =============================
# Create boxplots
# Use fill palette directly first
# =============================
g = sns.catplot(
    data=df_long,
    x="MI_louvain",
    y="Score",
    col="FunctionalState",
    kind="box",
    order=order,
    palette=fill_palette,
    showfliers=False,
    col_wrap=5,
    sharey=False,
    height=1.7,
    aspect=0.85,
    linewidth=0.8,
    saturation=1,
)

# =============================
# Helper to compute patch x-center
# =============================
def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2

# =============================
# Helper to compute patch x-range
# =============================
def get_patch_x_range(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return np.min(x_coords), np.max(x_coords)

# =============================
# Apply robust edge/fill colors
# =============================
for ax in g.axes.flatten():
    box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

    # Recolor each box based on its x position
    for patch in box_patches:
        x_center = get_patch_x_center(patch)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(order)) - x_center)))
        cluster = order[nearest_idx]

        patch.set_facecolor(fill_palette[cluster])
        patch.set_edgecolor(edge_palette[cluster])
        patch.set_linewidth(0.8)

    # Recolor associated lines based on overlap with each box x-range
    for line in ax.lines:
        xdata = np.asarray(line.get_xdata(), dtype=float)
        if xdata.size == 0 or np.any(~np.isfinite(xdata)):
            continue

        x_mid = np.mean(xdata)

        nearest_idx = int(np.argmin(np.abs(np.arange(len(order)) - x_mid)))
        cluster = order[nearest_idx]
        color = edge_palette[cluster]

        line.set_color(color)
        line.set_linewidth(0.8)

# =============================
# Significance annotation helpers
# =============================
def p_to_star(p):
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def add_star(ax, x1, x2, y, pval):
    stars = p_to_star(pval)
    if stars:
        ax.plot([x1, x2], [y, y], lw=0.8, c="black", clip_on=False)
        ax.text(
            (x1 + x2) / 2,
            y,
            stars,
            ha="center",
            va="bottom",
            fontsize=7,
            color="black",
        )

# =============================
# Add significance: C5 vs others
# =============================
ref_cluster = "5"

if ref_cluster not in order:
    print(f"[Warning] Reference cluster {ref_cluster} is not in the cluster order. No significance test added.")

for ax in g.axes.flatten():
    title_text = ax.get_title()
    fstate = title_text.split(" = ")[-1]
    subdf = df_long[df_long["FunctionalState"] == fstate]

    clusters_here = [c for c in order if c in subdf["MI_louvain"].unique()]

    if (ref_cluster not in clusters_here) or (len(clusters_here) < 2):
        continue

    ref_vals = subdf[subdf["MI_louvain"] == ref_cluster]["Score"].to_numpy()

    y_max = np.nanmax(subdf["Score"].to_numpy())
    y_min = np.nanmin(subdf["Score"].to_numpy())
    y_range = y_max - y_min
    step = 0.08 * y_range if y_range > 0 else 0.05

    x_ref = clusters_here.index(ref_cluster)

    k = 0
    for other in clusters_here:
        if other == ref_cluster:
            continue

        other_vals = subdf[subdf["MI_louvain"] == other]["Score"].to_numpy()

        if (ref_vals.size == 0) or (other_vals.size == 0):
            continue

        _, pval = ranksums(ref_vals, other_vals)

        x_other = clusters_here.index(other)
        k += 1
        y = y_max + step * (k + 0.1)

        add_star(ax, x_ref, x_other, y, pval)

# =============================
# Final formatting
# =============================
plt.draw()

for ax in g.axes.flatten():
    title = ax.get_title().split(" = ")[-1]
    title = title.replace("_score", "").replace("_", " ")
    ax.set_title(title, fontsize=8, pad=2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

    ax.set_xlabel("")
    ax.set_ylabel("Score", fontsize=7)

    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(cluster_labels, rotation=45, ha="right", fontsize=6)

g.fig.supxlabel("Cluster", fontsize=8, y=0.03)
g.fig.subplots_adjust(wspace=0.25, hspace=0.45, bottom=0.18)

# =============================
# Save figures
# =============================
plt.savefig(
    f"{run_dirs['run_dir']}/FunctionalState_boxplot_MILouvain.png",
    dpi=300,
    bbox_inches="tight"
)
plt.savefig(
    f"{run_dirs['run_dir']}/FunctionalState_boxplot_MILouvain.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


In [ ]:
# ==============================================================
# CancerSEA functional module scores across malignant clusters
# --------------------------------------------------------------
# Scores:
#   1. Angiogenesis_score
#   2. Hypoxia_score
#   3. Metastasis_score
#
# For each functional state:
#   - Load CancerSEA_OV/functional_geneset_list_df.csv
#   - Use genes overlapped with current HGSOC expression matrix
#   - z-score each gene across malignant cells
#   - average z-scored genes as module score
#   - aggregate at sample_id × malignant cluster level
#   - compare Malignant C5 vs each other cluster at sample level
#   - annotate one row: vs C5: ** ns * **** - ** *
# ==============================================================

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import PathPatch
from scipy import sparse
from scipy.stats import ranksums

# =============================
# Settings
# =============================
CLUSTER_COL = "MI_louvain"

FUNCTIONAL_STATES_TO_SCORE = [
    "Angiogenesis",
    "Hypoxia",
    "Metastasis",
]

GENESET_PATH = DATA_ROOT / "CancerSEA_OV/functional_geneset_list_df.csv"

OUT_PREFIX = "Malignant_C5_CancerSEA_Angiogenesis_Hypoxia_Metastasis_score_by_cluster"

MIN_CELLS_PER_SAMPLE_CLUSTER = 10

# Reference cluster for pairwise annotation
REF_CLUSTER = "Malignant C5"

# One-sided Wilcoxon rank-sum test direction:
#   "greater": test whether C5 > other cluster
#   "less":    test whether C5 < other cluster
TEST_ALTERNATIVE = "greater"

out_dir = run_dirs["run_dir"] if "run_dirs" in globals() and "run_dir" in run_dirs else "."
os.makedirs(out_dir, exist_ok=True)

# =============================
# Publication-style plotting settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Helper functions
# =============================
def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


# =============================
# Check malignant cluster column
# =============================
if CLUSTER_COL not in adata_choose.obs.columns:
    raise KeyError(
        f"Cannot find '{CLUSTER_COL}' in adata_choose.obs. "
        f"Available columns are:\n{list(adata_choose.obs.columns)}"
    )

# =============================
# Load CancerSEA gene sets
# =============================
if not input_path(Path(GENESET_PATH)).exists():
    raise FileNotFoundError(
        f"Cannot find CancerSEA gene-set file:\n{GENESET_PATH}"
    )

geneset_df = pd.read_csv(input_path(GENESET_PATH))

required_cols = {"GeneSet", "Gene"}
missing_cols = required_cols - set(geneset_df.columns)
if len(missing_cols) > 0:
    raise KeyError(
        f"geneset_df is missing required columns: {missing_cols}. "
        f"Available columns: {list(geneset_df.columns)}"
    )

geneset_df = geneset_df[["GeneSet", "Gene"]].dropna().copy()
geneset_df["GeneSet"] = geneset_df["GeneSet"].astype(str)
geneset_df["Gene"] = geneset_df["Gene"].astype(str)

available_gene_sets = set(geneset_df["GeneSet"].unique())

missing_states = [
    s for s in FUNCTIONAL_STATES_TO_SCORE
    if s not in available_gene_sets
]

if len(missing_states) > 0:
    raise ValueError(
        "The following requested functional states are not found in geneset_df['GeneSet']:\n"
        f"{missing_states}\n"
        f"Available GeneSet values include:\n{sorted(list(available_gene_sets))[:50]}"
    )

functional_gene_df = geneset_df.loc[
    geneset_df["GeneSet"].isin(FUNCTIONAL_STATES_TO_SCORE),
    ["GeneSet", "Gene"]
].drop_duplicates().copy()

if functional_gene_df.empty:
    raise ValueError("No genes found for selected CancerSEA functional states.")

print("CancerSEA gene counts before expression extraction:")
display(
    functional_gene_df
    .groupby("GeneSet", as_index=False)
    .agg(n_genes=("Gene", "nunique"))
)

# =============================
# Malignant cluster labels
# =============================
cluster_clean = adata_choose.obs[CLUSTER_COL].astype(str).map(_clean_cluster_label).to_numpy()

cluster_order = sorted(
    np.unique(cluster_clean),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

display_order = [f"Malignant C{x}" for x in cluster_order]
cluster_display = np.array([f"Malignant C{x}" for x in cluster_clean])

# =============================
# Resolve expression source
# Prefer adata_choose; fallback to processed.adata_all if needed
# =============================
all_functional_genes = sorted(functional_gene_df["Gene"].astype(str).unique())

resolved_choose, missing_choose = _resolve_gene_names(adata_choose, all_functional_genes)

adata_all = None
malignant_full_idx = None

# Use adata_choose if it contains all selected functional genes
if len(missing_choose) == 0:
    expr_source = "adata_choose"
    expr_adata = adata_choose
    expr_rows = np.arange(adata_choose.n_obs)
    resolved_genes = resolved_choose

# Otherwise use processed.adata_all
else:
    print(
        f"[Info] {len(missing_choose)} functional-state genes are missing from adata_choose. "
        "Trying processed.adata_all..."
    )

    if "processed" not in globals() or not hasattr(processed, "adata_all"):
        raise ValueError(
            "Some functional-state genes are missing from adata_choose, "
            "and processed.adata_all is not available."
        )

    adata_all = processed.adata_all
    resolved_all, missing_all = _resolve_gene_names(adata_all, all_functional_genes)

    if len(resolved_all) == 0:
        raise ValueError(
            "None of the selected CancerSEA functional-state genes are found in processed.adata_all."
        )

    if "barcode" in adata_choose.obs.columns:
        malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

    adata_all_index = pd.Index(adata_all.obs_names.astype(str))
    malignant_full_idx = adata_all_index.get_indexer(malignant_barcodes)

    if np.any(malignant_full_idx < 0) and "barcode" in adata_all.obs.columns:
        all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()

        barcode_to_idx = {}
        for i, b in enumerate(all_barcodes):
            if b not in barcode_to_idx:
                barcode_to_idx[b] = i

        malignant_full_idx = np.array(
            [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
            dtype=int
        )

    missing_match = malignant_full_idx < 0

    if np.any(missing_match):
        raise ValueError(
            f"{missing_match.sum()} malignant cells could not be matched to processed.adata_all. "
            "Please check whether adata_choose.obs['barcode'] matches processed.adata_all.obs_names "
            "or processed.adata_all.obs['barcode']."
        )

    expr_source = "processed.adata_all"
    expr_adata = adata_all
    expr_rows = malignant_full_idx
    resolved_genes = resolved_all

print(f"Expression source: {expr_source}")
print(f"Number of functional-state genes available for scoring: {len(resolved_genes)}")

# Keep only genes actually available in expression matrix
available_genes = sorted(resolved_genes.keys())

functional_gene_df_available = functional_gene_df.loc[
    functional_gene_df["Gene"].astype(str).isin(available_genes)
].drop_duplicates().copy()

if functional_gene_df_available.empty:
    raise ValueError("No selected CancerSEA genes are available in the expression matrix.")

functional_gene_count_df = (
    functional_gene_df_available
    .groupby("GeneSet", as_index=False)
    .agg(n_genes_used=("Gene", "nunique"))
)

print("CancerSEA gene counts used for module scoring:")
display(functional_gene_count_df)

# =============================
# Extract expression matrix
# =============================
actual_gene_names = [resolved_genes[g] for g in available_genes]
gene_idx = [expr_adata.var_names.get_loc(g) for g in actual_gene_names]

X_sub = expr_adata.X[expr_rows, :][:, gene_idx]

if sparse.issparse(X_sub):
    X_sub = X_sub.toarray()
else:
    X_sub = np.asarray(X_sub)

expr_gene_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=adata_choose.obs_names,
)

# =============================
# z-score genes across malignant cells
# =============================
gene_mean = expr_gene_df.mean(axis=0)
gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)

expr_z = (expr_gene_df - gene_mean) / gene_std
expr_z = expr_z.fillna(0)

# =============================
# Compute functional-state module scores
# average z-scored genes within each functional state
# =============================
score_records = []

for state in FUNCTIONAL_STATES_TO_SCORE:
    genes_cur = (
        functional_gene_df_available
        .loc[functional_gene_df_available["GeneSet"] == state, "Gene"]
        .astype(str)
        .unique()
        .tolist()
    )

    genes_cur = [g for g in genes_cur if g in expr_z.columns]

    if len(genes_cur) == 0:
        print(f"[Warning] No available genes for functional state: {state}. Skipping.")
        continue

    score_cur = expr_z[genes_cur].mean(axis=1).to_numpy()

    tmp = pd.DataFrame({
        "barcode": (
            adata_choose.obs["barcode"].astype(str).to_numpy()
            if "barcode" in adata_choose.obs.columns
            else adata_choose.obs_names.astype(str).to_numpy()
        ),
        "MI_louvain": cluster_clean,
        "MalignantCluster": cluster_display,
        "FunctionalState": state,
        "n_genes_used": len(genes_cur),
        "functional_state_module_score": score_cur,
    })

    score_records.append(tmp)

    # also save each functional-state score into adata_choose.obs
    adata_choose.obs[state] = score_cur

score_df = pd.concat(score_records, axis=0, ignore_index=True)

score_df["MalignantCluster"] = pd.Categorical(
    score_df["MalignantCluster"],
    categories=display_order,
    ordered=True,
)

score_df["FunctionalState"] = pd.Categorical(
    score_df["FunctionalState"],
    categories=FUNCTIONAL_STATES_TO_SCORE,
    ordered=True,
)

# =============================
# Add sample information
# =============================
sample_col_candidates = [
    "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
    "patient", "patients", "Patient", "Patient_ID", "library_id", "slide", "slice"
]

sample_col_choose = next(
    (col for col in sample_col_candidates if col in adata_choose.obs.columns),
    None
)

if sample_col_choose is not None:
    sample_values = adata_choose.obs[sample_col_choose].astype(str).to_numpy()
    print(f"Using sample column from adata_choose: {sample_col_choose}")
else:
    if adata_all is None and "processed" in globals() and hasattr(processed, "adata_all"):
        adata_all = processed.adata_all

        if "barcode" in adata_choose.obs.columns:
            malignant_barcodes = adata_choose.obs["barcode"].astype(str).to_numpy()
        else:
            malignant_barcodes = adata_choose.obs_names.astype(str).to_numpy()

        adata_all_index = pd.Index(adata_all.obs_names.astype(str))
        malignant_full_idx_tmp = adata_all_index.get_indexer(malignant_barcodes)

        if np.any(malignant_full_idx_tmp < 0) and "barcode" in adata_all.obs.columns:
            all_barcodes = adata_all.obs["barcode"].astype(str).to_numpy()
            barcode_to_idx = {}
            for i, b in enumerate(all_barcodes):
                if b not in barcode_to_idx:
                    barcode_to_idx[b] = i

            malignant_full_idx_tmp = np.array(
                [barcode_to_idx.get(b, -1) for b in malignant_barcodes],
                dtype=int
            )

        if not np.any(malignant_full_idx_tmp < 0):
            malignant_full_idx = malignant_full_idx_tmp

    sample_values = None

    if adata_all is not None and malignant_full_idx is not None:
        sample_col_all = next(
            (col for col in sample_col_candidates if col in adata_all.obs.columns),
            None
        )

        if sample_col_all is not None:
            sample_values = (
                adata_all.obs
                .iloc[malignant_full_idx][sample_col_all]
                .astype(str)
                .to_numpy()
            )
            print(f"Using sample column from processed.adata_all: {sample_col_all}")

    if sample_values is None:
        print(
            "[Warning] No sample column found. "
            "All malignant cells will be treated as one pseudo-sample."
        )
        sample_values = np.array(["__all_cells__"] * adata_choose.n_obs)

barcode_to_sample = pd.Series(
    sample_values,
    index=(
        adata_choose.obs["barcode"].astype(str).to_numpy()
        if "barcode" in adata_choose.obs.columns
        else adata_choose.obs_names.astype(str).to_numpy()
    )
)

score_df["sample_id"] = score_df["barcode"].map(barcode_to_sample).astype(str)

# =============================
# Sample-level aggregation
# =============================
sample_score_df = (
    score_df
    .groupby(["sample_id", "MalignantCluster", "FunctionalState"], observed=True)
    .agg(
        mean_functional_state_score=("functional_state_module_score", "mean"),
        median_functional_state_score=("functional_state_module_score", "median"),
        n_cells=("barcode", "count"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)

sample_score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_all.csv"),
    index=False,
)

sample_score_df_plot = sample_score_df.loc[
    sample_score_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_CLUSTER
].copy()

sample_score_df_plot["MalignantCluster"] = pd.Categorical(
    sample_score_df_plot["MalignantCluster"],
    categories=display_order,
    ordered=True,
)

sample_score_df_plot["FunctionalState"] = pd.Categorical(
    sample_score_df_plot["FunctionalState"],
    categories=FUNCTIONAL_STATES_TO_SCORE,
    ordered=True,
)

# =============================
# Summary + C5 vs others p-values
# =============================
cluster_summary_df = (
    sample_score_df_plot
    .groupby(["FunctionalState", "MalignantCluster"], observed=True)
    .agg(
        n_samples=("sample_id", "nunique"),
        mean_sample_level_score=("mean_functional_state_score", "mean"),
        median_sample_level_score=("mean_functional_state_score", "median"),
        std_sample_level_score=("mean_functional_state_score", "std"),
        total_cells=("n_cells", "sum"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)

pval_records = []

for state in FUNCTIONAL_STATES_TO_SCORE:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["FunctionalState"].astype(str) == state
    ].copy()

    for cluster in display_order:
        x = sub.loc[
            sub["MalignantCluster"].astype(str) == cluster,
            "mean_functional_state_score"
        ].to_numpy(dtype=float)

        y = sub.loc[
            sub["MalignantCluster"].astype(str) != cluster,
            "mean_functional_state_score"
        ].to_numpy(dtype=float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) > 0 and len(y) > 0:
            _, p_greater = ranksums(x, y, alternative="greater")
            _, p_less = ranksums(x, y, alternative="less")
            _, p_two = ranksums(x, y, alternative="two-sided")
        else:
            p_greater = np.nan
            p_less = np.nan
            p_two = np.nan

        pval_records.append({
            "FunctionalState": state,
            "MalignantCluster": cluster,
            "n_sample_cluster_values": len(x),
            "n_other_sample_cluster_values": len(y),
            "p_greater_than_other_clusters": p_greater,
            "p_less_than_other_clusters": p_less,
            "p_two_sided": p_two,
        })

pval_df = pd.DataFrame(pval_records)

# =============================
# Pairwise Wilcoxon rank-sum test:
# Malignant C5 vs each other malignant cluster
# one-sided, separately for each functional state
# =============================
pairwise_c5_records = []

for state in FUNCTIONAL_STATES_TO_SCORE:
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["FunctionalState"].astype(str) == state
    ].copy()

    if REF_CLUSTER not in sub["MalignantCluster"].astype(str).unique():
        print(f"[Warning] {REF_CLUSTER} is not found for {state}. Skip pairwise C5 tests.")
        continue

    ref_vals = sub.loc[
        sub["MalignantCluster"].astype(str) == REF_CLUSTER,
        "mean_functional_state_score"
    ].to_numpy(dtype=float)
    ref_vals = ref_vals[np.isfinite(ref_vals)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            p_one = np.nan
            star = "-"
            n_other = len(ref_vals)
        else:
            other_vals = sub.loc[
                sub["MalignantCluster"].astype(str) == cluster,
                "mean_functional_state_score"
            ].to_numpy(dtype=float)
            other_vals = other_vals[np.isfinite(other_vals)]

            if len(ref_vals) > 0 and len(other_vals) > 0:
                stat, p_one = ranksums(
                    ref_vals,
                    other_vals,
                    alternative=TEST_ALTERNATIVE,
                )
            else:
                stat = np.nan
                p_one = np.nan

            star = p_to_star_with_ns(p_one)
            n_other = len(other_vals)

        pairwise_c5_records.append({
            "FunctionalState": state,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "alternative": TEST_ALTERNATIVE,
            "n_reference": len(ref_vals),
            "n_other": n_other,
            "wilcoxon_rank_sum_statistic": stat,
            "p_one_sided": p_one,
            "star": star,
        })

pairwise_c5_pval_df = pd.DataFrame(pairwise_c5_records)

# =============================
# Save outputs
# =============================
score_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_celllevel_score.csv"),
    index=False,
)

sample_score_df_plot.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_score_min{MIN_CELLS_PER_SAMPLE_CLUSTER}cells.csv"),
    index=False,
)

cluster_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_summary.csv"),
    index=False,
)

pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_cluster_vs_others_ranksum_pvalues.csv"),
    index=False,
)

pairwise_c5_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_C5_vs_each_cluster_ranksum_pvalues_one_sided.csv"),
    index=False,
)

functional_gene_count_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_functional_gene_count_used_for_scoring.csv"),
    index=False,
)

functional_gene_df_available.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_functional_genes_used.csv"),
    index=False,
)

expr_z_out = expr_z.copy()
expr_z_out.columns = [f"{g}_z" for g in expr_z_out.columns]
expr_z_out["barcode"] = (
    adata_choose.obs["barcode"].astype(str).to_numpy()
    if "barcode" in adata_choose.obs.columns
    else adata_choose.obs_names.astype(str).to_numpy()
)
expr_z_out["sample_id"] = sample_values
expr_z_out["MalignantCluster"] = cluster_display

expr_z_out.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX}_gene_zscore_used_for_scoring.csv"),
    index=False,
)

if "adata_choose_path" in globals():
    try:
        adata_choose.write_h5ad(adata_choose_path)
    except Exception as e:
        print(f"[Warning] Could not save adata_choose to adata_choose_path: {e}")

print("Sample-level functional-state score table:")
display(sample_score_df_plot.head())

print("Cluster summary:")
display(cluster_summary_df)

print("Cluster vs others rank-sum p-values:")
display(pval_df)

print("Pairwise C5 vs each cluster one-sided rank-sum p-values:")
display(pairwise_c5_pval_df)

# =============================
# Plot: sample-level boxplot faceted by functional state
# =============================
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}

g = sns.catplot(
    data=sample_score_df_plot,
    x="MalignantCluster",
    y="mean_functional_state_score",
    col="FunctionalState",
    col_wrap=3,
    order=display_order,
    kind="box",
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    linewidth=0.8,
    width=0.65,
    height=1.45,
    aspect=1.15,
    sharey=False,
    saturation=1,
)

# =============================
# Apply robust fill and edge colors
# =============================
for ax in g.axes.flat:
    box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

    for patch in box_patches:
        x_center = get_patch_x_center(patch)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
        cluster = display_order[nearest_idx]

        patch.set_facecolor(fill_palette[cluster])
        patch.set_edgecolor(edge_palette[cluster])
        patch.set_linewidth(0.8)

    for line in ax.lines:
        xdata = np.asarray(line.get_xdata(), dtype=float)

        if xdata.size == 0 or np.any(~np.isfinite(xdata)):
            continue

        x_mid = np.mean(xdata)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
        cluster = display_order[nearest_idx]

        line.set_color(edge_palette[cluster])
        line.set_linewidth(0.8)

# =============================
# Final formatting + one-line annotation
# =============================
short_title_map = {
    "Angiogenesis": "Angiogenesis",
    "Hypoxia": "Hypoxia",
    "Metastasis": "Metastasis",
}

for ax, state in zip(g.axes.flat, FUNCTIONAL_STATES_TO_SCORE):
    sub = sample_score_df_plot.loc[
        sample_score_df_plot["FunctionalState"].astype(str) == state
    ].copy()

    if sub.shape[0] > 0:
        y_min = sub["mean_functional_state_score"].min()
        y_max = sub["mean_functional_state_score"].max()
        y_range = y_max - y_min if y_max > y_min else 1.0

        y_annot = y_max + 0.08 * y_range

        pairwise_plot_df = pairwise_c5_pval_df.loc[
            pairwise_c5_pval_df["FunctionalState"].astype(str) == state
        ].copy()

        pairwise_plot_df["other_cluster"] = pd.Categorical(
            pairwise_plot_df["other_cluster"],
            categories=display_order,
            ordered=True,
        )
        pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

        star_map = dict(zip(
            pairwise_plot_df["other_cluster"].astype(str),
            pairwise_plot_df["star"].astype(str),
        ))

        ax.text(
            -0.72,
            y_annot,
            "vs C5:",
            ha="right",
            va="center",
            fontsize=5.5,
            color="black",
            clip_on=False,
        )

        for i, cluster in enumerate(display_order):
            annot_text = star_map.get(cluster, "NA")
            ax.text(
                i,
                y_annot,
                annot_text,
                ha="center",
                va="center",
                fontsize=5.5,
                color="black",
                clip_on=False,
            )

        ax.set_ylim(
            y_min - 0.08 * y_range,
            y_max + 0.18 * y_range,
        )

    ax.set_title(short_title_map.get(state, state), fontsize=7)
    ax.set_xlabel("")
    ax.set_ylabel("Functional module score\nsample-level mean z-scored genes")

    ax.set_xticklabels(display_order, rotation=35, ha="right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    "CancerSEA functional module scores across malignant clusters",
    y=1.08,
    fontsize=8,
)

plt.tight_layout()

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    os.path.join(out_dir, f"{OUT_PREFIX}_samplelevel_boxplot.pdf"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()


In [ ]:
# ==============================================================
# Single-cell-level comparison: CancerSEA functional module scores
# --------------------------------------------------------------
# This complements the previous sample-level functional-state comparison.
# No points are drawn to keep the plot readable for many cells.
#
# Plot style:
#   - Same cluster-specific fill + edge colors as sample-level plot
#   - Box/median/whisker lines colored by malignant cluster
#   - One-line C5 comparison annotation for each functional state:
#       "vs C5: ** ns * **** - ** *"
#
# Statistics:
#   - Cell-level Wilcoxon rank-sum test
#   - REF_CLUSTER vs each other malignant cluster
# ==============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from scipy.stats import ranksums

OUT_PREFIX_SINGLECELL = f"{OUT_PREFIX}_singlecell"

REF_CLUSTER = "Malignant C5"
TEST_ALTERNATIVE_SINGLECELL = TEST_ALTERNATIVE if "TEST_ALTERNATIVE" in globals() else "greater"

if "score_df" not in globals() or "functional_state_module_score" not in score_df.columns:
    raise NameError(
        "Run the CancerSEA functional module score cell above before this single-cell comparison cell."
    )

if "FUNCTIONAL_STATES_TO_SCORE" not in globals():
    raise NameError("FUNCTIONAL_STATES_TO_SCORE is not found. Please run the score cell above first.")

if "display_order" not in globals():
    raise NameError("display_order is not found. Please run the score cell above first.")

if "out_dir" not in globals():
    out_dir = "."

os.makedirs(out_dir, exist_ok=True)


# =============================
# Helper functions
# =============================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star_with_ns(p):
    if not np.isfinite(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _p_to_label(p):
    if not np.isfinite(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


def get_patch_x_center(patch):
    verts = patch.get_path().vertices
    verts_disp = patch.get_transform().transform(verts)
    verts_data = patch.axes.transData.inverted().transform(verts_disp)
    x_coords = verts_data[:, 0]
    return (np.min(x_coords) + np.max(x_coords)) / 2


# =============================
# Prepare single-cell dataframe
# =============================
single_cell_df = score_df.copy()

single_cell_df["functional_state_module_score"] = pd.to_numeric(
    single_cell_df["functional_state_module_score"],
    errors="coerce",
)

single_cell_df = single_cell_df.loc[
    np.isfinite(single_cell_df["functional_state_module_score"].to_numpy(dtype=float))
].copy()

single_cell_df["MalignantCluster"] = pd.Categorical(
    single_cell_df["MalignantCluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_cell_df["FunctionalState"] = pd.Categorical(
    single_cell_df["FunctionalState"].astype(str),
    categories=FUNCTIONAL_STATES_TO_SCORE,
    ordered=True,
)

single_cell_df = single_cell_df.loc[
    single_cell_df["MalignantCluster"].notna()
    & single_cell_df["FunctionalState"].notna()
].copy()

single_cell_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_celllevel_scores.csv"),
    index=False,
)


# =============================
# Single-cell summary
# =============================
single_summary_df = (
    single_cell_df
    .groupby(["FunctionalState", "MalignantCluster"], observed=True)
    .agg(
        n_cells=("barcode", "count"),
        n_samples=("sample_id", "nunique"),
        mean_functional_state_score=("functional_state_module_score", "mean"),
        median_functional_state_score=("functional_state_module_score", "median"),
        std_functional_state_score=("functional_state_module_score", "std"),
        n_genes_used=("n_genes_used", "first"),
    )
    .reset_index()
)


# =============================
# Cell-level C5 vs each malignant cluster
# =============================
single_pval_records = []

for state in FUNCTIONAL_STATES_TO_SCORE:
    sub_state = single_cell_df.loc[
        single_cell_df["FunctionalState"].astype(str) == state
    ].copy()

    ref_values = sub_state.loc[
        sub_state["MalignantCluster"].astype(str) == REF_CLUSTER,
        "functional_state_module_score",
    ].to_numpy(dtype=float)

    ref_values = ref_values[np.isfinite(ref_values)]

    for cluster in display_order:
        if cluster == REF_CLUSTER:
            stat = np.nan
            pval = np.nan
            star = "-"
            p_label = "-"
            n_other = len(ref_values)
        else:
            other_values = sub_state.loc[
                sub_state["MalignantCluster"].astype(str) == cluster,
                "functional_state_module_score",
            ].to_numpy(dtype=float)

            other_values = other_values[np.isfinite(other_values)]

            if len(ref_values) > 0 and len(other_values) > 0:
                stat, pval = ranksums(
                    ref_values,
                    other_values,
                    alternative=TEST_ALTERNATIVE_SINGLECELL,
                )
            else:
                stat, pval = np.nan, np.nan

            star = p_to_star_with_ns(pval)
            p_label = _p_to_label(pval)
            n_other = len(other_values)

        single_pval_records.append({
            "FunctionalState": state,
            "comparison": f"{REF_CLUSTER} vs {cluster}",
            "reference_cluster": REF_CLUSTER,
            "other_cluster": cluster,
            "test": "cell-level Wilcoxon rank-sum",
            "alternative": TEST_ALTERNATIVE_SINGLECELL,
            "n_cells_ref": len(ref_values),
            "n_cells_other": n_other,
            "statistic": stat,
            "pvalue": pval,
            "pvalue_label": p_label,
            "star": star,
        })

single_pval_df = pd.DataFrame(single_pval_records)

single_pval_df["FunctionalState"] = pd.Categorical(
    single_pval_df["FunctionalState"].astype(str),
    categories=FUNCTIONAL_STATES_TO_SCORE,
    ordered=True,
)

single_pval_df["other_cluster"] = pd.Categorical(
    single_pval_df["other_cluster"].astype(str),
    categories=display_order,
    ordered=True,
)

single_pval_df = single_pval_df.sort_values(
    ["FunctionalState", "other_cluster"]
).reset_index(drop=True)


# =============================
# Save outputs
# =============================
single_summary_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_summary.csv"),
    index=False,
)

single_pval_df.to_csv(
    os.path.join(out_dir, f"{OUT_PREFIX_SINGLECELL}_C5_vs_each_cluster_pvalues.csv"),
    index=False,
)


# =============================
# Reference-style cluster color palettes
# =============================
edge_colors = [
    "#82CCE2", "#D1CABE", "#2E7FB9", "#FED881",
    "#9F3B38", "#519384", "#636491", "#8C7DB8"
]

fill_colors = [
    "#D4ECF1", "#EFE8E5", "#B7CCE5", "#FFF2D2",
    "#E1B6A7", "#B9CEC7", "#A6A2B9", "#D7CFE8"
]

edge_palette = {
    display_order[i]: edge_colors[i % len(edge_colors)]
    for i in range(len(display_order))
}

fill_palette = {
    display_order[i]: fill_colors[i % len(fill_colors)]
    for i in range(len(display_order))
}


# =============================
# Plot: single-cell boxplot faceted by functional state
# =============================
plt.close("all")

g = sns.catplot(
    data=single_cell_df,
    x="MalignantCluster",
    y="functional_state_module_score",
    col="FunctionalState",
    col_wrap=3,
    col_order=FUNCTIONAL_STATES_TO_SCORE,
    order=display_order,
    kind="box",
    palette=fill_palette,
    showfliers=False,
    showcaps=False,
    linewidth=0.8,
    width=0.65,
    height=1.65,
    aspect=1.20,
    sharey=False,
    saturation=1,
)


# =============================
# Apply robust fill and edge colors
# Also recolor median / whisker lines by cluster
# =============================
for ax in g.axes.flat:
    box_patches = [p for p in ax.patches if isinstance(p, PathPatch)]

    for patch in box_patches:
        x_center = get_patch_x_center(patch)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_center)))
        cluster = display_order[nearest_idx]

        patch.set_facecolor(fill_palette[cluster])
        patch.set_edgecolor(edge_palette[cluster])
        patch.set_linewidth(0.8)

    for line in ax.lines:
        xdata = np.asarray(line.get_xdata(), dtype=float)

        if xdata.size == 0 or np.any(~np.isfinite(xdata)):
            continue

        x_mid = np.mean(xdata)
        nearest_idx = int(np.argmin(np.abs(np.arange(len(display_order)) - x_mid)))
        cluster = display_order[nearest_idx]

        line.set_color(edge_palette[cluster])
        line.set_linewidth(0.8)


# =============================
# Final formatting + one-line annotation for each functional state
# =============================
short_title_map = {
    "Angiogenesis": "Angiogenesis",
    "Hypoxia": "Hypoxia",
    "Metastasis": "Metastasis",
}

for ax, state in zip(g.axes.flat, FUNCTIONAL_STATES_TO_SCORE):
    sub = single_cell_df.loc[
        single_cell_df["FunctionalState"].astype(str) == state
    ].copy()

    if sub.shape[0] > 0:
        y_min = sub["functional_state_module_score"].min()
        y_max = sub["functional_state_module_score"].max()
        y_range = y_max - y_min if y_max > y_min else 1.0

        y_annot = y_max + 0.08 * y_range

        pairwise_plot_df = single_pval_df.loc[
            single_pval_df["FunctionalState"].astype(str) == state
        ].copy()

        pairwise_plot_df["other_cluster"] = pd.Categorical(
            pairwise_plot_df["other_cluster"].astype(str),
            categories=display_order,
            ordered=True,
        )

        pairwise_plot_df = pairwise_plot_df.sort_values("other_cluster")

        star_map = dict(zip(
            pairwise_plot_df["other_cluster"].astype(str),
            pairwise_plot_df["star"].astype(str),
        ))

        ax.text(
            -0.72,
            y_annot,
            "vs C5:",
            ha="right",
            va="center",
            fontsize=5.5,
            color="black",
            clip_on=False,
        )

        for i, cluster in enumerate(display_order):
            annot_text = star_map.get(cluster, "NA")

            ax.text(
                i,
                y_annot,
                annot_text,
                ha="center",
                va="center",
                fontsize=5.5,
                color="black",
                clip_on=False,
            )

        ax.set_ylim(
            y_min - 0.08 * y_range,
            y_max + 0.18 * y_range,
        )

    ax.set_title(short_title_map.get(state, state), fontsize=7)
    ax.set_xlabel("")
    ax.set_ylabel("Functional module score\nsingle-cell mean z-scored genes")

    ax.set_xticklabels(display_order, rotation=35, ha="right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
    ax.xaxis.grid(False)

g.fig.suptitle(
    "Single-cell CancerSEA functional module scores across malignant clusters",
    y=1.08,
    fontsize=8,
)

plt.tight_layout()

for ext in ["png", "pdf"]:
    g.fig.savefig(
        os.path.join(
            out_dir,
            f"{OUT_PREFIX_SINGLECELL}_boxplot_no_points_with_C5_test.{ext}"
        ),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

plt.show()
plt.close()


# =============================
# Display outputs
# =============================
print("Single-cell CancerSEA functional-state comparison summary:")
_display_df(single_summary_df)

print("Cell-level C5 vs each malignant cluster rank-sum test:")
_display_df(single_pval_df)

In [ ]:
# ==============================================================
# Patient-level functional-state heatmaps by MI_louvain cluster
# --------------------------------------------------------------
# For each functional state, draw one heatmap:
#   rows    = malignant-cell clusters (MI_louvain)
#   columns = patients / samples
#   value   = patient-wise z-scored median functional-state score
#             of malignant cells assigned to that cluster
#
# NaN values are shown as blank cells, but patient columns are NOT removed.
# ============================================================== 

import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams

# =============================
# Publication-style global settings
# =============================
plt.close("all")
plt.style.use("default")
sns.set_theme(style="white", context="paper")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 8,
    "xtick.labelsize": 5,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Functional-state score columns
# =============================
# Uses the ten *_score columns created in the first CancerSEA scoring block.
# The separate three-state comparison writes columns without the _score suffix.
# Patient/sample identity is selected from the candidate columns below.
if "functional_states" not in globals():
    functional_states = [
        "Angiogenesis_score", "Apoptosis_score", "Cell Cycle_score",
        "Differentiation_score", "EMT_score", "Hypoxia_score",
        "Inflammation_score", "Metastasis_score",
        "Quiescence_score", "Stemness_score"
    ]

missing_states = [s for s in functional_states if s not in adata_choose.obs.columns]
if len(missing_states) > 0:
    raise KeyError(
        "The following functional-state score columns are missing from "
        f"adata_choose.obs: {missing_states}. Please run the previous "
        "functional-state scoring cell first."
    )

required_cluster_col = "MI_louvain"
if required_cluster_col not in adata_choose.obs.columns:
    raise KeyError("adata_choose.obs must contain an 'MI_louvain' cluster column.")

# =============================
# Patient/sample column
# =============================
patient_col_candidates = [
    "patient", "Patient", "patient_id", "Patient_ID", "patientID", "PatientID",
    "patients", "Patients", "sample", "samples", "sample_id", "sample_ID",
    "Sample", "Sample_ID", "orig.ident", "library_id", "slide", "slice"
]

patient_col = next(
    (col for col in patient_col_candidates if col in adata_choose.obs.columns),
    None
)

if patient_col is None:
    raise KeyError(
        "Could not automatically find a patient/sample column in adata_choose.obs. "
        "Please set patient_col manually. Available columns are:\n"
        + ", ".join(map(str, adata_choose.obs.columns))
    )

print(f"Using patient/sample column: {patient_col}")

# =============================
# Prepare table
# =============================
plot_df = adata_choose.obs[[patient_col, required_cluster_col] + functional_states].copy()
plot_df[patient_col] = plot_df[patient_col].astype(str)
plot_df[required_cluster_col] = plot_df[required_cluster_col].astype(str)
plot_df = plot_df.dropna(subset=[patient_col, required_cluster_col])

# Clean cluster labels:
# allow either "5" or "C5" in adata_choose.obs["MI_louvain"]
def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x

plot_df[required_cluster_col] = plot_df[required_cluster_col].map(_clean_cluster_label)

# Fixed cluster display order:
# C5, C6, C7, C3, C2, C4, C1
cluster_order = ["5", "6", "7", "3", "2", "4", "1"]
cluster_display_order = [f"C{x}" for x in cluster_order]

# Natural sorting helper for patient order.
def _natural_key(x):
    parts = re.split(r"(\d+)", str(x))
    return [int(p) if p.isdigit() else p.lower() for p in parts]

patient_order = sorted(plot_df[patient_col].unique(), key=_natural_key)

# Check missing / extra clusters
clusters_present = set(plot_df[required_cluster_col].unique())

missing_requested_clusters = [c for c in cluster_order if c not in clusters_present]
extra_clusters = sorted(
    [c for c in clusters_present if c not in cluster_order],
    key=_natural_key
)

if len(missing_requested_clusters) > 0:
    print(
        "[Warning] These requested clusters are not present in the data and will be shown as empty rows: "
        + ", ".join([f"C{x}" for x in missing_requested_clusters])
    )

if len(extra_clusters) > 0:
    print(
        "[Note] These clusters are present but excluded because the requested order only includes "
        "C5, C6, C7, C3, C2, C4, C1: "
        + ", ".join([f"C{x}" for x in extra_clusters])
    )

# Median score for each patient x cluster x functional state.
median_long = (
    plot_df
    .groupby([patient_col, required_cluster_col], observed=True)[functional_states]
    .median()
    .reset_index()
)

# Also save the number of malignant cells contributing to each median value.
count_df = (
    plot_df
    .groupby([patient_col, required_cluster_col], observed=True)
    .size()
    .reset_index(name="n_malignant_cells")
)

median_long = median_long.merge(
    count_df,
    on=[patient_col, required_cluster_col],
    how="left"
)

out_dir = Path(run_dirs["run_dir"]) / "FunctionalState_patient_cluster_heatmaps_patientwise_zscore"
out_dir.mkdir(parents=True, exist_ok=True)

median_long.to_csv(
    out_dir / "FunctionalState_median_scores_by_patient_MI_louvain_long.csv",
    index=False
)

count_mat = (
    median_long
    .pivot(index=required_cluster_col, columns=patient_col, values="n_malignant_cells")
    .reindex(index=cluster_order, columns=patient_order)
)
count_mat.index = cluster_display_order
count_mat.to_csv(out_dir / "Malignant_cell_counts_by_patient_MI_louvain.csv")

# =============================
# Helper: patient-wise z-score
# =============================
def patientwise_zscore(mat):
    """
    Z-score each patient column across clusters.

    For each patient:
        z = (cluster_median - patient_mean_across_clusters) / patient_std_across_clusters

    NaNs are ignored when calculating mean/std.
    Columns with zero std are set to NaN to avoid artificial values.
    These all-NaN columns are kept and shown as blank columns in the heatmap.
    """
    col_mean = mat.mean(axis=0, skipna=True)
    col_std = mat.std(axis=0, skipna=True, ddof=0)

    # Avoid division by zero.
    # If one patient's cluster medians are all identical, that whole column becomes NaN.
    col_std = col_std.replace(0, np.nan)

    zmat = mat.sub(col_mean, axis=1).div(col_std, axis=1)
    return zmat

# =============================
# Draw one heatmap per functional state
# =============================
for fstate in functional_states:
    # Raw median matrix: clusters x patients
    mat_raw = (
        median_long
        .pivot(index=required_cluster_col, columns=patient_col, values=fstate)
        .reindex(index=cluster_order, columns=patient_order)
    )
    mat_raw.index = cluster_display_order

    # Patient-wise z-score matrix
    mat_z = patientwise_zscore(mat_raw)

    # IMPORTANT:
    # Keep the same patient columns across all functional states.
    # NaN values are shown as blank cells, but columns are NOT dropped.
    mat_z = mat_z.reindex(index=cluster_display_order, columns=patient_order)

    state_name = fstate.replace("_score", "").replace("_", " ")
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", state_name).strip("_")

    # Save both raw median and patient-wise z-scored matrices
    mat_raw.to_csv(out_dir / f"{safe_name}_median_score_heatmap_matrix_raw.csv")
    mat_z.to_csv(out_dir / f"{safe_name}_median_score_heatmap_matrix_patientwise_zscore.csv")

    fig_w = max(5.0, 0.22 * len(patient_order) + 1.8)
    fig_h = max(2.2, 0.34 * len(cluster_order) + 0.9)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        mat_z,
        ax=ax,
        cmap="coolwarm",
        center=0,
        linewidths=0.25,
        linecolor="white",
        mask=mat_z.isna(),
        xticklabels=True,
        yticklabels=True,
        cbar_kws={
            "label": "Patient-wise z-scored median score",
            "shrink": 0.75
        },
    )

    ax.set_title(
        f"{state_name}: patient-wise z-scored median score",
        fontsize=8,
        pad=4
    )
    ax.set_xlabel("Patient", fontsize=7)
    ax.set_ylabel("Cluster", fontsize=7)
    ax.tick_params(axis="x", rotation=90, labelsize=5, length=2)
    ax.tick_params(axis="y", rotation=0, labelsize=6, length=2)

    plt.tight_layout()

    fig.savefig(
        out_dir / f"FunctionalState_{safe_name}_patientwise_zscore_by_patient_MI_louvain_heatmap.png",
        dpi=300,
        bbox_inches="tight"
    )
    fig.savefig(
        out_dir / f"FunctionalState_{safe_name}_patientwise_zscore_by_patient_MI_louvain_heatmap.pdf",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

print(f"Saved patient-wise z-scored heatmap matrices and figures to: {out_dir}")


### 5.6. Anatomical-site and disease-stage composition

Proportions are computed over cells within each malignant cluster.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

# -----------------------------
# Required font settings
# -----------------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

# -----------------------------
# Publication-style defaults
# -----------------------------
plt.close("all")
plt.style.use("default")
rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Barplot of Omentum proportion by cluster
# =============================

# Extract relevant columns
df = adata_choose.obs[['MI_louvain', 'sites_binary_x']].copy()
df["MI_louvain"] = df["MI_louvain"].astype(str)

# Compute the proportion of Omentum cells in each cluster
prop_table = (
    df.groupby('MI_louvain')['sites_binary_x']
      .value_counts(normalize=True)
      .rename("proportion")
      .reset_index()
)

# Filter for Omentum only
prop_oment = prop_table[prop_table['sites_binary_x'] == 'Omentum'].copy()

# Order clusters to match UMAP color palette
order = ['1', '2', '3', '4', '5', '6','7']
colors = ['#82CCE2', '#D1CABE', '#2E7FB9', '#FED881', '#9F3B38', '#519384', '#636491']

# Ensure all clusters exist (fill missing with 0)
prop_oment = (
    prop_oment.set_index('MI_louvain')
              .reindex(order)
              .reset_index()
)
prop_oment["proportion"] = prop_oment["proportion"].fillna(0.0)

# Format cluster labels as Malignant C1-C7.
prop_oment['ClusterName'] = [f"Malignant C{int(c)}" for c in prop_oment['MI_louvain']]

# -----------------------------
# Plot (publication style)
# -----------------------------
fig, ax = plt.subplots(figsize=(3.2, 2.6))

ax.bar(
    prop_oment['ClusterName'],
    prop_oment['proportion'],
    color=colors,
    edgecolor="black",
    linewidth=0.8
)

# Clean axes (Plot)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_ylabel("Proportion in Omentum")
ax.set_xlabel("Clusters")

# subtle y-grid only
ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.xticks(rotation=30, ha="right")

# # Optional: set y-limit to [0,1] since it's a proportion
# ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_Omentum_proportion_by_MI_louvain.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_Omentum_proportion_by_MI_louvain.pdf",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
plt.close()


In [ ]:
# Cell-weighted stage IV composition; no unique-patient aggregation is applied.

import matplotlib.pyplot as plt
from matplotlib import rcParams

# -----------------------------
# Required font settings
# -----------------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

# -----------------------------
# Publication-style defaults
# -----------------------------
plt.close("all")
plt.style.use("default")
rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =============================
# Barplot of Stage IV proportion by cluster
# =============================

# Extract relevant columns
df = adata_choose.obs[['MI_louvain', 'stage_x']].copy()
df["MI_louvain"] = df["MI_louvain"].astype(str)

# Compute the proportion of Stage IV cells in each cluster
prop_table = (
    df.groupby('MI_louvain')['stage_x']
      .value_counts(normalize=True)
      .rename("proportion")
      .reset_index()
)

# Filter for Stage IV only
prop_stage4 = prop_table[prop_table['stage_x'] == 'IV'].copy()

# Order clusters to match UMAP color palette
order = ['1', '2', '3', '4', '5', '6','7']
colors = ['#82CCE2', '#D1CABE', '#2E7FB9', '#FED881', '#9F3B38', '#519384', '#636491']
# Ensure all clusters exist (fill missing with 0)
prop_stage4 = (
    prop_stage4.set_index('MI_louvain')
               .reindex(order)
               .reset_index()
)
prop_stage4["proportion"] = prop_stage4["proportion"].fillna(0.0)

# Format cluster labels as Malignant C1-C7.
prop_stage4['ClusterName'] = [f"Malignant C{int(c)}" for c in prop_stage4['MI_louvain']]

# -----------------------------
# Plot (publication style)
# -----------------------------
fig, ax = plt.subplots(figsize=(3.2, 2.6))

ax.bar(
    prop_stage4['ClusterName'],
    prop_stage4['proportion'],
    color=colors,
    edgecolor="black",
    linewidth=0.8
)

# Clean axes (Plot)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", direction="out")

ax.set_ylabel("Proportion of Stage IV")
ax.set_xlabel("Clusters")

# subtle y-grid only
ax.set_axisbelow(True)
ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.5, color="0.88")
ax.xaxis.grid(False)

plt.xticks(rotation=30, ha="right")

# # Optional: set y-limit to [0,1] since it's a proportion
# ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_StageIV_proportion_by_MI_louvain.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.savefig(
    f"{run_dirs['run_dir']}/Barplot_StageIV_proportion_by_MI_louvain.pdf",
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
plt.close()


### 5.7. Cluster marker genes, KEGG enrichment and reference-gene export

Upregulated and downregulated markers are retained separately. Only the upregulated sets enter KEGG enrichment. The final export block produces the four-pathway reference table consumed by the earlier module-score analyses.


In [ ]:
# ==============================================================
# Step 1. Identify top up/down-regulated marker genes per cluster
# ==============================================================

adata_use = adata_choose.copy()
sc.tl.rank_genes_groups(adata_use, groupby='MI_louvain', method='wilcoxon')

up_genes_per_cluster, down_genes_per_cluster = {}, {}
groups = adata_use.uns['rank_genes_groups']['names'].dtype.names

for group in groups:
    df = sc.get.rank_genes_groups_df(adata_use, group=group)

    # Up-regulated genes
    df_up = df[(df['logfoldchanges'] > 0.5) & (df['pvals_adj'] < 0.05)]
    df_up = df_up.sort_values(['pvals_adj', 'logfoldchanges'], ascending=[True, False]).head(100)
    up_genes_per_cluster[group] = df_up

    # Down-regulated genes
    df_down = df[(df['logfoldchanges'] < -0.5) & (df['pvals_adj'] < 0.05)]
    df_down = df_down.sort_values(['pvals_adj', 'logfoldchanges'], ascending=[True, True]).head(100)
    down_genes_per_cluster[group] = df_down

print("Up-regulated gene counts per cluster:",
      [len(v) for v in up_genes_per_cluster.values()])
print("Down-regulated gene counts per cluster:",
      [len(v) for v in down_genes_per_cluster.values()])


In [ ]:
##Save the up_genes_per_cluster['5'] as csv
up_genes_per_cluster['5'].to_csv(f"{run_dirs['run_dir']}/Upregulated_genes_cluster_5.csv", index=False)


In [ ]:
# ==============================================================
# Step 2. KEGG enrichment analysis for upregulated genes
# ==============================================================

import time
import random
import gseapy as gp

organism = "hsa"  # 'hsa' for human, 'mmu' for mouse
gene_set = f"KEGG_2021_{'Human' if organism == 'hsa' else 'Mouse'}"
organism_label = 'human' if organism == 'hsa' else 'mouse'

kegg_results = {}
failed_clusters = {}

for cluster, df_up in up_genes_per_cluster.items():
    if df_up.empty or 'names' not in df_up.columns:
        continue

    gene_list = (
        df_up['names']
        .dropna()
        .astype(str)
        .tolist()
    )

    if len(gene_list) == 0:
        continue

    max_retries = 5
    success = False

    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_set,
                organism=organism_label,
                outdir=None,
                cutoff=0.05
            )

            result_df = enr.results.sort_values('Adjusted P-value', ascending=True)
            kegg_results[cluster] = result_df
            success = True
            print(f"[OK] {cluster}: {len(result_df)} terms")
            break

        except Exception as e:
            err_msg = str(e)

            if "429" in err_msg:
                wait_time = min(60, (2 ** attempt) + random.uniform(1, 3))
                print(f"[Retry {attempt+1}/{max_retries}] {cluster}: rate limited (429), sleep {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"[Failed] {cluster}: {e}")
                failed_clusters[cluster] = err_msg
                break

    if not success and cluster not in failed_clusters:
        failed_clusters[cluster] = "Failed after maximum retries"

    # add a small delay even after success to reduce API pressure
    time.sleep(random.uniform(1.5, 3.0))

print(f"\nKEGG enrichment completed for {len(kegg_results)} clusters.")
print(f"Failed clusters: {len(failed_clusters)}")


In [ ]:
# ==============================================================
# Step 3. Collect all KEGG terms across clusters
# ==============================================================

all_kegg_terms = sorted({term for df in kegg_results.values() if 'Term' in df.columns for term in df['Term']})
print(f"Total unique KEGG terms: {len(all_kegg_terms)}")


In [ ]:
# ==============================================================
# Step 4. Build -log10(FDR) matrix for selected KEGG pathways
# ==============================================================

selected_terms = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
    "Antigen processing and presentation",
    "Cell adhesion molecules"
]

clusters = list(up_genes_per_cluster.keys())
neglogp_matrix = pd.DataFrame(0.0, index=clusters, columns=selected_terms)

for cluster in clusters:
    if cluster not in kegg_results:
        continue
    df = kegg_results[cluster]
    for term in selected_terms:
        row = df.loc[df['Term'] == term, 'Adjusted P-value']
        if not row.empty:
            pval = max(row.iloc[0], 1e-300)
            neglogp_matrix.loc[cluster, term] = -np.log10(pval)

neglogp_matrix = neglogp_matrix.astype(np.float32)
##Cluster Order
neglogp_matrix = neglogp_matrix.iloc[np.array([5,6,7,3,2,4,1])-1,:]
##
neglogp_matrix_z = (neglogp_matrix - neglogp_matrix.mean()) / neglogp_matrix.std(ddof=0)
##Save neglogp_matrix_z as csv
neglogp_matrix_z.to_csv(f"{run_dirs['run_dir']}/KEGG_enrichment_matrix_Malignant.csv")


In [ ]:
# ==============================================================
# Step 5. Plot z-scored heatmap (annotated with original values)
# ==============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

# colormap
cmap_byr = LinearSegmentedColormap.from_list(
    "blue_yellow_red",
    ["#497AB7","#BCD9EA","#FAFAC1","#FDB876","#DC3B2E"]
)

plt.close()
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    neglogp_matrix_z,
    annot=neglogp_matrix.round(2),
    fmt=".2f",
    cmap=cmap_byr,
    vmin=-2,
    vmax=2,
    cbar_kws={'label': 'Column-wise z-score of -log10(FDR)'},
    ax=ax
)

ax.set_xlabel("KEGG Pathways", fontsize=14)
ax.set_ylabel("Malignant Clusters", fontsize=14)

plt.title("KEGG Pathway Enrichment (Z-scored across clusters)", fontsize=16)

plt.tight_layout()

plt.savefig(
    f"{run_dirs['run_dir']}/KEGG_Enrichment_Heatmap_Malignant_Clusters_Zscore.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close()


In [ ]:
# ==============================================================
# Step 5b. Export data-overlapped reference genes for the four C5-enriched KEGG pathways
# --------------------------------------------------------------
# NOTE:
#   We first load the full KEGG reference gene sets for the four pathways,
#   then intersect them with the genes available in this HGSOC dataset.
#   The Gene column contains the dataset-matched symbols; Reference_Gene
#   retains the corresponding symbols from the KEGG reference library.
# ==============================================================

from pathlib import Path
import re
import difflib
import pandas as pd
import gseapy as gp

# Four aggressiveness-associated KEGG pathways highlighted for C5
c5_kegg_terms_for_tcga = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

# Reuse the same KEGG library used above, if available
kegg_library_name = gene_set if "gene_set" in globals() else "KEGG_2021_Human"
kegg_organism_label = organism_label if "organism_label" in globals() else "human"

def _normalize_kegg_term(x):
    """Normalize KEGG term names for robust matching across Enrichr/GSEApy labels."""
    x = str(x).strip()
    x = re.sub(r"\s*\([^)]*\)\s*$", "", x)
    x = re.sub(r"\s+", " ", x)
    return x.lower()

def _get_dataset_gene_universe():
    """
    Get genes available in the current HGSOC dataset.
    Prefer processed.adata_list[0].var_names if available, otherwise adata_choose.var_names.
    """
    if "processed" in globals() and hasattr(processed, "adata_list") and len(processed.adata_list) > 0:
        return pd.Index(processed.adata_list[0].var_names.astype(str))
    elif "adata_choose" in globals():
        return pd.Index(adata_choose.var_names.astype(str))
    else:
        raise NameError(
            "Cannot find dataset gene universe. Expected processed.adata_list[0].var_names "
            "or adata_choose.var_names."
        )

dataset_genes = _get_dataset_gene_universe()
dataset_gene_upper_to_original = {
    str(g).upper(): str(g)
    for g in dataset_genes
}

print(f"Loading KEGG reference library: {kegg_library_name} ({kegg_organism_label})")
print(f"Number of genes in current dataset: {len(dataset_genes)}")

try:
    kegg_reference_library = gp.get_library(
        name=kegg_library_name,
        organism=kegg_organism_label,
    )
except TypeError:
    # Compatibility with older gseapy versions
    kegg_reference_library = gp.get_library(name=kegg_library_name)

available_terms = list(kegg_reference_library.keys())
term_lookup = {term: term for term in available_terms}
term_lookup.update({_normalize_kegg_term(term): term for term in available_terms})

resolved_terms = {}
missing_terms = []

for term in c5_kegg_terms_for_tcga:
    if term in kegg_reference_library:
        resolved_terms[term] = term
    elif _normalize_kegg_term(term) in term_lookup:
        resolved_terms[term] = term_lookup[_normalize_kegg_term(term)]
    else:
        missing_terms.append(term)

if len(missing_terms) > 0:
    print("Could not find the following requested KEGG terms:")
    for term in missing_terms:
        close_matches = difflib.get_close_matches(
            term,
            available_terms,
            n=5,
            cutoff=0.35,
        )
        print(f"  - {term}")
        print(f"    closest matches: {close_matches}")
    raise KeyError("Some requested KEGG terms were not found in the reference library.")

# Long-format pathway-gene table:
# keep only genes that are both in the KEGG reference set and in the current dataset
kegg_reference_gene_rows = []
reference_count_rows = []

for requested_term, library_term in resolved_terms.items():
    reference_genes = sorted(set(map(str, kegg_reference_library[library_term])))
    reference_count_rows.append({
        "Requested_Pathway": requested_term,
        "Library_Pathway": library_term,
        "n_full_reference_genes": len(reference_genes),
    })

    for ref_gene in reference_genes:
        ref_gene_upper = ref_gene.upper()

        if ref_gene_upper not in dataset_gene_upper_to_original:
            continue

        data_gene = dataset_gene_upper_to_original[ref_gene_upper]

        kegg_reference_gene_rows.append({
            "Requested_Pathway": requested_term,
            "Library_Pathway": library_term,
            "Gene": data_gene,
            "Reference_Gene": ref_gene,
        })

c5_kegg_reference_gene_df = pd.DataFrame(kegg_reference_gene_rows).drop_duplicates()

if c5_kegg_reference_gene_df.empty:
    raise ValueError(
        "No overlapping genes were found between the current dataset and the selected KEGG reference gene sets."
    )

c5_kegg_reference_gene_union = sorted(c5_kegg_reference_gene_df["Gene"].unique())

# Save outputs
kegg_gene_outdir = Path(run_dirs["run_dir"]) / "TCGA_OV_KEGG_gene_signature"
kegg_gene_outdir.mkdir(parents=True, exist_ok=True)

# Export the pathway table, union and summary.
long_outfile = kegg_gene_outdir / "C5_four_KEGG_pathway_reference_genes_long.csv"
union_outfile = kegg_gene_outdir / "C5_four_KEGG_pathway_reference_gene_union.csv"
union_txt_outfile = kegg_gene_outdir / "C5_four_KEGG_pathway_reference_gene_union.txt"
summary_outfile = kegg_gene_outdir / "C5_four_KEGG_pathway_reference_gene_summary.csv"

c5_kegg_reference_gene_df.to_csv(long_outfile, index=False)
pd.DataFrame({"Gene": c5_kegg_reference_gene_union}).to_csv(union_outfile, index=False)

with open(union_txt_outfile, "w", encoding="utf-8") as f:
    f.write("\n".join(c5_kegg_reference_gene_union))

summary_df = (
    c5_kegg_reference_gene_df
    .groupby(["Requested_Pathway", "Library_Pathway"], as_index=False)
    .agg(n_genes_after_dataset_intersection=("Gene", "nunique"))
)

reference_count_df = pd.DataFrame(reference_count_rows)
summary_df = summary_df.merge(
    reference_count_df,
    on=["Requested_Pathway", "Library_Pathway"],
    how="left"
)

summary_df["n_genes_removed_not_in_dataset"] = (
    summary_df["n_full_reference_genes"] -
    summary_df["n_genes_after_dataset_intersection"]
)

summary_df.loc[len(summary_df)] = [
    "UNION_OF_FOUR_PATHWAYS",
    "UNION_OF_FOUR_PATHWAYS",
    len(c5_kegg_reference_gene_union),
    sum(reference_count_df["n_full_reference_genes"]),
    sum(reference_count_df["n_full_reference_genes"]) - len(c5_kegg_reference_gene_union),
]

summary_df.to_csv(summary_outfile, index=False)

print("Saved KEGG gene outputs after intersecting with current dataset genes:")
print(f"  Long pathway-gene table: {long_outfile}")
print(f"  Union gene CSV:          {union_outfile}")
print(f"  Union gene TXT:          {union_txt_outfile}")
print(f"  Summary table:           {summary_outfile}")
print(f"  Number of union genes after dataset intersection: {len(c5_kegg_reference_gene_union)}")

display(summary_df)


### 5.8. Mean sending and receiving MI strengths by malignant cluster


In [ ]:
# Reload the saved malignant AnnData and align cells to the full expression object.
# Columns added only in memory since the last save are not carried through this reload.

# Load data
Factor_envir_list = pd.read_pickle(input_path(run_dirs["run_dir"] + "/Factor_envir_list.pkl"))
SpiderNet_data_pyg_list = processed.spidernet_data
adata_copy = sc.read_h5ad(input_path(PROCESSED_DATA_DIR / "adata_all.h5ad"))
adata_choose = sc.read_h5ad(input_path(run_dirs["run_dir"] + "/adata_choose_Malignant.h5ad"))
adata_list = processed.adata_list

cellclass_unique = np.unique(adata_copy.obs["cell.types"])
if np.min(adata_choose.obs["MI_louvain"].astype(int)) == 0:
    adata_choose.obs["MI_louvain"] = (adata_choose.obs["MI_louvain"].astype(int) + 1).astype(str)

match_index = adata_copy.obs.index.get_indexer(adata_choose.obs["barcode"])
barcode_all = adata_copy.obs.index.tolist()

malignant_cluster = ["Malignant_" + str(c) for c in adata_choose.obs["MI_louvain"]]
malignant_map = dict(zip(adata_choose.obs["barcode"], malignant_cluster))
cellclass = adata_copy.obs["cell.types"].values
cellclass_updated = np.array([malignant_map.get(b, c) for b, c in zip(barcode_all, cellclass)])
cellclass_updated_df = pd.DataFrame({"barcode": barcode_all, "cell.types.updated": cellclass_updated}).set_index("barcode")


In [ ]:
## Compute MI sender/receiver aggregation per cell
from SpiderNet.analysis import aggregate_mi_sender_receiver
MI_SR_agg_cur_all, MI_SR_agg_ct_all, MI_agg_meta, MI_agg_ct_meta = aggregate_mi_sender_receiver(
    Factor_envir_list=Factor_envir_list,
    SpiderNet_data_pyg_list=SpiderNet_data_pyg_list,
    cellclass_unique=cellclass_unique,
    device=device,
    cellclass_updated_df=cellclass_updated_df,
    celltype_col="cell.types.updated",
)


In [ ]:
## Cluster-level MI aggregation and visualization
# Prepare data and assign proper column names
MI_SR_agg_cur_all_choose = MI_SR_agg_cur_all[match_index, :]
cluster_label = adata_choose.obs["MI_louvain"].astype(str).values

MI_column_names = [f"{row.MI}_{row.SR}" for _, row in MI_agg_meta.iterrows()]
df_MI = pd.DataFrame(
    MI_SR_agg_cur_all_choose,
    index=adata_choose.obs["barcode"],
    columns=MI_column_names
)
df_MI["cluster"] = cluster_label

# Compute mean per cluster
cluster_mean = df_MI.groupby("cluster").mean()
cluster_mean.index = [f"Cluster_{i}" for i in cluster_mean.index]

# --------------------------------------------------------------
# Reorder clusters and split sender/receiver MIs
# --------------------------------------------------------------
cluster_order = ["Cluster_5", "Cluster_6", "Cluster_7", "Cluster_3", "Cluster_2", "Cluster_1","Cluster_4"]
cluster_mean_order = cluster_mean.loc[[c for c in cluster_order if c in cluster_mean.index], :]

cluster_mean_order_sender = cluster_mean_order.loc[:, [col for col in cluster_mean_order.columns if "Sender" in col]]
cluster_mean_order_receiver = cluster_mean_order.loc[:, [col for col in cluster_mean_order.columns if "Receiver" in col]]


In [ ]:
##Merge cluster_mean_order_sender and cluster_mean_order_receiver
cluster_mean_order_all = pd.concat([cluster_mean_order_sender, cluster_mean_order_receiver], axis=1)
##Save the cluster_mean_order_all as csv
cluster_mean_order_all.to_csv(f"{run_dirs['run_dir']}/Mean_MI_Intensity_cluster.csv")


In [ ]:
# ==============================================================
# Cluster MI activity: upper triangle = sending, lower = receiving
# ==============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

# Align both directions by MI and cluster labels before changing display order.
mi_heatmap_sender = cluster_mean_order_sender.rename(
    columns=lambda name: str(name).removesuffix("_Sender")
).copy()
mi_heatmap_receiver = cluster_mean_order_receiver.rename(
    columns=lambda name: str(name).removesuffix("_Receiver")
).copy()

if not all(
    frame.index.is_unique and frame.columns.is_unique
    for frame in (mi_heatmap_sender, mi_heatmap_receiver)
):
    raise ValueError("Cluster and MI labels must be unique for the triangular heatmap.")
if (
    set(mi_heatmap_sender.index) != set(mi_heatmap_receiver.index)
    or set(mi_heatmap_sender.columns) != set(mi_heatmap_receiver.columns)
):
    raise ValueError("Sending and receiving tables must contain the same clusters and MIs.")

# Figure display order; append additional clusters/MIs without dropping values.
mi_heatmap_preferred_clusters = [
    "Cluster_5", "Cluster_6", "Cluster_7", "Cluster_2",
    "Cluster_3", "Cluster_4", "Cluster_1",
]
mi_heatmap_preferred_mis = [
    "MI6", "MI13", "MI12", "MI10", "MI15", "MI11", "MI7", "MI1",
    "MI9", "MI2", "MI3", "MI14", "MI5", "MI4", "MI8",
]
mi_heatmap_clusters = [
    name for name in mi_heatmap_preferred_clusters if name in mi_heatmap_sender.index
] + [name for name in mi_heatmap_sender.index if name not in mi_heatmap_preferred_clusters]
mi_heatmap_mis = [
    name for name in mi_heatmap_preferred_mis if name in mi_heatmap_sender.columns
] + [name for name in mi_heatmap_sender.columns if name not in mi_heatmap_preferred_mis]

mi_heatmap_sender = mi_heatmap_sender.loc[mi_heatmap_clusters, mi_heatmap_mis]
mi_heatmap_receiver = mi_heatmap_receiver.loc[mi_heatmap_clusters, mi_heatmap_mis]
mi_heatmap_nrows, mi_heatmap_ncols = mi_heatmap_sender.shape

# Use a shared 0-0.4 display range without rescaling the input values.
mi_heatmap_norm = Normalize(vmin=0, vmax=0.4, clip=True)
mi_heatmap_sender_cmap = LinearSegmentedColormap.from_list(
    "sending_white_purple", ["#FFFFFF", "#CBC5D7", "#6B66AE"]
)
mi_heatmap_receiver_cmap = LinearSegmentedColormap.from_list(
    "receiving_white_red", ["#FFFFFF", "#FBD9D9", "#F35B61"]
)
mi_heatmap_sender_cmap.set_bad("#FFFFFF")
mi_heatmap_receiver_cmap.set_bad("#FFFFFF")

# With the y-axis inverted, each diagonal runs from bottom-left to top-right.
mi_heatmap_upper = [
    [(col, row), (col + 1, row), (col, row + 1)]
    for row in range(mi_heatmap_nrows) for col in range(mi_heatmap_ncols)
]
mi_heatmap_lower = [
    [(col + 1, row), (col + 1, row + 1), (col, row + 1)]
    for row in range(mi_heatmap_nrows) for col in range(mi_heatmap_ncols)
]

with plt.rc_context({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
}):
    mi_heatmap_fig = plt.figure(figsize=(10.5, 8.8), facecolor="white")
    mi_heatmap_ax = mi_heatmap_fig.add_axes([0.16, 0.24, 0.80, 0.46])

    mi_heatmap_sending_triangles = PolyCollection(
        mi_heatmap_upper,
        array=np.ma.masked_invalid(mi_heatmap_sender.to_numpy(dtype=float).ravel()),
        cmap=mi_heatmap_sender_cmap,
        norm=mi_heatmap_norm,
        edgecolors="#D0D0D0",
        linewidths=0.65,
    )
    mi_heatmap_receiving_triangles = PolyCollection(
        mi_heatmap_lower,
        array=np.ma.masked_invalid(mi_heatmap_receiver.to_numpy(dtype=float).ravel()),
        cmap=mi_heatmap_receiver_cmap,
        norm=mi_heatmap_norm,
        edgecolors="#D0D0D0",
        linewidths=0.65,
    )
    mi_heatmap_ax.add_collection(mi_heatmap_sending_triangles)
    mi_heatmap_ax.add_collection(mi_heatmap_receiving_triangles)
    mi_heatmap_ax.set_xlim(0, mi_heatmap_ncols)
    mi_heatmap_ax.set_ylim(mi_heatmap_nrows, 0)

    mi_heatmap_ax.set_xticks(np.arange(mi_heatmap_ncols) + 0.5)
    mi_heatmap_ax.set_xticklabels(
        [name.replace("MI", "MI-", 1) for name in mi_heatmap_mis],
        rotation=60, ha="left", rotation_mode="anchor", fontsize=24,
    )
    mi_heatmap_ax.tick_params(
        axis="x", top=True, labeltop=True, bottom=False, labelbottom=False,
        length=0, pad=8,
    )
    mi_heatmap_ax.xaxis.set_label_position("top")
    mi_heatmap_ax.set_xlabel("Meta-interaction IDs", fontsize=29, labelpad=26)

    mi_heatmap_ax.set_yticks(np.arange(mi_heatmap_nrows) + 0.5)
    mi_heatmap_ax.set_yticklabels(
        [name.replace("Cluster_", "C", 1) for name in mi_heatmap_clusters],
        fontsize=26,
    )
    mi_heatmap_ax.tick_params(axis="y", length=0, pad=12)
    mi_heatmap_ax.set_ylabel("Malignant cluster", fontsize=29, labelpad=14)
    for mi_heatmap_label in mi_heatmap_ax.get_yticklabels():
        if mi_heatmap_label.get_text() == "C5":
            mi_heatmap_label.set_color("#AC3539")
            mi_heatmap_label.set_fontweight("bold")
    for mi_heatmap_spine in mi_heatmap_ax.spines.values():
        mi_heatmap_spine.set_visible(False)

    # Two compact horizontal legends share the same endpoints.
    mi_heatmap_sender_cax = mi_heatmap_fig.add_axes([0.69, 0.14, 0.18, 0.035])
    mi_heatmap_receiver_cax = mi_heatmap_fig.add_axes([0.69, 0.082, 0.18, 0.035])
    mi_heatmap_sender_cbar = mi_heatmap_fig.colorbar(
        mi_heatmap_sending_triangles, cax=mi_heatmap_sender_cax,
        orientation="horizontal", ticks=[],
    )
    mi_heatmap_receiver_cbar = mi_heatmap_fig.colorbar(
        mi_heatmap_receiving_triangles, cax=mi_heatmap_receiver_cax,
        orientation="horizontal", ticks=[0, 0.4],
    )
    mi_heatmap_sender_cbar.outline.set_visible(False)
    mi_heatmap_receiver_cbar.outline.set_visible(False)
    mi_heatmap_receiver_cbar.set_ticklabels(["0", "0.4"])
    mi_heatmap_receiver_cbar.ax.tick_params(length=0, labelsize=23, pad=6)
    mi_heatmap_fig.text(
        0.64, 0.1575, "Sending MI activity", ha="right", va="center", fontsize=29,
    )
    mi_heatmap_fig.text(
        0.64, 0.0995, "Receiving MI activity", ha="right", va="center", fontsize=29,
    )

    for mi_heatmap_extension in ("png", "pdf"):
        mi_heatmap_fig.savefig(
            run_dirs["run_dir"] + "/MI_Sending_Receiving_Activity_per_Cluster."
            + mi_heatmap_extension,
            dpi=300, bbox_inches="tight", facecolor="white",
        )
    plt.show()
    plt.close(mi_heatmap_fig)


### 5.9. Receiving MI strengths by sender cell type and spatial MI-10 maps


In [ ]:
# Define MI–direction pairs
MI_OI_df = pd.DataFrame({
    "MI": ['MI10', 'MI13', 'MI6', 'MI9', 'MI12'],
    "direction": ['Receiver'] * 5
})

# Custom colormap: white → green
cmap_custom = LinearSegmentedColormap.from_list("white_to_green", ["#FFFFFF", "#CCEAA8", "#0C6338"])

# Cluster display order (skip missing ones automatically)
cluster_order = ["Cluster_5", "Cluster_6", "Cluster_7", "Cluster_3", "Cluster_2", "Cluster_1","Cluster_4"]

# Create subplot figure
fig, axes = plt.subplots(1, MI_OI_df.shape[0], figsize=(6 * MI_OI_df.shape[0], 6), sharey=True)

for i, (MI_OI, direction_OI) in enumerate(zip(MI_OI_df['MI'], MI_OI_df['direction'])):
    # Find indices for current MI and direction
    MI_ct_index = MI_agg_ct_meta[
        (MI_agg_ct_meta['MI'] == MI_OI) & (MI_agg_ct_meta['SR'] == direction_OI)
    ].index

    if len(MI_ct_index) == 0:
        axes[i].axis('off')
        axes[i].set_title(f"No data for {MI_OI} {direction_OI}", fontsize=12)
        continue

    MI_meta_OI = MI_agg_ct_meta.loc[MI_ct_index, :].copy()
    MI_values = MI_SR_agg_ct_all[match_index, :][:, MI_ct_index]

    df_MI_ct = pd.DataFrame(MI_values, index=adata_choose.obs['barcode'])
    df_MI_ct['cluster'] = cluster_label

    # Compute cluster-wise mean
    cluster_ct_mean = df_MI_ct.groupby('cluster').mean(numeric_only=True)
    cluster_ct_mean.index = [f'Cluster_{idx}' for idx in cluster_ct_mean.index]

    # Assign cell types as column names
    if len(MI_meta_OI['celltype'].values) == cluster_ct_mean.shape[1]:
        cluster_ct_mean.columns = MI_meta_OI['celltype'].values
    else:
        cluster_ct_mean.columns = [f"celltype_{j+1}" for j in range(cluster_ct_mean.shape[1])]

    # Reorder clusters
    cluster_ct_mean = cluster_ct_mean.loc[[c for c in cluster_order if c in cluster_ct_mean.index], :]

    # Plot heatmap
    sns.heatmap(
        cluster_ct_mean.astype(float),
        cmap=cmap_custom,
        annot=False,
        fmt=".2f",
        vmax=0.7,
        vmin=0.0,
        ax=axes[i],
        cbar=(i == MI_OI_df.shape[0] - 1),
        cbar_kws={'label': f"Mean {MI_OI} {direction_OI} Level"}
    )

    # Titles and labels
    if direction_OI.lower() == "receiver":
        title = f"Receiving {MI_OI}"
        xlabel = "Sender Cell Type"
    else:
        title = f"Sending {MI_OI}"
        xlabel = "Receiver Cell Type"

    axes[i].set_title(title, fontsize=14)
    axes[i].set_xlabel(xlabel, fontsize=12)
    axes[i].set_ylabel("Clusters" if i == 0 else "")

# Save figure
plt.tight_layout()
plt.savefig(
    run_dirs['run_dir'] + "/Combined_MI_Levels_per_Cluster_CellType.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.savefig(
    run_dirs['run_dir'] + "/Combined_MI_Levels_per_Cluster_CellType.pdf",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close()


In [ ]:
# Spatial display uses MI-10 >=0.20, with a global 95th-percentile fallback if
# no edge passes. Per-slice plots retain at most the 1,000 strongest edges.
# The summary n_edges_shown column counts threshold-passing edges before this cap.

# ==============================================================
# In-situ plots for all slices:
# Fibroblast -> Malignant C5 through MI10
# --------------------------------------------------------------
# Output:
#   run_dirs["run_dir"] / "Insitu_Fibroblast_to_MalignantC5_MI10_all_slices"
#
# Each slice will be saved as PNG and PDF.
# No plt.show().
# ==============================================================

import os
import re
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch

try:
    import torch
except Exception:
    torch = None


# -----------------------------
# Settings
# -----------------------------
SENDER_CELLTYPE = "Fibroblast"
RECEIVER_CLUSTER_ID = "5"

MI_NAME = "MI10"
MI_INDEX = 10 - 1

MI_STRENGTH_THRESHOLD = 0.20
FALLBACK_QUANTILE_IF_NO_EDGE_GLOBAL = 0.95

MAX_EDGES_TO_DRAW_PER_SLICE = 1000

BACKGROUND_CELL_SIZE = 2.0
CELLTYPE_CELL_SIZE = 6.0
EDGE_LINEWIDTH = 0.55 * 1.5

FIGSIZE = (8, 8)

FIBRO_COLOR = "#bb9d99"
MALIGNANT_C5_COLOR = "#ab4642"
BACKGROUND_COLOR = "#D9D9D9"

# Edge colormap: YlOrRd
EDGE_CMAP = plt.get_cmap("YlOrRd")

OUT_DIR = Path(run_dirs["run_dir"]) / "Insitu_Fibroblast_to_MalignantC5_MI10_all_slices"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Helper functions
# -----------------------------
def _to_numpy(x):
    if torch is not None and torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _load_pickle(path):
    with open(input_path(path), "rb") as f:
        return pickle.load(f)


def _normalize_edge_index(edge_index):
    edge_index = _to_numpy(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index.astype(int)

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(int)

    raise ValueError(f"Unsupported edge_index shape: {edge_index.shape}")


def _get_spatial_xy(adata):
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"])
        if spatial.ndim == 2 and spatial.shape[1] >= 2:
            return spatial[:, :2].astype(float)

    obs_col_pairs = [
        ("x", "y"),
        ("X", "Y"),
        ("x_centroid", "y_centroid"),
        ("X_centroid", "Y_centroid"),
        ("center_x", "center_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("global_x", "global_y"),
    ]

    for x_col, y_col in obs_col_pairs:
        if x_col in adata.obs.columns and y_col in adata.obs.columns:
            return adata.obs[[x_col, y_col]].to_numpy(dtype=float)

    raise KeyError(
        "Cannot find spatial coordinates. Expected adata.obsm['spatial'] "
        "or x/y-like columns in adata.obs."
    )


def _clean_cluster_label(x):
    x = str(x).strip()
    x = re.sub(r"^Cluster_", "", x, flags=re.IGNORECASE)
    x = re.sub(r"^C", "", x, flags=re.IGNORECASE)
    return x


def _get_sample_name(adata_cur, slice_index):
    for col in ["samples", "sample", "sample_name", "sample_id", "Sample", "library_id"]:
        if col in adata_cur.obs.columns:
            vals = adata_cur.obs[col].astype(str).unique()
            if len(vals) == 1:
                return vals[0]
            return f"{vals[0]}_mixed"
    return f"slice_{slice_index}"


def _safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")


def _load_factor_envir_list():
    if "Factor_envir_list" in globals():
        return Factor_envir_list

    factor_list_path = Path(run_dirs["run_dir"]) / "Factor_envir_list.pkl"
    if input_path(factor_list_path).exists():
        return pd.read_pickle(input_path(factor_list_path))

    factor_use_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"
    if input_path(factor_use_path).exists():
        factor_use = np.load(input_path(factor_use_path))
        factor_use = np.asarray(factor_use)

        edge_counts = [
            _normalize_edge_index(data_cur["edge_index"]).shape[0]
            for data_cur in SpiderNet_data_pyg_list
        ]

        edge_ends = np.cumsum(edge_counts)
        edge_starts = np.hstack([[0], edge_ends[:-1]])

        return [
            factor_use[start:end, :]
            for start, end in zip(edge_starts, edge_ends)
        ]

    raise FileNotFoundError(
        "Cannot find Factor_envir_list or Factor_envir_use.npy in run_dir."
    )


def _get_celltype_array(adata_cur, data_cur=None):
    for col in ["cell.types", "cell_type", "celltype", "CellType", "cell_class"]:
        if col in adata_cur.obs.columns:
            return adata_cur.obs[col].astype(str).to_numpy()

    if data_cur is not None and "cell_class" in data_cur:
        return np.asarray(data_cur["cell_class"]).astype(str)

    raise KeyError(
        "Cannot find cell-type labels. Expected adata.obs['cell.types'] "
        "or similar columns, or data_cur['cell_class']."
    )


# -----------------------------
# Load required objects
# -----------------------------
if "SpiderNet_data_pyg_list" not in globals():
    if "processed" in globals() and hasattr(processed, "spidernet_data"):
        SpiderNet_data_pyg_list = processed.spidernet_data
    else:
        SpiderNet_data_pyg_list = pd.read_pickle(input_path(PROCESSED_DATA_DIR / "SpiderNet_data_pyg_list.pkl"))

if "adata_list" not in globals():
    if "processed" in globals() and hasattr(processed, "adata_list"):
        adata_list = processed.adata_list
    else:
        adata_list = pd.read_pickle(input_path(PROCESSED_DATA_DIR / "adata_list.pkl"))

if "adata_choose" not in globals():
    adata_choose_path_candidates = [
        Path(run_dirs["run_dir"]) / "adata_choose_Malignant.h5ad",
        Path(run_dirs["run_dir"]) / "adata_choose.h5ad",
    ]

    loaded = False
    for p in adata_choose_path_candidates:
        if input_path(p).exists():
            import scanpy as sc
            adata_choose = sc.read_h5ad(input_path(p))
            loaded = True
            break

    if not loaded:
        raise FileNotFoundError(
            "Cannot find adata_choose. Please make sure adata_choose is already defined, "
            "or adata_choose_Malignant.h5ad exists in run_dir."
        )

Factor_envir_list = _load_factor_envir_list()


# -----------------------------
# Build Malignant C5 barcode set
# -----------------------------
if "barcode" not in adata_choose.obs.columns:
    raise KeyError("adata_choose.obs must contain a 'barcode' column.")

if "MI_louvain" not in adata_choose.obs.columns:
    raise KeyError("adata_choose.obs must contain 'MI_louvain' to define Malignant C5.")

malignant_cluster_clean = adata_choose.obs["MI_louvain"].astype(str).map(_clean_cluster_label)

malignant_c5_barcodes = frozenset(
    adata_choose.obs.loc[
        malignant_cluster_clean == RECEIVER_CLUSTER_ID,
        "barcode"
    ].astype(str).tolist()
)

if len(malignant_c5_barcodes) == 0:
    raise ValueError(
        "No Malignant C5 barcodes found from adata_choose.obs['MI_louvain'] == 5."
    )

print(f"Number of Malignant C5 cells: {len(malignant_c5_barcodes)}")
print(f"Output folder: {OUT_DIR}")


# -----------------------------
# First pass: collect edges
# -----------------------------
slice_edge_tables = {}
slice_summary_records = []
all_pair_mi_values = []

for slice_index in range(len(adata_list)):
    adata_cur = adata_list[slice_index]
    data_cur = SpiderNet_data_pyg_list[slice_index]

    edge_index_cur = _normalize_edge_index(data_cur["edge_index"])
    factor_cur = _to_numpy(Factor_envir_list[slice_index]).astype(float)

    if factor_cur.shape[0] != edge_index_cur.shape[0] and factor_cur.shape[1] == edge_index_cur.shape[0]:
        factor_cur = factor_cur.T

    if factor_cur.shape[0] != edge_index_cur.shape[0]:
        raise ValueError(
            f"Slice {slice_index}: Factor_envir and edge_index mismatch. "
            f"Factor shape={factor_cur.shape}, edge_index shape={edge_index_cur.shape}"
        )

    if MI_INDEX >= factor_cur.shape[1]:
        raise ValueError(
            f"{MI_NAME} requires column index {MI_INDEX}, "
            f"but Factor_envir has only {factor_cur.shape[1]} MI dimensions."
        )

    obs_names = np.asarray(adata_cur.obs_names.astype(str))
    celltypes = _get_celltype_array(adata_cur, data_cur=data_cur)

    sender_nodes = edge_index_cur[:, 0]
    receiver_nodes = edge_index_cur[:, 1]

    sender_is_fibro = celltypes[sender_nodes] == SENDER_CELLTYPE

    receiver_is_malignant_c5 = np.fromiter(
        (b in malignant_c5_barcodes for b in obs_names[receiver_nodes]),
        dtype=np.bool_,
        count=edge_index_cur.shape[0],
    )

    pair_mask = sender_is_fibro & receiver_is_malignant_c5
    mi_values = factor_cur[:, MI_INDEX]

    pair_mi_values = mi_values[pair_mask]
    pair_mi_values = pair_mi_values[np.isfinite(pair_mi_values)]

    if len(pair_mi_values) > 0:
        all_pair_mi_values.append(pair_mi_values)

    edge_df = pd.DataFrame({
        "slice_index": slice_index,
        "sample_name": _get_sample_name(adata_cur, slice_index),
        "edge_idx": np.where(pair_mask)[0],
        "sender_node": sender_nodes[pair_mask],
        "receiver_node": receiver_nodes[pair_mask],
        "sender_barcode": obs_names[sender_nodes[pair_mask]],
        "receiver_barcode": obs_names[receiver_nodes[pair_mask]],
        MI_NAME: mi_values[pair_mask],
    })

    slice_edge_tables[slice_index] = edge_df

    slice_summary_records.append({
        "slice_index": slice_index,
        "sample_name": _get_sample_name(adata_cur, slice_index),
        "n_pair_edges": int(pair_mask.sum()),
        "mean_pair_MI10": float(np.nanmean(pair_mi_values)) if len(pair_mi_values) > 0 else np.nan,
        "max_pair_MI10": float(np.nanmax(pair_mi_values)) if len(pair_mi_values) > 0 else np.nan,
    })


slice_summary_df = pd.DataFrame(slice_summary_records)

if len(all_pair_mi_values) > 0:
    all_pair_mi_values_concat = np.concatenate(all_pair_mi_values)
    fallback_threshold = float(np.nanquantile(all_pair_mi_values_concat, FALLBACK_QUANTILE_IF_NO_EDGE_GLOBAL))
else:
    fallback_threshold = MI_STRENGTH_THRESHOLD

plot_threshold = MI_STRENGTH_THRESHOLD

global_n_pass = 0
for edge_df in slice_edge_tables.values():
    if edge_df.shape[0] > 0:
        global_n_pass += int((edge_df[MI_NAME] >= plot_threshold).sum())

if global_n_pass == 0 and len(all_pair_mi_values) > 0:
    plot_threshold = fallback_threshold
    print(
        f"No Fibroblast -> Malignant C5 edge passed MI_STRENGTH_THRESHOLD={MI_STRENGTH_THRESHOLD}. "
        f"Using global {FALLBACK_QUANTILE_IF_NO_EDGE_GLOBAL:.2f} quantile threshold: {plot_threshold:.4f}"
    )
else:
    print(f"Using fixed MI10 threshold: {plot_threshold:.4f}")


# Add pass-threshold counts
n_strong_edges = []

for slice_index in range(len(adata_list)):
    edge_df = slice_edge_tables[slice_index]

    if edge_df.shape[0] == 0:
        edge_df["pass_threshold"] = pd.Series(dtype=bool)
        n_strong_edges.append(0)
    else:
        edge_df["pass_threshold"] = edge_df[MI_NAME] >= plot_threshold
        n_strong_edges.append(int(edge_df["pass_threshold"].sum()))

    slice_edge_tables[slice_index] = edge_df

slice_summary_df["MI10_threshold_used"] = plot_threshold
slice_summary_df["n_edges_shown"] = n_strong_edges

summary_path = OUT_DIR / "Fibroblast_to_MalignantC5_MI10_all_slice_summary.csv"
slice_summary_df.to_csv(summary_path, index=False)

print(f"Saved summary table: {summary_path}")
display(slice_summary_df.sort_values(["n_edges_shown", "n_pair_edges", "max_pair_MI10"], ascending=False).head(10))


# -----------------------------
# Global edge color scale
# -----------------------------
all_pass_values = []

for edge_df in slice_edge_tables.values():
    if edge_df.shape[0] > 0:
        vals = edge_df.loc[edge_df["pass_threshold"], MI_NAME].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            all_pass_values.append(vals)

if len(all_pass_values) > 0:
    all_pass_values = np.concatenate(all_pass_values)
    vmin = float(np.nanmin(all_pass_values))
    vmax = float(np.nanmax(all_pass_values))
else:
    vmin, vmax = 0.0, 1.0

if np.isclose(vmin, vmax):
    vmin = 0.0

edge_norm = Normalize(vmin=vmin, vmax=vmax)


# -----------------------------
# Plot and save all slices
# -----------------------------
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"

for slice_index in range(len(adata_list)):
    adata_cur = adata_list[slice_index]
    data_cur = SpiderNet_data_pyg_list[slice_index]

    sample_name = _get_sample_name(adata_cur, slice_index)
    spatial = _get_spatial_xy(adata_cur)
    obs_names = np.asarray(adata_cur.obs_names.astype(str))
    celltypes = _get_celltype_array(adata_cur, data_cur=data_cur)

    fibro_mask = celltypes == SENDER_CELLTYPE

    malignant_c5_mask = np.fromiter(
        (b in malignant_c5_barcodes for b in obs_names),
        dtype=np.bool_,
        count=adata_cur.n_obs,
    )

    edge_df_plot = slice_edge_tables[slice_index].copy()

    if edge_df_plot.shape[0] > 0:
        edge_df_plot = edge_df_plot.loc[edge_df_plot["pass_threshold"], :].copy()
        edge_df_plot = edge_df_plot.sort_values(MI_NAME, ascending=False)

    if edge_df_plot.shape[0] > MAX_EDGES_TO_DRAW_PER_SLICE:
        edge_df_plot = edge_df_plot.iloc[:MAX_EDGES_TO_DRAW_PER_SLICE, :].copy()

    edge_table_path = OUT_DIR / f"slice{slice_index:02d}_{_safe_filename(sample_name)}_MI10_edges_shown.csv"
    edge_df_plot.to_csv(edge_table_path, index=False)

    plt.close("all")
    fig, ax = plt.subplots(figsize=FIGSIZE)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Background cells
    ax.scatter(
        spatial[:, 0],
        spatial[:, 1],
        s=BACKGROUND_CELL_SIZE,
        c=BACKGROUND_COLOR,
        alpha=0.35,
        edgecolors="none",
        rasterized=True,
        label="Other cells",
    )

    # Fibroblast cells
    ax.scatter(
        spatial[fibro_mask, 0],
        spatial[fibro_mask, 1],
        s=CELLTYPE_CELL_SIZE,
        c=FIBRO_COLOR,
        alpha=0.95,
        edgecolors="none",
        rasterized=True,
        label="Fibroblast",
    )

    # Malignant C5 cells
    ax.scatter(
        spatial[malignant_c5_mask, 0],
        spatial[malignant_c5_mask, 1],
        s=CELLTYPE_CELL_SIZE,
        c=MALIGNANT_C5_COLOR,
        alpha=0.95,
        edgecolors="none",
        rasterized=True,
        label="Malignant C5",
    )

    # Directional MI10 edges
    for _, row in edge_df_plot.iterrows():
        s = int(row["sender_node"])
        r = int(row["receiver_node"])
        val = float(row[MI_NAME])

        x0, y0 = spatial[s, 0], spatial[s, 1]
        x1, y1 = spatial[r, 0], spatial[r, 1]

        arrow_color = EDGE_CMAP(edge_norm(val))

        arrow = FancyArrowPatch(
            (x0, y0),
            (x1, y1),
            arrowstyle="-|>",
            mutation_scale=5.2,
            linewidth=EDGE_LINEWIDTH,
            color=arrow_color,
            alpha=0.78,
            shrinkA=0.0,
            shrinkB=0.0,
            zorder=4,
        )
        ax.add_patch(arrow)

    # Colorbar
    sm = ScalarMappable(norm=edge_norm, cmap=EDGE_CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label(f"{MI_NAME} edge strength", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    # Style
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("Spatial X", fontsize=11)
    ax.set_ylabel("Spatial Y", fontsize=11)

    ax.set_title(
        f"{SENDER_CELLTYPE} → Malignant C5 through {MI_NAME}\n"
        f"Slice {slice_index}: {sample_name}; "
        f"edges shown={edge_df_plot.shape[0]}, threshold={plot_threshold:.3f}",
        fontsize=12,
        pad=12,
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            markerfacecolor=BACKGROUND_COLOR,
            markeredgecolor="none",
            markersize=5,
            alpha=0.6,
            label="Other cells",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            markerfacecolor=FIBRO_COLOR,
            markeredgecolor="none",
            markersize=6 * 0.75,
            label="Fibroblast",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            markerfacecolor=MALIGNANT_C5_COLOR,
            markeredgecolor="none",
            markersize=6 * 0.75,
            label="Malignant C5",
        ),
        Line2D(
            [0], [0],
            color=EDGE_CMAP(0.85),
            lw=1.2 * 1.5,
            label=f"{MI_NAME} Fibroblast→C5 edge",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        fontsize=9,
    )

    plt.tight_layout()

    save_base = OUT_DIR / f"slice{slice_index:02d}_{_safe_filename(sample_name)}_Fibroblast_to_MalignantC5_{MI_NAME}"

    plt.savefig(str(save_base) + ".png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(str(save_base) + ".pdf", dpi=300, bbox_inches="tight", facecolor="white")

    plt.close(fig)

    print(
        f"[Saved] slice {slice_index}: {sample_name}; "
        f"edges shown={edge_df_plot.shape[0]}"
    )

print(f"\nDone. All slice plots saved to:\n{OUT_DIR}")

### 5.10. Fibroblast CAF scores and MI sending strength toward C5


In [ ]:
# First retain edges with a malignant C5 receiver from any sender cell type.
# The next block selects fibroblasts for CAF-score comparisons.

import numpy as np
import torch
from torch_scatter import scatter_max

edge_keep_masks_cpu = []
edge_index_cpu_list = []
num_cell_list = []

barcode_MI5_set = frozenset(
    adata_choose.obs.loc[adata_choose.obs["MI_louvain"] == "5", "barcode"]
    .astype(str).tolist()
)

for slice_index in range(len(Factor_envir_list)):
    data = SpiderNet_data_pyg_list[slice_index]
    adata_cur = adata_list[slice_index]

    ei = data["edge_index"]
    ei_cpu = ei.detach().cpu().numpy() if isinstance(ei, torch.Tensor) else np.asarray(ei)

    # Edge indices are expected in (n_edges, 2) sender/receiver order.
    recv_nodes = ei_cpu[:, 1]

    n = int(data.x.shape[0])
    num_cell_list.append(n)
    edge_index_cpu_list.append(ei_cpu)

    # ---- node-level mask: length n
    obs_names = np.asarray(adata_cur.obs_names, dtype=object)

    # Identify C5 receiver nodes by barcode membership.
    node_is_MI5 = np.fromiter(
        (b in barcode_MI5_set for b in obs_names),
        dtype=np.bool_,
        count=obs_names.size
    )

    # edge mask (E,)
    keep = node_is_MI5[recv_nodes]
    edge_keep_masks_cpu.append(keep)


# ---------------------------------------
# 2) Aggregate on the selected device, transferring one slice at a time
# ---------------------------------------
MI_SR_agg_cur_all_FibrotoMC5 = []

for slice_index, Factor_envir_np in enumerate(Factor_envir_list):
    # print(f"Processing slice {slice_index + 1}/{len(Factor_envir_list)} with GPU scatter...")
    ei_cpu = edge_index_cpu_list[slice_index]
    keep_cpu = edge_keep_masks_cpu[slice_index]
    num_cell = num_cell_list[slice_index]

    # Transfer the edge indices and mask to the selected device.
    ei = torch.as_tensor(ei_cpu, device=device, dtype=torch.long)  # (E,2)
    keep = torch.as_tensor(keep_cpu, device=device, dtype=torch.float32).unsqueeze(1)  # (E,1)

    Factor_envir = torch.as_tensor(Factor_envir_np, device=device, dtype=torch.float32)  # (E,K)

    # Zero the MI strengths of excluded edges.
    F_masked = Factor_envir * keep

    recv_idx = ei[:, 1]
    send_idx = ei[:, 0]

    MI_recv_all, _ = scatter_max(F_masked, recv_idx, dim=0, dim_size=num_cell)
    MI_send_all, _ = scatter_max(F_masked, send_idx, dim=0, dim_size=num_cell)

    MI_SR_agg_cur_all_FibrotoMC5.append(
        torch.cat([MI_send_all, MI_recv_all], dim=1).detach().cpu().numpy()
    )

MI_SR_agg_cur_all_FibrotoMC5 = np.vstack(MI_SR_agg_cur_all_FibrotoMC5)


In [ ]:
from scipy import sparse
from scipy.stats import spearmanr

# ==============================================================
# CAF score vs sending MI correlation
# --------------------------------------------------------------
# CAF score is computed as global z-score + gene-set mean:
#   1. select all Fibroblast cells across all slices
#   2. z-score each CAF marker gene using this pooled Fibroblast reference
#   3. average z-scored CAF marker genes per cell
#   4. correlate the score in Fibroblast cells neighboring Malignant_5
#      with Fibroblast -> Malignant_5 sending MI levels
# ==============================================================

caf_modules = {
    "myCAF": ["ACTA2","TAGLN","MYL9","TPM2","CNN1","CALD1","COL1A1","COL1A2"],
    "iCAF": ["IL6","CXCL12","CXCL14","LIF","CCL2","PTGS2"],
    "apCAF": ["HLA-DRA","HLA-DRB1","CD74","CIITA"],
    "meCAF": ["COL11A1","THBS2","MMP11","ITGA11","FN1","VCAN","SPARC","SULF1","LOX","PLOD2"],
    "periCAF": ["RGS5","PDGFRB","MCAM","NOTCH3","TAGLN"],
    "prolCAF": ["MKI67","TOP2A","PCNA"]
}

adata_var_names = np.asarray(adata_copy.var_names).astype(str)
fibro_idx = np.where(cellclass_updated == "Fibroblast")[0]

caf_genes = np.unique([g for genes in caf_modules.values() for g in genes if g in adata_var_names])
caf_gene_mask = np.isin(adata_var_names, caf_genes)

if len(fibro_idx) == 0:
    raise ValueError("No Fibroblast cells found in cellclass_updated.")
if caf_gene_mask.sum() == 0:
    raise ValueError("No CAF marker genes were found in adata_copy.var_names.")

X_caf = adata_copy.X[fibro_idx, :][:, caf_gene_mask]
X_caf = X_caf.toarray() if sparse.issparse(X_caf) else np.asarray(X_caf)
X_caf = X_caf.astype(float, copy=False)

caf_gene_mean = np.nanmean(X_caf, axis=0)
caf_gene_std = np.nanstd(X_caf, axis=0)
caf_gene_std = np.where(caf_gene_std > 0, caf_gene_std, np.nan)

X_caf_z = (X_caf - caf_gene_mean) / caf_gene_std
X_caf_z = np.nan_to_num(X_caf_z, nan=0.0, posinf=0.0, neginf=0.0)

caf_score = np.full(adata_copy.n_obs, np.nan, dtype=float)
caf_score[fibro_idx] = np.mean(X_caf_z, axis=1)

# Further keep Fibroblast cells that are connected to Malignant_5
# based on Fibroblast -> Malignant_5 MI level.
fibro_idx_further = fibro_idx[np.max(MI_SR_agg_cur_all_FibrotoMC5[fibro_idx, :], axis=1) > 0]

MI_OI_df = pd.DataFrame({
    "MI": ["MI10", "MI6", "MI13"],
    "direction": ["Sender"] * 3
})

corr_CAF_df = pd.DataFrame(
    index=["CAF_Score"],
    columns=[f"{mi}_{d}" for mi, d in zip(MI_OI_df["MI"], MI_OI_df["direction"])]
)

for MI_OI, direction in zip(MI_OI_df["MI"], MI_OI_df["direction"]):
    idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index
    print(f"MI_OI: {MI_OI}, idx: {idx}")

    mi_vals = MI_SR_agg_cur_all_FibrotoMC5[fibro_idx_further, :][:, idx].flatten()
    scatter_df = pd.DataFrame({
        "MI_Level": mi_vals,
        "CAF_Score": caf_score[fibro_idx_further]
    }).dropna()

    rho, p = spearmanr(scatter_df["MI_Level"], scatter_df["CAF_Score"])
    corr_CAF_df.loc["CAF_Score", f"{MI_OI}_{direction}"] = rho

    # Save the scatter_df
    scatter_df.to_csv(
        run_dirs["run_dir"] + f"/Malignant_COI5_CAF_score_vs_Sending_{MI_OI}_scatter.csv",
        index=False
    )


### 5.11. Malignant C5 programs and fibroblast-derived MI receiving strength

This section exports cell-level Spearman-correlation inputs, compares CAF and CancerSEA scores at MI strength 0.5, and performs separate sample-level and cell-level KEGG comparisons.


In [ ]:
## Only consider edges whose SENDER is Fibroblast (across all slices)
import numpy as np
import torch
from torch_scatter import scatter_max

# ---------------------------------------
# 0) Build global Fibroblast barcode set (CPU)

# ---------------------------------------
barcode_Fibro_set = []
for slice_index in range(len(Factor_envir_list)):
    adata_cur = adata_list[slice_index]
    barcode_Fibro_set.append(
        frozenset(
            adata_cur.obs_names[adata_cur.obs["cell.types"] == "Fibroblast"]
            .astype(str).tolist()
        )
    )
barcode_Fibro_set = frozenset.union(*barcode_Fibro_set)  # global set (fast membership)

# ---------------------------------------
# 1) Precompute sender-based edge masks on the CPU
#    edge_keep_masks_cpu[slice] is bool array (E,) where sender is Fibroblast
# ---------------------------------------
edge_keep_masks_cpu = []
edge_index_cpu_list = []
num_cell_list = []

for slice_index in range(len(Factor_envir_list)):
    data = SpiderNet_data_pyg_list[slice_index]
    adata_cur = adata_list[slice_index]

    ei = data["edge_index"]
    ei_cpu = ei.detach().cpu().numpy() if isinstance(ei, torch.Tensor) else np.asarray(ei)

    # ---- assumes edge_index is (E, 2)
    send_nodes = ei_cpu[:, 0]

    n = int(data.x.shape[0])
    num_cell_list.append(n)
    edge_index_cpu_list.append(ei_cpu)

    # node-level sender barcode array (length n)
    # Use obs_names as the barcode identifiers.
    obs_names = np.asarray(adata_cur.obs_names, dtype=object)

    # fast node-level membership via set (avoid np.isin on object)
    node_is_fibro = np.fromiter(
        (b in barcode_Fibro_set for b in obs_names),
        dtype=np.bool_,
        count=obs_names.size
    )

    # edge mask: keep edges whose SENDER node is fibro
    keep = node_is_fibro[send_nodes]  # (E,)
    edge_keep_masks_cpu.append(keep)

# ---------------------------------------
# 2) Aggregate on the selected device, transferring one slice at a time
# ---------------------------------------
MI_SR_agg_cur_all_FibroOnly = []

for slice_index, Factor_envir_np in enumerate(Factor_envir_list):
    ei_cpu = edge_index_cpu_list[slice_index]
    keep_cpu = edge_keep_masks_cpu[slice_index]
    num_cell = num_cell_list[slice_index]

    # Transfer the edge indices and mask to the selected device.
    ei = torch.as_tensor(ei_cpu, device=device, dtype=torch.long)  # (E,2)
    keep = torch.as_tensor(keep_cpu, device=device, dtype=torch.float32).unsqueeze(1)  # (E,1)

    Factor_envir = torch.as_tensor(Factor_envir_np, device=device, dtype=torch.float32)  # (E,K)

    # Zero the MI strengths of excluded edges.
    F_masked = Factor_envir * keep

    recv_idx = ei[:, 1]
    send_idx = ei[:, 0]

    MI_recv_all, _ = scatter_max(F_masked, recv_idx, dim=0, dim_size=num_cell)
    MI_send_all, _ = scatter_max(F_masked, send_idx, dim=0, dim_size=num_cell)

    MI_SR_agg_cur_all_FibroOnly.append(
        torch.cat([MI_send_all, MI_recv_all], dim=1).detach().cpu().numpy()
    )

MI_SR_agg_cur_all_FibroOnly = np.vstack(MI_SR_agg_cur_all_FibroOnly)


In [ ]:
##Only consider edges whose SENDER is Fibroblast and RECEIVER is Malignant_5
from scipy import sparse
from scipy.stats import spearmanr

# ==============================================================
# Tumor functional-state score vs receiving MI correlation
# --------------------------------------------------------------
# Functional-state scores are computed as global z-score + gene-set mean:
#   1. select all Malignant_5 cells across all slices
#   2. z-score each gene using this pooled Malignant_5 reference
#   3. average z-scored genes within each CancerSEA gene set per cell
#   4. correlate the score in Malignant_5 cells neighboring Fibroblast
#      with Fibroblast -> Malignant_5 receiving MI levels
# ==============================================================

geneset_df = pd.read_csv(input_path(DATA_ROOT / "CancerSEA_OV/functional_geneset_list_df.csv"))
adata_var_names = np.asarray(adata_copy.var_names).astype(str)
geneset_df = geneset_df[geneset_df["Gene"].isin(adata_var_names)]

geneset_dict = {
    term: np.where(np.isin(
        adata_var_names,
        geneset_df.loc[geneset_df["GeneSet"] == term, "Gene"]
    ))[0]
    for term in np.unique(geneset_df["GeneSet"])
}

MI_OI_df = pd.DataFrame({
    "MI": ["MI10", "MI6", "MI13"],
    "direction": ["Receiver"] * 3
})

Malignant_COI = "Malignant_5"
malignant_idx = np.where(cellclass_updated == Malignant_COI)[0]

if len(malignant_idx) == 0:
    raise ValueError(f"No cells found for {Malignant_COI} in cellclass_updated.")

# Only keep malignant cells that are connected to Fibroblast
# based on Fibroblast -> Malignant_5 MI level.
malignant_idx_further = malignant_idx[np.max(MI_SR_agg_cur_all_FibroOnly[malignant_idx, :], axis=1) > 0]

# Global z-score reference: all Malignant_5 cells pooled across all slices.
X_malignant = adata_copy.X[malignant_idx, :]
X_malignant = X_malignant.toarray() if sparse.issparse(X_malignant) else np.asarray(X_malignant)
X_malignant = X_malignant.astype(float, copy=False)

malignant_gene_mean = np.nanmean(X_malignant, axis=0)
malignant_gene_std = np.nanstd(X_malignant, axis=0)
malignant_gene_std = np.where(malignant_gene_std > 0, malignant_gene_std, np.nan)

X_malignant_z = (X_malignant - malignant_gene_mean) / malignant_gene_std
X_malignant_z = np.nan_to_num(X_malignant_z, nan=0.0, posinf=0.0, neginf=0.0)

malignant_pos_lookup = pd.Series(
    np.arange(len(malignant_idx), dtype=int),
    index=malignant_idx
)
malignant_further_pos = malignant_pos_lookup.loc[malignant_idx_further].to_numpy()

corr_tumor_state_df = pd.DataFrame(
    index=geneset_dict.keys(),
    columns=[f"{mi}_{d}" for mi, d in zip(MI_OI_df["MI"], MI_OI_df["direction"])]
)

geneset_count = -1
for state, gene_idx in geneset_dict.items():
    geneset_count += 1

    gene_idx = np.asarray(gene_idx, dtype=int)
    gene_idx = gene_idx[(gene_idx >= 0) & (gene_idx < adata_copy.n_vars)]

    if len(gene_idx) == 0:
        exp_state_malignant = np.full(len(malignant_idx_further), np.nan, dtype=float)
    else:
        exp_state_all_malignant = np.mean(X_malignant_z[:, gene_idx], axis=1)
        exp_state_malignant = exp_state_all_malignant[malignant_further_pos]

    for MI_OI, direction in zip(MI_OI_df["MI"], MI_OI_df["direction"]):
        idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index
        if geneset_count == 0:
            print(f"State: {state}, MI_OI: {MI_OI}, idx: {idx}")

        mi_vals = MI_SR_agg_cur_all_FibroOnly[malignant_idx_further, :][:, idx].flatten()
        scatter_df = pd.DataFrame({
            "MI_Level": mi_vals,
            "State_Score": exp_state_malignant
        }).dropna()

        # Save the scatter_df
        scatter_df.to_csv(
            run_dirs["run_dir"] + f"/Malignant_COI5_{state}_vs_Receiving_{MI_OI}_scatter.csv",
            index=False
        )

        rho, p = spearmanr(scatter_df["MI_Level"], scatter_df["State_Score"])
        corr_tumor_state_df.loc[state, f"{MI_OI}_{direction}"] = rho

##
# corr_tumor_state_df_argmax = corr_tumor_state_df.astype(float).abs().idxmax(axis=1)
# corr_tumor_state_df = corr_tumor_state_df.loc[corr_tumor_state_df_argmax.sort_values(ascending=False).index, :]


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams

# -----------------------------
# Publication-style defaults
# -----------------------------
plt.close("all")
plt.style.use("default")

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# -----------------------------
# Prepare matrix: plot corr_tumor_state_df.T
# rows: MI(receiver), cols: functional state
# -----------------------------
mat = corr_tumor_state_df.copy().apply(pd.to_numeric, errors="coerce").T

# optional: keep MI order as MI_OI_df (recommended)
row_order = [f"{mi}_{d}" for mi, d in zip(MI_OI_df["MI"], MI_OI_df["direction"])]
row_order = [r for r in row_order if r in mat.index]
if len(row_order) > 0:
    mat = mat.loc[row_order]

# # optional: sort states by mean correlation (descending) for cleaner columns
# col_order = mat.mean(axis=0).sort_values(ascending=False).index
# mat = mat.loc[:, col_order]
# -----------------------------
# Column order:
#   1) primary: argmax row (which MI gives the max in this state)
#   2) secondary: colmax value (max correlation in this state), descending
# -----------------------------
mat_tmp = mat.copy()

# for each column, find which row index gives the max
col_argmax = mat_tmp.apply(lambda s: int(np.nanargmax(s.values)), axis=0)  # 0..nrow-1
# max value per column (for tie-break)
col_max = mat_tmp.max(axis=0, skipna=True)

col_order = (
    pd.DataFrame({
        "col": mat_tmp.columns,
        "argmax": col_argmax.values,
        "colmax": col_max.values
    })
    # primary: group by argmax (ascending: MI order in mat index)
    # secondary: within group, sort by colmax descending
    .sort_values(["argmax", "colmax"], ascending=[True, False])
    ["col"]
    .tolist()
)

mat = mat.loc[:, col_order]


# -----------------------------
# Plot heatmap (Publication-style)
# -----------------------------
fig_w = max(3.8, 0.35 * mat.shape[1] + 2.2)  # cols drive width
fig_h = max(2.4, 0.55 * mat.shape[0] + 1.2) * 0.8  # rows drive height
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

hm = sns.heatmap(
    mat,
    ax=ax,
    cmap="Reds",
    vmin=0, vmax=0.5,
    linewidths=0.6,
    linecolor="0.85",
    annot=True, fmt=".2f",
    annot_kws={"fontsize": 7, "color": "black"},
    cbar=True,
    cbar_kws={"shrink": 0.9, "pad": 0.02, "label": "Spearman ρ"}
)

ax.set_title("MI receiver vs tumor functional states (Malignant_5)", pad=6)
ax.set_xlabel("Tumor functional state")
ax.set_ylabel("MI (receiver)")

# Rotate tick labels and align them to the right.
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

# Clean spines
for s in ["top", "right", "left", "bottom"]:
    ax.spines[s].set_visible(False)

# Colorbar styling
cbar = hm.collections[0].colorbar
cbar.outline.set_linewidth(0.8)
cbar.ax.tick_params(width=0.8, length=3)

plt.tight_layout()
plt.savefig(run_dirs['run_dir'] + "/Corr_MI_Receiver_TumorFunctionalState_Malignant5_T.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(run_dirs['run_dir'] + "/Corr_MI_Receiver_TumorFunctionalState_Malignant5_T.pdf",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close()


In [ ]:
# Read the preceding CAF/CancerSEA scatter tables and compare MI >=0.5 vs <0.5.
# Tests are two-sided Mann-Whitney; log2(mean_high/mean_low) is reported only
# when both group means are positive. Z-score-based means can be nonpositive.

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.stats import mannwhitneyu

plt.close("all")
plt.style.use("default")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.35,
    "xtick.major.width": 0.35,
    "ytick.major.width": 0.35,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =========================================================
# 1. Global settings
# =========================================================
file_savepath_main = Path(run_dirs["run_dir"])
thr_use = 0.5
thr_tag = f"{thr_use:.2f}".replace(".", "p")

x_candidates = [
    "MI_Level",
    "Sending_MI10", "Sending_MI6", "Sending_MI13",
    "Receiving_MI10", "Receiving_MI6", "Receiving_MI13",
]
caf_y_candidates = ["CAF_Score", "CAF_score"]
state_y_candidates = ["State_Score", "state_score"]

caf_fill_low = "#C2E2FA"
caf_fill_high = "#B7A3E3"
state_fill_low = "#FFF1CB"
state_fill_high = "#FF8F8F"

# =========================================================
# 2. Utility helpers
# =========================================================
def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def pick_first_existing(df, candidates):
    for nm in candidates:
        if nm in df.columns:
            return nm
    raise KeyError(f"None of these columns exist: {', '.join(candidates)}")

def save_plot(fig, stem, out_dir=file_savepath_main, width=1.2, height=2.6):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = out_dir / f"{stem}.pdf"
    png_path = out_dir / f"{stem}.png"
    fig.set_size_inches(width, height)
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)
    return pdf_path, png_path

def _safe_nanmax(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.max()

def _safe_nanmin(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.min()

def make_box_plot(
    df,
    xcol,
    ycol,
    thr=0.25,
    fill_low="#C2E2FA",
    fill_high="#B7A3E3",
    title_txt=None,
    xlab=None,
    ylab=None,
    print_summary=True,
):
    lvl_low = f"<{thr}"
    lvl_high = f"\u2265{thr}"

    plot_df = df.loc[np.isfinite(pd.to_numeric(df[xcol], errors="coerce")) &
                     np.isfinite(pd.to_numeric(df[ycol], errors="coerce")),
                     [xcol, ycol]].copy()
    plot_df[xcol] = pd.to_numeric(plot_df[xcol], errors="coerce")
    plot_df[ycol] = pd.to_numeric(plot_df[ycol], errors="coerce")
    plot_df["MI_group"] = np.where(plot_df[xcol] >= thr, lvl_high, lvl_low)
    plot_df["MI_group"] = pd.Categorical(plot_df["MI_group"], categories=[lvl_high, lvl_low], ordered=True)

    high_vals = plot_df.loc[plot_df["MI_group"] == lvl_high, ycol].to_numpy(dtype=float)
    low_vals = plot_df.loc[plot_df["MI_group"] == lvl_low, ycol].to_numpy(dtype=float)

    m_high = np.nanmean(high_vals) if high_vals.size else np.nan
    m_low = np.nanmean(low_vals) if low_vals.size else np.nan
    mean_diff = m_high - m_low if np.isfinite(m_high) and np.isfinite(m_low) else np.nan

    if np.isfinite(m_high) and np.isfinite(m_low) and m_high > 0 and m_low > 0:
        log2_mean_change = np.log2(m_high / m_low)
    else:
        log2_mean_change = np.nan

    if print_summary:
        print("\n========================================")
        print("Boxplot summary")
        print(f"x column: {xcol}")
        print(f"y column: {ycol}")
        print(f"threshold: {thr}")
        print(f"Mean (High >= thr): {m_high:.4g}" if np.isfinite(m_high) else "Mean (High >= thr): NA")
        print(f"Mean (Low  < thr): {m_low:.4g}" if np.isfinite(m_low) else "Mean (Low  < thr): NA")
        print(f"Mean difference (High - Low): {mean_diff:.4g}" if np.isfinite(mean_diff) else "Mean difference (High - Low): NA")
        print(
            "log2 mean change [log2(mean_high / mean_low)]: "
            + (f"{log2_mean_change:.4g}" if np.isfinite(log2_mean_change) else "NA")
        )
        print("========================================")

    p_wilcox = np.nan
    if high_vals.size > 0 and low_vals.size > 0:
        try:
            # closest Python analogue to the two-sided Wilcoxon rank-sum / Mann–Whitney test
            p_wilcox = mannwhitneyu(high_vals, low_vals, alternative="two-sided", method="asymptotic").pvalue
        except TypeError:
            # compatibility with older SciPy
            p_wilcox = mannwhitneyu(high_vals, low_vals, alternative="two-sided").pvalue

    star = p_to_star(p_wilcox)
    p_lbl = "p = NA" if pd.isna(p_wilcox) else f"p = {p_wilcox:.2e}"

    y_top = _safe_nanmax(plot_df[ycol])
    y_min = _safe_nanmin(plot_df[ycol])
    if np.isfinite(y_top) and np.isfinite(y_min):
        y_rng = y_top - y_min + 1e-12
        y_anno = y_top + 0.15 * y_rng
        y_tick = 0.05 * y_anno
        y_text = y_anno + 0.03 * y_anno
        ylim_top = y_anno + 0.14 * max(abs(y_anno), y_rng)
    else:
        y_anno = y_tick = y_text = ylim_top = np.nan

    fig, ax = plt.subplots(figsize=(1.2, 2.6))

    grouped = [
        plot_df.loc[plot_df["MI_group"] == lvl_high, ycol].to_numpy(dtype=float),
        plot_df.loc[plot_df["MI_group"] == lvl_low, ycol].to_numpy(dtype=float),
    ]

    bp = ax.boxplot(
        grouped,
        positions=[1, 2],
        widths=0.58,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(linewidth=0.35, color="black"),
        medianprops=dict(linewidth=0.35, color="black"),
        whiskerprops=dict(linewidth=0.35, color="black"),
        capprops=dict(linewidth=0.35, color="black"),
    )

    for patch, color in zip(bp["boxes"], [fill_high, fill_low]):
        patch.set_facecolor(color)
        patch.set_edgecolor("black")

    if np.isfinite(y_anno):
        ax.plot([1, 2], [y_anno, y_anno], color="black", linewidth=0.35)
        ax.plot([1, 1], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)
        ax.plot([2, 2], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)
        ax.text(
            1.5,
            y_text,
            f"{p_lbl}  {star}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="black",
        )
        ax.set_ylim(top=ylim_top)

    ax.set_xticks([1, 2])
    ax.set_xticklabels([lvl_high, lvl_low], fontsize=8, color="black")
    ax.set_title("" if title_txt is None else title_txt, fontsize=9, loc="left", color="black")
    ax.set_xlabel("" if xlab is None else xlab, fontsize=9, color="black")
    ax.set_ylabel("" if ylab is None else ylab, fontsize=9, color="black")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.35)
        ax.spines[spine].set_color("black")
    ax.tick_params(axis="both", colors="black", width=0.35, length=2)
    ax.grid(False)
    
    fig.tight_layout()
    return fig, ax, plot_df, {
        "m_high": m_high,
        "m_low": m_low,
        "mean_diff": mean_diff,
        "log2_mean_change": log2_mean_change,
        "p_wilcox": p_wilcox,
    }

# =========================================================
# 3. Data loading
# =========================================================
data_list = {
    "CAF_MI10": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_CAF_score_vs_Sending_MI10_scatter.csv")),
    "CAF_MI6": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_CAF_score_vs_Sending_MI6_scatter.csv")),
    "CAF_MI13": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_CAF_score_vs_Sending_MI13_scatter.csv")),
    "Metastasis_MI10": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Metastasis_vs_Receiving_MI10_scatter.csv")),
    "Metastasis_MI6": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Metastasis_vs_Receiving_MI6_scatter.csv")),
    "Metastasis_MI13": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Metastasis_vs_Receiving_MI13_scatter.csv")),
    "Hypoxia_MI10": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Hypoxia_vs_Receiving_MI10_scatter.csv")),
    "Hypoxia_MI6": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Hypoxia_vs_Receiving_MI6_scatter.csv")),
    "Hypoxia_MI13": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Hypoxia_vs_Receiving_MI13_scatter.csv")),
    "Angiogenesis_MI10": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Angiogenesis_vs_Receiving_MI10_scatter.csv")),
    "Angiogenesis_MI6": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Angiogenesis_vs_Receiving_MI6_scatter.csv")),
    "Angiogenesis_MI13": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Angiogenesis_vs_Receiving_MI13_scatter.csv")),
    "Inflammation_MI6": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Inflammation_vs_Receiving_MI6_scatter.csv")),
    "Inflammation_MI10": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Inflammation_vs_Receiving_MI10_scatter.csv")),
    "Inflammation_MI13": pd.read_csv(input_path(file_savepath_main / "Malignant_COI5_Inflammation_vs_Receiving_MI13_scatter.csv")),
}

# =========================================================
# 4. Analysis specifications
# =========================================================
analysis_specs = [
    dict(key="CAF_MI10",          prefix="CAF_vs_SendingMI10",            y_candidates=caf_y_candidates,   fill_low=caf_fill_low,   fill_high=caf_fill_high),
    dict(key="CAF_MI6",           prefix="CAF_vs_SendingMI6",             y_candidates=caf_y_candidates,   fill_low=caf_fill_low,   fill_high=caf_fill_high),
    dict(key="CAF_MI13",          prefix="CAF_vs_SendingMI13",            y_candidates=caf_y_candidates,   fill_low=caf_fill_low,   fill_high=caf_fill_high),
    dict(key="Metastasis_MI10",   prefix="Metastasis_vs_ReceivingMI10",   y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Metastasis_MI6",    prefix="Metastasis_vs_ReceivingMI6",    y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Metastasis_MI13",   prefix="Metastasis_vs_ReceivingMI13",   y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Hypoxia_MI10",      prefix="Hypoxia_vs_ReceivingMI10",      y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Hypoxia_MI6",       prefix="Hypoxia_vs_ReceivingMI6",       y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Hypoxia_MI13",      prefix="Hypoxia_vs_ReceivingMI13",      y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Angiogenesis_MI10", prefix="Angiogenesis_vs_ReceivingMI10", y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Angiogenesis_MI6",  prefix="Angiogenesis_vs_ReceivingMI6",  y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Angiogenesis_MI13", prefix="Angiogenesis_vs_ReceivingMI13", y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Inflammation_MI6",  prefix="Inflammation_vs_ReceivingMI6",  y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Inflammation_MI10", prefix="Inflammation_vs_ReceivingMI10", y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
    dict(key="Inflammation_MI13", prefix="Inflammation_vs_ReceivingMI13", y_candidates=state_y_candidates, fill_low=state_fill_low, fill_high=state_fill_high),
]

# =========================================================
# 5. Build, save, and collect all boxplots
# =========================================================
plot_results = {}
all_boxplots = {}
summary_rows = []

for spec in analysis_specs:
    print(spec["prefix"])
    df = data_list[spec["key"]].copy()

    xcol = pick_first_existing(df, x_candidates)
    ycol = pick_first_existing(df, spec["y_candidates"])

    fig, ax, plot_df, stats_dict = make_box_plot(
        df=df,
        xcol=xcol,
        ycol=ycol,
        thr=thr_use,
        fill_low=spec["fill_low"],
        fill_high=spec["fill_high"],
        title_txt=spec["prefix"],
        xlab=None,
        ylab=None,
        print_summary=True,
    )

    stem = f'{spec["prefix"]}_box_wilcox_thr{thr_tag}'
    pdf_path, png_path = save_plot(fig, stem=stem, width=1.2, height=2.6)

    plot_results[spec["key"]] = {
        "data": df,
        "plot_df": plot_df,
        "xcol": xcol,
        "ycol": ycol,
        "fig": fig,
        "ax": ax,
        "pdf_path": pdf_path,
        "png_path": png_path,
        **stats_dict,
    }
    all_boxplots[spec["prefix"]] = fig

    summary_rows.append({
        "analysis": spec["prefix"],
        "xcol": xcol,
        "ycol": ycol,
        "m_high": stats_dict["m_high"],
        "m_low": stats_dict["m_low"],
        "mean_diff_high_minus_low": stats_dict["mean_diff"],
        "log2_mean_change_high_over_low": stats_dict["log2_mean_change"],
        "p_wilcox": stats_dict["p_wilcox"],
        "pdf_path": str(pdf_path),
        "png_path": str(png_path),
    })

boxplot_summary_df = pd.DataFrame(summary_rows)
boxplot_summary_df.to_csv(file_savepath_main / f"HGSOC_Malignant5_MI_boxplot_summary_thr{thr_tag}.csv", index=False)
display(boxplot_summary_df)

# =========================================================
# 6. Show all boxplots
# =========================================================
for nm in all_boxplots:
    print(f"Showing plot: {nm}")
    display(all_boxplots[nm])


In [ ]:
# Sample-level KEGG comparisons by fibroblast-derived MI receiving strength
# Select malignant C5 cells with positive fibroblast-associated MI strength,
# extract receiving MI-10/MI-6/MI-13 from the aggregated MI arrays, and compute
# four KEGG module scores from adata_copy. Retain sample/group combinations
# with at least ten cells; compare MI-high (>=0.5) and MI-low groups using paired
# Wilcoxon tests when possible, with the rank-sum fallback defined below.

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse
from scipy.stats import wilcoxon, ranksums, mannwhitneyu

# =========================================================
# 1. Settings
# =========================================================
file_savepath_main = Path(run_dirs["run_dir"]) if "run_dirs" in globals() else Path(".")
file_savepath_main.mkdir(parents=True, exist_ok=True)

KEGG_PATHWAYS_TO_SCORE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

MI_OI_DF = pd.DataFrame({
    "MI": ["MI10", "MI6", "MI13"],
    "direction": ["Receiver", "Receiver", "Receiver"],
})

Malignant_COI = "Malignant_5"

thr_use = 0.5
thr_tag = f"{thr_use:.2f}".replace(".", "p")

MIN_CELLS_PER_SAMPLE_GROUP = 10

# sample-level test direction:
#   "greater": MI-high group > MI-low group
#   "less":    MI-high group < MI-low group
#   "two-sided": two-sided comparison
TEST_ALTERNATIVE = "greater"

# Score method:
#   "zscore_mean_global": concatenate the selected malignant cells across all slices,
#                          compute one global mean/std per gene, then average
#                          global z-scored genes for each cell.
#   "maxnorm": legacy max-normalized expression
#   "zscore": legacy alias for zscore_mean_global
SCORE_METHOD = "zscore_mean_global"

OUT_PREFIX = "Malignant_COI5_KEGG_pathway_score_vs_ReceivingMI_direct_from_MI_arrays"

state_fill_low = "#FFF1CB"
state_fill_high = "#FF8F8F"

# =========================================================
# 2. Plot settings
# =========================================================
plt.close("all")
plt.style.use("default")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.35,
    "xtick.major.width": 0.35,
    "ytick.major.width": 0.35,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =========================================================
# 3. Helper functions
# =========================================================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def _safe_name(x):
    x = str(x)
    x = x.replace(" pathway", "")
    x = x.replace(" expression and ", "_")
    x = x.replace(" checkpoint pathway in cancer", "_checkpoint")
    x = x.replace(" signaling pathway", "")
    x = x.replace(" interaction", "")
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def _short_pathway_name(x):
    short_map = {
        "HIF-1 signaling pathway": "HIF1",
        "PD-L1 expression and PD-1 checkpoint pathway in cancer": "PD1_PDL1",
        "PI3K-Akt signaling pathway": "PI3K_Akt",
        "ECM-receptor interaction": "ECM_receptor",
    }
    return short_map.get(str(x), _safe_name(x))


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id", "Sample", "Sample_ID",
        "SAMPLE_ID", "patient", "patients", "Patient", "Patient_ID",
        "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _barcode_col_from_obs(adata):
    barcode_candidates = ["barcode", "Barcode", "cell_id", "cell", "CellID"]
    return next((col for col in barcode_candidates if col in adata.obs.columns), None)


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _safe_nanmax(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.max()


def _safe_nanmin(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.min()


def _p_to_label(p):
    if pd.isna(p):
        return "P=NA"
    if p < 1e-4:
        return "P<1e-4"
    return f"P={p:.2e}"


def save_plot(fig, stem, out_dir=file_savepath_main, width=1.35, height=2.6):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = out_dir / f"{stem}.pdf"
    png_path = out_dir / f"{stem}.png"

    fig.set_size_inches(width, height)
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)

    return pdf_path, png_path


def make_samplelevel_box_plot(
    sample_df,
    ycol,
    title_txt,
    fill_low="#FFF1CB",
    fill_high="#FF8F8F",
    pval=np.nan,
    width=1.35,
    height=2.6,
):
    lvl_low = f"<{thr_use}"
    lvl_high = f"\u2265{thr_use}"
    group_order = [lvl_high, lvl_low]

    plot_df = sample_df.loc[
        sample_df["MI_group"].isin(group_order)
        & np.isfinite(pd.to_numeric(sample_df[ycol], errors="coerce")),
        ["sample_id", "MI_group", ycol]
    ].copy()

    plot_df[ycol] = pd.to_numeric(plot_df[ycol], errors="coerce")
    plot_df["MI_group"] = pd.Categorical(
        plot_df["MI_group"],
        categories=group_order,
        ordered=True,
    )

    grouped = [
        plot_df.loc[plot_df["MI_group"] == lvl_high, ycol].to_numpy(dtype=float),
        plot_df.loc[plot_df["MI_group"] == lvl_low, ycol].to_numpy(dtype=float),
    ]

    fig, ax = plt.subplots(figsize=(width, height))

    bp = ax.boxplot(
        grouped,
        positions=[1, 2],
        widths=0.58,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(linewidth=0.35, color="black"),
        medianprops=dict(linewidth=0.35, color="black"),
        whiskerprops=dict(linewidth=0.35, color="black"),
        capprops=dict(linewidth=0.35, color="black"),
    )

    for patch, color in zip(bp["boxes"], [fill_high, fill_low]):
        patch.set_facecolor(color)
        patch.set_edgecolor("black")

    # paired sample lines when both groups exist within sample
    wide = (
        plot_df
        .pivot_table(
            index="sample_id",
            columns="MI_group",
            values=ycol,
            aggfunc="mean",
            observed=True,
        )
        .dropna(subset=group_order, how="any")
    )

    if wide.shape[0] > 0:
        for sample_id, row in wide.iterrows():
            ax.plot(
                [1, 2],
                [row[lvl_high], row[lvl_low]],
                color="black",
                linewidth=0.35,
                alpha=0.35,
                zorder=3,
            )

        ax.scatter(
            np.ones(wide.shape[0]),
            wide[lvl_high],
            s=9,
            color="black",
            alpha=0.65,
            zorder=4,
        )

        ax.scatter(
            np.ones(wide.shape[0]) * 2,
            wide[lvl_low],
            s=9,
            color="black",
            alpha=0.65,
            zorder=4,
        )

    star = p_to_star(pval)
    p_lbl = _p_to_label(pval)

    y_top = _safe_nanmax(plot_df[ycol])
    y_min = _safe_nanmin(plot_df[ycol])

    if np.isfinite(y_top) and np.isfinite(y_min):
        y_rng = y_top - y_min + 1e-12
        y_anno = y_top + 0.15 * y_rng
        y_tick = 0.05 * y_rng
        y_text = y_anno + 0.025 * y_rng
        ylim_top = y_text + 0.18 * y_rng

        ax.plot([1, 2], [y_anno, y_anno], color="black", linewidth=0.35)
        ax.plot([1, 1], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)
        ax.plot([2, 2], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)

        ax.text(
            1.5,
            y_text,
            f"{p_lbl}  {star}",
            ha="center",
            va="bottom",
            fontsize=7,
            color="black",
        )

        ax.set_ylim(top=ylim_top)

    ax.set_xticks([1, 2])
    ax.set_xticklabels([lvl_high, lvl_low], fontsize=8, color="black")
    ax.set_title(title_txt, fontsize=9, loc="left", color="black")
    ax.set_xlabel("", fontsize=9, color="black")
    ax.set_ylabel("Sample-level\npathway score", fontsize=9, color="black")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.35)
        ax.spines[spine].set_color("black")

    ax.tick_params(axis="both", colors="black", width=0.35, length=2)
    ax.grid(False)

    fig.tight_layout()

    return fig, ax, wide


# =========================================================
# 4. Check required variables from earlier notebook cells
# =========================================================
required_vars = [
    "adata_copy",
    "cellclass_updated",
    "MI_SR_agg_cur_all_FibroOnly",
    "MI_agg_meta",
]

missing_required = [v for v in required_vars if v not in globals()]
if len(missing_required) > 0:
    raise NameError(
        "Missing required variables from earlier notebook cells:\n"
        f"{missing_required}\n"
        "Please run the earlier MI/Fibroblast-to-malignant C5 analysis cells first."
    )

# =========================================================
# 5. Define malignant C5 cells and malignant_idx_further
#    This exactly follows the earlier Metastasis/Hypoxia/Angiogenesis logic.
# =========================================================
cellclass_updated_arr = np.asarray(cellclass_updated).astype(str)

malignant_idx = np.where(cellclass_updated_arr == Malignant_COI)[0]

if len(malignant_idx) == 0:
    raise ValueError(
        f"No cells found with cellclass_updated == '{Malignant_COI}'. "
        "Please check cellclass_updated labels."
    )

malignant_idx_further = malignant_idx[
    np.max(MI_SR_agg_cur_all_FibroOnly[malignant_idx, :], axis=1) > 0
]

if len(malignant_idx_further) == 0:
    raise ValueError(
        "No malignant C5 cells remain after filtering for Fibroblast-associated MI level > 0."
    )

print(f"Total malignant C5 cells: {len(malignant_idx)}")
print(f"Malignant C5 cells near Fibroblast by MI level > 0: {len(malignant_idx_further)}")

# Sample and barcode information from adata_copy
sample_col = _sample_col_from_obs(adata_copy)
if sample_col is not None:
    sample_values = adata_copy.obs.iloc[malignant_idx_further][sample_col].astype(str).to_numpy()
    print(f"Using sample column from adata_copy: {sample_col}")
else:
    print("[Warning] No sample column found in adata_copy.obs. All cells treated as one pseudo-sample.")
    sample_values = np.array(["__all_cells__"] * len(malignant_idx_further))

barcode_col = _barcode_col_from_obs(adata_copy)
if barcode_col is not None:
    barcode_values = adata_copy.obs.iloc[malignant_idx_further][barcode_col].astype(str).to_numpy()
else:
    barcode_values = adata_copy.obs_names[malignant_idx_further].astype(str).to_numpy()

# =========================================================
# 6. Load KEGG pathway genes
# =========================================================
if "c5_kegg_reference_gene_df" in globals():
    pathway_gene_df = c5_kegg_reference_gene_df.copy()
else:
    candidate_path = (
        file_savepath_main
        / "TCGA_OV_KEGG_gene_signature"
        / "C5_four_KEGG_pathway_reference_genes_long.csv"
    )

    if not input_path(candidate_path).exists():
        raise FileNotFoundError(
            "Cannot find c5_kegg_reference_gene_df in memory or saved pathway-gene CSV:\n"
            f"{candidate_path}\n"
            "Please run the KEGG reference gene export cell first."
        )

    pathway_gene_df = pd.read_csv(input_path(candidate_path))

if "Requested_Pathway" in pathway_gene_df.columns:
    pathway_col = "Requested_Pathway"
elif "Pathway" in pathway_gene_df.columns:
    pathway_col = "Pathway"
else:
    raise KeyError(
        "Cannot find pathway column. Expected 'Requested_Pathway' or 'Pathway'. "
        f"Available columns: {list(pathway_gene_df.columns)}"
    )

if "Gene" not in pathway_gene_df.columns:
    raise KeyError(
        f"Cannot find 'Gene' column in pathway_gene_df. Available columns: {list(pathway_gene_df.columns)}"
    )

pathway_gene_df = (
    pathway_gene_df
    .loc[pathway_gene_df[pathway_col].isin(KEGG_PATHWAYS_TO_SCORE), [pathway_col, "Gene"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

pathway_gene_df[pathway_col] = pathway_gene_df[pathway_col].astype(str)
pathway_gene_df["Gene"] = pathway_gene_df["Gene"].astype(str)

if pathway_gene_df.empty:
    raise ValueError("No KEGG genes found for the selected four pathways.")

print("KEGG pathway gene counts before expression extraction:")
_display_df(
    pathway_gene_df
    .groupby(pathway_col, as_index=False)
    .agg(n_genes=("Gene", "nunique"))
)

# =========================================================
# 7. Compute KEGG pathway scores directly in malignant_idx_further
# =========================================================
all_pathway_genes = sorted(pathway_gene_df["Gene"].astype(str).unique())
resolved_genes, missing_genes = _resolve_gene_names(adata_copy, all_pathway_genes)

available_genes = sorted(resolved_genes.keys())

if len(available_genes) == 0:
    raise ValueError("None of the selected KEGG genes are available in adata_copy.var_names.")

if len(missing_genes) > 0:
    print(f"[Warning] Missing KEGG genes skipped: {missing_genes}")

pathway_gene_df_available = pathway_gene_df.loc[
    pathway_gene_df["Gene"].isin(available_genes)
].drop_duplicates().copy()

print("KEGG pathway gene counts used:")
_display_df(
    pathway_gene_df_available
    .groupby(pathway_col, as_index=False)
    .agg(n_genes_used=("Gene", "nunique"))
)

actual_gene_names = [resolved_genes[g] for g in available_genes]
gene_idx = [adata_copy.var_names.get_loc(g) for g in actual_gene_names]

X_sub = adata_copy.X[malignant_idx_further, :][:, gene_idx]

if sparse.issparse(X_sub):
    X_sub = X_sub.toarray()
else:
    X_sub = np.asarray(X_sub)

expr_gene_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=barcode_values,
)

if SCORE_METHOD == "maxnorm":
    # Legacy option: exp_norm = exp / max(exp, axis=0)
    gene_max = expr_gene_df.max(axis=0).replace(0, np.nan)
    expr_score_base = expr_gene_df / gene_max
    expr_score_base = expr_score_base.fillna(0)
elif SCORE_METHOD in {"zscore", "zscore_mean_global"}:
    # zscore_mean_global:
    # Use all selected malignant cells across slices as one global reference.
    # Then apply the same gene-wise mean/std to every cell and average genes.
    gene_mean = expr_gene_df.mean(axis=0)
    gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)
    expr_score_base = (expr_gene_df - gene_mean) / gene_std
    expr_score_base = expr_score_base.fillna(0)
else:
    raise ValueError("SCORE_METHOD must be 'zscore_mean_global', 'zscore', or 'maxnorm'.")

pathway_score_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    genes_cur = (
        pathway_gene_df_available
        .loc[pathway_gene_df_available[pathway_col] == pathway, "Gene"]
        .astype(str)
        .unique()
        .tolist()
    )
    genes_cur = [g for g in genes_cur if g in expr_score_base.columns]

    if len(genes_cur) == 0:
        print(f"[Warning] No available genes for pathway: {pathway}. Skipping.")
        continue

    score_cur = expr_score_base[genes_cur].mean(axis=1).to_numpy()

    pathway_score_records.append(pd.DataFrame({
        "barcode": barcode_values,
        "sample_id": sample_values,
        "full_cell_index": malignant_idx_further,
        "Pathway": pathway,
        "Pathway_short": _short_pathway_name(pathway),
        "State_Score": score_cur,
        "n_genes_used": len(genes_cur),
        "score_method": SCORE_METHOD,
    }))

pathway_score_df = pd.concat(pathway_score_records, axis=0, ignore_index=True)

pathway_score_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_celllevel_pathway_scores.csv",
    index=False,
)

pathway_gene_df_available.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_KEGG_genes_used.csv",
    index=False,
)

score_base_out = expr_score_base.copy()
score_base_out.columns = [f"{g}_{SCORE_METHOD}" for g in score_base_out.columns]
score_base_out["barcode"] = barcode_values
score_base_out["sample_id"] = sample_values
score_base_out["full_cell_index"] = malignant_idx_further

score_base_out.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_gene_level_{SCORE_METHOD}_matrix.csv",
    index=False,
)

print("Cell-level KEGG pathway score table:")
_display_df(pathway_score_df.head())

# =========================================================
# 8. Extract MI levels directly from MI_SR_agg_cur_all_FibroOnly
#    This follows the earlier code exactly.
# =========================================================
mi_level_records = []

for MI_OI, direction in zip(MI_OI_DF["MI"], MI_OI_DF["direction"]):
    idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index

    print(f"MI_OI: {MI_OI}, direction: {direction}, idx: {list(idx)}")

    if len(idx) == 0:
        print(f"[Warning] No MI_agg_meta entry found for {MI_OI}, {direction}. Skipping.")
        continue

    mi_vals = MI_SR_agg_cur_all_FibroOnly[malignant_idx_further, :][:, idx]

    if mi_vals.ndim == 2 and mi_vals.shape[1] == 1:
        mi_vals = mi_vals[:, 0]
    else:
        # If multiple matching columns exist, average them.
        mi_vals = np.asarray(mi_vals).mean(axis=1)

    mi_level_records.append(pd.DataFrame({
        "barcode": barcode_values,
        "sample_id": sample_values,
        "full_cell_index": malignant_idx_further,
        "MI": MI_OI,
        "direction": direction,
        "MI_label": f"Receiving{MI_OI.replace('MI', 'MI')}",
        "MI_Level": np.asarray(mi_vals, dtype=float),
    }))

mi_level_df = pd.concat(mi_level_records, axis=0, ignore_index=True)

mi_level_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_direct_MI_levels.csv",
    index=False,
)

print("Direct MI-level table:")
_display_df(mi_level_df.head())

# =========================================================
# 9. Sample-level MI-high vs MI-low analysis
# =========================================================
lvl_low = f"<{thr_use}"
lvl_high = f"\u2265{thr_use}"
GROUP_ORDER = [lvl_high, lvl_low]

all_celllevel_rows = []
all_samplelevel_rows = []
summary_rows = []
plot_results = {}

for _, mi_meta in mi_level_df[["MI", "direction", "MI_label"]].drop_duplicates().iterrows():
    MI_OI = mi_meta["MI"]
    direction = mi_meta["direction"]
    mi_label = mi_meta["MI_label"]

    mi_sub = mi_level_df.loc[
        (mi_level_df["MI"] == MI_OI)
        & (mi_level_df["direction"] == direction)
    ].copy()

    for pathway in KEGG_PATHWAYS_TO_SCORE:
        pathway_short = _short_pathway_name(pathway)

        score_sub = pathway_score_df.loc[
            pathway_score_df["Pathway"] == pathway
        ].copy()

        df = score_sub.merge(
            mi_sub[["barcode", "full_cell_index", "MI_Level"]],
            on=["barcode", "full_cell_index"],
            how="inner",
        )

        df["MI_Level"] = pd.to_numeric(df["MI_Level"], errors="coerce")
        df["State_Score"] = pd.to_numeric(df["State_Score"], errors="coerce")

        df = df.loc[
            np.isfinite(df["MI_Level"].to_numpy(dtype=float))
            & np.isfinite(df["State_Score"].to_numpy(dtype=float))
        ].copy()

        if df.empty:
            print(f"[Warning] No valid rows for {pathway_short} vs {mi_label}. Skipping.")
            continue

        df["MI_group"] = np.where(df["MI_Level"] >= thr_use, lvl_high, lvl_low)
        df["MI_group"] = pd.Categorical(
            df["MI_group"],
            categories=GROUP_ORDER,
            ordered=True,
        )

        scatter_stem = f"Malignant_COI5_{pathway_short}_vs_{mi_label}_direct_scatter"
        scatter_csv = file_savepath_main / f"{scatter_stem}.csv"

        df_out = df[
            [
                "barcode",
                "sample_id",
                "full_cell_index",
                "Pathway",
                "Pathway_short",
                "State_Score",
                "MI_Level",
                "MI_group",
                "n_genes_used",
            ]
        ].copy()

        df_out.to_csv(scatter_csv, index=False)

        all_celllevel_rows.append(
            df_out.assign(
                MI=MI_OI,
                direction=direction,
                MI_label=mi_label,
            )
        )

        # sample-level aggregation
        sample_df = (
            df
            .groupby(["sample_id", "Pathway", "Pathway_short", "MI_group"], observed=True)
            .agg(
                mean_State_Score=("State_Score", "mean"),
                median_State_Score=("State_Score", "median"),
                mean_MI_Level=("MI_Level", "mean"),
                n_cells=("barcode", "count"),
                n_genes_used=("n_genes_used", "first"),
            )
            .reset_index()
        )

        sample_df["MI"] = MI_OI
        sample_df["direction"] = direction
        sample_df["MI_label"] = mi_label

        sample_df_all_path = file_savepath_main / f"Malignant_COI5_{pathway_short}_vs_{mi_label}_direct_samplelevel_all.csv"
        sample_df.to_csv(sample_df_all_path, index=False)

        sample_df_plot = sample_df.loc[
            sample_df["n_cells"] >= MIN_CELLS_PER_SAMPLE_GROUP
        ].copy()

        sample_df_plot["MI_group"] = pd.Categorical(
            sample_df_plot["MI_group"],
            categories=GROUP_ORDER,
            ordered=True,
        )

        sample_df_min_path = file_savepath_main / f"Malignant_COI5_{pathway_short}_vs_{mi_label}_direct_samplelevel_min{MIN_CELLS_PER_SAMPLE_GROUP}cells.csv"
        sample_df_plot.to_csv(sample_df_min_path, index=False)

        all_samplelevel_rows.append(sample_df_plot)

        # paired sample-level comparison if possible
        sample_wide = (
            sample_df_plot
            .pivot_table(
                index="sample_id",
                columns="MI_group",
                values="mean_State_Score",
                aggfunc="mean",
                observed=True,
            )
        )

        sample_wide = sample_wide.dropna(subset=GROUP_ORDER, how="any")

        pval_score = np.nan
        stat_score = np.nan
        test_name = "NA"

        if sample_wide.shape[0] >= 2:
            try:
                stat_score, pval_score = wilcoxon(
                    sample_wide[lvl_high],
                    sample_wide[lvl_low],
                    alternative=TEST_ALTERNATIVE,
                )
                test_name = "paired Wilcoxon"
            except ValueError:
                stat_score = np.nan
                pval_score = np.nan
                test_name = "paired Wilcoxon"
        else:
            high_vals = sample_df_plot.loc[
                sample_df_plot["MI_group"] == lvl_high,
                "mean_State_Score"
            ].to_numpy(dtype=float)

            low_vals = sample_df_plot.loc[
                sample_df_plot["MI_group"] == lvl_low,
                "mean_State_Score"
            ].to_numpy(dtype=float)

            high_vals = high_vals[np.isfinite(high_vals)]
            low_vals = low_vals[np.isfinite(low_vals)]

            if len(high_vals) > 0 and len(low_vals) > 0:
                stat_score, pval_score = ranksums(
                    high_vals,
                    low_vals,
                    alternative=TEST_ALTERNATIVE,
                )
                test_name = "sample-level ranksum"

        m_high = sample_df_plot.loc[
            sample_df_plot["MI_group"] == lvl_high,
            "mean_State_Score"
        ].mean()

        m_low = sample_df_plot.loc[
            sample_df_plot["MI_group"] == lvl_low,
            "mean_State_Score"
        ].mean()

        mean_diff = m_high - m_low if np.isfinite(m_high) and np.isfinite(m_low) else np.nan

        if np.isfinite(m_high) and np.isfinite(m_low) and m_high > 0 and m_low > 0:
            log2_mean_change = np.log2(m_high / m_low)
        else:
            log2_mean_change = np.nan

        summary_rows.append({
            "Pathway": pathway,
            "Pathway_short": pathway_short,
            "MI": MI_OI,
            "direction": direction,
            "MI_label": mi_label,
            "threshold": thr_use,
            "score_method": SCORE_METHOD,
            "test": test_name,
            "alternative": TEST_ALTERNATIVE,
            "n_paired_samples": sample_wide.shape[0],
            "n_sample_group_rows": sample_df_plot.shape[0],
            "mean_high": m_high,
            "mean_low": m_low,
            "mean_diff_high_minus_low": mean_diff,
            "log2_mean_change_high_over_low": log2_mean_change,
            "statistic": stat_score,
            "pvalue": pval_score,
            "star": p_to_star(pval_score),
            "scatter_csv": str(scatter_csv),
            "samplelevel_all_csv": str(sample_df_all_path),
            "samplelevel_min_cells_csv": str(sample_df_min_path),
        })

        # plot
        title_txt = f"{pathway_short} vs {mi_label}"

        fig, ax, sample_wide_for_plot = make_samplelevel_box_plot(
            sample_df=sample_df_plot.rename(columns={"mean_State_Score": "PlotScore"}),
            ycol="PlotScore",
            title_txt=title_txt,
            fill_low=state_fill_low,
            fill_high=state_fill_high,
            pval=pval_score,
            width=1.35,
            height=2.6,
        )

        stem = f"Malignant_COI5_{pathway_short}_vs_{mi_label}_direct_samplelevel_box_thr{thr_tag}_{SCORE_METHOD}"
        pdf_path, png_path = save_plot(fig, stem=stem, width=1.35, height=2.6)

        plot_results[f"{pathway_short}_{mi_label}"] = {
            "fig": fig,
            "ax": ax,
            "pdf_path": pdf_path,
            "png_path": png_path,
            "sample_wide": sample_wide,
        }

# =========================================================
# 10. Save combined outputs
# =========================================================
if len(all_celllevel_rows) > 0:
    all_celllevel_df = pd.concat(all_celllevel_rows, axis=0, ignore_index=True)
    all_celllevel_df.to_csv(
        file_savepath_main / f"{OUT_PREFIX}_all_celllevel_direct_scatter_compatible.csv",
        index=False,
    )
else:
    all_celllevel_df = pd.DataFrame()

if len(all_samplelevel_rows) > 0:
    all_samplelevel_df = pd.concat(all_samplelevel_rows, axis=0, ignore_index=True)
    all_samplelevel_df.to_csv(
        file_savepath_main / f"{OUT_PREFIX}_all_samplelevel_min{MIN_CELLS_PER_SAMPLE_GROUP}cells.csv",
        index=False,
    )
else:
    all_samplelevel_df = pd.DataFrame()

kegg_boxplot_summary_df = pd.DataFrame(summary_rows)
kegg_boxplot_summary_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_summary_thr{thr_tag}_{SCORE_METHOD}.csv",
    index=False,
)

print("KEGG pathway sample-level comparison summary:")
_display_df(kegg_boxplot_summary_df)

# =========================================================
# 11. Show all generated boxplots
# =========================================================
for nm, obj in plot_results.items():
    print(f"Showing plot: {nm}")
    _display_df(pd.DataFrame([{
        "pdf_path": str(obj["pdf_path"]),
        "png_path": str(obj["png_path"]),
    }]))
    display(obj["fig"])

plt.close("all")


In [ ]:
# Cell-level KEGG comparisons by fibroblast-derived MI receiving strength
# Compute four KEGG module scores in malignant C5 cells with positive
# fibroblast-associated MI strength. Compare MI-10/MI-6/MI-13 receiving-high
# (>=0.5) and receiving-low cells using two-sided Mann-Whitney tests.

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import sparse
from scipy.stats import mannwhitneyu

# =========================================================
# 1. Global settings
# =========================================================
file_savepath_main = Path(run_dirs["run_dir"]) if "run_dirs" in globals() else Path(".")
file_savepath_main.mkdir(parents=True, exist_ok=True)

thr_use = 0.5
thr_tag = f"{thr_use:.2f}".replace(".", "p")

Malignant_COI = "Malignant_5"

KEGG_PATHWAYS_TO_SCORE = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

MI_OI_DF = pd.DataFrame({
    "MI": ["MI10", "MI6", "MI13"],
    "direction": ["Receiver", "Receiver", "Receiver"],
})

# Same direction as the reference cell: two-sided cell-level Mann–Whitney test
TEST_ALTERNATIVE = "two-sided"

# Score method:
#   "zscore_mean_global": concatenate the selected malignant cells across all slices,
#                          compute one global mean/std per gene, then average
#                          global z-scored genes for each cell.
#   "maxnorm": legacy max-normalized expression
#   "zscore": legacy alias for zscore_mean_global
SCORE_METHOD = "zscore_mean_global"

OUT_PREFIX = "Malignant_COI5_KEGG_pathway_score_vs_ReceivingMI_celllevel_direct"

state_fill_low = "#FFF1CB"
state_fill_high = "#FF8F8F"

# =========================================================
# 2. Plot settings
# =========================================================
plt.close("all")
plt.style.use("default")

rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.35,
    "xtick.major.width": 0.35,
    "ytick.major.width": 0.35,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# =========================================================
# 3. Utility helpers
# =========================================================
def _display_df(df):
    if "display" in globals():
        display(df)
    else:
        print(df)


def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def _safe_nanmax(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.max()


def _safe_nanmin(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return np.nan if arr.size == 0 else arr.min()


def _safe_name(x):
    x = str(x)
    x = x.replace(" pathway", "")
    x = x.replace(" expression and ", "_")
    x = x.replace(" checkpoint pathway in cancer", "_checkpoint")
    x = x.replace(" signaling pathway", "")
    x = x.replace(" interaction", "")
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def _short_pathway_name(x):
    short_map = {
        "HIF-1 signaling pathway": "HIF1",
        "PD-L1 expression and PD-1 checkpoint pathway in cancer": "PD1_PDL1",
        "PI3K-Akt signaling pathway": "PI3K_Akt",
        "ECM-receptor interaction": "ECM_receptor",
    }
    return short_map.get(str(x), _safe_name(x))


def _sample_col_from_obs(adata):
    sample_col_candidates = [
        "samples", "sample", "sample_name", "sample_id",
        "Sample", "Sample_ID", "SAMPLE_ID",
        "patient", "patients", "Patient", "Patient_ID",
        "library_id", "slide", "slice"
    ]
    return next((col for col in sample_col_candidates if col in adata.obs.columns), None)


def _barcode_col_from_obs(adata):
    barcode_candidates = ["barcode", "Barcode", "cell_id", "cell", "CellID"]
    return next((col for col in barcode_candidates if col in adata.obs.columns), None)


def _resolve_gene_names(adata, genes):
    var_names = pd.Index(adata.var_names.astype(str))
    upper_to_actual = {str(g).upper(): str(g) for g in var_names}

    resolved = {}
    missing = []

    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _matrix_row_max(mat, rows):
    sub = mat[rows, :]

    if sparse.issparse(sub):
        return np.asarray(sub.max(axis=1)).ravel()

    return np.asarray(np.max(sub, axis=1)).ravel()


def _matrix_get_columns(mat, rows, cols):
    sub = mat[rows, :][:, cols]

    if sparse.issparse(sub):
        sub = sub.toarray()
    else:
        sub = np.asarray(sub)

    if sub.ndim == 1:
        return sub

    if sub.shape[1] == 1:
        return sub[:, 0]

    return np.nanmean(sub, axis=1)


def save_plot(fig, stem, out_dir=file_savepath_main, width=1.2, height=2.6):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = out_dir / f"{stem}.pdf"
    png_path = out_dir / f"{stem}.png"

    fig.set_size_inches(width, height)
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white", transparent=False)

    return pdf_path, png_path


def make_celllevel_box_plot(
    df,
    xcol,
    ycol,
    thr=0.5,
    fill_low="#FFF1CB",
    fill_high="#FF8F8F",
    title_txt=None,
    xlab=None,
    ylab=None,
    print_summary=True,
):
    lvl_low = f"<{thr}"
    lvl_high = f"\u2265{thr}"

    plot_df = df.loc[
        np.isfinite(pd.to_numeric(df[xcol], errors="coerce"))
        & np.isfinite(pd.to_numeric(df[ycol], errors="coerce")),
        [xcol, ycol]
    ].copy()

    plot_df[xcol] = pd.to_numeric(plot_df[xcol], errors="coerce")
    plot_df[ycol] = pd.to_numeric(plot_df[ycol], errors="coerce")

    plot_df["MI_group"] = np.where(plot_df[xcol] >= thr, lvl_high, lvl_low)
    plot_df["MI_group"] = pd.Categorical(
        plot_df["MI_group"],
        categories=[lvl_high, lvl_low],
        ordered=True,
    )

    high_vals = plot_df.loc[
        plot_df["MI_group"] == lvl_high,
        ycol
    ].to_numpy(dtype=float)

    low_vals = plot_df.loc[
        plot_df["MI_group"] == lvl_low,
        ycol
    ].to_numpy(dtype=float)

    high_vals = high_vals[np.isfinite(high_vals)]
    low_vals = low_vals[np.isfinite(low_vals)]

    m_high = np.nanmean(high_vals) if high_vals.size else np.nan
    m_low = np.nanmean(low_vals) if low_vals.size else np.nan
    mean_diff = m_high - m_low if np.isfinite(m_high) and np.isfinite(m_low) else np.nan

    if np.isfinite(m_high) and np.isfinite(m_low) and m_high > 0 and m_low > 0:
        log2_mean_change = np.log2(m_high / m_low)
    else:
        log2_mean_change = np.nan

    p_wilcox = np.nan

    if high_vals.size > 0 and low_vals.size > 0:
        try:
            p_wilcox = mannwhitneyu(
                high_vals,
                low_vals,
                alternative=TEST_ALTERNATIVE,
                method="asymptotic",
            ).pvalue
        except TypeError:
            p_wilcox = mannwhitneyu(
                high_vals,
                low_vals,
                alternative=TEST_ALTERNATIVE,
            ).pvalue

    star = p_to_star(p_wilcox)
    p_lbl = "p = NA" if pd.isna(p_wilcox) else f"p = {p_wilcox:.2e}"

    if print_summary:
        print("\n========================================")
        print("Cell-level boxplot summary")
        print(f"x column: {xcol}")
        print(f"y column: {ycol}")
        print(f"threshold: {thr}")
        print(f"n high cells: {len(high_vals)}")
        print(f"n low cells: {len(low_vals)}")
        print(f"Mean high: {m_high:.4g}" if np.isfinite(m_high) else "Mean high: NA")
        print(f"Mean low: {m_low:.4g}" if np.isfinite(m_low) else "Mean low: NA")
        print(f"Mean difference high - low: {mean_diff:.4g}" if np.isfinite(mean_diff) else "Mean difference high - low: NA")
        print(f"Mann–Whitney / rank-sum p: {p_wilcox:.3e}" if np.isfinite(p_wilcox) else "Mann–Whitney / rank-sum p: NA")
        print("========================================")

    y_top = _safe_nanmax(plot_df[ycol])
    y_min = _safe_nanmin(plot_df[ycol])

    if np.isfinite(y_top) and np.isfinite(y_min):
        y_rng = y_top - y_min + 1e-12
        y_anno = y_top + 0.15 * y_rng
        y_tick = 0.05 * y_rng
        y_text = y_anno + 0.03 * y_rng
        ylim_top = y_text + 0.18 * y_rng
    else:
        y_anno = y_tick = y_text = ylim_top = np.nan

    fig, ax = plt.subplots(figsize=(1.2, 2.6))

    grouped = [
        plot_df.loc[plot_df["MI_group"] == lvl_high, ycol].to_numpy(dtype=float),
        plot_df.loc[plot_df["MI_group"] == lvl_low, ycol].to_numpy(dtype=float),
    ]

    bp = ax.boxplot(
        grouped,
        positions=[1, 2],
        widths=0.58,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(linewidth=0.35, color="black"),
        medianprops=dict(linewidth=0.35, color="black"),
        whiskerprops=dict(linewidth=0.35, color="black"),
        capprops=dict(linewidth=0.35, color="black"),
    )

    for patch, color in zip(bp["boxes"], [fill_high, fill_low]):
        patch.set_facecolor(color)
        patch.set_edgecolor("black")

    if np.isfinite(y_anno):
        ax.plot([1, 2], [y_anno, y_anno], color="black", linewidth=0.35)
        ax.plot([1, 1], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)
        ax.plot([2, 2], [y_anno, y_anno - y_tick], color="black", linewidth=0.35)

        ax.text(
            1.5,
            y_text,
            f"{p_lbl}  {star}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="black",
        )

        ax.set_ylim(top=ylim_top)

    ax.set_xticks([1, 2])
    ax.set_xticklabels([lvl_high, lvl_low], fontsize=8, color="black")
    ax.set_title("" if title_txt is None else title_txt, fontsize=9, loc="left", color="black")
    ax.set_xlabel("" if xlab is None else xlab, fontsize=9, color="black")
    ax.set_ylabel("" if ylab is None else ylab, fontsize=9, color="black")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.35)
        ax.spines[spine].set_color("black")

    ax.tick_params(axis="both", colors="black", width=0.35, length=2)
    ax.grid(False)

    fig.tight_layout()

    stats_dict = {
        "n_high_cells": len(high_vals),
        "n_low_cells": len(low_vals),
        "m_high": m_high,
        "m_low": m_low,
        "mean_diff": mean_diff,
        "log2_mean_change": log2_mean_change,
        "p_wilcox": p_wilcox,
        "star": star,
    }

    return fig, ax, plot_df, stats_dict


# =========================================================
# 4. Check required variables from earlier notebook cells
# =========================================================
required_vars = [
    "adata_copy",
    "cellclass_updated",
    "MI_SR_agg_cur_all_FibroOnly",
    "MI_agg_meta",
]

missing_required = [v for v in required_vars if v not in globals()]

if len(missing_required) > 0:
    raise NameError(
        "Missing required variables from earlier notebook cells:\n"
        f"{missing_required}\n"
        "Please run the earlier MI/Fibroblast-to-malignant C5 analysis cells first."
    )

# =========================================================
# 5. Define malignant C5 cells and malignant_idx_further
# =========================================================
cellclass_updated_arr = np.asarray(cellclass_updated).astype(str)

malignant_idx = np.where(cellclass_updated_arr == Malignant_COI)[0]

if len(malignant_idx) == 0:
    raise ValueError(
        f"No cells found with cellclass_updated == '{Malignant_COI}'. "
        "Please check cellclass_updated labels."
    )

malignant_row_max = _matrix_row_max(MI_SR_agg_cur_all_FibroOnly, malignant_idx)

malignant_idx_further = malignant_idx[malignant_row_max > 0]

if len(malignant_idx_further) == 0:
    raise ValueError(
        "No malignant C5 cells remain after filtering for Fibroblast-associated MI level > 0."
    )

print(f"Total malignant C5 cells: {len(malignant_idx)}")
print(f"Malignant C5 cells with Fibroblast-associated MI level > 0: {len(malignant_idx_further)}")

sample_col = _sample_col_from_obs(adata_copy)

if sample_col is not None:
    sample_values = adata_copy.obs.iloc[malignant_idx_further][sample_col].astype(str).to_numpy()
    print(f"Using sample column from adata_copy: {sample_col}")
else:
    print("[Warning] No sample column found in adata_copy.obs. All cells treated as one pseudo-sample.")
    sample_values = np.array(["__all_cells__"] * len(malignant_idx_further))

barcode_col = _barcode_col_from_obs(adata_copy)

if barcode_col is not None:
    barcode_values = adata_copy.obs.iloc[malignant_idx_further][barcode_col].astype(str).to_numpy()
else:
    barcode_values = adata_copy.obs_names[malignant_idx_further].astype(str).to_numpy()

# =========================================================
# 6. Load KEGG pathway genes
# =========================================================
if "c5_kegg_reference_gene_df" in globals():
    pathway_gene_df = c5_kegg_reference_gene_df.copy()
else:
    candidate_path = (
        file_savepath_main
        / "TCGA_OV_KEGG_gene_signature"
        / "C5_four_KEGG_pathway_reference_genes_long.csv"
    )

    if not input_path(candidate_path).exists():
        raise FileNotFoundError(
            "Cannot find c5_kegg_reference_gene_df in memory or saved pathway-gene CSV:\n"
            f"{candidate_path}\n"
            "Please run the KEGG reference gene export cell first."
        )

    pathway_gene_df = pd.read_csv(input_path(candidate_path))

if "Requested_Pathway" in pathway_gene_df.columns:
    pathway_col = "Requested_Pathway"
elif "Pathway" in pathway_gene_df.columns:
    pathway_col = "Pathway"
else:
    raise KeyError(
        "Cannot find pathway column. Expected 'Requested_Pathway' or 'Pathway'. "
        f"Available columns: {list(pathway_gene_df.columns)}"
    )

if "Gene" not in pathway_gene_df.columns:
    raise KeyError(
        f"Cannot find 'Gene' column in pathway_gene_df. "
        f"Available columns: {list(pathway_gene_df.columns)}"
    )

pathway_gene_df = (
    pathway_gene_df
    .loc[pathway_gene_df[pathway_col].isin(KEGG_PATHWAYS_TO_SCORE), [pathway_col, "Gene"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

pathway_gene_df[pathway_col] = pathway_gene_df[pathway_col].astype(str)
pathway_gene_df["Gene"] = pathway_gene_df["Gene"].astype(str)

if pathway_gene_df.empty:
    raise ValueError("No KEGG genes found for the selected four pathways.")

print("KEGG pathway gene counts before expression extraction:")
_display_df(
    pathway_gene_df
    .groupby(pathway_col, as_index=False)
    .agg(n_genes=("Gene", "nunique"))
)

# =========================================================
# 7. Compute KEGG pathway scores in malignant_idx_further cells
# =========================================================
all_pathway_genes = sorted(pathway_gene_df["Gene"].astype(str).unique())

resolved_genes, missing_genes = _resolve_gene_names(adata_copy, all_pathway_genes)

available_genes = sorted(resolved_genes.keys())

if len(available_genes) == 0:
    raise ValueError("None of the selected KEGG genes are available in adata_copy.var_names.")

if len(missing_genes) > 0:
    print(f"[Warning] Missing KEGG genes skipped: {missing_genes}")

pathway_gene_df_available = pathway_gene_df.loc[
    pathway_gene_df["Gene"].isin(available_genes)
].drop_duplicates().copy()

print("KEGG pathway gene counts used:")
_display_df(
    pathway_gene_df_available
    .groupby(pathway_col, as_index=False)
    .agg(n_genes_used=("Gene", "nunique"))
)

actual_gene_names = [resolved_genes[g] for g in available_genes]
gene_idx = [adata_copy.var_names.get_loc(g) for g in actual_gene_names]

X_sub = adata_copy.X[malignant_idx_further, :][:, gene_idx]

if sparse.issparse(X_sub):
    X_sub = X_sub.toarray()
else:
    X_sub = np.asarray(X_sub)

expr_gene_df = pd.DataFrame(
    X_sub,
    columns=available_genes,
    index=barcode_values,
)

if SCORE_METHOD == "maxnorm":
    # Legacy option: exp_norm = exp / max(exp, axis=0)
    gene_max = expr_gene_df.max(axis=0).replace(0, np.nan)
    expr_score_base = expr_gene_df / gene_max
    expr_score_base = expr_score_base.fillna(0)
elif SCORE_METHOD in {"zscore", "zscore_mean_global"}:
    # zscore_mean_global:
    # Use all selected malignant cells across slices as one global reference.
    # Then apply the same gene-wise mean/std to every cell and average genes.
    gene_mean = expr_gene_df.mean(axis=0)
    gene_std = expr_gene_df.std(axis=0).replace(0, np.nan)
    expr_score_base = (expr_gene_df - gene_mean) / gene_std
    expr_score_base = expr_score_base.fillna(0)
else:
    raise ValueError("SCORE_METHOD must be 'zscore_mean_global', 'zscore', or 'maxnorm'.")

pathway_score_records = []

for pathway in KEGG_PATHWAYS_TO_SCORE:
    genes_cur = (
        pathway_gene_df_available
        .loc[pathway_gene_df_available[pathway_col] == pathway, "Gene"]
        .astype(str)
        .unique()
        .tolist()
    )

    genes_cur = [g for g in genes_cur if g in expr_score_base.columns]

    if len(genes_cur) == 0:
        print(f"[Warning] No available genes for pathway: {pathway}. Skipping.")
        continue

    score_cur = expr_score_base[genes_cur].mean(axis=1).to_numpy()

    pathway_score_records.append(pd.DataFrame({
        "barcode": barcode_values,
        "sample_id": sample_values,
        "full_cell_index": malignant_idx_further,
        "Pathway": pathway,
        "Pathway_short": _short_pathway_name(pathway),
        "State_Score": score_cur,
        "n_genes_used": len(genes_cur),
        "score_method": SCORE_METHOD,
    }))

pathway_score_df = pd.concat(pathway_score_records, axis=0, ignore_index=True)

pathway_score_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_celllevel_pathway_scores.csv",
    index=False,
)

pathway_gene_df_available.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_KEGG_genes_used.csv",
    index=False,
)

score_base_out = expr_score_base.copy()
score_base_out.columns = [f"{g}_{SCORE_METHOD}" for g in score_base_out.columns]
score_base_out["barcode"] = barcode_values
score_base_out["sample_id"] = sample_values
score_base_out["full_cell_index"] = malignant_idx_further

score_base_out.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_gene_level_{SCORE_METHOD}_matrix.csv",
    index=False,
)

print("Cell-level KEGG pathway score table:")
_display_df(pathway_score_df.head())

# =========================================================
# 8. Extract MI levels directly from MI_SR_agg_cur_all_FibroOnly
# =========================================================
mi_level_records = []

for MI_OI, direction in zip(MI_OI_DF["MI"], MI_OI_DF["direction"]):
    idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index

    print(f"MI_OI: {MI_OI}, direction: {direction}, idx: {list(idx)}")

    if len(idx) == 0:
        print(f"[Warning] No MI_agg_meta entry found for {MI_OI}, {direction}. Skipping.")
        continue

    mi_vals = _matrix_get_columns(
        MI_SR_agg_cur_all_FibroOnly,
        malignant_idx_further,
        list(idx),
    )

    mi_level_records.append(pd.DataFrame({
        "barcode": barcode_values,
        "sample_id": sample_values,
        "full_cell_index": malignant_idx_further,
        "MI": MI_OI,
        "direction": direction,
        "MI_label": f"Receiving{MI_OI}",
        "MI_Level": np.asarray(mi_vals, dtype=float),
    }))

if len(mi_level_records) == 0:
    raise ValueError("No MI levels were extracted. Please check MI_agg_meta and MI_OI_DF.")

mi_level_df = pd.concat(mi_level_records, axis=0, ignore_index=True)

mi_level_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_direct_MI_levels.csv",
    index=False,
)

print("Direct MI-level table:")
_display_df(mi_level_df.head())

# =========================================================
# 9. Cell-level MI-high vs MI-low analysis
# =========================================================
all_celllevel_rows = []
summary_rows = []
plot_results = {}
all_boxplots = {}

for _, mi_meta in mi_level_df[["MI", "direction", "MI_label"]].drop_duplicates().iterrows():
    MI_OI = mi_meta["MI"]
    direction = mi_meta["direction"]
    mi_label = mi_meta["MI_label"]

    mi_sub = mi_level_df.loc[
        (mi_level_df["MI"] == MI_OI)
        & (mi_level_df["direction"] == direction)
    ].copy()

    for pathway in KEGG_PATHWAYS_TO_SCORE:
        pathway_short = _short_pathway_name(pathway)

        score_sub = pathway_score_df.loc[
            pathway_score_df["Pathway"] == pathway
        ].copy()

        df = score_sub.merge(
            mi_sub[["barcode", "full_cell_index", "MI_Level"]],
            on=["barcode", "full_cell_index"],
            how="inner",
        )

        df["MI_Level"] = pd.to_numeric(df["MI_Level"], errors="coerce")
        df["State_Score"] = pd.to_numeric(df["State_Score"], errors="coerce")

        df = df.loc[
            np.isfinite(df["MI_Level"].to_numpy(dtype=float))
            & np.isfinite(df["State_Score"].to_numpy(dtype=float))
        ].copy()

        if df.empty:
            print(f"[Warning] No valid rows for {pathway_short} vs {mi_label}. Skipping.")
            continue

        # Save cell-level scatter-compatible CSV
        scatter_stem = f"Malignant_COI5_{pathway_short}_vs_{mi_label}_celllevel_scatter"
        scatter_csv = file_savepath_main / f"{scatter_stem}.csv"

        df_out = df[
            [
                "barcode",
                "sample_id",
                "full_cell_index",
                "Pathway",
                "Pathway_short",
                "State_Score",
                "MI_Level",
                "n_genes_used",
            ]
        ].copy()

        df_out.to_csv(scatter_csv, index=False)

        all_celllevel_rows.append(
            df_out.assign(
                MI=MI_OI,
                direction=direction,
                MI_label=mi_label,
            )
        )

        # Cell-level boxplot and Mann–Whitney test
        title_txt = f"{pathway_short} vs {mi_label}"

        fig, ax, plot_df, stats_dict = make_celllevel_box_plot(
            df=df,
            xcol="MI_Level",
            ycol="State_Score",
            thr=thr_use,
            fill_low=state_fill_low,
            fill_high=state_fill_high,
            title_txt=title_txt,
            xlab=None,
            ylab="Pathway score",
            print_summary=True,
        )

        stem = f"Malignant_COI5_{pathway_short}_vs_{mi_label}_celllevel_box_wilcox_thr{thr_tag}_{SCORE_METHOD}"
        pdf_path, png_path = save_plot(fig, stem=stem, width=1.2, height=2.6)

        plot_results[f"{pathway_short}_{mi_label}"] = {
            "data": df,
            "plot_df": plot_df,
            "fig": fig,
            "ax": ax,
            "pdf_path": pdf_path,
            "png_path": png_path,
            **stats_dict,
        }

        all_boxplots[f"{pathway_short}_vs_{mi_label}"] = fig

        summary_rows.append({
            "analysis": f"{pathway_short}_vs_{mi_label}",
            "Pathway": pathway,
            "Pathway_short": pathway_short,
            "MI": MI_OI,
            "direction": direction,
            "MI_label": mi_label,
            "threshold": thr_use,
            "score_method": SCORE_METHOD,
            "test": "cell-level Mann-Whitney/Wilcoxon rank-sum",
            "alternative": TEST_ALTERNATIVE,
            "n_high_cells": stats_dict["n_high_cells"],
            "n_low_cells": stats_dict["n_low_cells"],
            "m_high": stats_dict["m_high"],
            "m_low": stats_dict["m_low"],
            "mean_diff_high_minus_low": stats_dict["mean_diff"],
            "log2_mean_change_high_over_low": stats_dict["log2_mean_change"],
            "p_wilcox": stats_dict["p_wilcox"],
            "star": stats_dict["star"],
            "scatter_csv": str(scatter_csv),
            "pdf_path": str(pdf_path),
            "png_path": str(png_path),
        })

# =========================================================
# 10. Save combined outputs
# =========================================================
if len(all_celllevel_rows) > 0:
    all_celllevel_df = pd.concat(all_celllevel_rows, axis=0, ignore_index=True)

    all_celllevel_df.to_csv(
        file_savepath_main / f"{OUT_PREFIX}_all_celllevel_scatter_compatible.csv",
        index=False,
    )
else:
    all_celllevel_df = pd.DataFrame()

kegg_celllevel_boxplot_summary_df = pd.DataFrame(summary_rows)

kegg_celllevel_boxplot_summary_df.to_csv(
    file_savepath_main / f"{OUT_PREFIX}_boxplot_summary_thr{thr_tag}_{SCORE_METHOD}.csv",
    index=False,
)

print("KEGG pathway cell-level comparison summary:")
_display_df(kegg_celllevel_boxplot_summary_df)

# =========================================================
# 11. Show all generated boxplots
# =========================================================
for nm in all_boxplots:
    print(f"Showing plot: {nm}")
    display(all_boxplots[nm])

plt.close("all")


In [ ]:
kegg_celllevel_boxplot_summary_df.loc[:,["MI","Pathway_short",'p_wilcox', 'log2_mean_change_high_over_low']].sort_values(by="Pathway_short")


### 5.12. CAF and malignant C5 programs by MI strength

This cell-level summary combines MI-10, MI-6 and MI-13 across four programs.
CAF activation uses the fibroblast CAF-marker scores exported in Section 5.10;
PD-1/PD-L1 and ECM-receptor interaction use the preceding direct cell-level KEGG
scores; Hypoxia uses the CancerSEA scores exported in Section 5.11, rather than
the KEGG HIF-1 pathway. The existing scores and their reference populations are
preserved; no expression scoring is repeated here.

Cells are divided at MI strength 0.5 (high: >=0.5; low: <0.5). The figure shows
trimmed kernel-density violins with interquartile boxes, white medians and
1.5-IQR whiskers; individual outlier points are omitted. Stars denote unadjusted
two-sided cell-level Mann-Whitney rank-sum tests, and SMD is the high-minus-low
mean difference divided by the pooled within-group standard deviation.

The figure and its statistics are generated entirely in Python from the 12 CSVs
exported above. PNG, editable PDF and a summary CSV are saved in `run_dirs["run_dir"]`
with the prefix `MI_CAF_functionalstate_3x4_CAF_PD1_ECM_Hypoxia_violin_boxplot_SMD_python`.
The plotting function also accepts an optional `output_dir` for a separate export
location. Once the input CSVs exist, this cell can be run with only `run_dirs` defined.


In [ ]:
# Cell-level CAF and malignant C5 programs across MI-10, MI-6 and MI-13.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from scipy.stats import gaussian_kde, mannwhitneyu


def plot_mi_caf_program_summary(run_dir, output_dir=None, threshold=0.5):
    """Plot precomputed cell scores without changing their values or reference populations.

    Inputs are the CAF sending, CancerSEA Hypoxia receiving, and direct cell-level
    KEGG PD1_PDL1/ECM_receptor receiving CSVs exported in the preceding sections.
    Tests pool cells across slices, use two-sided Mann-Whitney rank-sum p-values
    without multiple-testing adjustment, and report signed pooled-SD SMDs.
    """
    input_dir = Path(run_dir)
    output_dir = input_dir if output_dir is None else Path(output_dir)
    mi_order = ("MI10", "MI6", "MI13")
    programs = (
        ("CAF activation", "CAF\nactivation", "CAF_Score",
         "Malignant_COI5_CAF_score_vs_Sending_{mi}_scatter.csv"),
        ("PD-1/PD-L1 checkpoint", "PD-1/PD-L1\ncheckpoint", "State_Score",
         "Malignant_COI5_PD1_PDL1_vs_Receiving{mi}_celllevel_scatter.csv"),
        ("ECM-receptor interaction", "ECM-receptor\ninteraction", "State_Score",
         "Malignant_COI5_ECM_receptor_vs_Receiving{mi}_celllevel_scatter.csv"),
        ("Hypoxia", "Hypoxia", "State_Score",
         "Malignant_COI5_Hypoxia_vs_Receiving_{mi}_scatter.csv"),
    )
    palettes = (("#3D91CE", "#81B6DE"), ("#F52E32", "#D9ADAC"))
    missing = [
        str(input_dir / template.format(mi=mi))
        for mi in mi_order for _, _, _, template in programs
        if not (input_path(input_dir / template.format(mi=mi))).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Run the preceding CAF, CancerSEA and direct cell-level KEGG export "
            "cells before generating this figure. Missing files:\n" + "\n".join(missing)
        )

    def significance(p_value):
        for cutoff, label in ((1e-4, "****"), (1e-3, "***"), (1e-2, "**"), (0.05, "*")):
            if p_value < cutoff:
                return label
        return "ns"

    def violin_density(values):
        # Normal-reference bandwidth (R's bw.nrd0 rule); density is trimmed to
        # the observed range. This affects only the violin silhouette.
        sd = np.std(values, ddof=1)
        if sd == 0:
            return np.array([values[0]]), np.array([1.0])
        iqr = np.subtract(*np.percentile(values, [75, 25]))
        scale = min(sd, iqr / 1.34) if iqr > 0 else sd
        bandwidth = 0.9 * scale * len(values) ** (-0.2)
        grid = np.linspace(values.min(), values.max(), 512)
        return grid, gaussian_kde(values, bw_method=bandwidth / sd)(grid)

    panel_values = {}
    statistics = []
    for row, mi in enumerate(mi_order):
        for col, (program, _, score_column, template) in enumerate(programs):
            source_path = input_path(input_dir / template.format(mi=mi))
            data = pd.read_csv(input_path(source_path))
            required = {"MI_Level", score_column}
            if not required.issubset(data.columns):
                raise ValueError(f"Missing {required - set(data.columns)} in {source_path.name}")
            mi_values = pd.to_numeric(data["MI_Level"], errors="coerce")
            scores = pd.to_numeric(data[score_column], errors="coerce")
            valid = np.isfinite(mi_values) & np.isfinite(scores)
            high = scores[valid & (mi_values >= threshold)].to_numpy(dtype=float)
            low = scores[valid & (mi_values < threshold)].to_numpy(dtype=float)
            if min(len(high), len(low)) < 2:
                raise ValueError(f"{mi} / {program} needs at least two valid cells in each group.")
            pooled_sd = np.sqrt(
                ((len(high) - 1) * np.var(high, ddof=1)
                 + (len(low) - 1) * np.var(low, ddof=1))
                / (len(high) + len(low) - 2)
            )
            mean_difference = high.mean() - low.mean()
            smd = mean_difference / pooled_sd if pooled_sd > 0 else np.nan
            # Match the large-sample, tie-corrected rank-sum test used for these
            # panels, including the continuity correction.
            p_value = mannwhitneyu(
                high, low, alternative="two-sided", method="asymptotic",
                use_continuity=True,
            ).pvalue
            panel_values[row, col] = (high, low)
            statistics.append({
                "MI": mi.replace("MI", "MI-"), "Program": program,
                "direction": "Sending" if col == 0 else "Receiving",
                "threshold": threshold, "n_high": len(high), "n_low": len(low),
                "mean_high": high.mean(), "mean_low": low.mean(),
                "sd_high": np.std(high, ddof=1), "sd_low": np.std(low, ddof=1),
                "mean_diff": mean_difference, "pooled_sd": pooled_sd,
                "SMD": smd, "p_wilcox": p_value, "significance": significance(p_value),
                "input_file": str(source_path), "x_column": "MI_Level",
                "y_column": score_column, "n_nonfinite_excluded": int((~valid).sum()),
            })
    summary = pd.DataFrame(statistics)

    style = {
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
        "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 120,
        "axes.grid": False, "axes.facecolor": "white", "figure.facecolor": "white",
    }
    with plt.rc_context(style):
        fig, axes = plt.subplots(3, 4, figsize=(8.2, 7.2))
        fig.subplots_adjust(left=0.105, right=0.90, bottom=0.105, top=0.835,
                            wspace=0.43, hspace=0.42)

        for col, (_, title, _, _) in enumerate(programs):
            all_scores = np.concatenate([
                group for row in range(3) for group in panel_values[row, col]
            ])
            y_min, y_max = all_scores.min(), all_scores.max()
            span = max(y_max - y_min, 0.1)
            bracket_y = y_max + 0.045 * span
            for row, mi in enumerate(mi_order):
                ax = axes[row, col]
                groups = panel_values[row, col]
                colors = palettes[0 if col == 0 else 1]
                densities = [violin_density(values) for values in groups]
                max_density = max(density.max() for _, density in densities)
                for xpos, (grid, density), color in zip((1, 2), densities, colors):
                    half_width = 0.42 * density / max_density
                    if len(grid) == 1:
                        ax.hlines(grid[0], xpos - 0.3, xpos + 0.3, colors=color)
                    else:
                        ax.fill_betweenx(
                            grid, xpos - half_width, xpos + half_width,
                            facecolor=color, edgecolor="none", alpha=0.45,
                        )
                        ax.plot(xpos - half_width, grid, color=color, linewidth=1.0)
                        ax.plot(xpos + half_width, grid, color=color, linewidth=1.0)
                boxes = ax.boxplot(
                    groups, positions=[1, 2], widths=0.36, patch_artist=True,
                    showfliers=False, showcaps=False, whis=1.5,
                    medianprops={"color": "white", "linewidth": 1.25},
                    boxprops={"linewidth": 0.9}, whiskerprops={"linewidth": 0.9},
                )
                for index, color in enumerate(colors):
                    boxes["boxes"][index].set(facecolor=color, edgecolor=color)
                    for whisker in boxes["whiskers"][2 * index:2 * index + 2]:
                        whisker.set_color(color)

                stats_row = summary.iloc[row * 4 + col]
                ax.plot([1, 1, 2, 2],
                        [bracket_y - 0.025 * span, bracket_y, bracket_y,
                         bracket_y - 0.025 * span], color="#222222", linewidth=0.8)
                smd_label = f"{stats_row['SMD']:.2f}" if np.isfinite(stats_row["SMD"]) else "NA"
                ax.text(1.5, bracket_y + 0.025 * span, f"(SMD = {smd_label})",
                        ha="center", va="bottom", fontsize=11.2, color="#747D86", clip_on=False)
                ax.text(1.5, bracket_y + 0.12 * span, stats_row["significance"],
                        ha="center", va="bottom", fontsize=12.0, color="black", clip_on=False)
                ax.set(xlim=(0.45, 2.55), ylim=(y_min - 0.055 * span, y_max + 0.14 * span))
                ax.set_xticks([])
                ax.yaxis.set_major_locator(MultipleLocator(0.5))
                ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
                ax.tick_params(axis="y", labelsize=11.5, width=0.8, length=3, pad=3)
                for side in ("left", "bottom", "right", "top"):
                    ax.spines[side].set_visible(side == "left" or (side == "bottom" and row == 2))
                    ax.spines[side].set_linewidth(0.9)
                if col == 0:
                    ax.set_ylabel("Module score", fontsize=14, labelpad=7)
                if col == 3:
                    ax.text(1.07, 0.5, mi.replace("MI", "MI-"), transform=ax.transAxes,
                            va="center", ha="left", fontsize=14)
                if row == 0:
                    ax.set_title(title, fontsize=15.5, pad=31, linespacing=1.1)
                    ax.plot([0, 1], [1.24, 1.24], transform=ax.transAxes,
                            color="#222222", linewidth=1.0, clip_on=False)

        legend_colors = palettes[0] + palettes[1]
        legend_handles = [
            (Line2D([], [], color=color, marker="|", markersize=13, linestyle="none"),
             Line2D([], [], color=color, marker="s", markersize=7, linestyle="none"),
             Line2D([], [], color="white", marker="_", markersize=5,
                    markeredgewidth=1.0, linestyle="none"))
            for color in legend_colors
        ]
        fig.legend(
            legend_handles,
            ["Sending MI-high fibroblast", "Sending MI-low fibroblast",
             "Receiving MI-high malignant C5", "Receiving MI-low malignant C5"],
            handler_map={tuple: HandlerTuple(ndivide=1)}, loc="lower center",
            bbox_to_anchor=(0.49, 0.018), ncol=2, frameon=False, fontsize=12.6,
            handlelength=0.8, handletextpad=0.3, columnspacing=2.0,
            labelspacing=0.2, borderaxespad=0,
        )
        output_dir.mkdir(parents=True, exist_ok=True)
        threshold_tag = f"{threshold:.2f}".replace(".", "p")
        stem = f"MI_CAF_functionalstate_3x4_CAF_PD1_ECM_Hypoxia_violin_boxplot_SMD_python_thr{threshold_tag}"
        for extension in ("png", "pdf"):
            fig.savefig(output_dir / f"{stem}.{extension}", dpi=300,
                        bbox_inches="tight", pad_inches=0.06, facecolor="white")
        summary.to_csv(output_dir / f"{stem}_summary.csv", index=False)
    return fig, summary


mi_caf_program_figure, mi_caf_program_summary = plot_mi_caf_program_summary(run_dirs["run_dir"])
display(mi_caf_program_summary[["MI", "Program", "n_high", "n_low", "SMD", "p_wilcox", "significance"]])
plt.show()
plt.close(mi_caf_program_figure)


### 5.13. Cross-method SMD comparison of CAF and malignant C5 programs

This section reconstructs the five-method comparison from upstream method
outputs and draws the paper-style grouped bar chart in Python. It incorporates
the analysis needed from `../Coupling_benchmark/HGSOC_Immunesuppression.ipynb`;
that notebook and its R visualization script no longer need to be run for this
barplot. The four programs are CAF activation, PD-1/PD-L1 checkpoint,
ECM-receptor interaction and CancerSEA Hypoxia.

**Inputs.** Run the preceding data-loading, malignant-cluster mapping, CAF,
CancerSEA and cell-level KEGG cells in this notebook. This section reuses
`adata_copy`, `adata_choose`, `adata_list`, `SpiderNet_data_pyg_list`, and the
MI-10 score CSVs and KEGG reference genes generated under `run_dirs["run_dir"]`.
Gene sets are read from `DATA_ROOT`. Raw method results must be available under
`OUTPUT_ROOT/COMMOT`, `OUTPUT_ROOT/ScCChain` and `OUTPUT_ROOT/Spacia` in their
existing formats. NMF-LR reads `run_dirs["run_dir"]/Factor_LR_list.pkl` when
present; otherwise it retains the benchmark's deterministic NMF fit from the
graph LR features. No pre-existing benchmark SMD or grouping tables are needed.

**Analysis.** Only fibroblast-to-malignant-C5 edge participants are evaluated,
with max edge-to-cell aggregation. The original gene sets, scoring reference
populations and score definitions are preserved. SpiderNet uses the preceding
MI-10 score exports with threshold 0.5. Each baseline uses a deterministic
median-rank split, including the original tie handling, and selects one feature
by the smallest mean SMD rank across all four programs. Ties are resolved by
the smallest maximum rank, then the largest mean SMD. Signed SMD is the
high-minus-low mean score difference divided by the pooled within-group sample
standard deviation. The plot independently recomputes all 20 SMDs from the
newly exported cell groups and checks them against the reconstructed summary.

**Outputs.** The reconstruction writes feature scans, feature ranks, selected
feature statistics and cell groups to
`run_dirs["run_dir"]/CCC_baseline_feature_SMD_comparison/`. The plotting cell
writes PNG, editable PDF and plotted statistics CSV with the prefix
`HGSOC_CCC_methods_4programs_SMD_barplot_python`, without a panel letter.


In [ ]:
def rebuild_hgsoc_method_smd(
    adata_all, adata_malignant, adata_slices, graph_slices, *,
    run_dir, data_root, baseline_root, n_components, output_dir=None,
):
    """Build the five-method, four-program SMD comparison from upstream results.

    Reuse this notebook's SpiderNet MI-10 score exports and rebuild the four
    baselines from raw method outputs, expression, gene sets and spatial graphs.
    Feature scanning, deterministic median-rank splits and signed SMDs follow
    the HGSOC benchmark. No existing benchmark SMD or grouping tables are read.
    Input AnnData objects and graph objects are treated as read-only.
    """
    import os
    import pickle
    from pathlib import Path

    import anndata as ad
    import h5py
    import numpy as np
    import pandas as pd
    import torch
    from scipy import sparse as sp
    from scipy.stats import mannwhitneyu
    from sklearn.decomposition import NMF

    adata_copy, adata_choose = adata_all, adata_malignant
    adata_list, SpiderNet_data_pyg_list = adata_slices, graph_slices
    RUN_DIR, DATA_ROOT = Path(run_dir), Path(data_root)
    OUT_DIR = RUN_DIR / "CCC_baseline_feature_SMD_comparison" if output_dir is None else Path(output_dir)
    COMMOT_PATH_MAIN = Path(baseline_root) / "COMMOT"
    SC_CCHAIN_PATH_MAIN = Path(baseline_root) / "ScCChain"
    SPACIA_PATH_MAIN = Path(baseline_root) / "Spacia"
    Factor_LR_list_path = RUN_DIR / "Factor_LR_list.pkl"
    FIBRO_LABEL, MALIGNANT_C5_LABEL = "Fibroblast", "Malignant_5"
    CELLTYPE_COL, MALIGNANT_CLUSTER_COL = "cell.types", "MI_louvain"
    SPIDERNET_MI_DIM_ONE_BASED, SPIDERNET_THRESHOLD = 10, 0.5
    BASELINE_THRESHOLD_MODE = "median"
    BASELINE_FEATURE_SELECTION_MODE = "mean_SMD_rank_across_programs"
    CELL_AGG_MODE = "max"
    MIN_CELLS_PER_GROUP = 5
    if len(adata_list) != len(SpiderNet_data_pyg_list):
        raise ValueError("Expression slices and graph slices must have identical order and length.")
    if sum(a.n_obs for a in adata_list) != adata_copy.n_obs:
        raise ValueError("Concatenated slices must match the full expression object.")


    def p_to_star(p):
        if pd.isna(p):
            return "NA"
        if p < 1e-4:
            return "****"
        if p < 1e-3:
            return "***"
        if p < 1e-2:
            return "**"
        if p < 0.05:
            return "*"
        return "ns"

    def _high_mask_from_split_rule(feature_values, threshold, split_rule="threshold_ge"):
        """Return a boolean high-group mask for finite feature values.

        split_rule options:
        - "threshold_ge": high = feature >= threshold; low = feature < threshold.
          This is used for SpiderNet MI-10 with the original threshold 0.5.
        - "median_rank_split_upper_half": high = upper half by feature rank; low = lower half.
          The numeric threshold stored in output is the feature median. This behaves
          like a median split when values are unique, but avoids pathological group
          sizes when many cells have identical feature values.
        """
        feature_values = np.asarray(feature_values, dtype=float)
        high_mask = np.zeros(feature_values.shape[0], dtype=bool)
        finite_mask = np.isfinite(feature_values)

        if not np.any(finite_mask):
            return high_mask

        if split_rule == "threshold_ge":
            high_mask[finite_mask] = feature_values[finite_mask] >= threshold
            return high_mask

        if split_rule == "median_rank_split_upper_half":
            finite_idx = np.where(finite_mask)[0]
            vals = feature_values[finite_idx]
            # Deterministic ascending order. np.lexsort uses the last key as primary;
            # therefore values are primary and original indices break ties.
            order = np.lexsort((finite_idx, vals))
            n = len(order)
            high_rel = order[n // 2:]
            high_mask[finite_idx[high_rel]] = True
            return high_mask

        raise ValueError(f"Unknown split_rule={split_rule!r}")

    def compute_smd_and_p(feature_values, scores, threshold, min_cells_per_group=5, split_rule="threshold_ge"):
        """Split scores into high/low groups and compute high-minus-low SMD."""
        feature_values = np.asarray(feature_values, dtype=float)
        scores = np.asarray(scores, dtype=float)

        ok = np.isfinite(feature_values) & np.isfinite(scores)
        feature_values = feature_values[ok]
        scores = scores[ok]

        if scores.size == 0:
            return {
                "n_high": 0, "n_low": 0,
                "mean_high": np.nan, "mean_low": np.nan,
                "sd_high": np.nan, "sd_low": np.nan,
                "pooled_sd": np.nan, "SMD": np.nan,
                "p_wilcox": np.nan, "significance": "NA"
            }

        high_mask = _high_mask_from_split_rule(feature_values, threshold, split_rule=split_rule)
        high = scores[high_mask]
        low = scores[~high_mask]

        n_high = int(high.size)
        n_low = int(low.size)

        mean_high = float(np.nanmean(high)) if n_high else np.nan
        mean_low = float(np.nanmean(low)) if n_low else np.nan
        sd_high = float(np.nanstd(high, ddof=1)) if n_high > 1 else np.nan
        sd_low = float(np.nanstd(low, ddof=1)) if n_low > 1 else np.nan

        pooled_sd = np.nan
        smd = np.nan
        if n_high > 1 and n_low > 1 and np.isfinite(sd_high) and np.isfinite(sd_low):
            pooled_var = ((n_high - 1) * sd_high**2 + (n_low - 1) * sd_low**2) / (n_high + n_low - 2)
            if np.isfinite(pooled_var) and pooled_var > 0:
                pooled_sd = float(np.sqrt(pooled_var))
                smd = float((mean_high - mean_low) / pooled_sd)

        p_val = np.nan
        if n_high >= min_cells_per_group and n_low >= min_cells_per_group:
            try:
                p_val = mannwhitneyu(high, low, alternative="two-sided", method="asymptotic").pvalue
            except TypeError:
                p_val = mannwhitneyu(high, low, alternative="two-sided").pvalue

        return {
            "n_high": n_high,
            "n_low": n_low,
            "mean_high": mean_high,
            "mean_low": mean_low,
            "sd_high": sd_high,
            "sd_low": sd_low,
            "pooled_sd": pooled_sd,
            "SMD": smd,
            "p_wilcox": p_val,
            "significance": p_to_star(p_val),
        }

    def make_long_group_table(method, program, feature_dim, feature_name, direction, feature_values, scores, threshold,
                              threshold_mode=None, split_rule="threshold_ge"):
        feature_values = np.asarray(feature_values, dtype=float)
        scores = np.asarray(scores, dtype=float)
        ok = np.isfinite(feature_values) & np.isfinite(scores)

        df = pd.DataFrame({
            "Method": method,
            "Program": program,
            "Feature_Dim": feature_dim,
            "Feature_Name": feature_name,
            "Direction": direction,
            "Feature_Value": feature_values[ok],
            "Score": scores[ok],
        })
        df["Threshold"] = threshold
        df["Threshold_Mode"] = threshold_mode if threshold_mode is not None else ("fixed_0.5" if method == "SpiderNet" else BASELINE_THRESHOLD_MODE)
        df["Grouping_Rule"] = split_rule
        high_mask = _high_mask_from_split_rule(df["Feature_Value"].to_numpy(dtype=float), threshold, split_rule=split_rule)
        df["Group"] = np.where(high_mask, "High", "Low")
        return df

    def edge_index_to_e2(edge_index):
        """Return edge_index as an E × 2 numpy integer array."""
        if torch is not None and hasattr(edge_index, "detach"):
            arr = edge_index.detach().cpu().numpy()
        else:
            arr = np.asarray(edge_index)

        if arr.ndim != 2:
            raise ValueError(f"edge_index must be 2D, got shape {arr.shape}")

        if arr.shape[1] == 2:
            out = arr
        elif arr.shape[0] == 2:
            out = arr.T
        else:
            raise ValueError(f"Cannot interpret edge_index shape as E×2 or 2×E: {arr.shape}")

        return out.astype(np.int64, copy=False)

    def get_data_item(data, key):
        """Get key from either dict-like PyG data or object-like PyG data."""
        try:
            return data[key]
        except Exception:
            return getattr(data, key)

    def get_num_cells_from_data(data, adata_sub):
        try:
            x = get_data_item(data, "x")
            return int(x.shape[0])
        except Exception:
            return int(adata_sub.n_obs)

    def to_dense_array(x):
        if sp.issparse(x):
            return x.toarray()
        if torch is not None and hasattr(x, "detach"):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    def edge_matrix_from_square(square_mat, rows, cols):
        if sp.issparse(square_mat):
            square_mat = square_mat.tocsr()
            return square_mat[rows, cols].A1.astype(np.float32, copy=False)
        return np.asarray(square_mat)[rows, cols].astype(np.float32, copy=False)

    def aggregate_edge_features_by_max(edge_features, edge_index_e2, num_cells, edge_mask):
        """Aggregate edge features to sender and receiver cells with max pooling.

        edge_mask selects the edges to keep. Cells with no selected edges get zero.
        """
        edge_features = np.asarray(edge_features, dtype=np.float32)
        if edge_features.ndim == 1:
            edge_features = edge_features.reshape(-1, 1)

        edge_features = np.nan_to_num(edge_features, nan=0.0, posinf=0.0, neginf=0.0)
        edge_index_e2 = np.asarray(edge_index_e2, dtype=np.int64)

        if edge_features.shape[0] != edge_index_e2.shape[0]:
            raise ValueError(
                f"edge_features rows ({edge_features.shape[0]}) != edge_index rows ({edge_index_e2.shape[0]})"
            )

        edge_mask = np.asarray(edge_mask, dtype=bool)
        if edge_mask.shape[0] != edge_features.shape[0]:
            raise ValueError("edge_mask length does not match number of edges")

        K = edge_features.shape[1]
        sender_out = np.zeros((num_cells, K), dtype=np.float32)
        receiver_out = np.zeros((num_cells, K), dtype=np.float32)

        if np.any(edge_mask):
            ef = edge_features[edge_mask, :]
            ei = edge_index_e2[edge_mask, :]
            send_idx = ei[:, 0]
            recv_idx = ei[:, 1]

            valid_s = (send_idx >= 0) & (send_idx < num_cells)
            valid_r = (recv_idx >= 0) & (recv_idx < num_cells)

            if np.any(valid_s):
                np.maximum.at(sender_out, send_idx[valid_s], ef[valid_s, :])
            if np.any(valid_r):
                np.maximum.at(receiver_out, recv_idx[valid_r], ef[valid_r, :])

        return sender_out, receiver_out

    def resolve_gene_names(adata, genes):
        var_names = pd.Index(adata.var_names.astype(str))
        upper_to_actual = {g.upper(): g for g in var_names}
        resolved = []
        missing = []
        seen = set()

        for g in genes:
            g = str(g).strip()
            if not g:
                continue

            if g in var_names:
                actual = g
            elif g.upper() in upper_to_actual:
                actual = upper_to_actual[g.upper()]
            else:
                missing.append(g)
                continue

            if actual not in seen:
                resolved.append(actual)
                seen.add(actual)

        return resolved, missing

    def get_X_dense(adata, rows, genes_actual):
        rows = np.asarray(rows, dtype=int)
        gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
        X = adata.X[rows, :][:, gene_idx]
        X = X.toarray() if sp.issparse(X) else np.asarray(X)
        return X.astype(float, copy=False)

    def zscore_mean_gene_set_score(adata, target_rows, reference_rows, genes, ddof=1):
        """Global z-score each gene using reference_rows, then average genes in target_rows."""
        genes_actual, missing = resolve_gene_names(adata, genes)
        target_rows = np.asarray(target_rows, dtype=int)
        out = np.full(target_rows.size, np.nan, dtype=float)

        if len(genes_actual) == 0 or target_rows.size == 0:
            return out, genes_actual, missing

        X_ref = get_X_dense(adata, reference_rows, genes_actual)
        gene_mean = np.nanmean(X_ref, axis=0)
        gene_std = np.nanstd(X_ref, axis=0, ddof=ddof)
        gene_std = np.where(np.isfinite(gene_std) & (gene_std > 0), gene_std, np.nan)

        X_tar = get_X_dense(adata, target_rows, genes_actual)
        with np.errstate(invalid="ignore", divide="ignore"):
            Z = (X_tar - gene_mean.reshape(1, -1)) / gene_std.reshape(1, -1)

        out[:] = np.nanmean(Z, axis=1)
        return out, genes_actual, missing

    def build_node_mask_from_id_set(adata_sub, id_set):
        """Build local node mask using obs_names and common barcode columns."""
        obs_names = adata_sub.obs_names.astype(str).to_numpy()
        mask = np.fromiter((x in id_set for x in obs_names), dtype=bool, count=obs_names.size)

        for col in ["barcode", "Barcode", "cell_id", "CellID"]:
            if col in adata_sub.obs.columns:
                vals = adata_sub.obs[col].astype(str).to_numpy()
                mask |= np.fromiter((x in id_set for x in vals), dtype=bool, count=vals.size)

        return mask

    def ids_for_rows(adata, rows):
        rows = np.asarray(rows, dtype=int)
        id_set = set(adata.obs_names[rows].astype(str).tolist())
        for col in ["barcode", "Barcode", "cell_id", "CellID"]:
            if col in adata.obs.columns:
                id_set.update(adata.obs.iloc[rows][col].astype(str).tolist())
        return id_set

    def first_existing_file(paths):
        for p in paths:
            p = Path(p)
            if input_path(p).exists():
                return p
        return None

    # Define C5 and the structural fibroblast-to-C5 evaluation populations.
    if CELLTYPE_COL not in adata_copy.obs.columns:
        raise KeyError(f"Cannot find {CELLTYPE_COL!r} in adata_copy.obs. Available columns: {list(adata_copy.obs.columns)}")

    if MALIGNANT_CLUSTER_COL not in adata_choose.obs.columns:
        raise KeyError(
            f"Cannot find {MALIGNANT_CLUSTER_COL!r} in adata_choose.obs. "
            "Run the malignant clustering cell first or update MALIGNANT_CLUSTER_COL."
        )

    # Normalize malignant cluster labels to 1-based strings if needed
    cluster_series = adata_choose.obs[MALIGNANT_CLUSTER_COL].astype(str)
    cluster_numeric = pd.to_numeric(cluster_series, errors="coerce")
    if cluster_numeric.notna().any() and int(np.nanmin(cluster_numeric)) == 0:
        cluster_series = (cluster_numeric.astype(int) + 1).astype(str)
    else:
        cluster_series = cluster_series.str.replace("^C", "", regex=True)

    # Determine malignant cell barcodes
    if "barcode" in adata_choose.obs.columns:
        malignant_ids = adata_choose.obs["barcode"].astype(str).to_numpy()
    else:
        malignant_ids = adata_choose.obs_names.astype(str).to_numpy()

    malignant_cluster = np.array([f"Malignant_{x}" for x in cluster_series.astype(str).to_numpy()], dtype=object)
    malignant_map = dict(zip(malignant_ids, malignant_cluster))

    # Map malignant subtype labels onto the full adata_copy cell type vector
    barcode_all = adata_copy.obs_names.astype(str).to_numpy()
    if "barcode" in adata_copy.obs.columns:
        barcode_all_alt = adata_copy.obs["barcode"].astype(str).to_numpy()
    else:
        barcode_all_alt = barcode_all

    cellclass_raw = adata_copy.obs[CELLTYPE_COL].astype(str).to_numpy()
    cellclass_updated = np.array([
        malignant_map.get(obs_id, malignant_map.get(bar_id, raw))
        for obs_id, bar_id, raw in zip(barcode_all, barcode_all_alt, cellclass_raw)
    ], dtype=object)

    fibro_idx = np.where(cellclass_updated == FIBRO_LABEL)[0]
    malignant_c5_idx = np.where(cellclass_updated == MALIGNANT_C5_LABEL)[0]

    if fibro_idx.size == 0:
        # fallback if exact label is not present
        fibro_idx = np.where(pd.Series(cellclass_updated).str.contains("fibro", case=False, regex=False).to_numpy())[0]

    if fibro_idx.size == 0:
        raise ValueError("No Fibroblast cells found.")
    if malignant_c5_idx.size == 0:
        raise ValueError("No malignant C5 cells found. Check MALIGNANT_CLUSTER_LABEL and malignant clustering.")

    fibro_id_set = ids_for_rows(adata_copy, fibro_idx)
    malignant_c5_id_set = ids_for_rows(adata_copy, malignant_c5_idx)

    print("Fibroblast cells:", len(fibro_idx))
    print("Malignant C5 cells:", len(malignant_c5_idx))

    # ------------------------------------------------------------
    # Evaluation cell universe for Fibroblast -> malignant C5 context
    # ------------------------------------------------------------
    # The original SpiderNet HGSOC analysis does not evaluate all Fibroblasts/all C5 cells.
    # It evaluates:
    #   - CAF: Fibroblasts with at least one outgoing spatial edge to malignant C5.
    #   - malignant programs: malignant C5 cells with at least one incoming spatial edge from Fibroblast.
    # We build this structural cell universe once and apply it to every baseline method.
    def build_structural_fibro_to_c5_eval_masks():
        fibro_to_c5_parts = []
        c5_from_fibro_parts = []

        for slice_index, adata_sub in enumerate(adata_list):
            data = SpiderNet_data_pyg_list[slice_index]
            edge_index = edge_index_to_e2(get_data_item(data, "edge_index"))
            n_cells = get_num_cells_from_data(data, adata_sub)

            node_is_fibro = build_node_mask_from_id_set(adata_sub, fibro_id_set)
            node_is_c5 = build_node_mask_from_id_set(adata_sub, malignant_c5_id_set)

            sender_nodes = edge_index[:, 0]
            receiver_nodes = edge_index[:, 1]
            valid_edges = (
                (sender_nodes >= 0) & (sender_nodes < n_cells)
                & (receiver_nodes >= 0) & (receiver_nodes < n_cells)
            )

            relation_edge = np.zeros(edge_index.shape[0], dtype=bool)
            relation_edge[valid_edges] = (
                node_is_fibro[sender_nodes[valid_edges]]
                & node_is_c5[receiver_nodes[valid_edges]]
            )

            fibro_has_c5_receiver = np.zeros(n_cells, dtype=bool)
            c5_has_fibro_sender = np.zeros(n_cells, dtype=bool)

            if np.any(relation_edge):
                fibro_has_c5_receiver[sender_nodes[relation_edge]] = True
                c5_has_fibro_sender[receiver_nodes[relation_edge]] = True

            fibro_to_c5_parts.append(fibro_has_c5_receiver)
            c5_from_fibro_parts.append(c5_has_fibro_sender)

        fibro_to_c5_mask = np.concatenate(fibro_to_c5_parts)
        c5_from_fibro_mask = np.concatenate(c5_from_fibro_parts)

        if fibro_to_c5_mask.size != adata_copy.n_obs:
            print("[Warning] structural fibro mask length != adata_copy.n_obs:", fibro_to_c5_mask.size, adata_copy.n_obs)
        if c5_from_fibro_mask.size != adata_copy.n_obs:
            print("[Warning] structural C5 mask length != adata_copy.n_obs:", c5_from_fibro_mask.size, adata_copy.n_obs)

        return fibro_to_c5_mask, c5_from_fibro_mask

    fibro_to_c5_struct_mask, c5_from_fibro_struct_mask = build_structural_fibro_to_c5_eval_masks()

    fibro_eval_idx = np.where(fibro_to_c5_struct_mask & (cellclass_updated == FIBRO_LABEL))[0]
    malignant_c5_eval_idx = np.where(c5_from_fibro_struct_mask & (cellclass_updated == MALIGNANT_C5_LABEL))[0]

    if fibro_eval_idx.size == 0:
        raise ValueError("No Fibroblast cells with outgoing spatial edges to malignant C5.")
    if malignant_c5_eval_idx.size == 0:
        raise ValueError("No malignant C5 cells with incoming spatial edges from Fibroblast.")

    print("Fibroblast cells used for CAF SMD:", len(fibro_eval_idx))
    print("Malignant C5 cells used for receiver-program SMD:", len(malignant_c5_eval_idx))

    # Rebuild the four baseline module scores from expression and gene sets.
    # CAF marker union used in the current HGSOC analysis
    caf_modules = {
        "myCAF": ["ACTA2", "TAGLN", "MYL9", "TPM2", "CNN1", "CALD1", "COL1A1", "COL1A2"],
        "iCAF": ["IL6", "CXCL12", "CXCL14", "LIF", "CCL2", "PTGS2"],
        "apCAF": ["HLA-DRA", "HLA-DRB1", "CD74", "CIITA"],
        "meCAF": ["COL11A1", "THBS2", "MMP11", "ITGA11", "FN1", "VCAN", "SPARC", "SULF1", "LOX", "PLOD2"],
        "periCAF": ["RGS5", "PDGFRB", "MCAM", "NOTCH3", "TAGLN"],
        "prolCAF": ["MKI67", "TOP2A", "PCNA"],
    }
    caf_genes = list(dict.fromkeys([g for genes in caf_modules.values() for g in genes]))

    # KEGG genes for PD-1/PD-L1 checkpoint and ECM-receptor interaction
    kegg_gene_path = RUN_DIR / "TCGA_OV_KEGG_gene_signature" / "C5_four_KEGG_pathway_reference_genes_long.csv"
    if not input_path(kegg_gene_path).exists():
        raise FileNotFoundError(
            "Cannot find the KEGG gene table generated by HGSOC_Malignantsubtype_analysis_V2:\n"
            f"{kegg_gene_path}"
        )

    kegg_gene_df = pd.read_csv(input_path(kegg_gene_path))
    if "Requested_Pathway" in kegg_gene_df.columns:
        kegg_pathway_col = "Requested_Pathway"
    elif "Pathway" in kegg_gene_df.columns:
        kegg_pathway_col = "Pathway"
    else:
        raise KeyError(f"Cannot find KEGG pathway column in {kegg_gene_path}. Columns: {list(kegg_gene_df.columns)}")

    if "Gene" not in kegg_gene_df.columns:
        raise KeyError(f"Cannot find Gene column in {kegg_gene_path}. Columns: {list(kegg_gene_df.columns)}")

    PD1_PDL1_PATHWAY = "PD-L1 expression and PD-1 checkpoint pathway in cancer"
    ECM_PATHWAY = "ECM-receptor interaction"

    pd1_genes = (
        kegg_gene_df.loc[kegg_gene_df[kegg_pathway_col] == PD1_PDL1_PATHWAY, "Gene"]
        .dropna().astype(str).drop_duplicates().tolist()
    )
    ecm_genes = (
        kegg_gene_df.loc[kegg_gene_df[kegg_pathway_col] == ECM_PATHWAY, "Gene"]
        .dropna().astype(str).drop_duplicates().tolist()
    )

    # CancerSEA Hypoxia genes
    cancersea_path = DATA_ROOT / "CancerSEA_OV" / "functional_geneset_list_df.csv"
    if not input_path(cancersea_path).exists():
        raise FileNotFoundError(f"Cannot find CancerSEA gene set table: {cancersea_path}")

    cancersea_df = pd.read_csv(input_path(cancersea_path))
    if not {"GeneSet", "Gene"}.issubset(cancersea_df.columns):
        raise KeyError(f"CancerSEA table needs GeneSet and Gene columns. Columns: {list(cancersea_df.columns)}")

    hypoxia_genes = (
        cancersea_df.loc[cancersea_df["GeneSet"].astype(str) == "Hypoxia", "Gene"]
        .dropna().astype(str).drop_duplicates().tolist()
    )

    PROGRAM_ORDER = [
        "CAF sig.",
        "PD-1/PD-L1 checkpoint",
        "ECM-receptor interaction",
        "Hypoxia",
    ]

    program_gene_sets = {
        "CAF sig.": caf_genes,
        "PD-1/PD-L1 checkpoint": pd1_genes,
        "ECM-receptor interaction": ecm_genes,
        "Hypoxia": hypoxia_genes,
    }

    program_cell_side = {
        "CAF sig.": "sender_fibroblast",
        "PD-1/PD-L1 checkpoint": "receiver_malignant_c5",
        "ECM-receptor interaction": "receiver_malignant_c5",
        "Hypoxia": "receiver_malignant_c5",
    }

    # Compute global z-score + gene-set mean scores on the same evaluation cell universe
    score_by_program = {}

    caf_score_fibro, caf_used, caf_missing = zscore_mean_gene_set_score(
        adata=adata_copy,
        target_rows=fibro_eval_idx,
        reference_rows=fibro_idx,
        genes=caf_genes,
        ddof=1,
    )
    score_by_program["CAF sig."] = (fibro_eval_idx, caf_score_fibro)

    print("CAF genes used:", len(caf_used), "missing:", len(caf_missing))

    for program in ["PD-1/PD-L1 checkpoint", "ECM-receptor interaction", "Hypoxia"]:
        scores, used, missing = zscore_mean_gene_set_score(
            adata=adata_copy,
            target_rows=malignant_c5_eval_idx,
            reference_rows=malignant_c5_eval_idx,
            genes=program_gene_sets[program],
            ddof=1,
        )
        score_by_program[program] = (malignant_c5_eval_idx, scores)
        print(program, "genes used:", len(used), "missing:", len(missing))

    def build_nmflr_factors(SpiderNet_data_pyg_list, n_components, random_state=0, max_iter=1000):
        factors = []
        for slice_index, data in enumerate(SpiderNet_data_pyg_list):
            print(f"Running NMF-LR for slice {slice_index + 1}/{len(SpiderNet_data_pyg_list)}")
            lr_mat = to_dense_array(get_data_item(data, "cellpair_LRpair_neigh"))

            nmf = NMF(
                n_components=n_components,
                init="nndsvda",
                random_state=random_state,
                max_iter=max_iter,
            )
            factor = nmf.fit_transform(lr_mat)

            colmax = np.max(factor, axis=0)
            colmax[colmax <= 0] = 1.0
            factor = factor / colmax.reshape(1, -1)

            factors.append(factor.astype(np.float32, copy=False))

        return factors

    def get_commot_file_for_slice(adata_sub):
        sample_values = adata_sub.obs["samples"].astype(str).unique() if "samples" in adata_sub.obs.columns else adata_sub.obs_names[:1].astype(str)
        sample_cur = sample_values[0]
        return COMMOT_PATH_MAIN / f"{sample_cur}_COMMOT_cellchat.h5ad"

    def load_commot_outputs():
        outputs = []
        for slice_index, adata_sub in enumerate(adata_list):
            print(f"Loading COMMOT for slice {slice_index + 1}/{len(adata_list)}")
            commot_path = get_commot_file_for_slice(adata_sub)
            if not input_path(commot_path).exists():
                raise FileNotFoundError(f"Missing COMMOT file: {commot_path}")

            commot_adata = ad.read_h5ad(input_path(commot_path))

            edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
            rows = edge_index[:, 0].astype(np.int64)
            cols = edge_index[:, 1].astype(np.int64)
            E = edge_index.shape[0]

            keys = list(commot_adata.obsp.keys())
            total_key = "commot-cellchat-total-total"
            pathway_keys = [
                k for k in keys
                if k.startswith("commot-cellchat-") and len(k.split("-")) == 3 and k != total_key
            ]

            arr = np.zeros((E, len(pathway_keys)), dtype=np.float32)
            for j, key in enumerate(pathway_keys):
                arr[:, j] = edge_matrix_from_square(commot_adata.obsp[key], rows, cols)

            outputs.append(arr)

        return outputs

    def load_sccchain_outputs():
        txt_path = SC_CCHAIN_PATH_MAIN / "h5ad_files.txt"
        if not input_path(txt_path).exists():
            raise FileNotFoundError(f"Missing scCChain h5ad list: {txt_path}")

        scc_h5ad_files = pd.read_csv(input_path(txt_path), header=None)[0].astype(str).tolist()

        n_obs_ref = np.array([ad.n_obs for ad in adata_list])
        n_obs_scc = []
        for p in scc_h5ad_files:
            slice_info = ad.read_h5ad(input_path(p), backed="r")
            n_obs_scc.append(slice_info.n_obs)
            slice_info.file.close()
        n_obs_scc = np.asarray(n_obs_scc)

        index_map = []
        for n in n_obs_ref:
            matched = np.where(n_obs_scc == n)[0]
            if len(matched) == 0:
                raise ValueError(f"Cannot match scCChain result by n_obs={n}")
            index_map.append(int(matched[0]))

        outputs = []
        for slice_index in range(len(adata_list)):
            print(f"Loading scCChain for slice {slice_index + 1}/{len(adata_list)}")
            src_path = Path(scc_h5ad_files[index_map[slice_index]])
            stem = src_path.stem
            score_path = SC_CCHAIN_PATH_MAIN / f"{stem}_ScCChain_edge_program_scores.csv"
            if not input_path(score_path).exists():
                raise FileNotFoundError(f"Missing scCChain score file: {score_path}")

            score_df = pd.read_csv(input_path(score_path))
            edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
            rows = edge_index[:, 0].astype(np.int64)
            cols = edge_index[:, 1].astype(np.int64)
            E = edge_index.shape[0]
            K_cols = score_df.columns.tolist()[2:]

            arr = np.zeros((E, len(K_cols)), dtype=np.float32)

            # The scCChain CSV uses 1-based sender/receiver indices.
            sender_index = score_df["sender_index"].to_numpy(dtype=np.int64) - 1
            receiver_index = score_df["receiver_index"].to_numpy(dtype=np.int64) - 1

            for j, col in enumerate(K_cols):
                A = np.zeros((adata_list[slice_index].n_obs, adata_list[slice_index].n_obs), dtype=np.float32)
                A[sender_index, receiver_index] = score_df[col].to_numpy(dtype=np.float32)
                arr[:, j] = edge_matrix_from_square(A, rows, cols)

            outputs.append(arr)

        return outputs

    def load_spacia_outputs():
        """Read only graph-edge entries of Spacia's sender x receiver x feature arrays."""
        spacia_dir = SPACIA_PATH_MAIN / "spacia_outputs"
        spacia_files = sorted(spacia_dir.glob("*.h5ad"))
        if not spacia_files:
            raise FileNotFoundError(f"No Spacia .h5ad results found in {spacia_dir}")
        n_obs_spacia = []
        for path in spacia_files:
            with h5py.File(input_path(path), "r") as handle:
                n_obs_spacia.append(handle["obsp/interaction_scores"].shape[0])
        n_obs_spacia = np.asarray(n_obs_spacia)
        outputs = []
        for slice_index, adata_sub in enumerate(adata_list):
            matched = np.where(n_obs_spacia == adata_sub.n_obs)[0]
            if len(matched) == 0:
                raise ValueError(f"Cannot match Spacia result by n_obs={adata_sub.n_obs}")
            # Preserve the benchmark's sorted-file, first-matching-size rule.
            path = spacia_files[int(matched[0])]
            edges = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
            rows, cols = edges[:, 0], edges[:, 1]
            with h5py.File(input_path(path), "r") as handle:
                scores = handle["obsp/interaction_scores"]
                if scores.ndim != 3 or scores.shape[0] != scores.shape[1]:
                    raise ValueError(f"Expected a square sender/receiver tensor in {path}")
                offset = scores.id.get_offset()
                if scores.chunks is None and scores.compression is None and offset is not None:
                    # A read-only view avoids loading multi-GB dense tensors into RAM.
                    mapped = np.memmap(input_path(path), mode="r", dtype=scores.dtype,
                                       offset=offset, shape=scores.shape, order="C")
                    arr = mapped[rows, cols, :].astype(np.float32, copy=True)
                    del mapped
                else:
                    arr = np.empty((len(rows), scores.shape[2]), dtype=np.float32)
                    for row in np.unique(rows):
                        positions = np.flatnonzero(rows == row)
                        unique_cols, inverse = np.unique(cols[positions], return_inverse=True)
                        arr[positions] = scores[int(row), unique_cols, :][inverse].astype(np.float32)
            outputs.append(arr)
        return outputs


    def load_baseline_features(method):
        if method == "NMF-LR":
            if input_path(Factor_LR_list_path).is_file():
                return pd.read_pickle(input_path(Factor_LR_list_path))
            # Keep the original deterministic NMF fallback when no fitted factors exist.
            return build_nmflr_factors(SpiderNet_data_pyg_list, n_components=int(n_components),
                                      random_state=0, max_iter=1000)
        if method == "COMMOT":
            return load_commot_outputs()
        if method == "ScCChain":
            return load_sccchain_outputs()
        if method == "Spacia":
            return load_spacia_outputs()
        raise ValueError(f"Unsupported baseline: {method}")


    def build_method_cell_feature_matrices(edge_feature_list):
        """Return two global cell × feature matrices.

        sender_to_c5:
            For each cell, max feature strength over outgoing edges whose receiver is malignant C5.
            This is used for CAF scores in sender Fibroblasts.

        receiver_from_fibro:
            For each cell, max feature strength over incoming edges whose sender is Fibroblast.
            This is used for malignant C5 receiver programs.

        Only cells that actually participate in Fibroblast -> malignant C5 edges are
        used later for SMD evaluation.
        """
        sender_to_c5_parts = []
        receiver_from_fibro_parts = []

        for slice_index, edge_features in enumerate(edge_feature_list):
            adata_sub = adata_list[slice_index]
            data = SpiderNet_data_pyg_list[slice_index]

            edge_index = edge_index_to_e2(get_data_item(data, "edge_index"))
            n_cells = get_num_cells_from_data(data, adata_sub)

            node_is_fibro = build_node_mask_from_id_set(adata_sub, fibro_id_set)
            node_is_c5 = build_node_mask_from_id_set(adata_sub, malignant_c5_id_set)

            if n_cells != adata_sub.n_obs:
                print(f"[Warning] slice {slice_index}: graph n_cells={n_cells}, adata n_obs={adata_sub.n_obs}")

            sender_nodes = edge_index[:, 0]
            receiver_nodes = edge_index[:, 1]

            valid_edges = (
                (sender_nodes >= 0) & (sender_nodes < n_cells)
                & (receiver_nodes >= 0) & (receiver_nodes < n_cells)
            )

            receiver_is_c5_edge = np.zeros(edge_index.shape[0], dtype=bool)
            sender_is_fibro_edge = np.zeros(edge_index.shape[0], dtype=bool)

            receiver_is_c5_edge[valid_edges] = node_is_c5[receiver_nodes[valid_edges]]
            sender_is_fibro_edge[valid_edges] = node_is_fibro[sender_nodes[valid_edges]]

            sender_to_c5, _ = aggregate_edge_features_by_max(
                edge_features=edge_features,
                edge_index_e2=edge_index,
                num_cells=n_cells,
                edge_mask=receiver_is_c5_edge,
            )

            _, receiver_from_fibro = aggregate_edge_features_by_max(
                edge_features=edge_features,
                edge_index_e2=edge_index,
                num_cells=n_cells,
                edge_mask=sender_is_fibro_edge,
            )

            sender_to_c5_parts.append(sender_to_c5)
            receiver_from_fibro_parts.append(receiver_from_fibro)

        sender_to_c5 = np.vstack(sender_to_c5_parts)
        receiver_from_fibro = np.vstack(receiver_from_fibro_parts)

        if sender_to_c5.shape[0] != adata_copy.n_obs:
            print(
                "[Warning] Concatenated feature matrix row count does not match adata_copy.n_obs:",
                sender_to_c5.shape[0], adata_copy.n_obs
            )

        return sender_to_c5, receiver_from_fibro

    def pick_first_existing_column(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"Cannot find any of {candidates}. Available columns: {list(df.columns)}")

    def load_existing_spidernet_csv_long():
        """Use the exact existing SpiderNet CSVs from HGSOC_Malignantsubtype_analysis_V2 if available."""
        file_specs = {
            "CAF sig.": {
                "files": [RUN_DIR / "Malignant_COI5_CAF_score_vs_Sending_MI10_scatter.csv"],
                "y_candidates": ["CAF_Score", "CAF_score"],
                "direction": "Sender",
            },
            "PD-1/PD-L1 checkpoint": {
                "files": [
                    RUN_DIR / "Malignant_COI5_PD1_PDL1_vs_ReceivingMI10_celllevel_scatter.csv",
                    RUN_DIR / "Malignant_COI5_PD1_PDL1_vs_Receiving_MI10_scatter.csv",
                ],
                "y_candidates": ["State_Score", "state_score", "pathway_module_score", "score"],
                "direction": "Receiver",
            },
            "ECM-receptor interaction": {
                "files": [
                    RUN_DIR / "Malignant_COI5_ECM_receptor_vs_ReceivingMI10_celllevel_scatter.csv",
                    RUN_DIR / "Malignant_COI5_ECM_receptor_vs_Receiving_MI10_scatter.csv",
                ],
                "y_candidates": ["State_Score", "state_score", "pathway_module_score", "score"],
                "direction": "Receiver",
            },
            "Hypoxia": {
                "files": [
                    RUN_DIR / "Malignant_COI5_Hypoxia_vs_Receiving_MI10_scatter.csv",
                    RUN_DIR / "Malignant_COI5_Hypoxia_vs_ReceivingMI10_celllevel_scatter.csv",
                ],
                "y_candidates": ["State_Score", "state_score", "Hypoxia_score", "score"],
                "direction": "Receiver",
            },
        }

        long_parts = []
        summary_rows = []

        for program, spec in file_specs.items():
            fn = first_existing_file(spec["files"])
            if fn is None:
                print(f"[Info] SpiderNet existing CSV not found for {program}.")
                return None, None

            df = pd.read_csv(input_path(fn))
            xcol = pick_first_existing_column(df, ["MI_Level", "Sending_MI10", "Receiving_MI10"])
            ycol = pick_first_existing_column(df, spec["y_candidates"])

            feature_values = pd.to_numeric(df[xcol], errors="coerce").to_numpy(dtype=float)
            scores = pd.to_numeric(df[ycol], errors="coerce").to_numpy(dtype=float)
            threshold = SPIDERNET_THRESHOLD

            stats = compute_smd_and_p(
                feature_values=feature_values,
                scores=scores,
                threshold=threshold,
                min_cells_per_group=MIN_CELLS_PER_GROUP,
            )

            long_df = make_long_group_table(
                method="SpiderNet",
                program=program,
                feature_dim=SPIDERNET_MI_DIM_ONE_BASED,
                feature_name=f"MI-{SPIDERNET_MI_DIM_ONE_BASED}",
                direction=spec["direction"],
                feature_values=feature_values,
                scores=scores,
                threshold=threshold,
            )
            long_df["Source"] = "existing_spidernet_scatter_csv"
            long_df["Input_File"] = str(fn)
            long_df["Threshold_Mode"] = "fixed_0.5"
            long_df["Grouping_Rule"] = "threshold_ge"
            long_df["Cell_Agg_Mode"] = CELL_AGG_MODE
            long_df["Eval_Cell_Filter"] = "structural_fibro_to_c5_edges"
            long_parts.append(long_df)

            summary_rows.append({
                "Method": "SpiderNet",
                "Program": program,
                "Feature_Dim": SPIDERNET_MI_DIM_ONE_BASED,
                "Feature_Name": f"MI-{SPIDERNET_MI_DIM_ONE_BASED}",
                "Direction": spec["direction"],
                "Threshold": threshold,
                "Threshold_Mode": "fixed_0.5",
                "Grouping_Rule": "threshold_ge",
                "Source": "existing_spidernet_scatter_csv",
                "Input_File": str(fn),
                "Cell_Agg_Mode": CELL_AGG_MODE,
                "Eval_Cell_Filter": "structural_fibro_to_c5_edges",
                **stats,
            })

        return pd.concat(long_parts, ignore_index=True), pd.DataFrame(summary_rows)

    def get_feature_values_for_program(sender_to_c5, receiver_from_fibro, feature_dim_zero_based, program):
        if program_cell_side[program] == "sender_fibroblast":
            rows, scores = score_by_program[program]
            feature_values = sender_to_c5[rows, feature_dim_zero_based]
            direction = "Sender"
        else:
            rows, scores = score_by_program[program]
            feature_values = receiver_from_fibro[rows, feature_dim_zero_based]
            direction = "Receiver"

        return rows, np.asarray(feature_values, dtype=float), np.asarray(scores, dtype=float), direction

    def threshold_and_grouping_for_feature(feature_values, method):
        feature_values = np.asarray(feature_values, dtype=float)
        finite = feature_values[np.isfinite(feature_values)]

        if finite.size == 0:
            return np.nan, "fixed_0.5" if method == "SpiderNet" else BASELINE_THRESHOLD_MODE, "threshold_ge"

        if method == "SpiderNet":
            return float(SPIDERNET_THRESHOLD), "fixed_0.5", "threshold_ge"

        return float(np.nanmedian(finite)), BASELINE_THRESHOLD_MODE, "median_rank_split_upper_half"

    def _add_program_smd_ranks(program_df):
        """Rank feature dimensions within each program by SMD.

        Rank 1 means the feature has the largest positive high-minus-low SMD for
        that program. NaN SMD values are left as NaN here and are excluded from
        candidate feature selection when REQUIRE_ALL_PROGRAMS_FOR_FEATURE_SELECTION
        is TRUE.
        """
        program_df = program_df.copy()
        program_df["SMD_rank_within_program"] = np.nan

        for program in PROGRAM_ORDER:
            mask = program_df["Program"].eq(program)
            vals = program_df.loc[mask, "SMD"]
            program_df.loc[mask, "SMD_rank_within_program"] = vals.rank(
                method="average",
                ascending=False,
                na_option="keep",
            )

        return program_df

    def scan_method_features(method, sender_to_c5, receiver_from_fibro):
        K = sender_to_c5.shape[1]
        program_rows = []

        for j in range(K):
            for program in PROGRAM_ORDER:
                rows, feature_values, scores, direction = get_feature_values_for_program(
                    sender_to_c5=sender_to_c5,
                    receiver_from_fibro=receiver_from_fibro,
                    feature_dim_zero_based=j,
                    program=program,
                )

                threshold, threshold_mode, split_rule = threshold_and_grouping_for_feature(feature_values, method)
                if not np.isfinite(threshold):
                    stats = compute_smd_and_p(
                        feature_values,
                        scores,
                        threshold=np.inf,
                        min_cells_per_group=MIN_CELLS_PER_GROUP,
                        split_rule="threshold_ge",
                    )
                else:
                    stats = compute_smd_and_p(
                        feature_values=feature_values,
                        scores=scores,
                        threshold=threshold,
                        min_cells_per_group=MIN_CELLS_PER_GROUP,
                        split_rule=split_rule,
                    )

                program_rows.append({
                    "Method": method,
                    "Program": program,
                    "Feature_Dim": j + 1,
                    "Feature_Name": f"Feature-{j + 1}",
                    "Direction": direction,
                    "Threshold": threshold,
                    "Threshold_Mode": threshold_mode,
                    "Grouping_Rule": split_rule,
                    "Cell_Agg_Mode": CELL_AGG_MODE,
                    "Eval_Cell_Filter": "structural_fibro_to_c5_edges",
                    **stats,
                })

        program_df = pd.DataFrame(program_rows)
        program_df = _add_program_smd_ranks(program_df)

        # Feature-level summary. mean_SMD is retained only for reporting; selection
        # for non-SpiderNet baselines uses mean_SMD_rank_across_programs.
        feature_df = (
            program_df
            .groupby(["Method", "Feature_Dim", "Feature_Name"], as_index=False)
            .agg(
                mean_SMD_across_programs=("SMD", "mean"),
                median_SMD_across_programs=("SMD", "median"),
                mean_SMD_rank_across_programs=("SMD_rank_within_program", "mean"),
                max_SMD_rank_across_programs=("SMD_rank_within_program", "max"),
                valid_program_count=("SMD", lambda x: int(np.isfinite(x).sum())),
                valid_rank_count=("SMD_rank_within_program", lambda x: int(np.isfinite(x).sum())),
            )
        )
        feature_df["n_programs"] = len(PROGRAM_ORDER)
        feature_df["Feature_Selection_Mode"] = (
            "fixed_MI10" if method == "SpiderNet" else BASELINE_FEATURE_SELECTION_MODE
        )

        return program_df, feature_df

    # SpiderNet's four scatter tables are generated earlier in this notebook.
    # They preserve the scoring reference populations used for the paper's MI-10 bars.
    spidernet_long, spidernet_summary = load_existing_spidernet_csv_long()
    if spidernet_long is None:
        raise FileNotFoundError("Run the preceding CAF, CancerSEA and cell-level KEGG export cells first.")
    selected_long_parts = [spidernet_long]
    selected_summary_parts = [spidernet_summary]
    all_scan_program_rows, all_scan_feature_rows = [], []

    for method in ("NMF-LR", "COMMOT", "ScCChain", "Spacia"):
        print(f"Rebuilding {method}: raw features, max aggregation and feature selection", flush=True)
        edge_features = load_baseline_features(method)
        if len(edge_features) != len(adata_list):
            raise ValueError(f"{method} does not contain every HGSOC slice.")
        sender_to_c5, receiver_from_fibro = build_method_cell_feature_matrices(edge_features)
        del edge_features
        program_df, feature_df = scan_method_features(method, sender_to_c5, receiver_from_fibro)
        all_scan_program_rows.append(program_df)
        all_scan_feature_rows.append(feature_df)
        candidates = feature_df.loc[(feature_df["valid_program_count"] == len(PROGRAM_ORDER))
                                    & (feature_df["valid_rank_count"] == len(PROGRAM_ORDER))].copy()
        if candidates.empty:
            # Preserve the source benchmark's fallback to valid available ranks.
            candidates = feature_df.copy()
        candidates = candidates.loc[np.isfinite(candidates["mean_SMD_rank_across_programs"])]
        if candidates.empty:
            raise ValueError(f"No feature with a valid SMD rank for {method}.")
        best = candidates.sort_values(
            ["mean_SMD_rank_across_programs", "max_SMD_rank_across_programs", "mean_SMD_across_programs"],
            ascending=[True, True, False], na_position="last",
        ).iloc[0]
        selected_dim = int(best["Feature_Dim"])
        selected_summary = program_df.loc[program_df["Feature_Dim"] == selected_dim].copy()
        selected_summary["Feature_Name"] = f"Feature-{selected_dim}"
        selected_summary["Source"] = "computed_from_edge_features"
        selected_summary["Feature_Selection_Mode"] = BASELINE_FEATURE_SELECTION_MODE
        selected_summary_parts.append(selected_summary)
        for program in PROGRAM_ORDER:
            rows, feature_values, scores, direction = get_feature_values_for_program(
                sender_to_c5, receiver_from_fibro, selected_dim - 1, program
            )
            threshold, threshold_mode, split_rule = threshold_and_grouping_for_feature(feature_values, method)
            long_df = make_long_group_table(
                method, program, selected_dim, f"Feature-{selected_dim}", direction,
                feature_values, scores, threshold, threshold_mode=threshold_mode, split_rule=split_rule,
            )
            long_df["Source"] = "computed_from_edge_features"
            long_df["Cell_Agg_Mode"] = CELL_AGG_MODE
            long_df["Eval_Cell_Filter"] = "structural_fibro_to_c5_edges"
            long_df["Feature_Selection_Mode"] = BASELINE_FEATURE_SELECTION_MODE
            selected_long_parts.append(long_df)
        print(f"Selected {method} feature {selected_dim}", flush=True)
        del sender_to_c5, receiver_from_fibro

    all_scan_program_df = pd.concat(all_scan_program_rows, ignore_index=True)
    all_scan_feature_df = pd.concat(all_scan_feature_rows, ignore_index=True)
    selected_long_df = pd.concat(selected_long_parts, ignore_index=True)
    selected_summary_df = pd.concat(selected_summary_parts, ignore_index=True)
    selected_feature_summary = (
        selected_summary_df.groupby(["Method", "Feature_Dim", "Feature_Name"], as_index=False)
        .agg(mean_SMD_across_programs=("SMD", "mean"), median_SMD_across_programs=("SMD", "median"),
             mean_SMD_rank_across_programs=("SMD_rank_within_program", "mean"),
             max_SMD_rank_across_programs=("SMD_rank_within_program", "max"),
             valid_program_count=("SMD", lambda x: int(np.isfinite(x).sum())),
             valid_rank_count=("SMD_rank_within_program", lambda x: int(np.isfinite(x).sum())))
    )
    selected_summary_df = selected_summary_df.merge(
        selected_feature_summary, on=["Method", "Feature_Dim", "Feature_Name"], how="left"
    )
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for filename, data in (
        ("HGSOC_CCC_method_feature_program_SMD_scan_all.csv", all_scan_program_df),
        ("HGSOC_CCC_method_feature_meanrank_SMD_scan_all.csv", all_scan_feature_df),
        ("HGSOC_CCC_method_selected_feature_program_SMD_summary.csv", selected_summary_df),
        ("HGSOC_CCC_method_selected_feature_program_score_long.csv", selected_long_df),
    ):
        data.to_csv(OUT_DIR / filename, index=False)
    return selected_summary_df, selected_long_df



In [ ]:
# Rebuild all five methods before drawing the SMD barplot.
hgsoc_smd_rebuilt_summary, hgsoc_smd_rebuilt_groups = rebuild_hgsoc_method_smd(
    adata_all=adata_copy,
    adata_malignant=adata_choose,
    adata_slices=adata_list,
    graph_slices=SpiderNet_data_pyg_list,
    run_dir=run_dirs["run_dir"],
    data_root=DATA_ROOT,
    baseline_root=OUTPUT_ROOT,
    n_components=dim_envir,
)
display(hgsoc_smd_rebuilt_summary[["Method", "Program", "Feature_Name", "n_high", "n_low", "SMD"]])


In [ ]:
# Method comparison using the exported, selected-feature cell groups.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter


def plot_hgsoc_method_smd(run_dir, output_dir=None):
    """Recompute and plot SMDs from the HGSOC benchmark's saved High/Low groups.

    The preceding reconstruction cell exports the input CSVs in this notebook.
    Feature selection, cell populations, scores and tie handling are preserved.
    No R code, expression rescoring or model fitting is required here.
    """
    input_dir = Path(run_dir) / "CCC_baseline_feature_SMD_comparison"
    output_dir = input_dir if output_dir is None else Path(output_dir)
    summary_path = input_dir / "HGSOC_CCC_method_selected_feature_program_SMD_summary.csv"
    cells_path = input_dir / "HGSOC_CCC_method_selected_feature_program_score_long.csv"
    for path in (summary_path, cells_path):
        if not input_path(path).is_file():
            raise FileNotFoundError(
                f"Missing {path}. Run the preceding SMD reconstruction cell "
                "to export the benchmark's selected-feature summary and cell groups."
            )
    reference = pd.read_csv(input_path(summary_path))
    cells = pd.read_csv(input_path(cells_path), low_memory=False)
    methods = ("SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Spacia")
    programs = ("CAF sig.", "PD-1/PD-L1 checkpoint", "ECM-receptor interaction", "Hypoxia")
    program_labels = ("CAF\nactivation", "PD-1/PD-L1\ncheckpoint", "ECM-receptor\ninteraction", "Hypoxia")
    method_labels = ("SpiderNet", "NMF-LR", "COMMOT", "scCChain", "Spacia")
    edge_colors = ("#9F3B38", "#82CCE2", "#519384", "#636491", "#FED881")
    fill_colors = ("#E1B6A7", "#D4ECF1", "#B9CEC7", "#A6A2B9", "#FFF2D2")

    required_summary = {
        "Method", "Program", "Feature_Dim", "Feature_Name", "Direction",
        "Threshold", "Threshold_Mode", "Grouping_Rule", "Cell_Agg_Mode",
        "Eval_Cell_Filter", "Feature_Selection_Mode", "n_high", "n_low",
        "mean_high", "mean_low", "sd_high", "sd_low", "pooled_sd", "SMD",
    }
    required_cells = {"Method", "Program", "Feature_Dim", "Group", "Score"}
    for data, required, path in ((reference, required_summary, summary_path),
                                 (cells, required_cells, cells_path)):
        if not required.issubset(data.columns):
            raise ValueError(f"Missing columns in {path.name}: {sorted(required - set(data.columns))}")
    reference = reference.loc[
        reference["Method"].isin(methods) & reference["Program"].isin(programs)
    ].copy()
    expected_pairs = pd.MultiIndex.from_product([methods, programs], names=["Method", "Program"])
    if reference.duplicated(["Method", "Program"]).any() or set(
        zip(reference["Method"], reference["Program"])
    ) != set(expected_pairs):
        raise ValueError("Expected one summary row for each of the five methods and four programs.")
    reference = reference.set_index(["Method", "Program"]).loc[expected_pairs].reset_index()
    baselines = reference["Method"] != "SpiderNet"
    if not (
        reference["Cell_Agg_Mode"].eq("max").all()
        and reference["Eval_Cell_Filter"].eq("structural_fibro_to_c5_edges").all()
        and reference.loc[baselines, "Threshold_Mode"].eq("median").all()
        and reference.loc[baselines, "Grouping_Rule"].eq("median_rank_split_upper_half").all()
        and reference.loc[baselines, "Feature_Selection_Mode"].eq("mean_SMD_rank_across_programs").all()
        and reference.loc[~baselines, "Feature_Dim"].eq(10).all()
        and reference.loc[~baselines, "Threshold"].eq(0.5).all()
    ):
        raise ValueError("Benchmark metadata do not match the MI-10/median-rank comparison settings.")

    rows = []
    metadata_columns = [
        "Method", "Program", "Feature_Dim", "Feature_Name", "Direction",
        "Threshold", "Threshold_Mode", "Grouping_Rule", "Cell_Agg_Mode",
        "Eval_Cell_Filter", "Feature_Selection_Mode",
    ]
    for _, saved in reference.iterrows():
        group_data = cells.loc[
            cells["Method"].eq(saved["Method"]) & cells["Program"].eq(saved["Program"])
        ]
        if group_data.empty or not group_data["Feature_Dim"].eq(saved["Feature_Dim"]).all():
            raise ValueError(f"Selected feature mismatch for {saved['Method']} / {saved['Program']}.")
        if not group_data["Group"].isin(["High", "Low"]).all():
            raise ValueError("Cell groups must be the saved High/Low assignments.")
        # Preserve the exported groups: reapplying >= median would change the
        # balanced rank split when many baseline feature values are tied.
        high = group_data.loc[group_data["Group"].eq("High"), "Score"].to_numpy(dtype=float)
        low = group_data.loc[group_data["Group"].eq("Low"), "Score"].to_numpy(dtype=float)
        if min(len(high), len(low)) < 2 or not (
            np.isfinite(high).all() and np.isfinite(low).all()
        ):
            raise ValueError("Each group needs at least two finite cell scores to compute SMD.")
        sd_high, sd_low = high.std(ddof=1), low.std(ddof=1)
        pooled_sd = np.sqrt(
            ((len(high) - 1) * sd_high**2 + (len(low) - 1) * sd_low**2)
            / (len(high) + len(low) - 2)
        )
        if pooled_sd <= 0:
            raise ValueError("SMD is undefined when the pooled within-group SD is zero.")
        calculated = {
            "n_high": len(high), "n_low": len(low),
            "mean_high": high.mean(), "mean_low": low.mean(),
            "sd_high": sd_high, "sd_low": sd_low, "pooled_sd": pooled_sd,
            "SMD": (high.mean() - low.mean()) / pooled_sd,
        }
        if not np.allclose(
            list(calculated.values()), saved[list(calculated)].to_numpy(dtype=float),
            rtol=1e-9, atol=1e-12,
        ):
            raise ValueError(
                f"Cell scores and summary disagree for {saved['Method']} / {saved['Program']}. "
                "Regenerate the two benchmark exports together."
            )
        rows.append({**saved[metadata_columns].to_dict(), **calculated})
    plot_data = pd.DataFrame(rows)
    smd_matrix = plot_data.pivot(index="Program", columns="Method", values="SMD").loc[
        list(programs), list(methods)
    ]

    with plt.rc_context({
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
        "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 120,
        "axes.grid": False, "axes.facecolor": "white", "figure.facecolor": "white",
    }):
        fig, ax = plt.subplots(figsize=(8.2, 3.05))
        fig.subplots_adjust(left=0.095, right=0.99, bottom=0.36, top=0.975)
        positions = np.arange(len(programs))
        for index, (method, label, edge, fill) in enumerate(zip(
            methods, method_labels, edge_colors, fill_colors
        )):
            ax.bar(
                positions + (index - 2) * 0.156, smd_matrix[method].to_numpy(),
                width=0.136, label=label, facecolor=fill, edgecolor=edge,
                linewidth=0.95, zorder=2,
            )
        ax.axhline(0, color="#999999", linewidth=0.9, zorder=1)
        ax.set_xticks(positions, program_labels)
        ax.set_ylabel("SMD", fontsize=14)
        ax.set_xlim(-0.6, len(programs) - 0.4)
        ax.set_ylim(min(-0.35, plot_data["SMD"].min() - 0.05),
                    max(0.93, plot_data["SMD"].max() + 0.05))
        ax.set_yticks(np.arange(-0.3, 1.0, 0.3))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.tick_params(axis="x", labelsize=13.3, pad=5, width=0.9, length=3)
        ax.tick_params(axis="y", labelsize=11.5, width=0.9, length=3)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_linewidth(0.9)
        fig.legend(
            *ax.get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.54, 0.015),
            ncol=5, frameon=False, fontsize=12.5, handlelength=0.8, handleheight=0.8,
            handletextpad=0.3, columnspacing=1.0, borderaxespad=0,
        )
        output_dir.mkdir(parents=True, exist_ok=True)
        stem = "HGSOC_CCC_methods_4programs_SMD_barplot_python"
        for extension in ("png", "pdf"):
            fig.savefig(output_dir / f"{stem}.{extension}", dpi=300,
                        bbox_inches="tight", pad_inches=0.05, facecolor="white")
        plot_data.to_csv(output_dir / f"{stem}_summary.csv", index=False)
    return fig, plot_data


hgsoc_method_smd_figure, hgsoc_method_smd_summary = plot_hgsoc_method_smd(run_dirs["run_dir"])
display(hgsoc_method_smd_summary[["Method", "Program", "Feature_Name", "n_high", "n_low", "SMD"]])
plt.show()
plt.close(hgsoc_method_smd_figure)


## 6. In silico perturbation of fibroblast-to-malignant C5 MI-10 signaling

The pretrained model reconstructs expression after full knockout or 50% reduction of MI-10-associated ligands in selected fibroblasts and receptors in selected malignant cells. A single random gene set, matched separately to the ligand and receptor counts with seed 123, supplies the knockout control. The targeted cells are endpoints of fibroblast-to-receiver edges with MI-10 strength >0.3.

The final sections calculate gene-program expression ratios for all three conditions. The plotted comparisons retain full knockout and random knockout; the partial-knockdown results remain in the exported tables.


### Load the trained model and supporting files


In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import torch

adata_list = processed.adata_list
SpiderNet_data_pyg_list = processed.spidernet_data

run_dir = Path(run_dirs["run_dir"])
model_dir = input_path(run_dir / "Model")

Factor_envir_list = pd.read_pickle(input_path(run_dir / "Factor_envir_list.pkl"))
loading_LR_use_df = pd.read_csv(input_path(run_dir / "loading_LR_use.csv"), index_col=0)
LR_list = processed.lr_list

model_config_path = model_dir / "SpiderNet_model_config.json"
with open(input_path(model_config_path), "r", encoding="utf-8") as f:
    model_config_dict = json.load(f)
train_cfg = TrainingConfig(**model_config_dict)

from SpiderNet.api import build_model
model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)

# Prefer epoch 19999; otherwise use the highest numbered available checkpoint.
model_ckpt_candidates = sorted(
    model_dir.glob("model_epoch*.pth"),
    key=lambda p: int(re.search(r"model_epoch(\d+)\.pth$", p.name).group(1))
)
if len(model_ckpt_candidates) == 0:
    raise FileNotFoundError(f"No model checkpoint found in: {model_dir}")

preferred_ckpt = model_dir / "model_epoch19999.pth"
model_ckpt_path = preferred_ckpt if input_path(preferred_ckpt).exists() else model_ckpt_candidates[-1]

model.load_state_dict(torch.load(input_path(model_ckpt_path), map_location=device))
model = model.to(device)
model.eval()

print(f"Loaded model config from: {model_config_path}")
print(f"Loaded checkpoint from: {model_ckpt_path.name}")


### Specify the sender population, receiver subtype, and MI of interest

In [ ]:
import re

sender_population = "Fibroblast"
MI_OI = "MI-10"
MI_strength_threshold = 0.3

# Prefer the malignant subtype annotation already created in this notebook
receiver_col_candidates = ["Malignant_C5", "Malignant_C4"]
receiver_population = next((c for c in receiver_col_candidates if c in adata_choose.obs.columns), None)

if receiver_population is None:
    raise KeyError(
        "Cannot find a receiver subtype column in adata_choose.obs. "
        "Expected one of: " + ", ".join(receiver_col_candidates)
    )

receiver_label_match = re.search(r"Malignant_(C\d+)", receiver_population)
receiver_label = receiver_label_match.group(1) if receiver_label_match else "C5"
MI_OI_numeric = int(MI_OI.replace("MI-", "")) - 1

receiver_barcodes = list(
    adata_choose.obs.loc[adata_choose.obs[receiver_population] == receiver_label, "barcode"]
)

print(f"Sender population: {sender_population}")
print(f"Receiver subtype column: {receiver_population}")
print(f"Receiver subtype label: {receiver_label}")
print(f"MI of interest: {MI_OI}")
print(f"Number of receiver cells in adata_choose: {len(receiver_barcodes)}")


### Define CAF genes, tumor functional-state genes, and MI-linked LR genes

In [ ]:
# -----------------------------
# CAF gene modules
# -----------------------------
caf_modules = {
    "myCAF": ["ACTA2", "TAGLN", "MYL9", "TPM2", "CNN1", "CALD1", "COL1A1", "COL1A2"],
    "iCAF": ["IL6", "CXCL12", "CXCL14", "LIF", "CCL2", "PTGS2"],
    "apCAF": ["HLA-DRA", "HLA-DRB1", "CD74", "CIITA"],
    "meCAF": ["COL11A1", "THBS2", "MMP11", "ITGA11", "FN1", "VCAN", "SPARC", "SULF1", "LOX", "PLOD2"],
    "periCAF": ["RGS5", "PDGFRB", "MCAM", "NOTCH3", "TAGLN"],
    "prolCAF": ["MKI67", "TOP2A", "PCNA"],
}

gene_index_list = adata_choose.var.index.tolist()

caf_genes_inadata = np.unique(
    [g for genes in caf_modules.values() for g in genes if g in gene_index_list]
)
caf_genes_inadata_geneindex = np.where(np.isin(gene_index_list, caf_genes_inadata))[0]

# -----------------------------
# Tumor functional-state gene sets
# -----------------------------
functional_geneset_list_df = pd.read_csv(input_path(DATA_ROOT / "CancerSEA_OV/functional_geneset_list_df.csv"))
functional_geneset_list_df = functional_geneset_list_df[
    functional_geneset_list_df["Gene"].isin(gene_index_list)
].copy()

functional_geneset_choose_geneindex_dict = {}
for term in np.unique(functional_geneset_list_df["GeneSet"]):
    genes_term = functional_geneset_list_df.loc[
        functional_geneset_list_df["GeneSet"] == term, "Gene"
    ].values
    functional_geneset_choose_geneindex_dict[term] = np.where(
        np.isin(gene_index_list, genes_term)
    )[0]

# -----------------------------
# Top LR genes associated with the selected MI
# -----------------------------
loading_LR_use_df_norm = loading_LR_use_df.div(loading_LR_use_df.sum(axis=0), axis=1)
loading_LR_use_df_norm_MIOI = loading_LR_use_df_norm.loc[MI_OI, :].sort_values(ascending=False)

# Keep strongly associated LR pairs
loading_LR_use_df_norm_MIOI_sorted_choose = loading_LR_use_df_norm_MIOI[
    loading_LR_use_df_norm_MIOI > 0.5
]

ligand_top_list = [x.split(" -> ")[0] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]
receptor_top_list = [x.split(" -> ")[1] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]

ligand_top_list_split = [
    sub for l in ligand_top_list for sub in l.replace("(", "").replace(")", "").split("+")
]
receptor_top_list_split = [
    sub for r in receptor_top_list for sub in r.replace("(", "").replace(")", "").split("+")
]

ligand_top_list_split_unique = sorted(set(ligand_top_list_split))
receptor_top_list_split_unique = sorted(set(receptor_top_list_split))

ligand_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in ligand_top_list_split_unique if g in gene_index_list
]
receptor_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in receptor_top_list_split_unique if g in gene_index_list
]

ligand_all = list({g for lr in LR_list for g in lr[0] if g in gene_index_list})
receptor_all = list({g for lr in LR_list for g in lr[1] if g in gene_index_list})

ligand_all_geneindex = [gene_index_list.index(g) for g in ligand_all]
receptor_all_geneindex = [gene_index_list.index(g) for g in receptor_all]

# Draw one count-matched random ligand/receptor gene control (seed 123).
rng = np.random.default_rng(123)

ligand_baseline_pool = np.setdiff1d(ligand_all_geneindex, ligand_top_list_split_unique_geneindex)
receptor_baseline_pool = np.setdiff1d(receptor_all_geneindex, receptor_top_list_split_unique_geneindex)

ligand_top_list_split_unique_geneindex_baseline = rng.choice(
    ligand_baseline_pool,
    size=len(ligand_top_list_split_unique_geneindex),
    replace=len(ligand_baseline_pool) < len(ligand_top_list_split_unique_geneindex),
)
receptor_top_list_split_unique_geneindex_baseline = rng.choice(
    receptor_baseline_pool,
    size=len(receptor_top_list_split_unique_geneindex),
    replace=len(receptor_baseline_pool) < len(receptor_top_list_split_unique_geneindex),
)

top_lr_summary_df = pd.DataFrame({
    "LR_pair": loading_LR_use_df_norm_MIOI_sorted_choose.index,
    "normalized_loading": loading_LR_use_df_norm_MIOI_sorted_choose.values,
})
top_lr_summary_df.to_csv(run_dir / f"Top_LR_pairs_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv", index=False)

print("Selected top LR pairs:")
display(top_lr_summary_df.head(20))
print(f"Top ligand genes kept: {len(ligand_top_list_split_unique_geneindex)}")
print(f"Top receptor genes kept: {len(receptor_top_list_split_unique_geneindex)}")


### Run in silico perturbation and reconstruct expression

In [ ]:
# -----------------------------------------
# Helper function
# -----------------------------------------
def run_model_on_perturbed_graph(
    graph,
    sender_idx,
    receiver_idx,
    ligand_idx,
    receptor_idx,
    scale=0.0,
    baseline=False,
):
    """
    Run model inference after perturbing ligand/receptor gene expression.
    - scale = 0.0: knockout of selected LR genes (zero expression)
    - scale = 0.5: 50% reduction of selected LR gene expression
    - baseline = True: zero the supplied control genes, independent of scale
    """
    g = graph.clone()

    sender_idx = np.asarray(sender_idx, dtype=int)
    receiver_idx = np.asarray(receiver_idx, dtype=int)
    ligand_idx = np.asarray(ligand_idx, dtype=int)
    receptor_idx = np.asarray(receptor_idx, dtype=int)

    if sender_idx.size > 0 and ligand_idx.size > 0:
        if baseline:
            g.x[sender_idx[:, None], ligand_idx] = 0
        else:
            g.x[sender_idx[:, None], ligand_idx] *= scale

    if receiver_idx.size > 0 and receptor_idx.size > 0:
        if baseline:
            g.x[receiver_idx[:, None], receptor_idx] = 0
        else:
            g.x[receiver_idx[:, None], receptor_idx] *= scale

    exp_recon, *_ = model(g)
    return exp_recon.detach().cpu().numpy().astype(np.float32)

# -----------------------------------------
# Main loop across slices
# -----------------------------------------
sender_cell_mask_list = [
    adata_cur.obs["cell.types"].eq(sender_population).to_numpy()
    for adata_cur in adata_list
]

receiver_barcode_idx_global = pd.Index(receiver_barcodes)

exp_recon_cur_ori_df_list = []
exp_recon_cur_perturb_df_receiver_list = []
exp_recon_cur_perturb_df_sender_list = []
exp_recon_cur_perturb_df_receiver_list_half = []
exp_recon_cur_perturb_df_sender_list_half = []
exp_recon_cur_perturb_df_receiver_list_baseline = []
exp_recon_cur_perturb_df_sender_list_baseline = []

slice_summary = []

for slice_index, (adata_cur, pyg_cur, Factor_envir_cur) in enumerate(
    zip(adata_list, SpiderNet_data_pyg_list, Factor_envir_list)
):
    pyg_cpu = pyg_cur.to("cpu")
    edge_index_cur = pyg_cpu.edge_index.detach().cpu().numpy()
    Factor_envir_cur = np.asarray(Factor_envir_cur)

    receiver_barcode_cur = adata_cur.obs.index.intersection(receiver_barcode_idx_global).tolist()
    receiver_barcode_idx = [adata_cur.obs.index.get_loc(x) for x in receiver_barcode_cur]

    if len(receiver_barcode_idx) == 0:
        slice_summary.append({
            "slice_index": slice_index,
            "n_receiver_cells": 0,
            "n_sender_cells_selected": 0,
            "n_receiver_cells_selected": 0,
            "status": "skip_no_receiver",
        })
        continue

    rel1 = np.where(np.isin(edge_index_cur[:, 1], receiver_barcode_idx))[0]
    if len(rel1) == 0:
        slice_summary.append({
            "slice_index": slice_index,
            "n_receiver_cells": len(receiver_barcode_idx),
            "n_sender_cells_selected": 0,
            "n_receiver_cells_selected": 0,
            "status": "skip_no_edges",
        })
        continue

    src_idx = edge_index_cur[rel1, 0]
    sender_mask = sender_cell_mask_list[slice_index][src_idx]
    rel2 = rel1[sender_mask]

    if len(rel2) == 0:
        slice_summary.append({
            "slice_index": slice_index,
            "n_receiver_cells": len(receiver_barcode_idx),
            "n_sender_cells_selected": 0,
            "n_receiver_cells_selected": 0,
            "status": "skip_no_sender_receiver_edges",
        })
        continue

    strong_MI_mask = Factor_envir_cur[rel2, MI_OI_numeric] > MI_strength_threshold
    rel3 = rel2[strong_MI_mask]

    if len(rel3) == 0:
        slice_summary.append({
            "slice_index": slice_index,
            "n_receiver_cells": len(receiver_barcode_idx),
            "n_sender_cells_selected": 0,
            "n_receiver_cells_selected": 0,
            "status": "skip_no_strong_MI_edges",
        })
        continue

    edge_strong = edge_index_cur[rel3, :]
    sender_idx = np.unique(edge_strong[:, 0])
    receiver_idx = np.unique(edge_strong[:, 1])

    sender_barcode = adata_cur.obs.index[sender_idx].tolist()
    receiver_barcode_selected = adata_cur.obs.index[receiver_idx].tolist()

    pyg_device = pyg_cpu.to(device)

    # Original reconstruction
    exp_ori, *_ = model(pyg_device)
    exp_ori_df = pd.DataFrame(
        exp_ori.detach().cpu().numpy().astype(np.float32),
        index=adata_cur.obs.index,
        columns=adata_cur.var.index,
    )
    exp_recon_cur_ori_df_list.append(exp_ori_df)

    # Full knockdown of MI-linked LR genes
    exp_full = run_model_on_perturbed_graph(
        pyg_device,
        sender_idx=sender_idx,
        receiver_idx=receiver_idx,
        ligand_idx=ligand_top_list_split_unique_geneindex,
        receptor_idx=receptor_top_list_split_unique_geneindex,
        scale=0.0,
        baseline=False,
    )
    exp_full_df = pd.DataFrame(exp_full, index=adata_cur.obs.index, columns=adata_cur.var.index)
    exp_recon_cur_perturb_df_receiver_list.append(exp_full_df.loc[receiver_barcode_selected])
    exp_recon_cur_perturb_df_sender_list.append(exp_full_df.loc[sender_barcode])

    # Half knockdown
    exp_half = run_model_on_perturbed_graph(
        pyg_device,
        sender_idx=sender_idx,
        receiver_idx=receiver_idx,
        ligand_idx=ligand_top_list_split_unique_geneindex,
        receptor_idx=receptor_top_list_split_unique_geneindex,
        scale=0.5,
        baseline=False,
    )
    exp_half_df = pd.DataFrame(exp_half, index=adata_cur.obs.index, columns=adata_cur.var.index)
    exp_recon_cur_perturb_df_receiver_list_half.append(exp_half_df.loc[receiver_barcode_selected])
    exp_recon_cur_perturb_df_sender_list_half.append(exp_half_df.loc[sender_barcode])

    # Random baseline knockdown
    exp_random = run_model_on_perturbed_graph(
        pyg_device,
        sender_idx=sender_idx,
        receiver_idx=receiver_idx,
        ligand_idx=ligand_top_list_split_unique_geneindex_baseline,
        receptor_idx=receptor_top_list_split_unique_geneindex_baseline,
        baseline=True,
    )
    exp_random_df = pd.DataFrame(exp_random, index=adata_cur.obs.index, columns=adata_cur.var.index)
    exp_recon_cur_perturb_df_receiver_list_baseline.append(exp_random_df.loc[receiver_barcode_selected])
    exp_recon_cur_perturb_df_sender_list_baseline.append(exp_random_df.loc[sender_barcode])

    slice_summary.append({
        "slice_index": slice_index,
        "n_receiver_cells": len(receiver_barcode_idx),
        "n_sender_cells_selected": len(sender_idx),
        "n_receiver_cells_selected": len(receiver_idx),
        "status": "used",
    })

def concat_or_empty(df_list, columns):
    if len(df_list) == 0:
        return pd.DataFrame(columns=columns)
    return pd.concat(df_list, axis=0)

exp_recon_cur_ori_df_list_all = concat_or_empty(exp_recon_cur_ori_df_list, gene_index_list)
exp_recon_cur_perturb_df_receiver_all = concat_or_empty(exp_recon_cur_perturb_df_receiver_list, gene_index_list)
exp_recon_cur_perturb_df_sender_all = concat_or_empty(exp_recon_cur_perturb_df_sender_list, gene_index_list)
exp_recon_cur_perturb_df_receiver_all_half = concat_or_empty(exp_recon_cur_perturb_df_receiver_list_half, gene_index_list)
exp_recon_cur_perturb_df_sender_all_half = concat_or_empty(exp_recon_cur_perturb_df_sender_list_half, gene_index_list)
exp_recon_cur_perturb_df_receiver_all_baseline = concat_or_empty(exp_recon_cur_perturb_df_receiver_list_baseline, gene_index_list)
exp_recon_cur_perturb_df_sender_all_baseline = concat_or_empty(exp_recon_cur_perturb_df_sender_list_baseline, gene_index_list)

slice_summary_df = pd.DataFrame(slice_summary)
slice_summary_df.to_csv(
    run_dir / f"InSilicoPerturbation_slice_summary_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv",
    index=False,
)

display(slice_summary_df)


### Quantify gene-program expression ratios and plot perturbation effects

For each cell, `compute_LFC` averages the per-gene perturbed/original expression ratios (with an offset of `1e-8`) and takes log2. The subsequent KEGG extension exports an additional table and a combined CAF, PD-1/PD-L1, ECM-receptor and hypoxia plot.


In [ ]:
# Export full, half and random knockout expression-ratio results; the plots
# compare full and random knockout for CAF and four CancerSEA programs.

import math
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.stats import mannwhitneyu

def compute_LFC(exp_perturb_df, exp_ori_df, genes_idx, func_name, label):
    """Return log2 of the mean per-gene perturbed/original ratio for each cell.

    Both expression matrices receive an offset of 1e-8 before division.
    """
    if len(exp_perturb_df) == 0 or len(genes_idx) == 0:
        return pd.DataFrame(columns=["LFC", "Functional_State", "Type"])

    exp_ori = exp_ori_df.loc[exp_perturb_df.index, :].iloc[:, genes_idx]
    ratio = np.mean(
        (exp_perturb_df.iloc[:, genes_idx].to_numpy() + 1e-8)
        / (exp_ori.to_numpy() + 1e-8),
        axis=1,
    )
    df = pd.DataFrame(np.log2(ratio), columns=["LFC"])
    df["Functional_State"] = func_name
    df["Type"] = label
    return df

LFC_perturb_list = []

# Sender-side CAF programs
for exp_df, label in [
    (exp_recon_cur_perturb_df_sender_all, "Perturbtop"),
    (exp_recon_cur_perturb_df_sender_all_half, "Perturbtop_half"),
    (exp_recon_cur_perturb_df_sender_all_baseline, "Perturbrandom"),
]:
    LFC_perturb_list.append(
        compute_LFC(exp_df, exp_recon_cur_ori_df_list_all, caf_genes_inadata_geneindex, "CAF_score", label)
    )

# Receiver-side malignant functional states
for func_name, genes_idx in functional_geneset_choose_geneindex_dict.items():
    for exp_df, label in [
        (exp_recon_cur_perturb_df_receiver_all, "Perturbtop"),
        (exp_recon_cur_perturb_df_receiver_all_half, "Perturbtop_half"),
        (exp_recon_cur_perturb_df_receiver_all_baseline, "Perturbrandom"),
    ]:
        LFC_perturb_list.append(
            compute_LFC(exp_df, exp_recon_cur_ori_df_list_all, genes_idx, func_name, label)
        )

LFC_perturb_all = pd.concat(LFC_perturb_list, axis=0, ignore_index=True)
lfc_path = run_dir / f"LFC_perturb_all_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv"
LFC_perturb_all.to_csv(lfc_path, index=False)
print(f"Saved LFC table to: {lfc_path}")

# -----------------------------
# Plot settings
# -----------------------------
plt.close("all")
plt.style.use("default")
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})
sns.set_theme(style="white")

df_caf = LFC_perturb_all.query("Functional_State == 'CAF_score'").copy()
df_others = LFC_perturb_all.query("Functional_State != 'CAF_score'").copy()

x_keep_order = ["Metastasis", "Angiogenesis", "Hypoxia", "Inflammation"]
df_others = df_others[df_others["Functional_State"].isin(x_keep_order)].copy()
df_others["Functional_State"] = pd.Categorical(
    df_others["Functional_State"], categories=x_keep_order, ordered=True
)

# Keep the main comparison for the final visualization
df_caf = df_caf[df_caf["Type"] != "Perturbtop_half"].copy()
df_others = df_others[df_others["Type"] != "Perturbtop_half"].copy()

palette = {
    "Perturbtop": "#b7a3ce",
    "Perturbrandom": "#c4e1f6",
}
hue_order = ["Perturbtop", "Perturbrandom"]

def pvalue_text(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return "p = NA"
    try:
        p = mannwhitneyu(a, b, alternative="two-sided", method="asymptotic").pvalue
    except TypeError:
        p = mannwhitneyu(a, b, alternative="two-sided").pvalue
    return f"p = {p:.2e}"

def save_fig(basepath_no_ext, fig):
    try:
        fig.savefig(basepath_no_ext + ".pdf", bbox_inches="tight", facecolor="white", backend="cairo")
    except Exception:
        fig.savefig(basepath_no_ext + ".pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(basepath_no_ext + ".png", bbox_inches="tight", facecolor="white", dpi=300)

def plot_box_nature(df, out_base, figsize=(4, 3.2), rotate_x=45, show_legend=False):
    fig, ax = plt.subplots(figsize=figsize)

    sns.boxplot(
        data=df,
        x="Functional_State",
        y="LFC",
        hue="Type",
        hue_order=hue_order,
        palette=palette,
        linewidth=0.6,
        fliersize=0,
        width=0.65,
        ax=ax,
    )

    ax.axhline(0, color="0.5", linewidth=0.8, linestyle="--")

    ax.set_xlabel("")
    ax.set_ylabel("log2 fold change")
    ax.tick_params(axis="x", rotation=rotate_x)
    if rotate_x:
        plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

    if not show_legend:
        ax.get_legend().remove()
    else:
        leg = ax.legend(frameon=False, title="")

    # Add pairwise p-value labels
    y_max = df["LFC"].max() if len(df) > 0 else 0
    y_min = df["LFC"].min() if len(df) > 0 else 0
    y_span = max(y_max - y_min, 1e-6)

    if df["Functional_State"].dtype.name == "category":
        states = list(df["Functional_State"].cat.categories)
    else:
        states = list(pd.unique(df["Functional_State"]))

    for i, state in enumerate(states):
        sub = df[df["Functional_State"] == state]
        a = sub.loc[sub["Type"] == "Perturbtop", "LFC"].values
        b = sub.loc[sub["Type"] == "Perturbrandom", "LFC"].values
        if len(a) == 0 or len(b) == 0:
            continue
        txt = pvalue_text(a, b)
        y_here = max(np.max(a), np.max(b)) + 0.08 * y_span
        ax.text(i, y_here, txt, ha="center", va="bottom", fontsize=6)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    save_fig(out_base, fig)
    plt.show()
    plt.close(fig)

plot_box_nature(
    df_caf,
    str(run_dir / f"InSilicoPerturbation_CAF_boxplot_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}"),
    figsize=(1.7, 3.0),
    rotate_x=0,
    show_legend=False,
)

plot_box_nature(
    df_others,
    str(run_dir / f"InSilicoPerturbation_FunctionalStates_boxplot_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}"),
    figsize=(4.1, 3.0),
    rotate_x=45,
    show_legend=False,
)


In [ ]:
# Extend the perturbation table with four KEGG programs, then plot the selected
# CAF, PD-1/PD-L1, ECM-receptor and hypoxia programs together.

import math
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.stats import mannwhitneyu


# ==============================================================
# 1. Helper: compute LFC for one gene set
# ==============================================================
def compute_LFC(exp_perturb_df, exp_ori_df, genes_idx, func_name, label):
    """Return log2 of the mean per-gene perturbed/original ratio for each cell.

    Both expression matrices receive an offset of 1e-8 before division.
    """
    genes_idx = pd.to_numeric(
        pd.Series(list(genes_idx)),
        errors="coerce"
    ).dropna().astype(int).to_numpy()

    # keep only valid gene indices
    genes_idx = genes_idx[
        (genes_idx >= 0) & (genes_idx < exp_perturb_df.shape[1])
    ]

    if len(exp_perturb_df) == 0 or len(genes_idx) == 0:
        return pd.DataFrame(columns=["LFC", "Functional_State", "Type"])

    # align original expression to perturbed rows
    exp_ori = exp_ori_df.loc[exp_perturb_df.index, :].iloc[:, genes_idx]

    ratio = np.mean(
        (exp_perturb_df.iloc[:, genes_idx].to_numpy() + 1e-8)
        / (exp_ori.to_numpy() + 1e-8),
        axis=1,
    )

    df = pd.DataFrame(np.log2(ratio), columns=["LFC"])
    df["Functional_State"] = func_name
    df["Type"] = label

    return df


# ==============================================================
# 2. Build KEGG gene-index dictionary
# ==============================================================

KEGG_PATHWAYS_TO_ADD = [
    "HIF-1 signaling pathway",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer",
    "PI3K-Akt signaling pathway",
    "ECM-receptor interaction",
]

KEGG_DISPLAY_NAME_MAP = {
    "HIF-1 signaling pathway": "KEGG HIF-1",
    "PD-L1 expression and PD-1 checkpoint pathway in cancer": "KEGG PD-1/PD-L1",
    "PI3K-Akt signaling pathway": "KEGG PI3K-Akt",
    "ECM-receptor interaction": "KEGG ECM-receptor",
}


def _load_kegg_reference_gene_df():
    """
    Load the 4 selected KEGG pathway reference genes.
    Prefer c5_kegg_reference_gene_df from memory; otherwise load saved CSV.
    """
    if "c5_kegg_reference_gene_df" in globals():
        return c5_kegg_reference_gene_df.copy()

    candidate_paths = []

    if "run_dir" in globals():
        candidate_paths.append(
            Path(run_dir)
            / "TCGA_OV_KEGG_gene_signature"
            / "C5_four_KEGG_pathway_reference_genes_long.csv"
        )

    if "run_dirs" in globals() and isinstance(run_dirs, dict) and "run_dir" in run_dirs:
        candidate_paths.append(
            Path(run_dirs["run_dir"])
            / "TCGA_OV_KEGG_gene_signature"
            / "C5_four_KEGG_pathway_reference_genes_long.csv"
        )

    for p in candidate_paths:
        if input_path(p).exists():
            print(f"Loaded KEGG reference genes from: {p}")
            return pd.read_csv(input_path(p))

    raise FileNotFoundError(
        "Cannot find c5_kegg_reference_gene_df in memory or saved KEGG CSV.\n"
        "Please run the KEGG reference gene export cell first.\n"
        "Tried:\n"
        + "\n".join([str(p) for p in candidate_paths])
    )


def _resolve_gene_indices_from_dataframe_columns(gene_list, gene_columns):
    """
    Resolve gene symbols to column indices in expression DataFrame.
    Uses exact match first, then case-insensitive match.
    """
    gene_columns = pd.Index(gene_columns.astype(str))
    upper_to_col = {}

    for i, g in enumerate(gene_columns):
        g_upper = str(g).upper()
        if g_upper not in upper_to_col:
            upper_to_col[g_upper] = i

    indices = []
    found_genes = []
    missing_genes = []

    for g in gene_list:
        g = str(g).strip()
        if len(g) == 0:
            continue

        if g in gene_columns:
            idx = gene_columns.get_loc(g)

            if isinstance(idx, slice):
                idx = idx.start
            elif isinstance(idx, np.ndarray):
                idx = np.where(idx)[0][0]

            indices.append(int(idx))
            found_genes.append(g)

        elif g.upper() in upper_to_col:
            indices.append(int(upper_to_col[g.upper()]))
            found_genes.append(g)

        else:
            missing_genes.append(g)

    return np.asarray(indices, dtype=int), found_genes, missing_genes


# Load KEGG gene table
kegg_reference_gene_df = _load_kegg_reference_gene_df()

# Identify pathway column
if "Requested_Pathway" in kegg_reference_gene_df.columns:
    kegg_pathway_col = "Requested_Pathway"
elif "Pathway" in kegg_reference_gene_df.columns:
    kegg_pathway_col = "Pathway"
elif "GeneSet" in kegg_reference_gene_df.columns:
    kegg_pathway_col = "GeneSet"
else:
    raise KeyError(
        "Cannot find KEGG pathway column. Expected one of: "
        "'Requested_Pathway', 'Pathway', 'GeneSet'. "
        f"Available columns: {list(kegg_reference_gene_df.columns)}"
    )

if "Gene" not in kegg_reference_gene_df.columns:
    raise KeyError(
        f"Cannot find 'Gene' column in KEGG gene table. "
        f"Available columns: {list(kegg_reference_gene_df.columns)}"
    )

kegg_reference_gene_df = (
    kegg_reference_gene_df
    .loc[
        kegg_reference_gene_df[kegg_pathway_col].isin(KEGG_PATHWAYS_TO_ADD),
        [kegg_pathway_col, "Gene"]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)

kegg_reference_gene_df[kegg_pathway_col] = kegg_reference_gene_df[kegg_pathway_col].astype(str)
kegg_reference_gene_df["Gene"] = kegg_reference_gene_df["Gene"].astype(str)

if kegg_reference_gene_df.empty:
    raise ValueError("No genes found for the selected 4 KEGG pathways.")

# Use reconstructed expression DataFrame columns as gene universe
gene_columns_for_recon = pd.Index(exp_recon_cur_ori_df_list_all.columns.astype(str))

kegg_geneindex_dict = {}
kegg_gene_use_records = []

for pathway in KEGG_PATHWAYS_TO_ADD:
    genes_cur = (
        kegg_reference_gene_df
        .loc[kegg_reference_gene_df[kegg_pathway_col] == pathway, "Gene"]
        .astype(str)
        .unique()
        .tolist()
    )

    idxs, found_genes, missing_genes = _resolve_gene_indices_from_dataframe_columns(
        gene_list=genes_cur,
        gene_columns=gene_columns_for_recon,
    )

    display_name = KEGG_DISPLAY_NAME_MAP[pathway]

    if len(idxs) == 0:
        print(f"[Warning] No genes found in reconstructed expression matrix for {display_name}. Skipping.")
        continue

    kegg_geneindex_dict[display_name] = idxs

    for g in found_genes:
        kegg_gene_use_records.append({
            "Functional_State": display_name,
            "Original_Pathway": pathway,
            "Gene": g,
            "Used": True,
        })

    for g in missing_genes:
        kegg_gene_use_records.append({
            "Functional_State": display_name,
            "Original_Pathway": pathway,
            "Gene": g,
            "Used": False,
        })

kegg_gene_use_df = pd.DataFrame(kegg_gene_use_records)

if "run_dir" in globals():
    kegg_gene_use_path = (
        Path(run_dir)
        / f"KEGG_genes_used_for_perturbation_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv"
    )
    kegg_gene_use_df.to_csv(kegg_gene_use_path, index=False)
    print(f"Saved KEGG gene-use table to: {kegg_gene_use_path}")

print("KEGG gene sets added to perturbation analysis:")
display(
    pd.DataFrame([
        {
            "Functional_State": k,
            "n_genes_used": len(v),
        }
        for k, v in kegg_geneindex_dict.items()
    ])
)


# ==============================================================
# 3. Merge functional gene sets + KEGG gene sets for receiver side
# ==============================================================

receiver_geneset_choose_geneindex_dict = {}

# Existing CancerSEA / functional states
for func_name, genes_idx in functional_geneset_choose_geneindex_dict.items():
    receiver_geneset_choose_geneindex_dict[func_name] = np.asarray(genes_idx, dtype=int)

# Four additional KEGG pathways
for func_name, genes_idx in kegg_geneindex_dict.items():
    receiver_geneset_choose_geneindex_dict[func_name] = np.asarray(genes_idx, dtype=int)

print("Receiver-side gene sets included in perturbation LFC:")
display(
    pd.DataFrame([
        {
            "Functional_State": k,
            "n_genes": len(v),
            "Source": "KEGG" if str(k).startswith("KEGG ") else "Functional"
        }
        for k, v in receiver_geneset_choose_geneindex_dict.items()
    ])
)


# ==============================================================
# 4. Compute perturbation LFC
# ==============================================================

LFC_perturb_list = []

# Sender-side CAF programs
for exp_df, label in [
    (exp_recon_cur_perturb_df_sender_all, "Perturbtop"),
    (exp_recon_cur_perturb_df_sender_all_half, "Perturbtop_half"),
    (exp_recon_cur_perturb_df_sender_all_baseline, "Perturbrandom"),
]:
    LFC_perturb_list.append(
        compute_LFC(
            exp_df,
            exp_recon_cur_ori_df_list_all,
            caf_genes_inadata_geneindex,
            "CAF_score",
            label,
        )
    )

# Receiver-side malignant functional states + 4 KEGG pathways
for func_name, genes_idx in receiver_geneset_choose_geneindex_dict.items():
    for exp_df, label in [
        (exp_recon_cur_perturb_df_receiver_all, "Perturbtop"),
        (exp_recon_cur_perturb_df_receiver_all_half, "Perturbtop_half"),
        (exp_recon_cur_perturb_df_receiver_all_baseline, "Perturbrandom"),
    ]:
        LFC_perturb_list.append(
            compute_LFC(
                exp_df,
                exp_recon_cur_ori_df_list_all,
                genes_idx,
                func_name,
                label,
            )
        )

LFC_perturb_all = pd.concat(LFC_perturb_list, axis=0, ignore_index=True)

lfc_path = run_dir / f"LFC_perturb_all_with_KEGG_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv"
LFC_perturb_all.to_csv(lfc_path, index=False)
print(f"Saved LFC table to: {lfc_path}")


# ==============================================================
# 5. Plot settings
# ==============================================================

plt.close("all")
plt.style.use("default")

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

sns.set_theme(style="white")


# ==============================================================
# 6. Prepare one combined plotting DataFrame
# ==============================================================

# Keep the main comparison only
LFC_plot_all = LFC_perturb_all.loc[
    LFC_perturb_all["Type"].isin(["Perturbtop", "Perturbrandom"])
].copy()

# Select the CAF and malignant programs for the combined panel.
plot_state_order_raw = [
    "CAF_score",
    "KEGG PD-1/PD-L1",
    "KEGG ECM-receptor",
    "Hypoxia",
]

plot_label_map = {
    "CAF_score": "CAF score",
    "KEGG PD-1/PD-L1": "PD-1/PD-L1",
    "KEGG ECM-receptor": "ECM-receptor\nsignaling",
    "Hypoxia": "Hypoxia",
}

type_label_map = {
    "Perturbtop": "Top KO",
    "Perturbrandom": "Random KO",
}

df_plot_combined = LFC_plot_all.loc[
    LFC_plot_all["Functional_State"].isin(plot_state_order_raw)
].copy()

df_plot_combined["Functional_State_label"] = (
    df_plot_combined["Functional_State"].map(plot_label_map)
)

df_plot_combined["Type_label"] = (
    df_plot_combined["Type"].map(type_label_map)
)

plot_state_order_label = [plot_label_map[x] for x in plot_state_order_raw]
hue_order = ["Top KO", "Random KO"]

df_plot_combined["Functional_State_label"] = pd.Categorical(
    df_plot_combined["Functional_State_label"],
    categories=plot_state_order_label,
    ordered=True,
)

df_plot_combined["Type_label"] = pd.Categorical(
    df_plot_combined["Type_label"],
    categories=hue_order,
    ordered=True,
)

df_plot_combined.to_csv(
    run_dir / f"LFC_perturb_plot_CAF_PD1_ECM_Hypoxia_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}.csv",
    index=False,
)

print("Combined plotting states:")
display(
    df_plot_combined
    .groupby(["Functional_State", "Functional_State_label", "Type_label"], observed=True)
    .agg(
        n=("LFC", "size"),
        mean_LFC=("LFC", "mean"),
        median_LFC=("LFC", "median"),
    )
    .reset_index()
)


# ==============================================================
# 7. Manual plotting helper
# ==============================================================

TOP_FILL = "#FFF3D3"
TOP_EDGE = "#FED881"

RANDOM_FILL = "#EEE7E4"
RANDOM_EDGE = "#D1CABD"

fill_palette = {
    "Top KO": TOP_FILL,
    "Random KO": RANDOM_FILL,
}

edge_palette = {
    "Top KO": TOP_EDGE,
    "Random KO": RANDOM_EDGE,
}


def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def pvalue_text_and_value(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) == 0 or len(b) == 0:
        return "p = NA", np.nan

    try:
        p = mannwhitneyu(
            a,
            b,
            alternative="two-sided",
            method="asymptotic",
        ).pvalue
    except TypeError:
        p = mannwhitneyu(
            a,
            b,
            alternative="two-sided",
        ).pvalue

    if p < 1e-4:
        return "p < 1e-4", p

    return f"p = {p:.2e}", p


def save_fig(basepath_no_ext, fig):
    try:
        fig.savefig(
            basepath_no_ext + ".pdf",
            bbox_inches="tight",
            facecolor="white",
            backend="cairo",
        )
    except Exception:
        fig.savefig(
            basepath_no_ext + ".pdf",
            bbox_inches="tight",
            facecolor="white",
        )

    fig.savefig(
        basepath_no_ext + ".png",
        bbox_inches="tight",
        facecolor="white",
        dpi=300,
    )


def plot_combined_caf_malignant_boxplot_manual(
    df,
    out_base,
    figsize=(4.3, 3.0),
    rotate_x=35,
    show_legend=True,
):
    """
    Plot paired knockout conditions at fixed positions for each gene program.

    Fill and edge colors are assigned explicitly from the condition palettes.
    """

    fig, ax = plt.subplots(figsize=figsize)

    x_centers = np.arange(len(plot_state_order_label), dtype=float)
    offset = 0.18
    box_width = 0.28

    positions = {
        "Top KO": x_centers - offset,
        "Random KO": x_centers + offset,
    }

    all_y_values = []

    # -----------------------------
    # Draw boxes manually
    # -----------------------------
    for type_label in hue_order:
        grouped_values = []

        for state_label in plot_state_order_label:
            vals = df.loc[
                (df["Functional_State_label"].astype(str) == state_label)
                & (df["Type_label"].astype(str) == type_label),
                "LFC"
            ].to_numpy(dtype=float)

            vals = vals[np.isfinite(vals)]
            grouped_values.append(vals)

            if len(vals) > 0:
                all_y_values.extend(vals.tolist())

        bp = ax.boxplot(
            grouped_values,
            positions=positions[type_label],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops={
                "linewidth": 0.8,
                "edgecolor": edge_palette[type_label],
                "facecolor": fill_palette[type_label],
            },
            medianprops={
                "linewidth": 0.8,
                "color": edge_palette[type_label],
            },
            whiskerprops={
                "linewidth": 0.8,
                "color": edge_palette[type_label],
            },
            capprops={
                "linewidth": 0.8,
                "color": edge_palette[type_label],
            },
        )

        for patch in bp["boxes"]:
            patch.set_facecolor(fill_palette[type_label])
            patch.set_edgecolor(edge_palette[type_label])
            patch.set_linewidth(0.8)

    ax.axhline(0, color="0.55", linewidth=0.8, linestyle="--", zorder=0)

    # -----------------------------
    # P-value labels
    # -----------------------------
    if len(all_y_values) > 0:
        y_min = np.nanmin(all_y_values)
        y_max = np.nanmax(all_y_values)
        y_span = max(y_max - y_min, 1e-6)
    else:
        y_min, y_max, y_span = -1, 1, 1

    p_records = []

    for i, state_label in enumerate(plot_state_order_label):
        sub = df[df["Functional_State_label"].astype(str) == state_label]

        a = sub.loc[
            sub["Type_label"].astype(str) == "Top KO",
            "LFC"
        ].to_numpy(dtype=float)

        b = sub.loc[
            sub["Type_label"].astype(str) == "Random KO",
            "LFC"
        ].to_numpy(dtype=float)

        a = a[np.isfinite(a)]
        b = b[np.isfinite(b)]

        txt, p_val = pvalue_text_and_value(a, b)

        if len(a) == 0 or len(b) == 0:
            y_here = y_max + 0.08 * y_span
        else:
            y_here = max(np.max(a), np.max(b)) + 0.08 * y_span

        ax.text(
            i,
            y_here,
            txt,
            ha="center",
            va="bottom",
            fontsize=6,
            color="black",
        )

        p_records.append({
            "Functional_State_label": state_label,
            "n_top_KO": len(a),
            "n_random_KO": len(b),
            "mean_top_KO": np.nanmean(a) if len(a) > 0 else np.nan,
            "mean_random_KO": np.nanmean(b) if len(b) > 0 else np.nan,
            "median_top_KO": np.nanmedian(a) if len(a) > 0 else np.nan,
            "median_random_KO": np.nanmedian(b) if len(b) > 0 else np.nan,
            "p_value": p_val,
            "p_label": txt,
            "star": p_to_star(p_val),
        })

    p_df = pd.DataFrame(p_records)

    # -----------------------------
    # Axis style
    # -----------------------------
    ax.set_xticks(x_centers)
    ax.set_xticklabels(plot_state_order_label)

    ax.set_xlabel("")
    ax.set_ylabel("log2 fold change")

    ax.tick_params(axis="x", rotation=rotate_x)

    if rotate_x:
        plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.tick_params(axis="both", direction="out")

    # -----------------------------
    # Legend
    # -----------------------------
    if show_legend:
        import matplotlib.patches as mpatches

        handles = [
            mpatches.Patch(
                facecolor=fill_palette["Top KO"],
                edgecolor=edge_palette["Top KO"],
                linewidth=0.8,
                label="Top KO",
            ),
            mpatches.Patch(
                facecolor=fill_palette["Random KO"],
                edgecolor=edge_palette["Random KO"],
                linewidth=0.8,
                label="Random KO",
            ),
        ]

        ax.legend(
            handles=handles,
            frameon=False,
            title="",
            loc="best",
        )

    ax.set_ylim(
        y_min - 0.08 * y_span,
        y_max + 0.22 * y_span,
    )

    plt.tight_layout()

    save_fig(out_base, fig)

    p_df.to_csv(out_base + "_pvalues.csv", index=False)

    plt.show()
    plt.close(fig)

    return fig, ax, p_df


# ==============================================================
# 8. Plot combined CAF + malignant-side selected scores
# ==============================================================

fig_combined, ax_combined, pvalue_combined_df = plot_combined_caf_malignant_boxplot_manual(
    df=df_plot_combined,
    out_base=str(
        run_dir
        / f"InSilicoPerturbation_CAF_PD1_ECM_Hypoxia_combined_boxplot_FIXED_{MI_OI}_{sender_population}_to_{receiver_population}_{receiver_label}"
    ),
    figsize=(4.3, 3.0),
    rotate_x=35,
    show_legend=True,
)

print("P-values for Top KO vs Random KO:")
display(pvalue_combined_df)
